# **Environment Initialisation**


## **Load Dependencies**

### **Install All Required Libraries**

In [1]:
# =============================================================================
# Retriever Implementation
# INSTALL DEPENDENCIES
# =============================================================================

# Install the libraries required for telecom RAG data acquisition,
# document processing, embeddings and vector retrieval.
#
# The current runtime is CPU-based because inference is not yet being
# performed. GPU-specific acceleration can be enabled later when required.

!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    safetensors \
    sentencepiece \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    python-pptx \
    beautifulsoup4 \
    pyarrow \
    tqdm

print("All required Module 2 libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 11.9 MB/s eta 0:00:00
All required Module 2 libraries installed successfully.


### **Import All Required Library**

In [2]:
# =============================================================================
# Retriever Implementation
# IMPORT REQUIRED LIBRARIES
# =============================================================================

# ---------------------------------------------------------------------------
# Core scientific stack
# ---------------------------------------------------------------------------
import numpy as np
import scipy
import pandas as pd

# ---------------------------------------------------------------------------
# Standard Python libraries
# ---------------------------------------------------------------------------
import os
import gc
import json
import shutil
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Progress monitoring
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm

# Data / document processing
# ---------------------------------------------------------------------------
import pyarrow
import pyarrow.parquet as pq
from docx import Document
from bs4 import BeautifulSoup
from pypdf import PdfReader

# ---------------------------------------------------------------------------
# HTTP / source acquisition
# ---------------------------------------------------------------------------
import requests

# ---------------------------------------------------------------------------
# PyTorch
# ---------------------------------------------------------------------------
import torch

# ---------------------------------------------------------------------------
# Hugging Face / LLM inference
# ---------------------------------------------------------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from huggingface_hub import login, HfApi, snapshot_download

# ---------------------------------------------------------------------------
# Embeddings
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Vector similarity search
# ---------------------------------------------------------------------------
import faiss

# ---------------------------------------------------------------------------
# PDF document processing
# ---------------------------------------------------------------------------
from pypdf import PdfReader

print("All required libraries imported successfully.")

# Display key package versions for reproducibility
print("\nPackage Versions")
print("-" * 40)
print(f"NumPy           : {np.__version__}")
print(f"SciPy           : {scipy.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"FAISS           : {faiss.__version__}")
print(f"PyArrow         : {pyarrow.__version__}")

All required libraries imported successfully.

Package Versions
----------------------------------------
NumPy           : 2.1.3
SciPy           : 1.16.3
Pandas          : 2.2.3
PyTorch         : 2.11.0+cpu
FAISS           : 1.15.0
PyArrow         : 18.1.0


### **Kaggle Login**

In [3]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


### **Dataset Import**

In [15]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

cliffordimaguezegie_telecom_chunks_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-reconciled-chunks')
cliffordimaguezegie_telecom_bge_m3_embeddings_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-bge-m3-embeddings')
cliffordimaguezegie_telecom_bge_m3_faiss_path = kagglehub.dataset_download('cliffordimaguezegie/telecom-bge-m3-faiss')
cliffordimaguezegie_bench_responses_path = kagglehub.dataset_download('cliffordimaguezegie/bench-responses')
cliffordimaguezegie_benchmark_path = kagglehub.dataset_download('cliffordimaguezegie/benchmark')
cliffordimaguezegie_relevance_path = kagglehub.dataset_download('cliffordimaguezegie/relevance')

print('Data source import complete.')


Data source import complete.


In [16]:
print("Chunks:")
print(cliffordimaguezegie_telecom_chunks_path)

print("\nEmbeddings:")
print(cliffordimaguezegie_telecom_bge_m3_embeddings_path)

print("\nFAISS:")
print(cliffordimaguezegie_telecom_bge_m3_faiss_path)

print("\nRESPONSES:")
print(cliffordimaguezegie_bench_responses_path)

print("\nQUESTIONS:")
print(cliffordimaguezegie_benchmark_path)

Chunks:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-reconciled-chunks/versions/1

Embeddings:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-embeddings/versions/1

FAISS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-faiss/versions/1

RESPONSES:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/bench-responses/versions/1

QUESTIONS:
/root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1


In [17]:
from pathlib import Path

CHUNK_DIR = Path(
    cliffordimaguezegie_telecom_chunks_path
)

EMBED_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_embeddings_path
)

FAISS_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_faiss_path
)

BENCHMARK_DIR = Path(
    cliffordimaguezegie_benchmark_path
)

RESPONSE_DIR = Path(
    cliffordimaguezegie_bench_responses_path
)

print("BENCHMARK :", BENCHMARK_DIR)
print("RESPONSE :", RESPONSE_DIR)
print("Chunks    :", CHUNK_DIR)
print("Embeddings:", EMBED_DIR)
print("FAISS     :", FAISS_DIR)

BENCHMARK : /root/.cache/kagglehub/datasets/cliffordimaguezegie/benchmark/versions/1
RESPONSE : /root/.cache/kagglehub/datasets/cliffordimaguezegie/bench-responses/versions/1
Chunks    : /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-reconciled-chunks/versions/1
Embeddings: /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-embeddings/versions/1
FAISS     : /root/.cache/kagglehub/datasets/cliffordimaguezegie/telecom-bge-m3-faiss/versions/1


In [18]:
# =============================================================================
# INSPECT BENCHMARK DATASET
# =============================================================================

print("=" * 90)
print("BENCHMARK DATASET CONTENTS")
print("=" * 90)

for file_path in sorted(
    BENCHMARK_DIR.rglob("*")
):

    if file_path.is_file():

        print(
            file_path.relative_to(
                BENCHMARK_DIR
            )
        )

print("=" * 90)

BENCHMARK DATASET CONTENTS
track1_20_questions.json
track2_final_32_questions.json


In [19]:
# =============================================================================
# INSPECT RESPONSE DATASET
# =============================================================================

print("=" * 90)
print("RESPONSE DATASET CONTENTS")
print("=" * 90)

for file_path in sorted(
    RESPONSE_DIR.rglob("*")
):

    if file_path.is_file():

        print(
            file_path.relative_to(
                RESPONSE_DIR
            )
        )

print("=" * 90)

RESPONSE DATASET CONTENTS
track1_essentialai_llm_results_20260828_101033.json
track1_essentialai_rag_results_20260828_125927.json
track1_gemma4_llm_results.json
track1_gemma4_vllm_rag_results_20260827_193305.json
track1_otel2_llm_results_20260826_232923.json
track2_essentialai_llm_results_20260828_103230.json
track2_essentialai_rag_results_20260828_132208.json
track2_gemma4_llm_result.json
track2_gemma4_vllm_rag_results_20260827_193951.json
track2_otel2_llm_results_20260827_004654.json


In [20]:
# =============================================================================
# LOAD BENCHMARK QUESTION BANKS
# =============================================================================

import json


TRACK1_FILE = (
    BENCHMARK_DIR
    / "track1_20_questions.json"
)

TRACK2_FILE = (
    BENCHMARK_DIR
    / "track2_final_32_questions.json"
)


# =============================================================================
# LOAD TRACK 1
# =============================================================================

with open(
    TRACK1_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions = json.load(
        file
    )


# =============================================================================
# LOAD TRACK 2
# =============================================================================

with open(
    TRACK2_FILE,
    "r",
    encoding="utf-8",
) as file:

    benchmark_questions_track2 = json.load(
        file
    )


# =============================================================================
# VALIDATION
# =============================================================================

print("=" * 90)
print("BENCHMARK QUESTION BANKS LOADED")
print("=" * 90)

print("\nTRACK 1")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions[0]['id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions[-1]['id']}"
)


print("\nTRACK 2")
print("-" * 60)

print(
    f"Questions : "
    f"{len(benchmark_questions_track2)}"
)

print(
    f"First ID  : "
    f"{benchmark_questions_track2[0]['evaluation_id']}"
)

print(
    f"Last ID   : "
    f"{benchmark_questions_track2[-1]['evaluation_id']}"
)


# =============================================================================
# COUNT CHECKS
# =============================================================================

if len(benchmark_questions) != 20:

    raise RuntimeError(
        f"Track 1 expected 20 questions, "
        f"found {len(benchmark_questions)}."
    )


if len(benchmark_questions_track2) != 32:

    raise RuntimeError(
        f"Track 2 expected 32 questions, "
        f"found {len(benchmark_questions_track2)}."
    )


print("\n" + "=" * 90)
print("TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED")
print("=" * 90)

BENCHMARK QUESTION BANKS LOADED

TRACK 1
------------------------------------------------------------
Questions : 20
First ID  : Q01
Last ID   : Q20

TRACK 2
------------------------------------------------------------
Questions : 32
First ID  : T2-01
Last ID   : T2-32

TRACK 1 + TRACK 2 QUESTION BANKS VALIDATED


In [21]:
# =============================================================================
# ROBUST MODEL RESPONSE DATASET LOADER
# =============================================================================

response_files = {
    "Track 1": {
        "Essential AI + RAG": RESPONSE_DIR / "track1_essentialai_rag_results_20260828_125927.json",
        "Essential AI Only": RESPONSE_DIR / "track1_essentialai_llm_results_20260828_101033.json",
        "Otel 2.0 Only": RESPONSE_DIR / "track1_otel2_llm_results_20260826_232923.json",
        "Gemma 4 Only": RESPONSE_DIR / "track1_gemma4_llm_results.json",
        "Gemma 4 + RAG": RESPONSE_DIR / "track1_gemma4_vllm_rag_results_20260827_193305.json",
    },
    "Track 2": {
        "Essential AI + RAG": RESPONSE_DIR / "track2_essentialai_rag_results_20260828_132208.json",
        "Essential AI Only": RESPONSE_DIR / "track2_essentialai_llm_results_20260828_103230.json",
        "Otel 2.0 Only": RESPONSE_DIR / "track2_otel2_llm_results_20260827_004654.json",
        "Gemma 4 Only": RESPONSE_DIR / "track2_gemma4_llm_result.json",
        "Gemma 4 + RAG": RESPONSE_DIR / "track2_gemma4_vllm_rag_results_20260827_193951.json",
    }
}

loaded_responses = {"Track 1": {}, "Track 2": {}}

print("=" * 90)
print("LOADING & NORMALIZING MODEL RESPONSE PAYLOADS")
print("=" * 90)

for track_name, models in response_files.items():
    print(f"\n{track_name}")
    print("-" * 60)

    for model_name, file_path in models.items():
        if not file_path.exists():
            raise FileNotFoundError(f"Missing response file for {track_name} - {model_name}:\n{file_path}")

        with open(file_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        # Handle both flat list formats and nested dict formats {'results': [...]}
        if isinstance(raw_data, dict) and "results" in raw_data:
            data_list = raw_data["results"]
        elif isinstance(raw_data, list):
            data_list = raw_data
        else:
            raise ValueError(f"Unrecognized structure in {file_path.name}")

        loaded_responses[track_name][model_name] = data_list

        expected_len = 20 if track_name == "Track 1" else 32
        print(f"  [Loaded] {model_name:<20} : {len(data_list)} records (Source: {file_path.name})")

        if len(data_list) != expected_len:
            print(f"  ⚠️ Warning: Expected {expected_len} records, found {len(data_list)}.")

print("\n" + "=" * 90)
print("ALL RESPONSE PAYLOADS NORMALIZED & LOADED SUCCESSFULLY")
print("=" * 90)

LOADING & NORMALIZING MODEL RESPONSE PAYLOADS

Track 1
------------------------------------------------------------
  [Loaded] Essential AI + RAG   : 20 records (Source: track1_essentialai_rag_results_20260828_125927.json)
  [Loaded] Essential AI Only    : 20 records (Source: track1_essentialai_llm_results_20260828_101033.json)
  [Loaded] Otel 2.0 Only        : 20 records (Source: track1_otel2_llm_results_20260826_232923.json)
  [Loaded] Gemma 4 Only         : 20 records (Source: track1_gemma4_llm_results.json)
  [Loaded] Gemma 4 + RAG        : 20 records (Source: track1_gemma4_vllm_rag_results_20260827_193305.json)

Track 2
------------------------------------------------------------
  [Loaded] Essential AI + RAG   : 32 records (Source: track2_essentialai_rag_results_20260828_132208.json)
  [Loaded] Essential AI Only    : 32 records (Source: track2_essentialai_llm_results_20260828_103230.json)
  [Loaded] Otel 2.0 Only        : 32 records (Source: track2_otel2_llm_results_20260827_0046

## **Load Rag Retriever**

### **Load Retriever**

In [23]:
# =============================================================================
# RETRIEVER V1 — STANDARD IN-MEMORY IMPLEMENTATION
# =============================================================================

import json
import time
from pathlib import Path

import faiss
import numpy as np
import torch

from sentence_transformers import SentenceTransformer


# =============================================================================
# RETRIEVER V1 CONFIGURATION
# =============================================================================

MODEL_NAME = "BAAI/bge-m3"

MAX_SEQ_LENGTH = 1280

DEFAULT_K = 7

EXPECTED_VECTORS = 1_506_367

EXPECTED_DIMENSION = 1024

RETRIEVER_VERSION = "V1"


# =============================================================================
# RUNTIME OBJECTS
# =============================================================================

faiss_index = None

query_encoder = None

vector_metadata = None

chunk_text_map = None

faiss_manifest = None


# =============================================================================
# INITIALIZE RETRIEVER
# =============================================================================

def initialize_retriever(
    faiss_dir,
    chunk_dir,
    device="cuda",
):
    """
    Initialize Retriever V1.

    Loads:
        - FAISS index
        - BGE-M3 query encoder
        - Vector metadata
        - Chunk text
    """

    global faiss_index
    global query_encoder
    global vector_metadata
    global chunk_text_map
    global faiss_manifest


    # =========================================================================
    # VALIDATE DEVICE
    # =========================================================================

    if device.startswith("cuda"):

        if not torch.cuda.is_available():

            raise RuntimeError(
                "CUDA GPU is required for Retriever V1."
            )


    # =========================================================================
    # RESOLVE PATHS
    # =========================================================================

    faiss_dir = Path(
        faiss_dir
    )

    chunk_dir = Path(
        chunk_dir
    )


    # =========================================================================
    # FAISS ARTIFACTS
    # =========================================================================

    faiss_index_path = (
        faiss_dir
        / "faiss_index_flat_ip.index"
    )

    faiss_manifest_path = (
        faiss_dir
        / "faiss_manifest.json"
    )


    for path in [
        faiss_dir,
        chunk_dir,
        faiss_index_path,
        faiss_manifest_path,
    ]:

        if not path.exists():

            raise FileNotFoundError(
                f"Required Retriever V1 resource not found:\n"
                f"{path}"
            )


    # =========================================================================
    # LOAD FAISS
    # =========================================================================

    faiss_index = faiss.read_index(
        str(faiss_index_path)
    )


    if faiss_index.ntotal != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Unexpected FAISS vector count: "
            f"{faiss_index.ntotal:,}"
        )


    if faiss_index.d != EXPECTED_DIMENSION:

        raise RuntimeError(
            f"Unexpected FAISS dimension: "
            f"{faiss_index.d}"
        )


    # =========================================================================
    # LOAD FAISS MANIFEST
    # =========================================================================

    with open(
        faiss_manifest_path,
        "r",
        encoding="utf-8",
    ) as file:

        faiss_manifest = json.load(
            file
        )


    # =========================================================================
    # LOAD BGE-M3
    # =========================================================================

    query_encoder = SentenceTransformer(
        MODEL_NAME,
        device=device,
    )

    query_encoder.max_seq_length = (
        MAX_SEQ_LENGTH
    )

    query_encoder.half()

    query_encoder.eval()


    # =========================================================================
    # LOAD VECTOR METADATA
    # =========================================================================

    vector_mapping_path = (
        faiss_dir
        / "vector_mapping.jsonl"
    )

    if not vector_mapping_path.exists():

        raise FileNotFoundError(
            f"Vector mapping not found:\n"
            f"{vector_mapping_path}"
        )


    vector_metadata = {}


    with open(
        vector_mapping_path,
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if not line.strip():
                continue

            record = json.loads(
                line
            )

            vector_id = int(
                record["vector_id"]
            )

            vector_metadata[
                vector_id
            ] = record


    if len(vector_metadata) != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Expected {EXPECTED_VECTORS:,} "
            f"metadata records, found "
            f"{len(vector_metadata):,}"
        )


    # =========================================================================
    # LOAD CHUNK TEXT
    # =========================================================================

    chunk_shards = sorted(
        chunk_dir.glob(
            "chunks_*.jsonl"
        )
    )


    if not chunk_shards:

        raise FileNotFoundError(
            "No chunk shards found in:\n"
            f"{chunk_dir}"
        )


    chunk_text_map = {}


    for shard_path in chunk_shards:

        with open(
            shard_path,
            "r",
            encoding="utf-8",
        ) as file:

            for line in file:

                if not line.strip():
                    continue

                record = json.loads(
                    line
                )

                chunk_id = record.get(
                    "chunk_id"
                )

                if chunk_id is not None:

                    chunk_text_map[
                        chunk_id
                    ] = record["text"]


    if len(chunk_text_map) != EXPECTED_VECTORS:

        raise RuntimeError(
            f"Expected {EXPECTED_VECTORS:,} "
            f"chunk texts, found "
            f"{len(chunk_text_map):,}"
        )


    # =========================================================================
    # RETURN STATUS
    # =========================================================================

    return {
        "version": RETRIEVER_VERSION,
        "model": MODEL_NAME,
        "device": device,
        "embedding_dimension":
            EXPECTED_DIMENSION,
        "faiss_vectors":
            int(faiss_index.ntotal),
        "faiss_index":
            faiss_manifest.get(
                "index_type"
            ),
        "similarity":
            faiss_manifest.get(
                "similarity"
            ),
        "metadata_rows":
            len(vector_metadata),
        "chunk_rows":
            len(chunk_text_map),
        "default_k":
            DEFAULT_K,
    }


# =============================================================================
# RETRIEVE
# =============================================================================

def retrieve(
    query: str,
    k: int = DEFAULT_K,
):
    """
    Execute Retriever V1.
    """

    global faiss_index
    global query_encoder
    global vector_metadata
    global chunk_text_map


    # =========================================================================
    # VALIDATE INITIALIZATION
    # =========================================================================

    if (
        faiss_index is None
        or query_encoder is None
        or vector_metadata is None
        or chunk_text_map is None
    ):

        raise RuntimeError(
            "Retriever V1 is not initialized. "
            "Call initialize_retriever() first."
        )


    # =========================================================================
    # INPUT VALIDATION
    # =========================================================================

    if not isinstance(
        query,
        str,
    ):

        raise TypeError(
            "query must be a string."
        )


    query = query.strip()


    if not query:

        raise ValueError(
            "query cannot be empty."
        )


    if not isinstance(
        k,
        int,
    ):

        raise TypeError(
            "k must be an integer."
        )


    if k <= 0:

        raise ValueError(
            "k must be greater than zero."
        )


    # =========================================================================
    # QUERY EMBEDDING
    # =========================================================================

    embedding_start = time.time()


    query_embedding = (
        query_encoder.encode(
            [query],
            batch_size=1,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )


    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32,
    )


    query_embedding /= np.linalg.norm(
        query_embedding,
        axis=1,
        keepdims=True,
    )


    embedding_time = (
        time.time()
        - embedding_start
    )


    # =========================================================================
    # FAISS SEARCH
    # =========================================================================

    search_start = time.time()


    scores, vector_ids = (
        faiss_index.search(
            query_embedding,
            k,
        )
    )


    search_time = (
        time.time()
        - search_start
    )


    # =========================================================================
    # RESULT RESOLUTION
    # =========================================================================

    results = []


    for rank, (
        vector_id,
        score,
    ) in enumerate(
        zip(
            vector_ids[0],
            scores[0],
        ),
        start=1,
    ):

        vector_id = int(
            vector_id
        )


        if vector_id < 0:
            continue


        metadata = vector_metadata.get(
            vector_id
        )


        if metadata is None:

            continue


        chunk_id = metadata.get(
            "chunk_id"
        )


        text = chunk_text_map.get(
            chunk_id
        )


        if text is None:

            continue


        results.append(
            {
                "rank": rank,
                "score": float(score),
                "vector_id": vector_id,
                "chunk_id": chunk_id,
                "document_id":
                    metadata.get(
                        "document_id"
                    ),
                "source":
                    metadata.get(
                        "source"
                    ),
                "title":
                    metadata.get(
                        "title"
                    ),
                "path":
                    metadata.get(
                        "path"
                    ),
                "text": text,
            }
        )


    # =========================================================================
    # TOTAL LATENCY
    # =========================================================================

    total_time = (
        embedding_time
        + search_time
    )


    return {
        "query": query,
        "k": k,
        "results": results,
        "timing": {
            "query_embedding_sec":
                embedding_time,
            "faiss_search_sec":
                search_time,
            "total_sec":
                total_time,
        },
    }

### **Initialize Retriever**

In [24]:
FAISS_DIR = Path(
    cliffordimaguezegie_telecom_bge_m3_faiss_path
)

CHUNK_DIR = Path(
    cliffordimaguezegie_telecom_chunks_path
)

retriever_status = initialize_retriever(
    faiss_dir=FAISS_DIR,
    chunk_dir=CHUNK_DIR,
    device="cuda",
)

print("=" * 90)
print("RETRIEVER V1 INITIALIZED")
print("=" * 90)

for key, value in retriever_status.items():
    print(
        f"{key:<25}: {value}"
    )

print("=" * 90)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RETRIEVER V1 INITIALIZED
version                  : V1
model                    : BAAI/bge-m3
device                   : cuda
embedding_dimension      : 1024
faiss_vectors            : 1506367
faiss_index              : IndexFlatIP
similarity               : inner_product
metadata_rows            : 1506367
chunk_rows               : 1506367
default_k                : 7


### **Validate Retriever**

In [25]:
# =============================================================================
# RAG V1 — RETRIEVER V1 FINAL VALIDATION
# =============================================================================

TEST_QUERY = (
    "What are the primary responsibilities of the AMF "
    "in a 5G Standalone network?"
)

results = retrieve(
    TEST_QUERY,
    k=7,
)

print("=" * 90)
print("RAG V1 — RETRIEVER V1 FINAL VALIDATION")
print("=" * 90)

print(
    f"Query              : "
    f"{results['query']}"
)

print(
    f"K                  : "
    f"{results['k']}"
)

print(
    f"Results returned   : "
    f"{len(results['results'])}"
)

print(
    f"Query embedding    : "
    f"{results['timing']['query_embedding_sec']:.4f} sec"
)

print(
    f"FAISS search       : "
    f"{results['timing']['faiss_search_sec']:.4f} sec"
)

print(
    f"Total retrieval    : "
    f"{results['timing']['total_sec']:.4f} sec"
)

if len(results["results"]) != 7:
    raise RuntimeError(
        "Retriever V1 did not return 7 results."
    )

top = results["results"][0]

print("\nTop result:")

print(
    f"Rank       : {top['rank']}"
)

print(
    f"Score      : {top['score']:.4f}"
)

print(
    f"Chunk ID   : {top['chunk_id']}"
)

print(
    f"Source     : {top['source']}"
)

print(
    f"Title      : {top['title']}"
)

print(
    f"Text       : "
    f"{top['text'][:500]}"
)

print("\n" + "=" * 90)
print("RETRIEVER V1 FINAL VALIDATION PASSED")
print("=" * 90)

RAG V1 — RETRIEVER V1 FINAL VALIDATION
Query              : What are the primary responsibilities of the AMF in a 5G Standalone network?
K                  : 7
Results returned   : 7
Query embedding    : 0.5983 sec
FAISS search       : 0.2700 sec
Total retrieval    : 0.8683 sec

Top result:
Rank       : 1
Score      : 0.7065
Chunk ID   : standards/3gpp_rel18/original/rel_15.docx::chunk_0013
Source     : standards
Title      : rel_15
Text       : the Core Network side, the AMF ("Access and Mobility management Function") oversees all the signalling which is not specific to User Data, such as mobility or security. The SMF ("Session Management Function"), takes care of the signalling related to User Data traffic, such as session establishment. Finally, The UPF ("User Plane Function") represents the handling of user data.
On the Access Network side, the gNB (5G Node B) performs all the main AN-related tasks, including Radio Resource Manageme

RETRIEVER V1 FINAL VALIDATION PASSED


# **Evaluating Relevance to Corpus of the Track 1 Questions**

In [26]:
# =============================================================================
# MODULE 2 — TRACK 1 CORPUS RELEVANCE AUDIT
# Q01-Q20 vs ACTUAL RAG CORPUS
# =============================================================================
#
# PURPOSE
# -------
# Determine whether each existing Track 1 benchmark question is adequately
# supported by the ACTUAL RAG corpus before interpreting RAG model performance.
#
# IMPORTANT
# ---------
# - No model responses are generated.
# - No benchmark questions are modified.
# - No corpus files are modified.
# - No embeddings are regenerated.
# - Uses the EXISTING Retriever V1.
# - Similarity score alone does NOT determine corpus relevance.
# - Actual retrieved chunk text must be reviewed.
#
# OUTPUT
# ------
# For each Q01-Q20:
#   - Top-10 retrieval evidence
#   - Top-1 / Top-3 / Top-7 score summary
#   - Source distribution
#   - Retrieved chunk text
#   - Machine-readable audit dataframe
#
# Final manual classification:
#
#   STRONG  = Corpus contains sufficient evidence to answer the question.
#   PARTIAL = Corpus contains meaningful but incomplete evidence.
#   WEAK    = Corpus contains only indirect/peripheral evidence.
#   NONE    = No meaningful supporting evidence identified.
#
# =============================================================================


from collections import Counter
import numpy as np
import pandas as pd


# =============================================================================
# 1. VALIDATE REQUIRED OBJECTS
# =============================================================================

required_objects = [
    "benchmark_questions",
    "retrieve",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Required objects are missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )


# =============================================================================
# 2. VALIDATE TRACK 1 QUESTION BANK
# =============================================================================

if len(benchmark_questions) != 20:

    raise RuntimeError(
        f"Track 1 expected 20 questions, "
        f"found {len(benchmark_questions)}."
    )


expected_ids = [
    f"Q{i:02d}"
    for i in range(1, 21)
]

actual_ids = [
    item["id"]
    for item in benchmark_questions
]

if actual_ids != expected_ids:

    raise RuntimeError(
        "Track 1 question IDs are not aligned "
        "with expected Q01-Q20 sequence.\n"
        f"Expected: {expected_ids}\n"
        f"Actual  : {actual_ids}"
    )


# =============================================================================
# 3. AUDIT CONFIGURATION
# =============================================================================
#
# Production Retriever V1 uses K=7.
#
# For corpus relevance auditing we inspect K=10:
#
# Rank 1-7:
#   Evidence available to the production RAG.
#
# Rank 8-10:
#   Evidence may exist in the corpus but fall just outside production K=7.
#
# =============================================================================

AUDIT_K = 10
PRODUCTION_K = 7
TEXT_PREVIEW_CHARS = 1200


print("=" * 100)
print("MODULE 2 — TRACK 1 CORPUS RELEVANCE AUDIT")
print("=" * 100)

print(
    f"{'Questions':<30}: "
    f"{len(benchmark_questions)}"
)

print(
    f"{'Production Retriever K':<30}: "
    f"{PRODUCTION_K}"
)

print(
    f"{'Audit Retrieval K':<30}: "
    f"{AUDIT_K}"
)

print("=" * 100)


# =============================================================================
# 4. AUDIT CONTAINERS
# =============================================================================

audit_rows = []

audit_evidence = {}


# =============================================================================
# 5. RETRIEVE CORPUS EVIDENCE FOR Q01-Q20
# =============================================================================

for question_item in benchmark_questions:

    question_id = question_item["id"]

    category = question_item.get(
        "category",
        "UNKNOWN"
    )

    question = question_item["question"]


    # =========================================================================
    # QUESTION HEADER
    # =========================================================================

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"{question_id} | "
        f"{category}"
    )

    print(
        "=" * 100
    )

    print(
        f"\nQUESTION:\n"
        f"{question}"
    )


    # =========================================================================
    # USE EXISTING RETRIEVER V1
    # =========================================================================

    retrieval = retrieve(
        query=question,
        k=AUDIT_K,
    )

    results = retrieval.get(
        "results",
        []
    )


    # Preserve complete evidence for later analysis.
    audit_evidence[
        question_id
    ] = results


    # =========================================================================
    # RETRIEVAL EVIDENCE
    # =========================================================================

    print(
        "\nTOP RETRIEVED CORPUS EVIDENCE"
    )

    print(
        "-" * 100
    )


    source_counter = Counter()


    for result in results:

        rank = result.get(
            "rank"
        )

        score = result.get(
            "score",
            np.nan
        )

        source = (
            result.get("source")
            or "UNKNOWN"
        )

        title = (
            result.get("title")
            or "UNKNOWN"
        )

        document_id = (
            result.get("document_id")
            or "UNKNOWN"
        )

        chunk_id = (
            result.get("chunk_id")
            or "UNKNOWN"
        )

        text = (
            result.get("text")
            or ""
        )


        source_counter[
            source
        ] += 1


        production_marker = (
            "PRODUCTION K=7"
            if rank <= PRODUCTION_K
            else "AUDIT ONLY"
        )


        print(
            f"\nRank {rank:>2}"
            f" | Score {score:.4f}"
            f" | {production_marker}"
        )

        print(
            f"Source      : "
            f"{source}"
        )

        print(
            f"Title       : "
            f"{title[:160]}"
        )

        print(
            f"Document ID : "
            f"{str(document_id)[:160]}"
        )

        print(
            f"Chunk ID    : "
            f"{chunk_id}"
        )

        print(
            "\nCHUNK TEXT:"
        )

        print(
            text[
                :TEXT_PREVIEW_CHARS
            ]
        )

        print(
            "-" * 100
        )


    # =========================================================================
    # SCORE SUMMARY
    # =========================================================================

    scores = [
        result["score"]
        for result in results
        if result.get("score") is not None
    ]


    top1_score = (
        scores[0]
        if len(scores) >= 1
        else np.nan
    )


    mean_top3_score = (
        float(
            np.mean(
                scores[:3]
            )
        )
        if len(scores) >= 1
        else np.nan
    )


    mean_top7_score = (
        float(
            np.mean(
                scores[:7]
            )
        )
        if len(scores) >= 1
        else np.nan
    )


    mean_top10_score = (
        float(
            np.mean(
                scores[:10]
            )
        )
        if len(scores) >= 1
        else np.nan
    )


    # =========================================================================
    # TOP RESULT METADATA
    # =========================================================================

    top_result = (
        results[0]
        if results
        else {}
    )


    # =========================================================================
    # STORE QUESTION-LEVEL AUDIT SUMMARY
    # =========================================================================

    audit_rows.append(
        {
            "question_id":
                question_id,

            "category":
                category,

            "question":
                question,

            "top1_score":
                top1_score,

            "mean_top3_score":
                mean_top3_score,

            "mean_top7_score":
                mean_top7_score,

            "mean_top10_score":
                mean_top10_score,

            "top1_source":
                top_result.get(
                    "source"
                ),

            "top1_title":
                top_result.get(
                    "title"
                ),

            "top1_document_id":
                top_result.get(
                    "document_id"
                ),

            "unique_sources_top10":
                len(
                    source_counter
                ),

            "retrieved_chunks":
                len(
                    results
                ),

            # Intentionally blank.
            # Complete after reviewing actual chunk evidence.
            "corpus_relevance":
                None,

            "production_k7_support":
                None,

            "evaluation_recommendation":
                None,

            "review_notes":
                None,
        }
    )


# =============================================================================
# 6. BUILD AUDIT DATAFRAME
# =============================================================================

track1_corpus_audit_df = pd.DataFrame(
    audit_rows
)


# =============================================================================
# 7. DISPLAY RETRIEVAL SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TRACK 1 — CORPUS RELEVANCE RETRIEVAL SUMMARY"
)

print(
    "=" * 100
)


summary_columns = [
    "question_id",
    "category",
    "top1_score",
    "mean_top3_score",
    "mean_top7_score",
    "top1_source",
    "unique_sources_top10",
    "retrieved_chunks",
]


display(
    track1_corpus_audit_df[
        summary_columns
    ]
)


# =============================================================================
# 8. SOURCE DISTRIBUTION ACROSS TRACK 1
# =============================================================================

track1_source_counter = Counter()


for question_id, results in (
    audit_evidence.items()
):

    question_sources = set()

    for result in results:

        source = (
            result.get("source")
            or "UNKNOWN"
        )

        question_sources.add(
            source
        )

    for source in question_sources:

        track1_source_counter[
            source
        ] += 1


print(
    "\n"
    + "=" * 100
)

print(
    "SOURCE APPEARANCE ACROSS TRACK 1 TOP-10 RETRIEVAL"
)

print(
    "=" * 100
)


for source, count in (
    track1_source_counter
    .most_common()
):

    print(
        f"{source:<40}"
        f": {count:>2} / 20 questions"
    )


# =============================================================================
# 9. CREATE REVIEW TABLE
# =============================================================================
#
# These columns are intentionally left blank.
#
# They should be completed only AFTER inspecting the retrieved text.
#
# Suggested values:
#
# corpus_relevance:
#   STRONG
#   PARTIAL
#   WEAK
#   NONE
#
# production_k7_support:
#   YES
#   PARTIAL
#   NO
#
# evaluation_recommendation:
#   KEEP
#   REVIEW
#   EXCLUDE
#   GENERAL_REASONING_ONLY
#
# =============================================================================

review_columns = [
    "question_id",
    "category",
    "top1_score",
    "mean_top3_score",
    "mean_top7_score",
    "corpus_relevance",
    "production_k7_support",
    "evaluation_recommendation",
    "review_notes",
]


print(
    "\n"
    + "=" * 100
)

print(
    "TRACK 1 — CORPUS RELEVANCE REVIEW TABLE"
)

print(
    "=" * 100
)


display(
    track1_corpus_audit_df[
        review_columns
    ]
)


# =============================================================================
# 10. COMPLETION MESSAGE
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TRACK 1 CORPUS RETRIEVAL AUDIT COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nINTERPRETATION RULE:"
)

print(
    "\nDo NOT classify corpus relevance "
    "from similarity score alone."
)

print(
    "\nReview the actual retrieved chunk text "
    "for each Q01-Q20 before assigning:"
)

print(
    "\n  STRONG"
    "\n  PARTIAL"
    "\n  WEAK"
    "\n  NONE"
)

print(
    "\nThen determine whether each question "
    "should be retained for RAG-vs-LLM evaluation."
)

print(
    "\nThe existing model responses should NOT "
    "be regenerated at this stage."
)

print(
    "=" * 100
)

Streaming output truncated to the last 5000 lines.
![Figure 5: Network architecture diagram showing the 5G core network components and their connections.](562f471e8153729557e6a4ee6343c32c_img.jpg)

The diagram illustrates the 5G network architecture. On the left, a green vertical bar represents the User Equipment (UE). A dashed green line connects the UE to the NG RAN (Next Generation Radio Access Network), which is a blue vertical bar containing the gNB (gNodeB). A solid green line labeled N1 connects the gNB to the 5G core. A solid blue line labeled N2 connects the gNB to the AMF (Access and Mobility Management Function) within the 5G core. The 5G core is enclosed in an orange rounded rectangle. Inside, the AMF is connected to the SMF (Session Management Function). The SMF is connected to the UPF (User Plane Function). The SMF is also connected to a stack of network functions: NRF (Network Repository Function), NSSF (Network Slice Selection Function), AUSF (Authentication Server Func

,question_id,category,top1_score,mean_top3_score,mean_top7_score,top1_source,unique_sources_top10,retrieved_chunks
0,Q01,5G Core,0.706545,0.692330,0.673813,standards,2,10
1,Q02,5G Core,0.721201,0.708249,0.698088,standards,1,10
2,Q03,5G RAN,0.746015,0.738995,0.724302,standards,3,10
3,Q04,5G RAN,0.701377,0.693252,0.685002,academic,2,10
4,Q05,5G SA Procedures,0.732167,0.730501,0.726462,standards,1,10
5,Q06,5G SA Procedures,0.744180,0.740568,0.733703,standards,1,10
6,Q07,Open RAN,0.713507,0.697138,0.678701,standards,2,10
7,Q08,Open RAN,0.746895,0.723349,0.709581,standards,2,10
8,Q09,Cloud-Native Telecom,0.676047,0.666029,0.653445,standards,4,10
9,Q10,Cloud-Native Telecom,0.699503,0.691766,0.685535,cloud_native,1,10



SOURCE APPEARANCE ACROSS TRACK 1 TOP-10 RETRIEVAL
standards                               : 19 / 20 questions
academic                                :  6 / 20 questions
open_source                             :  5 / 20 questions
cloud_native                            :  2 / 20 questions
vendor_network                          :  1 / 20 questions
cloud_platform                          :  1 / 20 questions

TRACK 1 — CORPUS RELEVANCE REVIEW TABLE


,question_id,category,top1_score,mean_top3_score,mean_top7_score,corpus_relevance,production_k7_support,evaluation_recommendation,review_notes
0,Q01,5G Core,0.706545,0.692330,0.673813,None,None,None,None
1,Q02,5G Core,0.721201,0.708249,0.698088,None,None,None,None
2,Q03,5G RAN,0.746015,0.738995,0.724302,None,None,None,None
3,Q04,5G RAN,0.701377,0.693252,0.685002,None,None,None,None
4,Q05,5G SA Procedures,0.732167,0.730501,0.726462,None,None,None,None
5,Q06,5G SA Procedures,0.744180,0.740568,0.733703,None,None,None,None
6,Q07,Open RAN,0.713507,0.697138,0.678701,None,None,None,None
7,Q08,Open RAN,0.746895,0.723349,0.709581,None,None,None,None
8,Q09,Cloud-Native Telecom,0.676047,0.666029,0.653445,None,None,None,None
9,Q10,Cloud-Native Telecom,0.699503,0.691766,0.685535,None,None,None,None



TRACK 1 CORPUS RETRIEVAL AUDIT COMPLETE

INTERPRETATION RULE:

Do NOT classify corpus relevance from similarity score alone.

Review the actual retrieved chunk text for each Q01-Q20 before assigning:

  STRONG
  PARTIAL
  WEAK
  NONE

Then determine whether each question should be retained for RAG-vs-LLM evaluation.

The existing model responses should NOT be regenerated at this stage.


In [27]:
# =============================================================================
# TRACK 1 — EXPERT CORPUS RELEVANCE CLASSIFICATION
# =============================================================================

expert_classification = {

    "Q01": ("STRONG", "YES", "KEEP"),
    "Q02": ("STRONG", "YES", "KEEP"),
    "Q03": ("STRONG", "YES", "KEEP"),
    "Q04": ("STRONG", "YES", "KEEP"),
    "Q05": ("STRONG", "YES", "KEEP"),
    "Q06": ("STRONG", "YES", "KEEP"),
    "Q07": ("STRONG", "YES", "KEEP"),
    "Q08": ("STRONG", "YES", "KEEP"),

    "Q09": ("PARTIAL-STRONG", "YES", "KEEP"),
    "Q10": ("PARTIAL-STRONG", "YES", "KEEP"),

    "Q11": ("WEAK-PARTIAL", "PARTIAL", "REVIEW"),
    "Q12": ("PARTIAL", "YES", "KEEP"),
    "Q13": ("PARTIAL", "YES", "REVIEW"),
    "Q14": ("PARTIAL-STRONG", "YES", "KEEP"),
    "Q15": ("PARTIAL", "YES", "KEEP"),

    "Q16": ("STRONG", "YES", "KEEP"),

    "Q17": ("PARTIAL-STRONG", "YES", "KEEP"),
    "Q18": ("WEAK", "PARTIAL", "REVIEW"),
    "Q19": ("PARTIAL", "PARTIAL", "REVIEW"),
    "Q20": ("PARTIAL", "PARTIAL", "REVIEW"),
}


for qid, (
    relevance,
    k7_support,
    recommendation,
) in expert_classification.items():

    mask = (
        track1_corpus_audit_df[
            "question_id"
        ] == qid
    )

    track1_corpus_audit_df.loc[
        mask,
        "corpus_relevance"
    ] = relevance

    track1_corpus_audit_df.loc[
        mask,
        "production_k7_support"
    ] = k7_support

    track1_corpus_audit_df.loc[
        mask,
        "evaluation_recommendation"
    ] = recommendation


display(
    track1_corpus_audit_df[
        [
            "question_id",
            "category",
            "top1_score",
            "mean_top3_score",
            "corpus_relevance",
            "production_k7_support",
            "evaluation_recommendation",
        ]
    ]
)

,question_id,category,top1_score,mean_top3_score,corpus_relevance,production_k7_support,evaluation_recommendation
0,Q01,5G Core,0.706545,0.692330,STRONG,YES,KEEP
1,Q02,5G Core,0.721201,0.708249,STRONG,YES,KEEP
2,Q03,5G RAN,0.746015,0.738995,STRONG,YES,KEEP
3,Q04,5G RAN,0.701377,0.693252,STRONG,YES,KEEP
4,Q05,5G SA Procedures,0.732167,0.730501,STRONG,YES,KEEP
5,Q06,5G SA Procedures,0.744180,0.740568,STRONG,YES,KEEP
6,Q07,Open RAN,0.713507,0.697138,STRONG,YES,KEEP
7,Q08,Open RAN,0.746895,0.723349,STRONG,YES,KEEP
8,Q09,Cloud-Native Telecom,0.676047,0.666029,PARTIAL-STRONG,YES,KEEP
9,Q10,Cloud-Native Telecom,0.699503,0.691766,PARTIAL-STRONG,YES,KEEP


In [28]:
# =============================================================================
# TRACK 1 — ASSIGN EVALUATION TYPE
# =============================================================================

engineering_synthesis_questions = {
    "Q11",
    "Q13",
    "Q18",
    "Q19",
    "Q20",
}

track1_corpus_audit_df[
    "evaluation_type"
] = track1_corpus_audit_df[
    "question_id"
].apply(
    lambda qid: (
        "ENGINEERING_SYNTHESIS"
        if qid in engineering_synthesis_questions
        else "CORPUS_GROUNDED"
    )
)


display(
    track1_corpus_audit_df[
        [
            "question_id",
            "category",
            "top1_score",
            "mean_top3_score",
            "corpus_relevance",
            "production_k7_support",
            "evaluation_recommendation",
            "evaluation_type",
        ]
    ]
)

,question_id,category,top1_score,mean_top3_score,corpus_relevance,production_k7_support,evaluation_recommendation,evaluation_type
0,Q01,5G Core,0.706545,0.692330,STRONG,YES,KEEP,CORPUS_GROUNDED
1,Q02,5G Core,0.721201,0.708249,STRONG,YES,KEEP,CORPUS_GROUNDED
2,Q03,5G RAN,0.746015,0.738995,STRONG,YES,KEEP,CORPUS_GROUNDED
3,Q04,5G RAN,0.701377,0.693252,STRONG,YES,KEEP,CORPUS_GROUNDED
4,Q05,5G SA Procedures,0.732167,0.730501,STRONG,YES,KEEP,CORPUS_GROUNDED
5,Q06,5G SA Procedures,0.744180,0.740568,STRONG,YES,KEEP,CORPUS_GROUNDED
6,Q07,Open RAN,0.713507,0.697138,STRONG,YES,KEEP,CORPUS_GROUNDED
7,Q08,Open RAN,0.746895,0.723349,STRONG,YES,KEEP,CORPUS_GROUNDED
8,Q09,Cloud-Native Telecom,0.676047,0.666029,PARTIAL-STRONG,YES,KEEP,CORPUS_GROUNDED
9,Q10,Cloud-Native Telecom,0.699503,0.691766,PARTIAL-STRONG,YES,KEEP,CORPUS_GROUNDED


In [29]:
from pathlib import Path

# =============================================================================
# SAVE TRACK 1 CORPUS / EVALUATION CLASSIFICATION
# =============================================================================

RESULTS_DIR = Path("./results")

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRACK1_AUDIT_FILE = (
    RESULTS_DIR
    / "track1_corpus_relevance_classification.csv"
)

track1_corpus_audit_df.to_csv(
    TRACK1_AUDIT_FILE,
    index=False,
    encoding="utf-8",
)

print("=" * 90)
print("TRACK 1 CLASSIFICATION SAVED")
print("=" * 90)
print(f"Saved to: {TRACK1_AUDIT_FILE}")
print("=" * 90)

TRACK 1 CLASSIFICATION SAVED
Saved to: results/track1_corpus_relevance_classification.csv


In [30]:
from pathlib import Path
import pandas as pd

TRACK1_AUDIT_DIR = Path(
    cliffordimaguezegie_relevance_path
)

TRACK1_AUDIT_FILE = (
    TRACK1_AUDIT_DIR
    / "track1_corpus_relevance_classification.csv"
)

track1_corpus_audit_df = pd.read_csv(
    TRACK1_AUDIT_FILE
)

print("=" * 90)
print("TRACK 1 CORPUS RELEVANCE CLASSIFICATION LOADED")
print("=" * 90)

print(
    f"Questions : "
    f"{len(track1_corpus_audit_df)}"
)

print(
    f"First ID  : "
    f"{track1_corpus_audit_df['question_id'].iloc[0]}"
)

print(
    f"Last ID   : "
    f"{track1_corpus_audit_df['question_id'].iloc[-1]}"
)

print("\nEvaluation Type")
print("-" * 60)

print(
    track1_corpus_audit_df[
        "evaluation_type"
    ].value_counts()
)

print("\nCorpus Relevance")
print("-" * 60)

print(
    track1_corpus_audit_df[
        "corpus_relevance"
    ].value_counts()
)

print("=" * 90)

TRACK 1 CORPUS RELEVANCE CLASSIFICATION LOADED
Questions : 20
First ID  : Q01
Last ID   : Q20

Evaluation Type
------------------------------------------------------------
evaluation_type
CORPUS_GROUNDED          15
ENGINEERING_SYNTHESIS     5
Name: count, dtype: int64

Corpus Relevance
------------------------------------------------------------
corpus_relevance
STRONG            9
PARTIAL           5
PARTIAL-STRONG    4
WEAK-PARTIAL      1
WEAK              1
Name: count, dtype: int64


# **Track 1 Evaluation Dataset Preparation**

In [31]:
# =============================================================================
# TRACK 1 — RESPONSE SCHEMA INSPECTION
# =============================================================================

print("=" * 90)
print("TRACK 1 RESPONSE SCHEMA INSPECTION")
print("=" * 90)

for model_name, records in loaded_responses["Track 1"].items():

    print(f"\n{model_name}")
    print("-" * 60)

    print(f"Records : {len(records)}")

    if not records:
        print("No records found.")
        continue

    first_record = records[0]

    print(
        f"Record type : "
        f"{type(first_record).__name__}"
    )

    if isinstance(first_record, dict):

        print(
            f"Keys        : "
            f"{list(first_record.keys())}"
        )

        print("\nFirst record preview:")

        for key, value in first_record.items():

            value_preview = str(value)

            if len(value_preview) > 300:
                value_preview = (
                    value_preview[:300]
                    + " ..."
                )

            print(
                f"  {key:<25}: "
                f"{value_preview}"
            )

    else:

        print(
            f"First record value: "
            f"{str(first_record)[:500]}"
        )


print("\n" + "=" * 90)
print("SCHEMA INSPECTION COMPLETE")
print("=" * 90)

TRACK 1 RESPONSE SCHEMA INSPECTION

Essential AI + RAG
------------------------------------------------------------
Records : 20
Record type : dict
Keys        : ['question_id', 'category', 'question', 'expected_points', 'retriever', 'status', 'answer', 'input_tokens', 'output_tokens', 'generation_time_sec', 'retrieval', 'generation_config', 'error', 'timestamp_utc']

First record preview:
  question_id              : Q01
  category                 : 5G Core
  question                 : What are the primary responsibilities of the AMF in a 5G Standalone network?
  expected_points          : ['Access and mobility management', 'UE registration management', 'NAS signalling termination', 'UE authentication and security context handling', 'Mobility and reachability management', 'Interaction with other 5G Core network functions']
  retriever                : {'version': 'V1', 'k': 7}
  status                   : PASS
  answer                   : The primary responsibilities of the AMF (Acces

In [32]:
# =============================================================================
# MODULE 3 — TRACK 1 EVALUATION DATASET PREPARATION
# =============================================================================
#
# PURPOSE
# -------
# Build one unified Track 1 evaluation dataset:
#
#   20 questions × 5 systems = 100 rows
#
# Handles the known schema difference:
#
#   Essential AI + RAG  -> question_id / question / answer
#   Essential AI Only   -> question_id / question / answer
#   Otel 2.0 Only       -> question_id / question / answer
#   Gemma 4 Only        -> id / prompt / response
#   Gemma 4 + RAG       -> question_id / question / answer
#
# =============================================================================

import pandas as pd


# =============================================================================
# 1. EXPECTED MODELS
# =============================================================================

EXPECTED_MODELS = [
    "Essential AI + RAG",
    "Essential AI Only",
    "Otel 2.0 Only",
    "Gemma 4 Only",
    "Gemma 4 + RAG",
]


# =============================================================================
# 2. VALIDATE REQUIRED OBJECTS
# =============================================================================

required_objects = [
    "benchmark_questions",
    "loaded_responses",
    "track1_corpus_audit_df",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Required objects are missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )


# =============================================================================
# 3. TRACK 1 QUESTION METADATA
# =============================================================================

track1_question_df = pd.DataFrame(
    [
        {
            "question_id": item["id"],
            "category": item["category"],
            "question": item["question"],
            "expected_points": item.get(
                "expected_points"
            ),
        }
        for item in benchmark_questions
    ]
)


if len(track1_question_df) != 20:

    raise RuntimeError(
        f"Expected 20 Track 1 questions, "
        f"found {len(track1_question_df)}."
    )


# =============================================================================
# 4. CORPUS RELEVANCE METADATA
# =============================================================================

audit_columns = [
    "question_id",
    "corpus_relevance",
    "production_k7_support",
    "evaluation_recommendation",
    "evaluation_type",
]


track1_metadata_df = (
    track1_corpus_audit_df[
        audit_columns
    ]
    .copy()
)


# =============================================================================
# 5. NORMALIZE MODEL RESPONSE RECORD
# =============================================================================

def normalize_track1_record(
    model_name,
    record,
):

    # -------------------------------------------------------------------------
    # GEMMA 4 ONLY
    # -------------------------------------------------------------------------

    if model_name == "Gemma 4 Only":

        question_id = record.get(
            "id"
        )

        question = record.get(
            "prompt"
        )

        response = record.get(
            "response"
        )

        status = record.get(
            "status"
        )

        input_tokens = None
        output_tokens = None
        generation_time_sec = None

        retrieval_k = None
        retrieval_top1_score = None


    # -------------------------------------------------------------------------
    # ALL OTHER SYSTEMS
    # -------------------------------------------------------------------------

    else:

        question_id = record.get(
            "question_id"
        )

        question = record.get(
            "question"
        )

        response = record.get(
            "answer"
        )

        status = record.get(
            "status"
        )

        input_tokens = record.get(
            "input_tokens"
        )

        output_tokens = record.get(
            "output_tokens"
        )

        generation_time_sec = record.get(
            "generation_time_sec"
        )


        # ---------------------------------------------------------------------
        # RAG RETRIEVAL METADATA
        # ---------------------------------------------------------------------

        retrieval = record.get(
            "retrieval"
        )

        if isinstance(
            retrieval,
            dict,
        ):

            retrieval_k = retrieval.get(
                "k"
            )

            retrieval_results = (
                retrieval.get(
                    "results"
                )
                or []
            )

            retrieval_top1_score = (
                retrieval_results[0].get(
                    "score"
                )
                if retrieval_results
                else None
            )

        else:

            retrieval_k = None
            retrieval_top1_score = None


    # -------------------------------------------------------------------------
    # BASIC VALIDATION
    # -------------------------------------------------------------------------

    if not question_id:

        raise RuntimeError(
            f"{model_name}: "
            "missing question ID."
        )


    if not isinstance(
        response,
        str,
    ) or not response.strip():

        raise RuntimeError(
            f"{model_name} | {question_id}: "
            "missing generated response."
        )


    return {
        "question_id":
            question_id,

        "model_name":
            model_name,

        "response":
            response.strip(),

        "response_status":
            status,

        "source_question":
            question,

        "input_tokens":
            input_tokens,

        "output_tokens":
            output_tokens,

        "generation_time_sec":
            generation_time_sec,

        "retrieval_k":
            retrieval_k,

        "retrieval_top1_score":
            retrieval_top1_score,
    }


# =============================================================================
# 6. NORMALIZE ALL TRACK 1 RESPONSES
# =============================================================================

response_rows = []


for model_name in EXPECTED_MODELS:

    records = loaded_responses[
        "Track 1"
    ][model_name]


    if len(records) != 20:

        raise RuntimeError(
            f"{model_name}: "
            f"expected 20 records, "
            f"found {len(records)}."
        )


    for record in records:

        response_rows.append(
            normalize_track1_record(
                model_name=model_name,
                record=record,
            )
        )


track1_response_df = pd.DataFrame(
    response_rows
)


# =============================================================================
# 7. VALIDATE RESPONSE DATASET
# =============================================================================

if len(track1_response_df) != 100:

    raise RuntimeError(
        f"Expected 100 response rows, "
        f"found {len(track1_response_df)}."
    )


duplicate_pairs = (
    track1_response_df[
        [
            "question_id",
            "model_name",
        ]
    ]
    .duplicated()
)


if duplicate_pairs.any():

    raise RuntimeError(
        "Duplicate question/model pairs detected."
    )


# =============================================================================
# 8. MERGE QUESTION BANK
# =============================================================================

track1_evaluation_df = (
    track1_response_df
    .merge(
        track1_question_df,
        on="question_id",
        how="left",
        validate="many_to_one",
    )
)


# =============================================================================
# 9. MERGE CORPUS RELEVANCE CLASSIFICATION
# =============================================================================

track1_evaluation_df = (
    track1_evaluation_df
    .merge(
        track1_metadata_df,
        on="question_id",
        how="left",
        validate="many_to_one",
    )
)


# =============================================================================
# 10. QUESTION TEXT CONSISTENCY CHECK
# =============================================================================

question_mismatch = (
    track1_evaluation_df[
        "source_question"
    ].fillna("").str.strip()
    !=
    track1_evaluation_df[
        "question"
    ].fillna("").str.strip()
)


if question_mismatch.any():

    print(
        "\n[WARNING] Question text mismatch detected "
        "in some model payloads."
    )

    display(
        track1_evaluation_df.loc[
            question_mismatch,
            [
                "question_id",
                "model_name",
                "source_question",
                "question",
            ]
        ]
    )

else:

    print(
        "\n[OK] Question text aligned "
        "across all model payloads."
    )


# =============================================================================
# 11. VALIDATE METADATA COMPLETENESS
# =============================================================================

required_columns = [
    "category",
    "question",
    "corpus_relevance",
    "production_k7_support",
    "evaluation_recommendation",
    "evaluation_type",
]


if (
    track1_evaluation_df[
        required_columns
    ]
    .isna()
    .any()
    .any()
):

    raise RuntimeError(
        "Missing Track 1 metadata after merge."
    )


# =============================================================================
# 12. VALIDATE MODEL COVERAGE
# =============================================================================

responses_per_model = (
    track1_evaluation_df
    .groupby(
        "model_name"
    )[
        "question_id"
    ]
    .nunique()
)


if not (
    responses_per_model == 20
).all():

    raise RuntimeError(
        "Each model must have exactly "
        "20 Track 1 questions."
    )


responses_per_question = (
    track1_evaluation_df
    .groupby(
        "question_id"
    )[
        "model_name"
    ]
    .nunique()
)


if not (
    responses_per_question == 5
).all():

    raise RuntimeError(
        "Each question must have exactly "
        "5 model responses."
    )


# =============================================================================
# 13. VALIDATE EVALUATION TYPE SPLIT
# =============================================================================

evaluation_counts = (
    track1_evaluation_df[
        "evaluation_type"
    ]
    .value_counts()
)


if (
    evaluation_counts.get(
        "CORPUS_GROUNDED",
        0
    )
    != 75
):

    raise RuntimeError(
        "Expected 75 CORPUS_GROUNDED rows."
    )


if (
    evaluation_counts.get(
        "ENGINEERING_SYNTHESIS",
        0
    )
    != 25
):

    raise RuntimeError(
        "Expected 25 ENGINEERING_SYNTHESIS rows."
    )


# =============================================================================
# 14. ORDER DATASET
# =============================================================================

model_order = {
    model_name: idx
    for idx, model_name
    in enumerate(
        EXPECTED_MODELS
    )
}


track1_evaluation_df[
    "_model_order"
] = (
    track1_evaluation_df[
        "model_name"
    ]
    .map(
        model_order
    )
)


track1_evaluation_df[
    "_question_order"
] = (
    track1_evaluation_df[
        "question_id"
    ]
    .str.extract(
        r"(\d+)"
    )[0]
    .astype(int)
)


track1_evaluation_df = (
    track1_evaluation_df
    .sort_values(
        [
            "_question_order",
            "_model_order",
        ]
    )
    .drop(
        columns=[
            "_question_order",
            "_model_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 15. FINAL VALIDATION REPORT
# =============================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "TRACK 1 — UNIFIED EVALUATION DATASET"
)

print(
    "=" * 90
)


print(
    f"\nTotal rows               : "
    f"{len(track1_evaluation_df)}"
)

print(
    f"Unique questions         : "
    f"{track1_evaluation_df['question_id'].nunique()}"
)

print(
    f"Evaluation systems       : "
    f"{track1_evaluation_df['model_name'].nunique()}"
)


print(
    "\nResponses per Model"
)

print(
    "-" * 60
)

print(
    responses_per_model
)


print(
    "\nEvaluation Type"
)

print(
    "-" * 60
)

print(
    evaluation_counts
)


print(
    "\nCorpus Relevance"
)

print(
    "-" * 60
)

print(
    track1_evaluation_df[
        "corpus_relevance"
    ]
    .value_counts()
)


print(
    "\nRAG Retrieval Metadata"
)

print(
    "-" * 60
)

print(
    track1_evaluation_df[
        [
            "model_name",
            "retrieval_k",
            "retrieval_top1_score",
        ]
    ]
    .groupby(
        "model_name"
    )
    .agg(
        {
            "retrieval_k":
                "count",

            "retrieval_top1_score":
                "count",
        }
    )
)


print(
    "\n"
    + "=" * 90
)

print(
    "TRACK 1 EVALUATION DATASET "
    "VALIDATED SUCCESSFULLY"
)

print(
    "=" * 90
)


# =============================================================================
# 16. DISPLAY COMPACT VIEW
# =============================================================================

display(
    track1_evaluation_df[
        [
            "question_id",
            "model_name",
            "category",
            "corpus_relevance",
            "production_k7_support",
            "evaluation_recommendation",
            "evaluation_type",
            "response_status",
        ]
    ]
)


[OK] Question text aligned across all model payloads.

TRACK 1 — UNIFIED EVALUATION DATASET

Total rows               : 100
Unique questions         : 20
Evaluation systems       : 5

Responses per Model
------------------------------------------------------------
model_name
Essential AI + RAG    20
Essential AI Only     20
Gemma 4 + RAG         20
Gemma 4 Only          20
Otel 2.0 Only         20
Name: question_id, dtype: int64

Evaluation Type
------------------------------------------------------------
evaluation_type
CORPUS_GROUNDED          75
ENGINEERING_SYNTHESIS    25
Name: count, dtype: int64

Corpus Relevance
------------------------------------------------------------
corpus_relevance
STRONG            45
PARTIAL           25
PARTIAL-STRONG    20
WEAK-PARTIAL       5
WEAK               5
Name: count, dtype: int64

RAG Retrieval Metadata
------------------------------------------------------------
                    retrieval_k  retrieval_top1_score
model_name              

,question_id,model_name,category,corpus_relevance,production_k7_support,evaluation_recommendation,evaluation_type,response_status
0,Q01,Essential AI + RAG,5G Core,STRONG,YES,KEEP,CORPUS_GROUNDED,PASS
1,Q01,Essential AI Only,5G Core,STRONG,YES,KEEP,CORPUS_GROUNDED,PASS
2,Q01,Otel 2.0 Only,5G Core,STRONG,YES,KEEP,CORPUS_GROUNDED,PASS
3,Q01,Gemma 4 Only,5G Core,STRONG,YES,KEEP,CORPUS_GROUNDED,success
4,Q01,Gemma 4 + RAG,5G Core,STRONG,YES,KEEP,CORPUS_GROUNDED,PASS
...,...,...,...,...,...,...,...,...
95,Q20,Essential AI + RAG,End-to-End Network Design,PARTIAL,PARTIAL,REVIEW,ENGINEERING_SYNTHESIS,PASS
96,Q20,Essential AI Only,End-to-End Network Design,PARTIAL,PARTIAL,REVIEW,ENGINEERING_SYNTHESIS,PASS
97,Q20,Otel 2.0 Only,End-to-End Network Design,PARTIAL,PARTIAL,REVIEW,ENGINEERING_SYNTHESIS,PASS
98,Q20,Gemma 4 Only,End-to-End Network Design,PARTIAL,PARTIAL,REVIEW,ENGINEERING_SYNTHESIS,success


In [33]:
print(
    "\nRetrieval Metadata Availability"
)

print(
    "-" * 60
)

retrieval_metadata_summary = (
    track1_evaluation_df
    .groupby("model_name")
    .agg(
        responses=("question_id", "count"),
        retrieval_k_present=("retrieval_k", "count"),
        retrieval_score_present=("retrieval_top1_score", "count"),
    )
)

retrieval_metadata_summary[
    "retrieval_expected"
] = retrieval_metadata_summary.index.isin(
    [
        "Essential AI + RAG",
        "Gemma 4 + RAG",
    ]
)

display(
    retrieval_metadata_summary
)


Retrieval Metadata Availability
------------------------------------------------------------


,responses,retrieval_k_present,retrieval_score_present,retrieval_expected
model_name,,,,
Essential AI + RAG,20,20,20,True
Essential AI Only,20,0,0,False
Gemma 4 + RAG,20,20,20,True
Gemma 4 Only,20,0,0,False
Otel 2.0 Only,20,0,0,False


In [46]:
# =============================================================================
# MODULE 4.1 + 4.2 — TRACK 1 SCORING RUBRIC AND JUDGE PROMPT
# =============================================================================
#
# PURPOSE
# -------
# Define:
#
#   1. A consistent 1–10 scoring rubric for all Track 1 responses.
#   2. Evaluation-type-specific judge instructions for:
#
#        CORPUS_GROUNDED
#        ENGINEERING_SYNTHESIS
#
#   3. Critique-first judge output:
#        - technical errors
#        - missing points
#        - strengths
#        - summary
#        - scores LAST
#
# IMPORTANT
# ---------
# - No judge inference is executed in this cell.
# - No existing model responses are modified.
# - The same core scoring dimensions are used across all five systems.
# - Corpus relevance is contextual metadata, NOT a model score.
# - RAG models are not automatically rewarded merely for using retrieval.
# - Non-RAG models are not automatically penalized for lacking retrieval.
# - Main response quality and later RAG grounding analysis remain separate.
#
# =============================================================================


import json
from textwrap import dedent


# =============================================================================
# 1. CORE SCORING DIMENSIONS
# =============================================================================

TRACK1_SCORING_DIMENSIONS = {

    "technical_accuracy": {
        "label": "Technical Accuracy",
        "description": (
            "Correctness of telecom concepts, procedures, architecture, "
            "interfaces, calculations, terminology and engineering claims."
        ),
    },

    "completeness": {
        "label": "Completeness",
        "description": (
            "Extent to which the response addresses the major elements "
            "required by the question and expected points."
        ),
    },

    "relevance": {
        "label": "Relevance",
        "description": (
            "How directly the response addresses the actual question "
            "without unnecessary, off-topic or misleading material."
        ),
    },

    "engineering_reasoning": {
        "label": "Engineering Reasoning",
        "description": (
            "Quality of causal reasoning, diagnosis, design logic, "
            "assumptions, trade-offs and technical justification."
        ),
    },

    "practical_applicability": {
        "label": "Practical Applicability",
        "description": (
            "Whether the response would be useful in a realistic telecom "
            "engineering, deployment, troubleshooting or operational context."
        ),
    },

    "factual_reliability": {
        "label": "Factual Reliability",
        "description": (
            "Degree to which the response avoids hallucinations, invented "
            "facts, unsupported specificity and internally inconsistent claims."
        ),
    },

    "overall_score": {
        "label": "Overall Score",
        "description": (
            "Holistic assessment of the response after considering all "
            "dimensions above. This is not required to be a mathematical "
            "average of the individual scores."
        ),
    },
}


# =============================================================================
# 2. SCORE SCALE
# =============================================================================

TRACK1_SCORE_SCALE = dedent(
    """
    Use integer scores from 1 to 10.

    10 = Excellent
         Technically correct, complete for the question asked, directly
         relevant, well reasoned where reasoning is required, practically
         useful and highly reliable.

         A 10 is appropriate when there are:
         - no material technical errors;
         - no important omissions;
         - no meaningful reasoning weakness;
         - and no substantive limitation relative to the task.

         Do not avoid a 10 merely for the sake of score distribution.


    9  = Very strong
         Technically excellent and highly complete, with only minor
         limitations, slight imprecision, or small opportunities for
         improvement. No material technical error.


    8  = Strong
         Correct and useful overall, with no major technical errors, but with
         meaningful omissions, limited depth, weaker reasoning, or some
         technical imprecision.


    7  = Good
         Generally correct and relevant, but incomplete or containing
         noticeable technical or reasoning weaknesses.


    6  = Adequate
         Provides a useful core answer, but important omissions, weak
         reasoning, or some questionable statements are present.


    5  = Mixed
         Partially correct, but substantial omissions or technical weaknesses
         limit usefulness.


    4  = Weak
         Some correct material exists, but major errors, gaps or poor reasoning
         significantly reduce reliability.


    3  = Poor
         Limited correct content and substantial misunderstanding.


    2  = Very poor
         Mostly incorrect, misleading or largely non-responsive.


    1  = Failed / Non-responsive
         No meaningful answer, refusal without justification, irrelevant
         output, or essentially unusable technical content.
    """
).strip()


# =============================================================================
# 2A. SCORE CALIBRATION
# =============================================================================

TRACK1_SCORE_CALIBRATION = dedent(
    """
    SCORE CALIBRATION

    Apply the scoring scale consistently and conservatively.

    Do not assume that a response is excellent or weak before examining it.

    Do NOT:
    - start from an assumed score of 10;
    - start from an assumed score of 5;
    - manufacture weaknesses merely to avoid a high score;
    - reward fluency, verbosity or formatting as substitutes for correctness.

    A technically excellent response may legitimately receive a 10.

    A response with minor limitations should normally receive a 9 rather than
    being artificially forced lower.

    A response with meaningful omissions but no major technical error will
    normally fall in the 7–8 range depending on severity.

    Material telecom errors must have a clear effect on technical_accuracy,
    factual_reliability and, where appropriate, overall_score.


    ============================================================
    ENGINEERING REASONING CALIBRATION
    ============================================================

    Do not equate technical detail with engineering reasoning.


    For CORPUS_GROUNDED questions:

    Engineering reasoning measures how well the response explains:
    - functional relationships;
    - technical dependencies;
    - architectural logic;
    - cause-and-effect where relevant;
    - and relationships between telecom network functions.

    A correct factual list can score highly on technical accuracy and
    completeness without automatically receiving a perfect
    engineering_reasoning score.


    For ENGINEERING_SYNTHESIS questions:

    Engineering reasoning measures:
    - hypothesis formation;
    - fault-domain isolation;
    - diagnostic sequencing;
    - evidence correlation;
    - explicit assumptions where appropriate;
    - trade-offs;
    - causal reasoning;
    - and discipline in identifying root cause only after evidence supports it.


    ============================================================
    ROOT-CAUSE DISCIPLINE
    ============================================================

    For troubleshooting and fault-isolation questions:

    A strong answer should:
    - preserve plausible competing hypotheses;
    - collect evidence across relevant domains;
    - correlate observations;
    - narrow the fault domain progressively;
    - and identify root cause only when evidence supports it.

    Prematurely declaring a likely root cause without sufficient evidence is a
    reasoning weakness.

    However, mentioning a plausible working hypothesis is acceptable when it is
    clearly identified as a hypothesis to be validated rather than a confirmed
    root cause.


    ============================================================
    EVIDENCE DISCIPLINE
    ============================================================

    For this MAIN response-quality evaluation, assess:
    - technical correctness;
    - completeness;
    - relevance;
    - engineering reasoning;
    - practical applicability;
    - and factual reliability.

    Do not describe a response as:
    - "fully grounded";
    - "fully supported";
    - "proven by the corpus";
    - "free of unsupported claims";
    - or equivalent absolute evidence statements

    unless the evidence presented in the evaluation prompt is sufficient to
    establish that conclusion.

    Retrieved-document grounding will be evaluated separately for RAG systems.
    """
).strip()


# =============================================================================
# 2B. SCORING CONSISTENCY RULES
# =============================================================================

TRACK1_SCORING_CONSISTENCY_RULES = dedent(
    """
    SCORING CONSISTENCY RULES

    Complete the qualitative critique before assigning numerical scores.

    Your final scores must logically follow from:
    - significant_technical_errors;
    - missing_important_points;
    - strengths;
    - and judge_summary.


    ============================================================
    TECHNICAL ERRORS
    ============================================================

    Distinguish between:

    MATERIAL TECHNICAL ERROR
    A claim that meaningfully misrepresents telecom architecture,
    procedures, interfaces, protocols, responsibilities, calculations,
    causal relationships or engineering behavior.

    MINOR IMPRECISION
    A wording issue, simplification, lack of qualification, or limited
    specificity that does not materially change the technical meaning.


    Material errors must reduce:
    - technical_accuracy;
    - factual_reliability;
    - and normally overall_score.

    Multiple material errors should normally prevent a very high overall score.

    Minor imprecision must not be treated as equivalent to a major technical
    error.


    ============================================================
    OMISSIONS
    ============================================================

    Distinguish between:

    IMPORTANT OMISSION
    Missing content that is necessary to adequately answer a major requirement
    of the question.

    OPTIONAL DETAIL
    Additional detail that could improve the answer but is not necessary for
    the requested task.

    Important omissions must reduce completeness.

    Optional detail should not prevent an otherwise excellent response from
    receiving a high score.


    ============================================================
    SCORE / CRITIQUE CONSISTENCY
    ============================================================

    Do not produce internally inconsistent evaluations.

    Example of INVALID logic:

    significant_technical_errors:
    - "AMF incorrectly described as performing SMF session management."

    technical_accuracy: 10
    factual_reliability: 10
    overall_score: 10

    This is inconsistent because a material technical error was documented.


    Another INVALID pattern:

    significant_technical_errors: []
    missing_important_points: []
    judge_summary:
    "Technically excellent, complete and fully addresses the task."

    overall_score: 6

    This is also inconsistent unless the summary clearly explains another
    material weakness.


    Scores must be justified by the critique actually written.
    """
).strip()


# =============================================================================
# 3. EVALUATION-TYPE-SPECIFIC INSTRUCTIONS
# =============================================================================

CORPUS_GROUNDED_INSTRUCTIONS = dedent(
    """
    EVALUATION TYPE: CORPUS_GROUNDED

    This question has been classified as sufficiently supported by the
    telecom corpus.

    Judge the response primarily on:

    - technical correctness;
    - coverage of the expected technical points;
    - correct use of telecom terminology;
    - correct functional boundaries and responsibilities;
    - completeness and relevance;
    - technical relationships and architectural logic;
    - whether claims are factually dependable.


    For RAG responses:

    - Do NOT award extra points merely because retrieval was used.
    - Judge the generated answer itself.
    - If retrieved information appears to have caused an incorrect claim,
      score the generated response according to the incorrect claim.


    For non-RAG responses:

    - Do NOT penalize the response merely because it does not cite or use
      retrieved evidence.
    - A technically correct answer can score highly regardless of architecture.


    Expected points are guidance for coverage, not a rigid word-matching key.

    Equivalent technically correct explanations should receive full credit
    where appropriate.

    Do not require every optional detail if the core question is already
    answered completely and correctly.
    """
).strip()


ENGINEERING_SYNTHESIS_INSTRUCTIONS = dedent(
    """
    EVALUATION TYPE: ENGINEERING_SYNTHESIS

    This question requires applied engineering synthesis, design,
    troubleshooting, calculation, fault isolation or trade-off reasoning.

    The corpus may contain only partial or distributed supporting knowledge.

    Judge the response primarily on:

    - technical correctness;
    - quality of engineering reasoning;
    - logical sequencing;
    - explicit assumptions where appropriate;
    - evidence collection;
    - correlation of evidence across domains;
    - consideration of trade-offs;
    - practical feasibility;
    - completeness relative to the requested engineering task.


    IMPORTANT:

    - Do NOT penalize a RAG response merely because the corpus was classified
      as PARTIAL, WEAK-PARTIAL or WEAK.

    - Corpus relevance is contextual metadata, not a model-performance score.

    - A model may legitimately synthesize a strong engineering answer from
      distributed or incomplete evidence.

    - Equally, a fluent answer that invents technically unsupported details
      must be penalized for factual reliability.

    - Do not require one exact architecture, calculation method or
      troubleshooting sequence when multiple technically valid approaches
      exist.

    - For troubleshooting questions, distinguish between:
        * a hypothesis;
        * an observed symptom;
        * an isolated fault domain;
        * and a confirmed root cause.

    - Reward evidence-first engineering methodology.

    Expected points are anchor criteria, not a rigid answer template.
    """
).strip()


# =============================================================================
# 4. GENERAL JUDGE RULES
# =============================================================================

TRACK1_GENERAL_JUDGE_RULES = dedent(
    """
    GENERAL JUDGING RULES

    1. Evaluate the response that was actually produced.
       Do not improve, rewrite or complete it mentally.

    2. Technical fluency is not evidence of technical correctness.

    3. Penalize material telecom errors even when the answer sounds confident.

    4. Do not require exact wording from the expected points.
       Accept technically equivalent explanations.

    5. Distinguish omission from error:

       - Missing an important point primarily reduces completeness.
       - Stating something technically wrong reduces technical_accuracy and
         factual_reliability.

    6. Distinguish material error from minor imprecision.

    7. Do not infer reasoning that is not present in the answer.

    8. Do not award points based on:
       - model identity;
       - model size;
       - RAG/non-RAG architecture;
       - response length;
       - formatting;
       - or confidence.

    9. Concise answers may score highly if they fully answer the question.

    10. Long answers should not score highly if they contain unnecessary,
        speculative or incorrect material.

    11. Use response_status = "NON_RESPONSIVE" when the model:

        - does not meaningfully answer the requested task;
        - refuses without a valid task-specific reason;
        - says information is unavailable instead of performing an engineering
          synthesis task it was asked to perform;
        - produces essentially irrelevant content;
        - or provides no usable technical response.

    12. Use response_status = "SUBSTANTIVE" for any meaningful attempt,
        even if the technical score is low.

    13. The overall_score is holistic.

        It does NOT have to equal the arithmetic mean of the six component
        dimensions.

    14. Scores of 9 and 10 are legitimate when supported by the critique.

        Do not suppress high scores merely to create artificial separation
        between models.

    15. Equally, do not award high scores simply because a response is fluent,
        detailed or confidently written.

    16. The qualitative critique must be completed before numerical scoring.

    17. The final numerical scores must be consistent with the documented
        errors, omissions, strengths and judge summary.

    18. Return only the requested JSON structure.

        Do not include markdown fences or text outside the JSON object.
    """
).strip()


# =============================================================================
# 5. JUDGE OUTPUT SCHEMA — CRITIQUE FIRST, SCORES LAST
# =============================================================================
#
# IMPORTANT
# ---------
# Qualitative evaluation is generated before numerical scores.
#
# This helps reduce score-first anchoring and makes the final scores easier
# to audit for internal consistency.
#
# =============================================================================

TRACK1_JUDGE_OUTPUT_SCHEMA = {

    "question_id": "Q01",

    "model_name": "Example Model",

    "evaluation_type": "CORPUS_GROUNDED",

    "response_status": "SUBSTANTIVE",

    # -------------------------------------------------------------------------
    # QUALITATIVE EVALUATION FIRST
    # -------------------------------------------------------------------------

    "significant_technical_errors": [],

    "missing_important_points": [],

    "strengths": [],

    "judge_summary": "",

    # -------------------------------------------------------------------------
    # NUMERICAL SCORES LAST
    # -------------------------------------------------------------------------

    "scores": {

        "technical_accuracy": 1,

        "completeness": 1,

        "relevance": 1,

        "engineering_reasoning": 1,

        "practical_applicability": 1,

        "factual_reliability": 1,

        "overall_score": 1,
    },
}


# =============================================================================
# 6. BUILD SYSTEM PROMPT
# =============================================================================

TRACK1_JUDGE_SYSTEM_PROMPT = dedent(
    f"""
    You are an independent senior telecom engineering evaluator.

    Your task is to evaluate AI-generated responses to advanced telecom
    questions involving areas such as:

    - 5G Core;
    - 5G Standalone procedures;
    - NG-RAN;
    - Open RAN;
    - cloud-native telecom;
    - Kubernetes-based telecom infrastructure;
    - RAN capacity and performance;
    - troubleshooting;
    - network design;
    - engineering trade-offs.

    You must judge technical quality, not writing style or model identity.


    ============================================================
    SCORE SCALE
    ============================================================

    {TRACK1_SCORE_SCALE}


    ============================================================
    SCORE CALIBRATION
    ============================================================

    {TRACK1_SCORE_CALIBRATION}


    ============================================================
    SCORING CONSISTENCY RULES
    ============================================================

    {TRACK1_SCORING_CONSISTENCY_RULES}


    ============================================================
    GENERAL RULES
    ============================================================

    {TRACK1_GENERAL_JUDGE_RULES}


    ============================================================
    REQUIRED OUTPUT
    ============================================================

    Return exactly one valid JSON object matching this structure:

    {json.dumps(
        TRACK1_JUDGE_OUTPUT_SCHEMA,
        indent=2,
    )}

    Populate the JSON fields in the order shown.

    First evaluate:
    - response_status;
    - significant_technical_errors;
    - missing_important_points;
    - strengths;
    - judge_summary.

    Only AFTER completing that critique should you assign the numerical scores.

    All scores must be integers from 1 to 10.

    response_status must be exactly one of:

    "SUBSTANTIVE"
    "NON_RESPONSIVE"


    significant_technical_errors:

    - Include only genuine material telecom or engineering errors.
    - Do not place simple omissions or optional improvements here.
    - Be specific about incorrect functions, interfaces, protocols,
      architecture, calculations, causal claims or engineering conclusions.


    missing_important_points:

    - Include important missing elements required by the question.
    - Do not list every possible optional improvement.
    - Expected points are anchors, not a requirement for verbatim coverage.


    strengths:

    - Identify specific technically meaningful strengths.
    - Do not praise formatting or verbosity unless it materially improves
      engineering clarity.


    judge_summary:

    - Briefly explain the evaluation.
    - State the main reasons for the final score.
    - Ensure the summary is consistent with the listed errors and omissions.


    scores:

    - Assign scores only after completing the critique above.
    - Ensure numerical scores logically follow from the critique.
    - Do not include text outside the JSON object.
    - Do not include markdown fences.
    """
).strip()


# =============================================================================
# 7. BUILD QUESTION-SPECIFIC JUDGE PROMPT
# =============================================================================

def build_track1_judge_prompt(row):
    """
    Construct one evaluation prompt from one row of
    track1_evaluation_df.
    """

    evaluation_type = row[
        "evaluation_type"
    ]


    # -------------------------------------------------------------------------
    # SELECT EVALUATION-SPECIFIC INSTRUCTIONS
    # -------------------------------------------------------------------------

    if evaluation_type == "CORPUS_GROUNDED":

        evaluation_instructions = (
            CORPUS_GROUNDED_INSTRUCTIONS
        )


    elif evaluation_type == "ENGINEERING_SYNTHESIS":

        evaluation_instructions = (
            ENGINEERING_SYNTHESIS_INSTRUCTIONS
        )


    else:

        raise ValueError(
            f"Unknown evaluation_type: "
            f"{evaluation_type}"
        )


    # -------------------------------------------------------------------------
    # EXPECTED POINTS
    # -------------------------------------------------------------------------

    expected_points = row.get(
        "expected_points"
    )


    if isinstance(
        expected_points,
        list,
    ):

        expected_points_text = "\n".join(
            f"- {point}"
            for point in expected_points
        )


    elif expected_points is None:

        expected_points_text = (
            "No explicit expected points provided."
        )


    else:

        expected_points_text = str(
            expected_points
        )


    # -------------------------------------------------------------------------
    # RESPONSE
    # -------------------------------------------------------------------------

    response_text = row.get(
        "response"
    )


    if response_text is None:

        response_text = ""


    response_text = str(
        response_text
    ).strip()


    if not response_text:

        response_text = (
            "[EMPTY RESPONSE]"
        )


    # -------------------------------------------------------------------------
    # BUILD PROMPT
    # -------------------------------------------------------------------------

    prompt = dedent(
        f"""
        Evaluate the following Track 1 telecom response.

        ============================================================
        QUESTION METADATA
        ============================================================

        Question ID:
        {row['question_id']}

        Category:
        {row['category']}

        Evaluation Type:
        {row['evaluation_type']}

        Corpus Relevance:
        {row['corpus_relevance']}

        Production K=7 Support:
        {row['production_k7_support']}

        Evaluation Recommendation:
        {row['evaluation_recommendation']}


        ============================================================
        QUESTION
        ============================================================

        {row['question']}


        ============================================================
        EXPECTED TECHNICAL POINTS
        ============================================================

        {expected_points_text}


        ============================================================
        RESPONSE TO EVALUATE
        ============================================================

        Model:
        {row['model_name']}

        Response:

        {response_text}


        ============================================================
        EVALUATION-SPECIFIC INSTRUCTIONS
        ============================================================

        {evaluation_instructions}


        ============================================================
        SCORING DIMENSIONS
        ============================================================

        Technical Accuracy:
        {TRACK1_SCORING_DIMENSIONS['technical_accuracy']['description']}


        Completeness:
        {TRACK1_SCORING_DIMENSIONS['completeness']['description']}


        Relevance:
        {TRACK1_SCORING_DIMENSIONS['relevance']['description']}


        Engineering Reasoning:
        {TRACK1_SCORING_DIMENSIONS['engineering_reasoning']['description']}


        Practical Applicability:
        {TRACK1_SCORING_DIMENSIONS['practical_applicability']['description']}


        Factual Reliability:
        {TRACK1_SCORING_DIMENSIONS['factual_reliability']['description']}


        Overall Score:
        {TRACK1_SCORING_DIMENSIONS['overall_score']['description']}


        ============================================================
        EVALUATION ORDER
        ============================================================

        Perform the evaluation in this order:

        1. Determine response_status.

        2. Identify genuine significant technical errors.

        3. Identify important missing points.

        4. Identify technically meaningful strengths.

        5. Write a concise judge_summary.

        6. Assign the seven numerical scores.

        The numerical scores must be consistent with the critique above.


        ============================================================
        FINAL OUTPUT
        ============================================================

        Return only the required JSON object.

        Do not include markdown fences.

        Do not include explanatory text outside the JSON.
        """
    ).strip()


    return prompt


# =============================================================================
# 8. VALIDATE PROMPT GENERATION
# =============================================================================

test_row = (
    track1_evaluation_df
    .iloc[0]
)


test_prompt = build_track1_judge_prompt(
    test_row
)


# =============================================================================
# 9. VALIDATE ACTIVE SYSTEM PROMPT
# =============================================================================

required_prompt_markers = {

    "SCORE CALIBRATION":
        "SCORE CALIBRATION"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "ENGINEERING REASONING CALIBRATION":
        "ENGINEERING REASONING CALIBRATION"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "ROOT-CAUSE DISCIPLINE":
        "ROOT-CAUSE DISCIPLINE"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "EVIDENCE DISCIPLINE":
        "EVIDENCE DISCIPLINE"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "SCORING CONSISTENCY RULES":
        "SCORING CONSISTENCY RULES"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "CRITIQUE BEFORE SCORES":
        "Only AFTER completing that critique"
        in TRACK1_JUDGE_SYSTEM_PROMPT,
}


# =============================================================================
# 10. DISPLAY VALIDATION
# =============================================================================

print("=" * 90)
print("TRACK 1 SCORING RUBRIC + JUDGE PROMPT INITIALIZED")
print("=" * 90)

print(
    f"\nScoring dimensions : "
    f"{len(TRACK1_SCORING_DIMENSIONS)}"
)

print(
    f"System prompt chars: "
    f"{len(TRACK1_JUDGE_SYSTEM_PROMPT):,}"
)

print(
    f"Test prompt chars  : "
    f"{len(test_prompt):,}"
)

print(
    f"Test question      : "
    f"{test_row['question_id']}"
)

print(
    f"Evaluation type    : "
    f"{test_row['evaluation_type']}"
)


print(
    "\n"
    + "=" * 90
)

print(
    "ACTIVE JUDGE CONTROL VALIDATION"
)

print(
    "=" * 90
)


for control_name, present in required_prompt_markers.items():

    status = (
        "✅ PRESENT"
        if present
        else "❌ MISSING"
    )

    print(
        f"{control_name:<40} : "
        f"{status}"
    )


if not all(
    required_prompt_markers.values()
):

    raise RuntimeError(
        "One or more required judge controls "
        "are missing from the active system prompt."
    )


print(
    "\n"
    + "=" * 90
)

print(
    "JUDGE PROMPT PREVIEW"
)

print(
    "=" * 90
)

print(
    test_prompt[:4000]
)


print(
    "\n"
    + "=" * 90
)

print(
    "MODULE 4.1 + 4.2 COMPLETE"
)

print(
    "=" * 90
)

TRACK 1 SCORING RUBRIC + JUDGE PROMPT INITIALIZED

Scoring dimensions : 7
System prompt chars: 12,792
Test prompt chars  : 6,041
Test question      : Q01
Evaluation type    : CORPUS_GROUNDED

ACTIVE JUDGE CONTROL VALIDATION
SCORE CALIBRATION                        : ✅ PRESENT
ENGINEERING REASONING CALIBRATION        : ✅ PRESENT
ROOT-CAUSE DISCIPLINE                    : ✅ PRESENT
EVIDENCE DISCIPLINE                      : ✅ PRESENT
SCORING CONSISTENCY RULES                : ✅ PRESENT
CRITIQUE BEFORE SCORES                   : ✅ PRESENT

JUDGE PROMPT PREVIEW
Evaluate the following Track 1 telecom response.

        QUESTION METADATA

        Question ID:
        Q01

        Category:
        5G Core

        Evaluation Type:
        CORPUS_GROUNDED

        Corpus Relevance:
        STRONG

        Production K=7 Support:
        YES

        Evaluation Recommendation:
        KEEP


        QUESTION

        What are the primary responsibilities of the AMF in a 5G Standalone network?


# **Load Judge LLM**

In [1]:
# =============================================================================
# vLLM LOADING
# =============================================================================

# 1. Install packages
!pip install vllm openai nest_asyncio requests hf_transfer huggingface_hub --upgrade

# TorchAudio is only required if your environment needs the matching
# PyTorch CUDA 13.0 nightly build.
!pip install --upgrade torchaudio \
    --index-url https://download.pytorch.org/whl/nightly/cu130 \
    --force-reinstall


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.5 MB/s eta 0:00:00
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0.dev20260828+cu130
    Uninstalling torchaudio-2.11.0.dev20260828+cu130:
      Successfully uninstalled torchaudio-2.11.0.dev20260828+cu130
Looking in indexes: https://download.pytorch.org/whl/nightly/cu130
  Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torchaudio-2.11.0.dev20260828%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (7.4 kB)
Using cached https://download-r2.pytorch.org/whl/nightly/cu130/torchaudio-2.11.0.dev20260828%2Bcu130-cp313-cp313-manylinux_2_28_x86_64.whl (1.7 MB)
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0
    Uninstalling torchaudio-2.11.0:
      Successfully uninstalled torchaudio-2.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following 

In [2]:
# =============================================================================
# MODULE 4.3A — JUDGE LLM ENVIRONMENT CHECK
# =============================================================================

import subprocess
import torch
import vllm

print("=" * 90)
print("JUDGE LLM — ENVIRONMENT CHECK")
print("=" * 90)

print(f"PyTorch version : {torch.__version__}")
print(f"vLLM version    : {vllm.__version__}")

print("\nGPU")
print("-" * 60)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

print(f"GPU Name        : {torch.cuda.get_device_name(0)}")

props = torch.cuda.get_device_properties(0)

gpu_memory_gb = props.total_memory / (1024 ** 3)

print(f"GPU Memory      : {gpu_memory_gb:.2f} GB")

print("\nNVIDIA-SMI")
print("-" * 60)

subprocess.run(
    ["nvidia-smi"],
    check=False,
)

print("=" * 90)

JUDGE LLM — ENVIRONMENT CHECK
PyTorch version : 2.13.0+cu130
vLLM version    : 0.28.0

GPU
------------------------------------------------------------
GPU Name        : NVIDIA L4
GPU Memory      : 22.03 GB

NVIDIA-SMI
------------------------------------------------------------


In [7]:
from huggingface_hub import notebook_login, get_token
import os
import signal
import subprocess
import time
import requests
import nest_asyncio
from openai import OpenAI

# Apply nest_asyncio to prevent event loop conflicts in Jupyter
nest_asyncio.apply()

# Log in interactively (Run this once)
notebook_login()


In [11]:
hf_token = get_token()
if not hf_token:
    raise ValueError("⚠️ No Hugging Face token found! Run Cell 1 successfully first.")

# Setup environment
env = os.environ.copy()
env["HF_TOKEN"] = hf_token
env["HF_XET_HIGH_PERFORMANCE"] = "1"
env["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "1800"
env["VLLM_WORKER_TIMEOUT_S"] = "1800"
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# =============================================================================
# THE REAL CORRECT ENV VARIABLE TO DISABLE THE EXPERIMENTAL V1 ENGINE
# =============================================================================
env["VLLM_USE_V1"] = "0"

VLLM_MODEL = "cyankiwi/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit"
VLLM_PORT = "8000"
VLLM_BASE_URL = f"http://localhost:{VLLM_PORT}"

# Clean up stale servers running on this port
try:
    existing_response = requests.get(f"{VLLM_BASE_URL}/v1/models", timeout=3)
    if existing_response.status_code == 200:
        print(f"⚠️ Existing service detected on port {VLLM_PORT}. Cleaning up...")
        subprocess.run(["pkill", "-f", "vllm serve"], check=False)
        time.sleep(5)
except requests.exceptions.RequestException:
    pass

# vLLM Server Command (Cleaned from invalid arguments)
vllm_command = [
    "vllm", "serve", VLLM_MODEL,
    "--port", VLLM_PORT,
    "--tensor-parallel-size", "1",

    # 22GB L4 VRAM Optimized constraints
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.85",
    "--max-num-seqs", "4",

    "--trust-remote-code",
    "--quantization", "compressed-tensors"
]

VLLM_LOG_FILE = "vllm_qwen3_judge_server.log"
log_file = open(VLLM_LOG_FILE, "w", encoding="utf-8")

# Launch completely in the background
vllm_process = subprocess.Popen(
    vllm_command,
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"⏳ vLLM background process started (PID: {vllm_process.pid})")
print(f"Tracking log: {VLLM_LOG_FILE}")

# Monitor readiness loop
max_attempts = 120
attempt = 0
server_ready = False

while attempt < max_attempts:
    if vllm_process.poll() is not None:
        log_file.flush()
        log_file.close()
        print("\n❌ vLLM process terminated unexpectedly!")
        with open(VLLM_LOG_FILE, "r", encoding="utf-8", errors="replace") as f:
            lines = f.readlines()
            print("".join(lines[-40:]))  # Will now print legitimate V0 logging details
        raise RuntimeError("vLLM Judge Server crashed during initialization.")

    try:
        response = requests.get(f"{VLLM_BASE_URL}/v1/models", timeout=2)
        if response.status_code == 200:
            print("\n✅ vLLM Server is fully up and running!")
            server_ready = True
            break
    except requests.exceptions.RequestException:
        pass

    attempt += 1
    time.sleep(5)

if not server_ready:
    log_file.close()
    raise TimeoutError("Server did not respond within the allocated time.")


⏳ vLLM background process started (PID: 29589)
Tracking log: vllm_qwen3_judge_server.log

✅ vLLM Server is fully up and running!


In [12]:
# Instantiate the OpenAI client targeting your local vLLM instance
client = OpenAI(
    base_url=f"{VLLM_BASE_URL}/v1",
    api_key="token-not-needed-for-local-vllm"
)

# Run a quick test inference to make sure the pipeline handles requests
try:
    response = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[{"role": "user", "content": "Hello! Confirm you are working as a judge LLM."}],
        max_tokens=50
    )
    print("🤖 Response from Judge Model:")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"❌ Failed to communicate with client: {e}")


🤖 Response from Judge Model:
Hello! I'm Qwen, a large-scale language model developed by Alibaba Cloud. I'm not a judge LLM, but I can assist you with various tasks such as answering questions, writing stories, creating documents, coding, and more. How


# **Dry Run and Prompt Tuning**

In [39]:
# =============================================================================
# MODULE 4.3B — QWEN3 JUDGE CLIENT + SANITY TEST
# =============================================================================

from openai import OpenAI
import json


# =============================================================================
# 1. OPENAI-COMPATIBLE CLIENT
# =============================================================================

client = OpenAI(
    base_url=f"{VLLM_BASE_URL}/v1",
    api_key="not-needed",
)


# =============================================================================
# 2. VERIFY EXPOSED MODEL
# =============================================================================

models_response = client.models.list()

available_models = [
    model.id
    for model in models_response.data
]

print("=" * 90)
print("QWEN3 JUDGE — AVAILABLE MODELS")
print("=" * 90)

for model_id in available_models:
    print(f"• {model_id}")

print("=" * 90)


# =============================================================================
# 3. JSON SANITY TEST
# =============================================================================

test_response = client.chat.completions.create(

    model=VLLM_MODEL,

    messages=[
        {
            "role": "system",
            "content": (
                "You are an independent senior telecommunications "
                "engineering evaluator. "
                "Follow output-format instructions exactly. "
                "Return only valid JSON."
            ),
        },
        {
            "role": "user",
            "content": (
                'Return exactly one JSON object with two fields: '
                '"status" and "score". '
                'Set "status" to "OK" and "score" to 10.'
            ),
        },
    ],

    temperature=0.0,
    max_tokens=100,
)


test_text = (
    test_response
    .choices[0]
    .message
    .content
    .strip()
)


# =============================================================================
# 4. DISPLAY RAW RESPONSE
# =============================================================================

print("\n" + "=" * 90)
print("QWEN3 JUDGE — RAW SANITY RESPONSE")
print("=" * 90)

print(test_text)

print("=" * 90)


# =============================================================================
# 5. VALIDATE JSON
# =============================================================================

try:

    parsed_test = json.loads(
        test_text
    )

except json.JSONDecodeError as exc:

    raise RuntimeError(
        "Judge returned invalid JSON.\n\n"
        f"Raw output:\n{test_text}"
    ) from exc


expected_keys = {
    "status",
    "score",
}


if set(parsed_test.keys()) != expected_keys:

    raise RuntimeError(
        "Unexpected JSON keys.\n"
        f"Expected: {expected_keys}\n"
        f"Actual  : {set(parsed_test.keys())}"
    )


if parsed_test["status"] != "OK":

    raise RuntimeError(
        f"Unexpected status: "
        f"{parsed_test['status']}"
    )


if parsed_test["score"] != 10:

    raise RuntimeError(
        f"Unexpected score: "
        f"{parsed_test['score']}"
    )


print("\n✅ Judge JSON sanity test passed.")

print(
    "\n"
    + "=" * 90
)

print(
    "QWEN3 JUDGE CLIENT VALIDATED SUCCESSFULLY"
)

print(
    "=" * 90
)

QWEN3 JUDGE — AVAILABLE MODELS
• cyankiwi/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit

QWEN3 JUDGE — RAW SANITY RESPONSE
{"status": "OK", "score": 10}

✅ Judge JSON sanity test passed.

QWEN3 JUDGE CLIENT VALIDATED SUCCESSFULLY


In [40]:
# =============================================================================
# MODULE 4.3C — JUDGE DRY RUN
# Q01 + Q19 ACROSS ALL FIVE SYSTEMS
# =============================================================================

import json
import pandas as pd


# =============================================================================
# 1. DRY-RUN QUESTION SELECTION
# =============================================================================

DRY_RUN_QUESTION_IDS = [
    "Q01",
    "Q19",
]


dry_run_df = (
    track1_evaluation_df[
        track1_evaluation_df[
            "question_id"
        ].isin(
            DRY_RUN_QUESTION_IDS
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# Expected:
# 2 questions × 5 models = 10 responses

if len(dry_run_df) != 10:

    raise RuntimeError(
        f"Expected 10 dry-run rows, "
        f"found {len(dry_run_df)}."
    )


print("=" * 90)
print("TRACK 1 JUDGE DRY RUN")
print("=" * 90)

print(
    f"\nQuestions : "
    f"{DRY_RUN_QUESTION_IDS}"
)

print(
    f"Responses : "
    f"{len(dry_run_df)}"
)

print(
    f"Models    : "
    f"{dry_run_df['model_name'].nunique()}"
)

print("=" * 90)


# =============================================================================
# 2. JUDGE RESPONSE PARSER
# =============================================================================

def validate_judge_result(
    result,
    expected_question_id,
    expected_model_name,
    expected_evaluation_type,
):

    required_top_level = {
        "question_id",
        "model_name",
        "evaluation_type",
        "response_status",
        "scores",
        "significant_technical_errors",
        "missing_important_points",
        "strengths",
        "judge_summary",
    }


    missing_fields = (
        required_top_level
        - set(
            result.keys()
        )
    )


    if missing_fields:

        raise ValueError(
            f"Missing judge fields: "
            f"{sorted(missing_fields)}"
        )


    # -------------------------------------------------------------------------
    # IDENTITY CHECKS
    # -------------------------------------------------------------------------

    if result["question_id"] != expected_question_id:

        raise ValueError(
            f"Question ID mismatch. "
            f"Expected {expected_question_id}, "
            f"got {result['question_id']}."
        )


    if result["model_name"] != expected_model_name:

        raise ValueError(
            f"Model name mismatch. "
            f"Expected {expected_model_name}, "
            f"got {result['model_name']}."
        )


    if (
        result["evaluation_type"]
        != expected_evaluation_type
    ):

        raise ValueError(
            "Evaluation type mismatch."
        )


    # -------------------------------------------------------------------------
    # RESPONSE STATUS
    # -------------------------------------------------------------------------

    if result[
        "response_status"
    ] not in {
        "SUBSTANTIVE",
        "NON_RESPONSIVE",
    }:

        raise ValueError(
            "Invalid response_status."
        )


    # -------------------------------------------------------------------------
    # SCORE VALIDATION
    # -------------------------------------------------------------------------

    expected_score_fields = {
        "technical_accuracy",
        "completeness",
        "relevance",
        "engineering_reasoning",
        "practical_applicability",
        "factual_reliability",
        "overall_score",
    }


    score_fields = set(
        result["scores"].keys()
    )


    if score_fields != expected_score_fields:

        raise ValueError(
            "Unexpected score fields.\n"
            f"Expected: {expected_score_fields}\n"
            f"Actual  : {score_fields}"
        )


    for score_name, score in (
        result["scores"].items()
    ):

        if not isinstance(
            score,
            int,
        ):

            raise ValueError(
                f"{score_name} must be integer."
            )


        if not 1 <= score <= 10:

            raise ValueError(
                f"{score_name} outside 1–10: "
                f"{score}"
            )


    # -------------------------------------------------------------------------
    # LIST FIELDS
    # -------------------------------------------------------------------------

    for list_field in [
        "significant_technical_errors",
        "missing_important_points",
        "strengths",
    ]:

        if not isinstance(
            result[list_field],
            list,
        ):

            raise ValueError(
                f"{list_field} must be a list."
            )


    if not isinstance(
        result["judge_summary"],
        str,
    ):

        raise ValueError(
            "judge_summary must be a string."
        )


    return True


# =============================================================================
# 3. RUN JUDGE DRY RUN
# =============================================================================

judge_dry_run_results = []


for idx, row in dry_run_df.iterrows():

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"Judging "
        f"{row['question_id']} "
        f"| {row['model_name']}"
    )

    print(
        f"Evaluation Type: "
        f"{row['evaluation_type']}"
    )

    print(
        "=" * 90
    )


    # -------------------------------------------------------------------------
    # BUILD USER PROMPT
    # -------------------------------------------------------------------------

    judge_user_prompt = (
        build_track1_judge_prompt(
            row
        )
    )


    # -------------------------------------------------------------------------
    # CALL QWEN3 JUDGE
    # -------------------------------------------------------------------------

    response = (
        client
        .chat
        .completions
        .create(
            model=VLLM_MODEL,

            messages=[
                {
                    "role": "system",
                    "content":
                        TRACK1_JUDGE_SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content":
                        judge_user_prompt,
                },
            ],

            temperature=0.0,

            max_tokens=1200,
        )
    )


    raw_judge_text = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )


    # -------------------------------------------------------------------------
    # PARSE JSON
    # -------------------------------------------------------------------------

    try:

        judge_result = (
            json.loads(
                raw_judge_text
            )
        )

    except json.JSONDecodeError as exc:

        print(
            "\n❌ Invalid JSON returned."
        )

        print(
            "\nRAW RESPONSE:\n"
        )

        print(
            raw_judge_text
        )

        raise RuntimeError(
            f"Judge returned invalid JSON for "
            f"{row['question_id']} | "
            f"{row['model_name']}"
        ) from exc


    # -------------------------------------------------------------------------
    # VALIDATE RESULT
    # -------------------------------------------------------------------------

    validate_judge_result(

        result=judge_result,

        expected_question_id=
            row["question_id"],

        expected_model_name=
            row["model_name"],

        expected_evaluation_type=
            row["evaluation_type"],
    )


    # -------------------------------------------------------------------------
    # STORE RESULT
    # -------------------------------------------------------------------------

    scores = (
        judge_result[
            "scores"
        ]
    )


    judge_dry_run_results.append(
        {
            "question_id":
                row["question_id"],

            "model_name":
                row["model_name"],

            "evaluation_type":
                row["evaluation_type"],

            "corpus_relevance":
                row["corpus_relevance"],

            "response_status":
                judge_result[
                    "response_status"
                ],

            "technical_accuracy":
                scores[
                    "technical_accuracy"
                ],

            "completeness":
                scores[
                    "completeness"
                ],

            "relevance":
                scores[
                    "relevance"
                ],

            "engineering_reasoning":
                scores[
                    "engineering_reasoning"
                ],

            "practical_applicability":
                scores[
                    "practical_applicability"
                ],

            "factual_reliability":
                scores[
                    "factual_reliability"
                ],

            "overall_score":
                scores[
                    "overall_score"
                ],

            "significant_technical_errors":
                judge_result[
                    "significant_technical_errors"
                ],

            "missing_important_points":
                judge_result[
                    "missing_important_points"
                ],

            "strengths":
                judge_result[
                    "strengths"
                ],

            "judge_summary":
                judge_result[
                    "judge_summary"
                ],

            "raw_judge_json":
                judge_result,
        }
    )


    print(
        f"\n✅ Valid judge result"
    )

    print(
        f"Overall score : "
        f"{scores['overall_score']}/10"
    )

    print(
        f"Status        : "
        f"{judge_result['response_status']}"
    )


# =============================================================================
# 4. BUILD DRY-RUN RESULTS DATAFRAME
# =============================================================================

track1_judge_dry_run_df = (
    pd.DataFrame(
        judge_dry_run_results
    )
)


# =============================================================================
# 5. DISPLAY COMPACT RESULTS
# =============================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "TRACK 1 JUDGE DRY RUN — RESULTS"
)

print(
    "=" * 90
)


display(
    track1_judge_dry_run_df[
        [
            "question_id",
            "model_name",
            "evaluation_type",
            "technical_accuracy",
            "completeness",
            "engineering_reasoning",
            "factual_reliability",
            "overall_score",
            "response_status",
        ]
    ]
)


# =============================================================================
# 6. QUESTION-LEVEL COMPARISON
# =============================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "DRY RUN — OVERALL SCORE BY QUESTION / MODEL"
)

print(
    "=" * 90
)


dry_run_score_pivot = (
    track1_judge_dry_run_df
    .pivot(
        index="question_id",
        columns="model_name",
        values="overall_score",
    )
)


display(
    dry_run_score_pivot
)


# =============================================================================
# 7. VALIDATION SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "DRY RUN VALIDATION SUMMARY"
)

print(
    "=" * 90
)

print(
    f"Judge outputs          : "
    f"{len(track1_judge_dry_run_df)}"
)

print(
    f"Questions evaluated    : "
    f"{track1_judge_dry_run_df['question_id'].nunique()}"
)

print(
    f"Models evaluated       : "
    f"{track1_judge_dry_run_df['model_name'].nunique()}"
)

print(
    f"Invalid JSON outputs   : 0"
)

print(
    f"Schema validation      : PASS"
)

print(
    "=" * 90
)

TRACK 1 JUDGE DRY RUN

Questions : ['Q01', 'Q19']
Responses : 10
Models    : 5

Judging Q01 | Essential AI + RAG
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 6/10
Status        : SUBSTANTIVE

Judging Q01 | Essential AI Only
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 5/10
Status        : SUBSTANTIVE

Judging Q01 | Otel 2.0 Only
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 10/10
Status        : SUBSTANTIVE

Judging Q01 | Gemma 4 Only
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 10/10
Status        : SUBSTANTIVE

Judging Q01 | Gemma 4 + RAG
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 10/10
Status        : SUBSTANTIVE

Judging Q19 | Essential AI + RAG
Evaluation Type: ENGINEERING_SYNTHESIS

✅ Valid judge result
Overall score : 1/10
Status        : NON_RESPONSIVE

Judging Q19 | Essential AI Only
Evaluation Type: ENGINEERING_SYNTHESIS

✅ Valid judge result
Over

,question_id,model_name,evaluation_type,technical_accuracy,completeness,engineering_reasoning,factual_reliability,overall_score,response_status
0,Q01,Essential AI + RAG,CORPUS_GROUNDED,5,6,5,5,6,SUBSTANTIVE
1,Q01,Essential AI Only,CORPUS_GROUNDED,4,5,4,4,5,SUBSTANTIVE
2,Q01,Otel 2.0 Only,CORPUS_GROUNDED,10,10,10,10,10,SUBSTANTIVE
3,Q01,Gemma 4 Only,CORPUS_GROUNDED,10,10,10,10,10,SUBSTANTIVE
4,Q01,Gemma 4 + RAG,CORPUS_GROUNDED,10,10,9,10,10,SUBSTANTIVE
5,Q19,Essential AI + RAG,ENGINEERING_SYNTHESIS,1,1,1,1,1,NON_RESPONSIVE
6,Q19,Essential AI Only,ENGINEERING_SYNTHESIS,9,9,9,9,9,SUBSTANTIVE
7,Q19,Otel 2.0 Only,ENGINEERING_SYNTHESIS,10,10,10,10,10,SUBSTANTIVE
8,Q19,Gemma 4 Only,ENGINEERING_SYNTHESIS,10,10,10,10,10,SUBSTANTIVE
9,Q19,Gemma 4 + RAG,ENGINEERING_SYNTHESIS,10,10,10,10,10,SUBSTANTIVE



DRY RUN — OVERALL SCORE BY QUESTION / MODEL


model_name,Essential AI + RAG,Essential AI Only,Gemma 4 + RAG,Gemma 4 Only,Otel 2.0 Only
question_id,,,,,
Q01,6,5,10,10,10
Q19,1,9,10,10,10



DRY RUN VALIDATION SUMMARY
Judge outputs          : 10
Questions evaluated    : 2
Models evaluated       : 5
Invalid JSON outputs   : 0
Schema validation      : PASS


In [37]:
# =============================================================================
# MODULE 4.3D — DRY RUN DETAILED JUDGE REVIEW
# =============================================================================

for question_id in [
    "Q01",
    "Q19",
]:

    print(
        "\n"
        + "=" * 100
    )

    print(
        f"DETAILED JUDGE REVIEW — {question_id}"
    )

    print(
        "=" * 100
    )


    question_results = (
        track1_judge_dry_run_df[
            track1_judge_dry_run_df[
                "question_id"
            ] == question_id
        ]
    )


    for _, row in question_results.iterrows():

        print(
            "\n"
            + "-" * 100
        )

        print(
            f"{row['model_name']}"
        )

        print(
            "-" * 100
        )

        print(
            f"Overall Score       : "
            f"{row['overall_score']}/10"
        )

        print(
            f"Technical Accuracy  : "
            f"{row['technical_accuracy']}/10"
        )

        print(
            f"Completeness        : "
            f"{row['completeness']}/10"
        )

        print(
            f"Engineering Reasoning: "
            f"{row['engineering_reasoning']}/10"
        )

        print(
            f"Factual Reliability : "
            f"{row['factual_reliability']}/10"
        )

        print(
            f"Response Status     : "
            f"{row['response_status']}"
        )


        print(
            "\nSIGNIFICANT TECHNICAL ERRORS"
        )

        errors = row[
            "significant_technical_errors"
        ]

        if errors:

            for item in errors:

                print(
                    f"  - {item}"
                )

        else:

            print(
                "  None identified"
            )


        print(
            "\nMISSING IMPORTANT POINTS"
        )

        missing = row[
            "missing_important_points"
        ]

        if missing:

            for item in missing:

                print(
                    f"  - {item}"
                )

        else:

            print(
                "  None identified"
            )


        print(
            "\nSTRENGTHS"
        )

        strengths = row[
            "strengths"
        ]

        if strengths:

            for item in strengths:

                print(
                    f"  - {item}"
                )

        else:

            print(
                "  None identified"
            )


        print(
            "\nJUDGE SUMMARY"
        )

        print(
            row[
                "judge_summary"
            ]
        )


print(
    "\n"
    + "=" * 100
)

print(
    "DETAILED DRY RUN REVIEW COMPLETE"
)

print(
    "=" * 100
)


DETAILED JUDGE REVIEW — Q01

----------------------------------------------------------------------------------------------------
Essential AI + RAG
----------------------------------------------------------------------------------------------------
Overall Score       : 6/10
Technical Accuracy  : 5/10
Completeness        : 6/10
Engineering Reasoning: 5/10
Factual Reliability : 5/10
Response Status     : SUBSTANTIVE

SIGNIFICANT TECHNICAL ERRORS
  - The AMF does not handle UE IP address allocation for PDU sessions — this is the responsibility of the SMF (Session Management Function).
  - The AMF does not control traffic steering at the UPF — this is managed by the SMF and UPF based on PDU session policies.
  - The AMF does not enforce QoS at the user plane level — QoS enforcement is handled by the UPF and SMF.
  - The AMF does not directly manage policy rules or enforce them at the user plane — policy control is handled by the PCF and enforced by the SMF and UPF.
  - The AMF does not 

In [44]:
# =============================================================================
# VERIFY ACTIVE JUDGE CALIBRATION PROMPT
# =============================================================================

print("=" * 90)
print("VERIFY ACTIVE TRACK 1 JUDGE SYSTEM PROMPT")
print("=" * 90)

checks = {
    "STRICT SCORE CALIBRATION":
        "STRICT SCORE CALIBRATION"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "CEILING CONTROL":
        "CEILING CONTROL"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "ENGINEERING REASONING CALIBRATION":
        "ENGINEERING REASONING CALIBRATION"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "EVIDENCE DISCIPLINE":
        "EVIDENCE DISCIPLINE"
        in TRACK1_JUDGE_SYSTEM_PROMPT,

    "ROOT-CAUSE DISCIPLINE":
        "ROOT-CAUSE DISCIPLINE"
        in TRACK1_JUDGE_SYSTEM_PROMPT,
}

for name, present in checks.items():

    status = "✅ PRESENT" if present else "❌ MISSING"

    print(
        f"{name:<40} : {status}"
    )


print(
    f"\nSystem prompt length : "
    f"{len(TRACK1_JUDGE_SYSTEM_PROMPT):,} characters"
)


print("\n" + "=" * 90)
print("CALIBRATION SECTION PREVIEW")
print("=" * 90)


marker = (
    TRACK1_JUDGE_SYSTEM_PROMPT.find(
        "STRICT SCORE CALIBRATION"
    )
)


if marker >= 0:

    print(
        TRACK1_JUDGE_SYSTEM_PROMPT[
            marker:
            marker + 3500
        ]
    )

else:

    print(
        "❌ Calibration section not found "
        "inside active system prompt."
    )


print("\n" + "=" * 90)

VERIFY ACTIVE TRACK 1 JUDGE SYSTEM PROMPT
STRICT SCORE CALIBRATION                 : ✅ PRESENT
CEILING CONTROL                          : ✅ PRESENT
ENGINEERING REASONING CALIBRATION        : ✅ PRESENT
EVIDENCE DISCIPLINE                      : ✅ PRESENT
ROOT-CAUSE DISCIPLINE                    : ✅ PRESENT

System prompt length : 9,091 characters

CALIBRATION SECTION PREVIEW
STRICT SCORE CALIBRATION

    STRICT SCORE CALIBRATION

Apply the 1–10 scale conservatively.

10 — EXCEPTIONAL
A score of 10 is reserved for responses that are essentially exemplary
for the question asked.

The response must have:
- no material technical errors;
- no important omissions;
- precise functional boundaries and terminology;
- appropriate depth for the question;
- strong reasoning where reasoning is required;
- and no meaningful weakness that would justify improvement.

Do not award 10 merely because the response is correct, detailed,
fluent, well structured, or covers the expected points.


9 — VERY STRO

In [42]:
# =============================================================================
# 6. BUILD SYSTEM PROMPT
# =============================================================================

TRACK1_JUDGE_SYSTEM_PROMPT = dedent(
    f"""
    You are an independent senior telecom engineering evaluator.

    Your task is to evaluate AI-generated responses to advanced telecom
    questions involving areas such as:

    - 5G Core;
    - 5G Standalone procedures;
    - NG-RAN;
    - Open RAN;
    - cloud-native telecom;
    - Kubernetes-based telecom infrastructure;
    - RAN capacity and performance;
    - troubleshooting;
    - network design;
    - engineering trade-offs.

    You must judge technical quality, not writing style or model identity.

    ============================================================
    SCORE SCALE
    ============================================================

    {TRACK1_SCORE_SCALE}

    ============================================================
    GENERAL RULES
    ============================================================

    {TRACK1_GENERAL_JUDGE_RULES}

    ============================================================
    REQUIRED OUTPUT
    ============================================================

    Return one valid JSON object matching this structure:

    {json.dumps(
        TRACK1_JUDGE_OUTPUT_SCHEMA,
        indent=2,
    )}

    All scores must be integers from 1 to 10.

    response_status must be exactly one of:

    "SUBSTANTIVE"
    "NON_RESPONSIVE"

    significant_technical_errors must contain only genuine material
    technical errors.

    missing_important_points must contain major omissions relative to
    the question.

    strengths should identify the strongest technical aspects of
    the response.

    judge_summary should briefly explain the scoring decision.
    """
).strip()

In [43]:
# =============================================================================
# 6. BUILD SYSTEM PROMPT
# =============================================================================

TRACK1_JUDGE_SYSTEM_PROMPT = dedent(
    f"""
    You are an independent senior telecom engineering evaluator.

    Your task is to evaluate AI-generated responses to advanced telecom
    questions involving areas such as:

    - 5G Core;
    - 5G Standalone procedures;
    - NG-RAN;
    - Open RAN;
    - cloud-native telecom;
    - Kubernetes-based telecom infrastructure;
    - RAN capacity and performance;
    - troubleshooting;
    - network design;
    - engineering trade-offs.

    You must judge technical quality, not writing style or model identity.

    ============================================================
    SCORE SCALE
    ============================================================

    {TRACK1_SCORE_SCALE}

    ============================================================
    STRICT SCORE CALIBRATION
    ============================================================

    {TRACK1_SCORE_CALIBRATION}

    ============================================================
    GENERAL RULES
    ============================================================

    {TRACK1_GENERAL_JUDGE_RULES}

    ============================================================
    REQUIRED OUTPUT
    ============================================================

    Return one valid JSON object matching this structure:

    {json.dumps(
        TRACK1_JUDGE_OUTPUT_SCHEMA,
        indent=2,
    )}

    All scores must be integers from 1 to 10.

    response_status must be exactly one of:

    "SUBSTANTIVE"
    "NON_RESPONSIVE"

    significant_technical_errors must contain only genuine material
    technical errors.

    missing_important_points must contain major omissions relative to
    the question.

    strengths should identify the strongest technical aspects of
    the response.

    judge_summary should briefly explain the scoring decision.
    """
).strip()

In [47]:
# =============================================================================
# MODULE 4.3C — JUDGE DRY RUN
# Q01 + Q19 ACROSS ALL FIVE SYSTEMS
# =============================================================================

import json
import pandas as pd


# =============================================================================
# 1. DRY-RUN QUESTION SELECTION
# =============================================================================

DRY_RUN_QUESTION_IDS = [
    "Q01",
    "Q19",
]


dry_run_df = (
    track1_evaluation_df[
        track1_evaluation_df[
            "question_id"
        ].isin(
            DRY_RUN_QUESTION_IDS
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# Expected:
# 2 questions × 5 models = 10 responses

if len(dry_run_df) != 10:

    raise RuntimeError(
        f"Expected 10 dry-run rows, "
        f"found {len(dry_run_df)}."
    )


print("=" * 90)
print("TRACK 1 JUDGE DRY RUN")
print("=" * 90)

print(
    f"\nQuestions : "
    f"{DRY_RUN_QUESTION_IDS}"
)

print(
    f"Responses : "
    f"{len(dry_run_df)}"
)

print(
    f"Models    : "
    f"{dry_run_df['model_name'].nunique()}"
)

print("=" * 90)


# =============================================================================
# 2. JUDGE RESPONSE PARSER
# =============================================================================

def validate_judge_result(
    result,
    expected_question_id,
    expected_model_name,
    expected_evaluation_type,
):

    required_top_level = {
        "question_id",
        "model_name",
        "evaluation_type",
        "response_status",
        "scores",
        "significant_technical_errors",
        "missing_important_points",
        "strengths",
        "judge_summary",
    }


    missing_fields = (
        required_top_level
        - set(
            result.keys()
        )
    )


    if missing_fields:

        raise ValueError(
            f"Missing judge fields: "
            f"{sorted(missing_fields)}"
        )


    # -------------------------------------------------------------------------
    # IDENTITY CHECKS
    # -------------------------------------------------------------------------

    if result["question_id"] != expected_question_id:

        raise ValueError(
            f"Question ID mismatch. "
            f"Expected {expected_question_id}, "
            f"got {result['question_id']}."
        )


    if result["model_name"] != expected_model_name:

        raise ValueError(
            f"Model name mismatch. "
            f"Expected {expected_model_name}, "
            f"got {result['model_name']}."
        )


    if (
        result["evaluation_type"]
        != expected_evaluation_type
    ):

        raise ValueError(
            "Evaluation type mismatch."
        )


    # -------------------------------------------------------------------------
    # RESPONSE STATUS
    # -------------------------------------------------------------------------

    if result[
        "response_status"
    ] not in {
        "SUBSTANTIVE",
        "NON_RESPONSIVE",
    }:

        raise ValueError(
            "Invalid response_status."
        )


    # -------------------------------------------------------------------------
    # SCORE VALIDATION
    # -------------------------------------------------------------------------

    expected_score_fields = {
        "technical_accuracy",
        "completeness",
        "relevance",
        "engineering_reasoning",
        "practical_applicability",
        "factual_reliability",
        "overall_score",
    }


    score_fields = set(
        result["scores"].keys()
    )


    if score_fields != expected_score_fields:

        raise ValueError(
            "Unexpected score fields.\n"
            f"Expected: {expected_score_fields}\n"
            f"Actual  : {score_fields}"
        )


    for score_name, score in (
        result["scores"].items()
    ):

        if not isinstance(
            score,
            int,
        ):

            raise ValueError(
                f"{score_name} must be integer."
            )


        if not 1 <= score <= 10:

            raise ValueError(
                f"{score_name} outside 1–10: "
                f"{score}"
            )


    # -------------------------------------------------------------------------
    # LIST FIELDS
    # -------------------------------------------------------------------------

    for list_field in [
        "significant_technical_errors",
        "missing_important_points",
        "strengths",
    ]:

        if not isinstance(
            result[list_field],
            list,
        ):

            raise ValueError(
                f"{list_field} must be a list."
            )


    if not isinstance(
        result["judge_summary"],
        str,
    ):

        raise ValueError(
            "judge_summary must be a string."
        )


    return True


# =============================================================================
# 3. RUN JUDGE DRY RUN
# =============================================================================

judge_dry_run_results = []


for idx, row in dry_run_df.iterrows():

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"Judging "
        f"{row['question_id']} "
        f"| {row['model_name']}"
    )

    print(
        f"Evaluation Type: "
        f"{row['evaluation_type']}"
    )

    print(
        "=" * 90
    )


    # -------------------------------------------------------------------------
    # BUILD USER PROMPT
    # -------------------------------------------------------------------------

    judge_user_prompt = (
        build_track1_judge_prompt(
            row
        )
    )


    # -------------------------------------------------------------------------
    # CALL QWEN3 JUDGE
    # -------------------------------------------------------------------------

    response = (
        client
        .chat
        .completions
        .create(
            model=VLLM_MODEL,

            messages=[
                {
                    "role": "system",
                    "content":
                        TRACK1_JUDGE_SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content":
                        judge_user_prompt,
                },
            ],

            temperature=0.0,

            max_tokens=1200,
        )
    )


    raw_judge_text = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )


    # -------------------------------------------------------------------------
    # PARSE JSON
    # -------------------------------------------------------------------------

    try:

        judge_result = (
            json.loads(
                raw_judge_text
            )
        )

    except json.JSONDecodeError as exc:

        print(
            "\n❌ Invalid JSON returned."
        )

        print(
            "\nRAW RESPONSE:\n"
        )

        print(
            raw_judge_text
        )

        raise RuntimeError(
            f"Judge returned invalid JSON for "
            f"{row['question_id']} | "
            f"{row['model_name']}"
        ) from exc


    # -------------------------------------------------------------------------
    # VALIDATE RESULT
    # -------------------------------------------------------------------------

    validate_judge_result(

        result=judge_result,

        expected_question_id=
            row["question_id"],

        expected_model_name=
            row["model_name"],

        expected_evaluation_type=
            row["evaluation_type"],
    )


    # -------------------------------------------------------------------------
    # STORE RESULT
    # -------------------------------------------------------------------------

    scores = (
        judge_result[
            "scores"
        ]
    )


    judge_dry_run_results.append(
        {
            "question_id":
                row["question_id"],

            "model_name":
                row["model_name"],

            "evaluation_type":
                row["evaluation_type"],

            "corpus_relevance":
                row["corpus_relevance"],

            "response_status":
                judge_result[
                    "response_status"
                ],

            "technical_accuracy":
                scores[
                    "technical_accuracy"
                ],

            "completeness":
                scores[
                    "completeness"
                ],

            "relevance":
                scores[
                    "relevance"
                ],

            "engineering_reasoning":
                scores[
                    "engineering_reasoning"
                ],

            "practical_applicability":
                scores[
                    "practical_applicability"
                ],

            "factual_reliability":
                scores[
                    "factual_reliability"
                ],

            "overall_score":
                scores[
                    "overall_score"
                ],

            "significant_technical_errors":
                judge_result[
                    "significant_technical_errors"
                ],

            "missing_important_points":
                judge_result[
                    "missing_important_points"
                ],

            "strengths":
                judge_result[
                    "strengths"
                ],

            "judge_summary":
                judge_result[
                    "judge_summary"
                ],

            "raw_judge_json":
                judge_result,
        }
    )


    print(
        f"\n✅ Valid judge result"
    )

    print(
        f"Overall score : "
        f"{scores['overall_score']}/10"
    )

    print(
        f"Status        : "
        f"{judge_result['response_status']}"
    )


# =============================================================================
# 4. BUILD DRY-RUN RESULTS DATAFRAME
# =============================================================================

track1_judge_dry_run_df = (
    pd.DataFrame(
        judge_dry_run_results
    )
)


# =============================================================================
# 5. DISPLAY COMPACT RESULTS
# =============================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "TRACK 1 JUDGE DRY RUN — RESULTS"
)

print(
    "=" * 90
)


display(
    track1_judge_dry_run_df[
        [
            "question_id",
            "model_name",
            "evaluation_type",
            "technical_accuracy",
            "completeness",
            "engineering_reasoning",
            "factual_reliability",
            "overall_score",
            "response_status",
        ]
    ]
)


# =============================================================================
# 6. QUESTION-LEVEL COMPARISON
# =============================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "DRY RUN — OVERALL SCORE BY QUESTION / MODEL"
)

print(
    "=" * 90
)


dry_run_score_pivot = (
    track1_judge_dry_run_df
    .pivot(
        index="question_id",
        columns="model_name",
        values="overall_score",
    )
)


display(
    dry_run_score_pivot
)


# =============================================================================
# 7. VALIDATION SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "DRY RUN VALIDATION SUMMARY"
)

print(
    "=" * 90
)

print(
    f"Judge outputs          : "
    f"{len(track1_judge_dry_run_df)}"
)

print(
    f"Questions evaluated    : "
    f"{track1_judge_dry_run_df['question_id'].nunique()}"
)

print(
    f"Models evaluated       : "
    f"{track1_judge_dry_run_df['model_name'].nunique()}"
)

print(
    f"Invalid JSON outputs   : 0"
)

print(
    f"Schema validation      : PASS"
)

print(
    "=" * 90
)

TRACK 1 JUDGE DRY RUN

Questions : ['Q01', 'Q19']
Responses : 10
Models    : 5

Judging Q01 | Essential AI + RAG
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 5/10
Status        : SUBSTANTIVE

Judging Q01 | Essential AI Only
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 4/10
Status        : SUBSTANTIVE

Judging Q01 | Otel 2.0 Only
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 10/10
Status        : SUBSTANTIVE

Judging Q01 | Gemma 4 Only
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 10/10
Status        : SUBSTANTIVE

Judging Q01 | Gemma 4 + RAG
Evaluation Type: CORPUS_GROUNDED

✅ Valid judge result
Overall score : 9/10
Status        : SUBSTANTIVE

Judging Q19 | Essential AI + RAG
Evaluation Type: ENGINEERING_SYNTHESIS

✅ Valid judge result
Overall score : 1/10
Status        : NON_RESPONSIVE

Judging Q19 | Essential AI Only
Evaluation Type: ENGINEERING_SYNTHESIS

✅ Valid judge result
Overa

,question_id,model_name,evaluation_type,technical_accuracy,completeness,engineering_reasoning,factual_reliability,overall_score,response_status
0,Q01,Essential AI + RAG,CORPUS_GROUNDED,4,6,5,4,5,SUBSTANTIVE
1,Q01,Essential AI Only,CORPUS_GROUNDED,3,4,4,3,4,SUBSTANTIVE
2,Q01,Otel 2.0 Only,CORPUS_GROUNDED,10,10,10,10,10,SUBSTANTIVE
3,Q01,Gemma 4 Only,CORPUS_GROUNDED,10,10,10,10,10,SUBSTANTIVE
4,Q01,Gemma 4 + RAG,CORPUS_GROUNDED,8,9,9,8,9,SUBSTANTIVE
5,Q19,Essential AI + RAG,ENGINEERING_SYNTHESIS,1,1,1,1,1,NON_RESPONSIVE
6,Q19,Essential AI Only,ENGINEERING_SYNTHESIS,9,7,8,10,8,SUBSTANTIVE
7,Q19,Otel 2.0 Only,ENGINEERING_SYNTHESIS,10,10,10,10,10,SUBSTANTIVE
8,Q19,Gemma 4 Only,ENGINEERING_SYNTHESIS,10,7,9,10,8,SUBSTANTIVE
9,Q19,Gemma 4 + RAG,ENGINEERING_SYNTHESIS,10,8,9,10,9,SUBSTANTIVE



DRY RUN — OVERALL SCORE BY QUESTION / MODEL


model_name,Essential AI + RAG,Essential AI Only,Gemma 4 + RAG,Gemma 4 Only,Otel 2.0 Only
question_id,,,,,
Q01,5,4,9,10,10
Q19,1,8,9,8,10



DRY RUN VALIDATION SUMMARY
Judge outputs          : 10
Questions evaluated    : 2
Models evaluated       : 5
Invalid JSON outputs   : 0
Schema validation      : PASS


# **Track 1 Full Evaluation**

In [22]:
# =============================================================================
# MODULE 4.4 — FULL TRACK 1 JUDGE EVALUATION
# 20 QUESTIONS × 5 SYSTEMS = 100 RESPONSES
# =============================================================================

import json
import time
import pandas as pd


# =============================================================================
# 1. INPUT DATASET
# =============================================================================

full_eval_df = (
    track1_evaluation_df
    .copy()
    .reset_index(drop=True)
)


expected_rows = (
    full_eval_df["question_id"].nunique()
    * full_eval_df["model_name"].nunique()
)


if len(full_eval_df) != expected_rows:

    raise RuntimeError(
        f"Unexpected Track 1 row count. "
        f"Expected {expected_rows}, found {len(full_eval_df)}."
    )


print("=" * 90)
print("TRACK 1 — FULL JUDGE EVALUATION")
print("=" * 90)

print(
    f"\nQuestions : "
    f"{full_eval_df['question_id'].nunique()}"
)

print(
    f"Models    : "
    f"{full_eval_df['model_name'].nunique()}"
)

print(
    f"Responses : "
    f"{len(full_eval_df)}"
)

print("=" * 90)


# =============================================================================
# 2. JUDGE RESULT VALIDATOR
# =============================================================================

def validate_judge_result(
    result,
    expected_question_id,
    expected_model_name,
    expected_evaluation_type,
):

    required_top_level = {
        "question_id",
        "model_name",
        "evaluation_type",
        "response_status",
        "significant_technical_errors",
        "missing_important_points",
        "strengths",
        "judge_summary",
        "scores",
    }


    missing_fields = (
        required_top_level
        - set(result.keys())
    )


    if missing_fields:

        raise ValueError(
            f"Missing judge fields: "
            f"{sorted(missing_fields)}"
        )


    # -------------------------------------------------------------------------
    # IDENTITY CHECKS
    # -------------------------------------------------------------------------

    if result["question_id"] != expected_question_id:

        raise ValueError(
            f"Question ID mismatch. "
            f"Expected {expected_question_id}, "
            f"got {result['question_id']}."
        )


    if result["model_name"] != expected_model_name:

        raise ValueError(
            f"Model name mismatch. "
            f"Expected {expected_model_name}, "
            f"got {result['model_name']}."
        )


    if (
        result["evaluation_type"]
        != expected_evaluation_type
    ):

        raise ValueError(
            "Evaluation type mismatch."
        )


    # -------------------------------------------------------------------------
    # RESPONSE STATUS
    # -------------------------------------------------------------------------

    if result["response_status"] not in {
        "SUBSTANTIVE",
        "NON_RESPONSIVE",
    }:

        raise ValueError(
            "Invalid response_status."
        )


    # -------------------------------------------------------------------------
    # QUALITATIVE FIELD VALIDATION
    # -------------------------------------------------------------------------

    for list_field in [
        "significant_technical_errors",
        "missing_important_points",
        "strengths",
    ]:

        if not isinstance(
            result[list_field],
            list,
        ):

            raise ValueError(
                f"{list_field} must be a list."
            )


    if not isinstance(
        result["judge_summary"],
        str,
    ):

        raise ValueError(
            "judge_summary must be a string."
        )


    # -------------------------------------------------------------------------
    # SCORE VALIDATION
    # -------------------------------------------------------------------------

    expected_score_fields = {
        "technical_accuracy",
        "completeness",
        "relevance",
        "engineering_reasoning",
        "practical_applicability",
        "factual_reliability",
        "overall_score",
    }


    score_fields = set(
        result["scores"].keys()
    )


    if score_fields != expected_score_fields:

        raise ValueError(
            "Unexpected score fields.\n"
            f"Expected: {expected_score_fields}\n"
            f"Actual  : {score_fields}"
        )


    for score_name, score in result["scores"].items():

        if not isinstance(
            score,
            int,
        ):

            raise ValueError(
                f"{score_name} must be integer."
            )


        if not 1 <= score <= 10:

            raise ValueError(
                f"{score_name} outside 1–10: "
                f"{score}"
            )


    return True


# =============================================================================
# 3. RUN FULL JUDGE EVALUATION
# =============================================================================

track1_judge_results = []

start_time = time.time()


for idx, row in full_eval_df.iterrows():

    current_number = idx + 1
    total_number = len(full_eval_df)


    print(
        f"\n[{current_number:03d}/{total_number:03d}] "
        f"{row['question_id']} | "
        f"{row['model_name']} | "
        f"{row['evaluation_type']}"
    )


    # -------------------------------------------------------------------------
    # BUILD JUDGE PROMPT
    # -------------------------------------------------------------------------

    judge_user_prompt = (
        build_track1_judge_prompt(
            row
        )
    )


    # -------------------------------------------------------------------------
    # CALL QWEN3 JUDGE
    # -------------------------------------------------------------------------

    response = (
        client
        .chat
        .completions
        .create(

            model=VLLM_MODEL,

            messages=[
                {
                    "role": "system",
                    "content":
                        TRACK1_JUDGE_SYSTEM_PROMPT,
                },
                {
                    "role": "user",
                    "content":
                        judge_user_prompt,
                },
            ],

            temperature=0.0,

            max_tokens=1400,
        )
    )


    raw_judge_text = (
        response
        .choices[0]
        .message
        .content
        .strip()
    )


    # -------------------------------------------------------------------------
    # PARSE JSON
    # -------------------------------------------------------------------------

    try:

        judge_result = (
            json.loads(
                raw_judge_text
            )
        )

    except json.JSONDecodeError as exc:

        print(
            "\n❌ Invalid JSON returned."
        )

        print(
            raw_judge_text
        )

        raise RuntimeError(
            f"Judge returned invalid JSON for "
            f"{row['question_id']} | "
            f"{row['model_name']}"
        ) from exc


    # -------------------------------------------------------------------------
    # VALIDATE RESULT
    # -------------------------------------------------------------------------

    validate_judge_result(

        result=judge_result,

        expected_question_id=
            row["question_id"],

        expected_model_name=
            row["model_name"],

        expected_evaluation_type=
            row["evaluation_type"],
    )


    scores = (
        judge_result["scores"]
    )


    # -------------------------------------------------------------------------
    # STORE RESULT
    # -------------------------------------------------------------------------

    track1_judge_results.append(
        {

            "question_id":
                row["question_id"],

            "category":
                row["category"],

            "model_name":
                row["model_name"],

            "evaluation_type":
                row["evaluation_type"],

            "corpus_relevance":
                row["corpus_relevance"],

            "production_k7_support":
                row["production_k7_support"],

            "evaluation_recommendation":
                row["evaluation_recommendation"],

            "response_status":
                judge_result["response_status"],

            "technical_accuracy":
                scores["technical_accuracy"],

            "completeness":
                scores["completeness"],

            "relevance":
                scores["relevance"],

            "engineering_reasoning":
                scores["engineering_reasoning"],

            "practical_applicability":
                scores["practical_applicability"],

            "factual_reliability":
                scores["factual_reliability"],

            "overall_score":
                scores["overall_score"],

            "significant_technical_errors":
                judge_result[
                    "significant_technical_errors"
                ],

            "missing_important_points":
                judge_result[
                    "missing_important_points"
                ],

            "strengths":
                judge_result[
                    "strengths"
                ],

            "judge_summary":
                judge_result[
                    "judge_summary"
                ],

            "raw_judge_json":
                judge_result,
        }
    )


    print(
        f"    ✅ Overall "
        f"{scores['overall_score']}/10 "
        f"| {judge_result['response_status']}"
    )


# =============================================================================
# 4. BUILD FULL RESULTS DATAFRAME
# =============================================================================

track1_judge_df = (
    pd.DataFrame(
        track1_judge_results
    )
)


# =============================================================================
# 5. EXECUTION SUMMARY
# =============================================================================

elapsed_sec = (
    time.time()
    - start_time
)


print(
    "\n"
    + "=" * 90
)

print(
    "TRACK 1 FULL JUDGE RUN COMPLETE"
)

print(
    "=" * 90
)

print(
    f"\nJudge outputs       : "
    f"{len(track1_judge_df)}"
)

print(
    f"Questions evaluated : "
    f"{track1_judge_df['question_id'].nunique()}"
)

print(
    f"Models evaluated    : "
    f"{track1_judge_df['model_name'].nunique()}"
)

print(
    f"Elapsed time        : "
    f"{elapsed_sec / 60:.2f} minutes"
)

print("=" * 90)


# =============================================================================
# 6. RESPONSE STATUS CHECK
# =============================================================================

print(
    "\nRESPONSE STATUS"
)

print(
    track1_judge_df[
        "response_status"
    ]
    .value_counts()
)


# =============================================================================
# 7. OVERALL SCORE SUMMARY BY MODEL
# =============================================================================

print(
    "\n"
    + "=" * 90
)

print(
    "OVERALL SCORE SUMMARY BY MODEL"
)

print(
    "=" * 90
)


overall_summary = (
    track1_judge_df
    .groupby(
        "model_name"
    )["overall_score"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
            "std",
        ]
    )
    .sort_values(
        "mean",
        ascending=False,
    )
)


display(
    overall_summary.round(3)
)


# =============================================================================
# 8. DIMENSION SCORE SUMMARY BY MODEL
# =============================================================================

score_columns = [

    "technical_accuracy",
    "completeness",
    "relevance",
    "engineering_reasoning",
    "practical_applicability",
    "factual_reliability",
    "overall_score",
]


dimension_summary = (
    track1_judge_df
    .groupby(
        "model_name"
    )[score_columns]
    .mean()
    .sort_values(
        "overall_score",
        ascending=False,
    )
)


print(
    "\n"
    + "=" * 90
)

print(
    "MEAN SCORE BY DIMENSION"
)

print(
    "=" * 90
)


display(
    dimension_summary.round(3)
)


# =============================================================================
# 9. OVERALL SCORE BY QUESTION / MODEL
# =============================================================================

question_score_pivot = (
    track1_judge_df
    .pivot(
        index="question_id",
        columns="model_name",
        values="overall_score",
    )
)


print(
    "\n"
    + "=" * 90
)

print(
    "OVERALL SCORE BY QUESTION / MODEL"
)

print(
    "=" * 90
)


display(
    question_score_pivot
)


# =============================================================================
# 10. VALIDATION
# =============================================================================

validation_checks = {

    "Total outputs = 100":
        len(track1_judge_df) == 100,

    "20 unique questions":
        track1_judge_df[
            "question_id"
        ].nunique() == 20,

    "5 unique models":
        track1_judge_df[
            "model_name"
        ].nunique() == 5,

    "No missing overall scores":
        track1_judge_df[
            "overall_score"
        ].notna().all(),

    "All scores within 1–10":
        track1_judge_df[
            score_columns
        ]
        .apply(
            lambda col:
                col.between(
                    1,
                    10,
                ).all()
        )
        .all(),
}


print(
    "\n"
    + "=" * 90
)

print(
    "TRACK 1 FULL EVALUATION VALIDATION"
)

print(
    "=" * 90
)


for check_name, passed in validation_checks.items():

    status = (
        "✅ PASS"
        if passed
        else "❌ FAIL"
    )

    print(
        f"{check_name:<35} : "
        f"{status}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Track 1 full judge evaluation "
        "failed one or more validation checks."
    )


print(
    "\nTRACK 1 FULL JUDGE DATASET "
    "VALIDATED SUCCESSFULLY"
)

print("=" * 90)

NameError: name 'track1_evaluation_df' is not defined

In [49]:
# =============================================================================
# SAVE EXISTING TRACK 1 JUDGE RESULTS BEFORE RESUME
# =============================================================================

import json

CURRENT_CHECKPOINT_FILE = (
    "track1_judge_results_checkpoint.json"
)

print(
    f"Existing successful judge results : "
    f"{len(track1_judge_results)}"
)

with open(
    CURRENT_CHECKPOINT_FILE,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        track1_judge_results,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    f"✅ Saved checkpoint: "
    f"{CURRENT_CHECKPOINT_FILE}"
)

Existing successful judge results : 52
✅ Saved checkpoint: track1_judge_results_checkpoint.json


In [50]:
# =============================================================================
# MODULE 4.4B — RESUME TRACK 1 FULL JUDGE EVALUATION
# ROBUST CHECKPOINTING + JSON RETRY
# =============================================================================

import json
import time
import os
import pandas as pd


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

CHECKPOINT_FILE = (
    "track1_judge_results_checkpoint.json"
)

FINAL_JSON_FILE = (
    "track1_judge_results_complete.json"
)

FINAL_CSV_FILE = (
    "track1_judge_results_complete.csv"
)

JUDGE_MAX_TOKENS = 3000

MAX_JSON_ATTEMPTS = 2


# =============================================================================
# 2. RECOVER RESULTS ALREADY COMPLETED IN MEMORY
# =============================================================================
#
# The failed Module 4.4 run already populated track1_judge_results
# with all successful evaluations before Q11 | Otel 2.0 Only.
#
# We preserve those results and resume only the missing evaluations.
#
# =============================================================================

if "track1_judge_results" not in globals():

    track1_judge_results = []


print("=" * 90)
print("TRACK 1 JUDGE — RESUME PREPARATION")
print("=" * 90)

print(
    f"\nResults currently in memory : "
    f"{len(track1_judge_results)}"
)


# =============================================================================
# 3. OPTIONAL CHECKPOINT RECOVERY
# =============================================================================

if os.path.exists(
    CHECKPOINT_FILE
):

    with open(
        CHECKPOINT_FILE,
        "r",
        encoding="utf-8",
    ) as f:

        checkpoint_results = json.load(
            f
        )


    if len(checkpoint_results) > len(
        track1_judge_results
    ):

        track1_judge_results = (
            checkpoint_results
        )

        print(
            f"✅ Recovered "
            f"{len(track1_judge_results)} "
            f"results from checkpoint."
        )


# =============================================================================
# 4. IDENTIFY COMPLETED QUESTION/MODEL PAIRS
# =============================================================================

completed_pairs = {

    (
        result["question_id"],
        result["model_name"],
    )

    for result in track1_judge_results
}


print(
    f"Completed pairs             : "
    f"{len(completed_pairs)}"
)


# =============================================================================
# 5. BUILD REMAINING DATASET
# =============================================================================

remaining_rows = []


for _, row in track1_evaluation_df.iterrows():

    pair = (
        row["question_id"],
        row["model_name"],
    )

    if pair not in completed_pairs:

        remaining_rows.append(
            row
        )


remaining_df = pd.DataFrame(
    remaining_rows
).reset_index(
    drop=True
)


print(
    f"Remaining evaluations       : "
    f"{len(remaining_df)}"
)

print("=" * 90)


# =============================================================================
# 6. JSON VALIDATOR
# =============================================================================

def validate_judge_result(
    result,
    expected_question_id,
    expected_model_name,
    expected_evaluation_type,
):

    required_top_level = {
        "question_id",
        "model_name",
        "evaluation_type",
        "response_status",
        "significant_technical_errors",
        "missing_important_points",
        "strengths",
        "judge_summary",
        "scores",
    }


    missing_fields = (
        required_top_level
        - set(result.keys())
    )


    if missing_fields:

        raise ValueError(
            f"Missing judge fields: "
            f"{sorted(missing_fields)}"
        )


    if (
        result["question_id"]
        != expected_question_id
    ):

        raise ValueError(
            "Question ID mismatch."
        )


    if (
        result["model_name"]
        != expected_model_name
    ):

        raise ValueError(
            "Model name mismatch."
        )


    if (
        result["evaluation_type"]
        != expected_evaluation_type
    ):

        raise ValueError(
            "Evaluation type mismatch."
        )


    if result["response_status"] not in {
        "SUBSTANTIVE",
        "NON_RESPONSIVE",
    }:

        raise ValueError(
            "Invalid response_status."
        )


    for list_field in [
        "significant_technical_errors",
        "missing_important_points",
        "strengths",
    ]:

        if not isinstance(
            result[list_field],
            list,
        ):

            raise ValueError(
                f"{list_field} must be a list."
            )


    if not isinstance(
        result["judge_summary"],
        str,
    ):

        raise ValueError(
            "judge_summary must be a string."
        )


    expected_score_fields = {
        "technical_accuracy",
        "completeness",
        "relevance",
        "engineering_reasoning",
        "practical_applicability",
        "factual_reliability",
        "overall_score",
    }


    if (
        set(result["scores"].keys())
        != expected_score_fields
    ):

        raise ValueError(
            "Unexpected score fields."
        )


    for score_name, score in (
        result["scores"].items()
    ):

        if not isinstance(
            score,
            int,
        ):

            raise ValueError(
                f"{score_name} must be integer."
            )


        if not 1 <= score <= 10:

            raise ValueError(
                f"{score_name} outside 1–10."
            )


    return True


# =============================================================================
# 7. RUN REMAINING EVALUATIONS
# =============================================================================

resume_start_time = time.time()


for idx, row in remaining_df.iterrows():

    completed_count = len(
        track1_judge_results
    )

    overall_number = (
        completed_count + 1
    )


    print(
        f"\n[{overall_number:03d}/100] "
        f"{row['question_id']} | "
        f"{row['model_name']} | "
        f"{row['evaluation_type']}"
    )


    judge_user_prompt = (
        build_track1_judge_prompt(
            row
        )
    )


    judge_result = None

    last_raw_text = None


    # =========================================================================
    # RETRY LOOP
    # =========================================================================

    for attempt in range(
        1,
        MAX_JSON_ATTEMPTS + 1,
    ):

        response = (
            client
            .chat
            .completions
            .create(

                model=VLLM_MODEL,

                messages=[
                    {
                        "role": "system",
                        "content":
                            TRACK1_JUDGE_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content":
                            judge_user_prompt,
                    },
                ],

                temperature=0.0,

                max_tokens=JUDGE_MAX_TOKENS,
            )
        )


        raw_judge_text = (
            response
            .choices[0]
            .message
            .content
            .strip()
        )


        last_raw_text = (
            raw_judge_text
        )


        try:

            parsed_result = (
                json.loads(
                    raw_judge_text
                )
            )


            validate_judge_result(

                result=
                    parsed_result,

                expected_question_id=
                    row["question_id"],

                expected_model_name=
                    row["model_name"],

                expected_evaluation_type=
                    row["evaluation_type"],
            )


            judge_result = (
                parsed_result
            )

            break


        except (
            json.JSONDecodeError,
            ValueError,
        ) as exc:

            print(
                f"    ⚠️ Attempt "
                f"{attempt}/{MAX_JSON_ATTEMPTS} "
                f"failed JSON/schema validation."
            )

            print(
                f"    Reason: {exc}"
            )


            if attempt < MAX_JSON_ATTEMPTS:

                print(
                    "    ↻ Retrying judge response..."
                )


    # =========================================================================
    # FAIL ONLY AFTER RETRIES
    # =========================================================================

    if judge_result is None:

        print(
            "\n❌ Judge failed after all retry attempts."
        )

        print(
            "\nLast raw response:\n"
        )

        print(
            last_raw_text
        )

        raise RuntimeError(
            f"Judge failed for "
            f"{row['question_id']} | "
            f"{row['model_name']}"
        )


    # =========================================================================
    # STORE VALID RESULT
    # =========================================================================

    scores = (
        judge_result["scores"]
    )


    result_record = {

        "question_id":
            row["question_id"],

        "category":
            row["category"],

        "model_name":
            row["model_name"],

        "evaluation_type":
            row["evaluation_type"],

        "corpus_relevance":
            row["corpus_relevance"],

        "production_k7_support":
            row["production_k7_support"],

        "evaluation_recommendation":
            row["evaluation_recommendation"],

        "response_status":
            judge_result[
                "response_status"
            ],

        "technical_accuracy":
            scores[
                "technical_accuracy"
            ],

        "completeness":
            scores[
                "completeness"
            ],

        "relevance":
            scores[
                "relevance"
            ],

        "engineering_reasoning":
            scores[
                "engineering_reasoning"
            ],

        "practical_applicability":
            scores[
                "practical_applicability"
            ],

        "factual_reliability":
            scores[
                "factual_reliability"
            ],

        "overall_score":
            scores[
                "overall_score"
            ],

        "significant_technical_errors":
            judge_result[
                "significant_technical_errors"
            ],

        "missing_important_points":
            judge_result[
                "missing_important_points"
            ],

        "strengths":
            judge_result[
                "strengths"
            ],

        "judge_summary":
            judge_result[
                "judge_summary"
            ],

        "raw_judge_json":
            judge_result,
    }


    track1_judge_results.append(
        result_record
    )


    print(
        f"    ✅ Overall "
        f"{scores['overall_score']}/10 "
        f"| "
        f"{judge_result['response_status']}"
    )


    # =========================================================================
    # SAVE CHECKPOINT AFTER EVERY SUCCESSFUL RESPONSE
    # =========================================================================

    with open(
        CHECKPOINT_FILE,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            track1_judge_results,
            f,
            indent=2,
            ensure_ascii=False,
        )


# =============================================================================
# 8. BUILD COMPLETE DATAFRAME
# =============================================================================

track1_judge_df = (
    pd.DataFrame(
        track1_judge_results
    )
)


# =============================================================================
# 9. SORT INTO ORIGINAL BENCHMARK ORDER
# =============================================================================

question_order = {
    f"Q{i:02d}": i
    for i in range(
        1,
        21,
    )
}


model_order = {

    "Essential AI + RAG": 1,

    "Essential AI Only": 2,

    "Otel 2.0 Only": 3,

    "Gemma 4 Only": 4,

    "Gemma 4 + RAG": 5,
}


track1_judge_df[
    "_question_order"
] = (
    track1_judge_df[
        "question_id"
    ].map(
        question_order
    )
)


track1_judge_df[
    "_model_order"
] = (
    track1_judge_df[
        "model_name"
    ].map(
        model_order
    )
)


track1_judge_df = (
    track1_judge_df
    .sort_values(
        [
            "_question_order",
            "_model_order",
        ]
    )
    .drop(
        columns=[
            "_question_order",
            "_model_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 10. FINAL VALIDATION
# =============================================================================

score_columns = [

    "technical_accuracy",

    "completeness",

    "relevance",

    "engineering_reasoning",

    "practical_applicability",

    "factual_reliability",

    "overall_score",
]


validation_checks = {

    "Total outputs = 100":
        len(
            track1_judge_df
        ) == 100,

    "20 unique questions":
        track1_judge_df[
            "question_id"
        ].nunique() == 20,

    "5 unique models":
        track1_judge_df[
            "model_name"
        ].nunique() == 5,

    "No duplicate question/model pairs":
        not track1_judge_df
        .duplicated(
            subset=[
                "question_id",
                "model_name",
            ]
        )
        .any(),

    "No missing overall scores":
        track1_judge_df[
            "overall_score"
        ]
        .notna()
        .all(),

    "All scores within 1–10":
        track1_judge_df[
            score_columns
        ]
        .apply(
            lambda col:
                col.between(
                    1,
                    10,
                ).all()
        )
        .all(),
}


print(
    "\n"
    + "=" * 90
)

print(
    "TRACK 1 FULL EVALUATION VALIDATION"
)

print(
    "=" * 90
)


for check_name, passed in (
    validation_checks.items()
):

    status = (
        "✅ PASS"
        if passed
        else "❌ FAIL"
    )

    print(
        f"{check_name:<40} : "
        f"{status}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Track 1 evaluation failed "
        "one or more final checks."
    )


# =============================================================================
# 11. SAVE FINAL RESULTS
# =============================================================================

with open(
    FINAL_JSON_FILE,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        track1_judge_df
        .to_dict(
            orient="records"
        ),
        f,
        indent=2,
        ensure_ascii=False,
    )


track1_judge_df.to_csv(
    FINAL_CSV_FILE,
    index=False,
)


# =============================================================================
# 12. SUMMARY
# =============================================================================

elapsed_sec = (
    time.time()
    - resume_start_time
)


print(
    "\n"
    + "=" * 90
)

print(
    "TRACK 1 FULL JUDGE RUN COMPLETE"
)

print(
    "=" * 90
)

print(
    f"\nJudge outputs       : "
    f"{len(track1_judge_df)}"
)

print(
    f"Questions evaluated : "
    f"{track1_judge_df['question_id'].nunique()}"
)

print(
    f"Models evaluated    : "
    f"{track1_judge_df['model_name'].nunique()}"
)

print(
    f"Resume elapsed time : "
    f"{elapsed_sec / 60:.2f} minutes"
)

print(
    f"\nJSON saved          : "
    f"{FINAL_JSON_FILE}"
)

print(
    f"CSV saved           : "
    f"{FINAL_CSV_FILE}"
)

print("=" * 90)


# =============================================================================
# 13. MODEL SCORE SUMMARY
# =============================================================================

overall_summary = (
    track1_judge_df
    .groupby(
        "model_name"
    )["overall_score"]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
            "std",
        ]
    )
    .sort_values(
        "mean",
        ascending=False,
    )
)


print(
    "\nOVERALL SCORE SUMMARY BY MODEL"
)

display(
    overall_summary.round(3)
)


print(
    "\n✅ TRACK 1 FULL JUDGE DATASET "
    "VALIDATED SUCCESSFULLY"
)

TRACK 1 JUDGE — RESUME PREPARATION

Results currently in memory : 52
Completed pairs             : 52
Remaining evaluations       : 48

[053/100] Q11 | Otel 2.0 Only | ENGINEERING_SYNTHESIS
    ✅ Overall 7/10 | SUBSTANTIVE

[054/100] Q11 | Gemma 4 Only | ENGINEERING_SYNTHESIS
    ✅ Overall 5/10 | SUBSTANTIVE

[055/100] Q11 | Gemma 4 + RAG | ENGINEERING_SYNTHESIS
    ✅ Overall 3/10 | SUBSTANTIVE

[056/100] Q12 | Essential AI + RAG | CORPUS_GROUNDED
    ✅ Overall 8/10 | SUBSTANTIVE

[057/100] Q12 | Essential AI Only | CORPUS_GROUNDED
    ✅ Overall 8/10 | SUBSTANTIVE

[058/100] Q12 | Otel 2.0 Only | CORPUS_GROUNDED
    ✅ Overall 9/10 | SUBSTANTIVE

[059/100] Q12 | Gemma 4 Only | CORPUS_GROUNDED
    ✅ Overall 8/10 | SUBSTANTIVE

[060/100] Q12 | Gemma 4 + RAG | CORPUS_GROUNDED
    ✅ Overall 8/10 | SUBSTANTIVE

[061/100] Q13 | Essential AI + RAG | ENGINEERING_SYNTHESIS
    ✅ Overall 6/10 | SUBSTANTIVE

[062/100] Q13 | Essential AI Only | ENGINEERING_SYNTHESIS
    ✅ Overall 3/10 | SUBSTANTIVE

,count,mean,median,min,max,std
model_name,,,,,,
Otel 2.0 Only,20,9.20,9.5,7,10,0.951
Gemma 4 Only,20,9.00,9.0,5,10,1.214
Gemma 4 + RAG,20,6.75,7.5,1,10,2.633
Essential AI Only,20,5.80,5.0,3,9,1.852
Essential AI + RAG,20,5.65,6.0,1,10,2.434



✅ TRACK 1 FULL JUDGE DATASET VALIDATED SUCCESSFULLY


# **Track 1 Performance Analysis**

In [51]:
# =============================================================================
# MODULE 5.1 — TRACK 1 MODEL PERFORMANCE SUMMARY
# =============================================================================
#
# PURPOSE
# -------
# Summarize the completed Track 1 judge results across all five systems.
#
# ANALYSIS INCLUDED
# -----------------
# 1. Overall ranking by mean score
# 2. Mean score by evaluation dimension
# 3. CORPUS_GROUNDED vs ENGINEERING_SYNTHESIS performance
# 4. Per-question winners
# 5. RAG vs non-RAG comparison within model families
#
# INPUT
# -----
# track1_judge_df
#
# Expected:
#   20 questions × 5 systems = 100 judged responses
#
# =============================================================================


import pandas as pd
import numpy as np


# =============================================================================
# 1. VALIDATE INPUT DATASET
# =============================================================================

required_columns = {

    "question_id",
    "model_name",
    "evaluation_type",

    "technical_accuracy",
    "completeness",
    "relevance",
    "engineering_reasoning",
    "practical_applicability",
    "factual_reliability",
    "overall_score",

    "response_status",
}


missing_columns = (
    required_columns
    - set(track1_judge_df.columns)
)


if missing_columns:

    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )


if len(track1_judge_df) != 100:

    raise ValueError(
        f"Expected 100 Track 1 judge rows, "
        f"found {len(track1_judge_df)}."
    )


if (
    track1_judge_df["question_id"].nunique()
    != 20
):

    raise ValueError(
        "Expected 20 unique Track 1 questions."
    )


if (
    track1_judge_df["model_name"].nunique()
    != 5
):

    raise ValueError(
        "Expected 5 evaluation systems."
    )


print("=" * 100)
print("MODULE 5.1 — TRACK 1 MODEL PERFORMANCE SUMMARY")
print("=" * 100)

print(
    f"\nResponses : "
    f"{len(track1_judge_df)}"
)

print(
    f"Questions : "
    f"{track1_judge_df['question_id'].nunique()}"
)

print(
    f"Systems   : "
    f"{track1_judge_df['model_name'].nunique()}"
)


# =============================================================================
# 2. SCORE COLUMNS
# =============================================================================

score_columns = [

    "technical_accuracy",
    "completeness",
    "relevance",
    "engineering_reasoning",
    "practical_applicability",
    "factual_reliability",
    "overall_score",
]


# =============================================================================
# 3. OVERALL MODEL RANKING
# =============================================================================

overall_ranking = (

    track1_judge_df

    .groupby(
        "model_name"
    )

    .agg(

        responses=(
            "overall_score",
            "count",
        ),

        mean_overall=(
            "overall_score",
            "mean",
        ),

        median_overall=(
            "overall_score",
            "median",
        ),

        min_overall=(
            "overall_score",
            "min",
        ),

        max_overall=(
            "overall_score",
            "max",
        ),

        std_overall=(
            "overall_score",
            "std",
        ),
    )

    .sort_values(
        [
            "mean_overall",
            "median_overall",
        ],
        ascending=False,
    )

    .reset_index()
)


overall_ranking.insert(

    0,

    "rank",

    range(
        1,
        len(overall_ranking) + 1,
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "1. OVERALL MODEL RANKING"
)

print(
    "=" * 100
)


display(
    overall_ranking.round(3)
)


# =============================================================================
# 4. MEAN SCORE BY DIMENSION
# =============================================================================

dimension_summary = (

    track1_judge_df

    .groupby(
        "model_name"
    )[score_columns]

    .mean()

    .sort_values(
        "overall_score",
        ascending=False,
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "2. MEAN SCORE BY EVALUATION DIMENSION"
)

print(
    "=" * 100
)


display(
    dimension_summary.round(3)
)


# =============================================================================
# 5. PERFORMANCE BY EVALUATION TYPE
# =============================================================================

evaluation_type_summary = (

    track1_judge_df

    .groupby(
        [
            "evaluation_type",
            "model_name",
        ]
    )

    .agg(

        responses=(
            "overall_score",
            "count",
        ),

        mean_overall=(
            "overall_score",
            "mean",
        ),

        median_overall=(
            "overall_score",
            "median",
        ),

        mean_technical_accuracy=(
            "technical_accuracy",
            "mean",
        ),

        mean_engineering_reasoning=(
            "engineering_reasoning",
            "mean",
        ),

        mean_factual_reliability=(
            "factual_reliability",
            "mean",
        ),
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "3. PERFORMANCE BY EVALUATION TYPE"
)

print(
    "=" * 100
)


display(
    evaluation_type_summary.round(3)
)


# =============================================================================
# 6. OVERALL SCORE PIVOT BY EVALUATION TYPE
# =============================================================================

evaluation_type_pivot = (

    track1_judge_df

    .pivot_table(

        index="model_name",

        columns="evaluation_type",

        values="overall_score",

        aggfunc="mean",
    )

    .round(3)
)


evaluation_type_pivot[
    "ALL_TRACK1"
] = (

    track1_judge_df

    .groupby(
        "model_name"
    )["overall_score"]

    .mean()
)


evaluation_type_pivot = (

    evaluation_type_pivot

    .sort_values(
        "ALL_TRACK1",
        ascending=False,
    )

    .round(3)
)


print(
    "\n"
    + "=" * 100
)

print(
    "4. CORPUS_GROUNDED vs ENGINEERING_SYNTHESIS"
)

print(
    "=" * 100
)


display(
    evaluation_type_pivot
)


# =============================================================================
# 7. QUESTION-LEVEL SCORE MATRIX
# =============================================================================

question_score_matrix = (

    track1_judge_df

    .pivot(

        index="question_id",

        columns="model_name",

        values="overall_score",
    )

    .sort_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "5. OVERALL SCORE BY QUESTION / SYSTEM"
)

print(
    "=" * 100
)


display(
    question_score_matrix
)


# =============================================================================
# 8. IDENTIFY QUESTION WINNERS
# =============================================================================

winner_records = []


for question_id, group in (

    track1_judge_df

    .groupby(
        "question_id"
    )
):

    max_score = (
        group[
            "overall_score"
        ].max()
    )


    winners = (

        group[
            group[
                "overall_score"
            ] == max_score
        ][
            "model_name"
        ]

        .tolist()
    )


    evaluation_type = (

        group[
            "evaluation_type"
        ]

        .iloc[0]
    )


    winner_records.append(
        {

            "question_id":
                question_id,

            "evaluation_type":
                evaluation_type,

            "winning_score":
                max_score,

            "winner_count":
                len(winners),

            "winner_models":
                " | ".join(
                    winners
                ),
        }
    )


question_winners_df = (

    pd.DataFrame(
        winner_records
    )

    .sort_values(
        "question_id"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "6. QUESTION-LEVEL WINNERS"
)

print(
    "=" * 100
)


display(
    question_winners_df
)


# =============================================================================
# 9. NUMBER OF QUESTION WINS PER MODEL
# =============================================================================
#
# Ties count as a win for each tied model.
#
# =============================================================================

win_counts = {

    model_name: 0

    for model_name
    in track1_judge_df[
        "model_name"
    ].unique()
}


for _, row in (
    question_winners_df.iterrows()
):

    winners = (
        row[
            "winner_models"
        ].split(
            " | "
        )
    )


    for winner in winners:

        win_counts[
            winner
        ] += 1


win_count_df = (

    pd.DataFrame(
        [
            {
                "model_name":
                    model_name,

                "question_wins":
                    wins,
            }

            for model_name, wins
            in win_counts.items()
        ]
    )

    .sort_values(
        "question_wins",
        ascending=False,
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "7. QUESTION WINS PER MODEL"
)

print(
    "=" * 100
)


display(
    win_count_df
)


# =============================================================================
# 10. RAG vs NON-RAG — MODEL FAMILY COMPARISON
# =============================================================================
#
# Compare only within the same model family:
#
# Essential AI:
#   Essential AI + RAG
#   Essential AI Only
#
# Gemma 4:
#   Gemma 4 + RAG
#   Gemma 4 Only
#
# Otel 2.0 has no RAG counterpart in this experiment.
#
# =============================================================================

rag_family_pairs = {

    "Essential AI": {
        "rag":
            "Essential AI + RAG",

        "base":
            "Essential AI Only",
    },

    "Gemma 4": {
        "rag":
            "Gemma 4 + RAG",

        "base":
            "Gemma 4 Only",
    },
}


rag_comparison_records = []


for family_name, pair in (
    rag_family_pairs.items()
):

    rag_df = (

        track1_judge_df[

            track1_judge_df[
                "model_name"
            ] == pair["rag"]

        ][
            [
                "question_id",
                "evaluation_type",
                "overall_score",
            ]
        ]

        .rename(
            columns={
                "overall_score":
                    "rag_score"
            }
        )
    )


    base_df = (

        track1_judge_df[

            track1_judge_df[
                "model_name"
            ] == pair["base"]

        ][
            [
                "question_id",
                "overall_score",
            ]
        ]

        .rename(
            columns={
                "overall_score":
                    "base_score"
            }
        )
    )


    comparison_df = (

        rag_df

        .merge(

            base_df,

            on="question_id",

            how="inner",
        )
    )


    comparison_df[
        "delta_rag_minus_base"
    ] = (

        comparison_df[
            "rag_score"
        ]

        - comparison_df[
            "base_score"
        ]
    )


    rag_comparison_records.append(
        {

            "model_family":
                family_name,

            "questions":
                len(
                    comparison_df
                ),

            "rag_mean":
                comparison_df[
                    "rag_score"
                ].mean(),

            "base_mean":
                comparison_df[
                    "base_score"
                ].mean(),

            "mean_delta":
                comparison_df[
                    "delta_rag_minus_base"
                ].mean(),

            "rag_better":
                (
                    comparison_df[
                        "delta_rag_minus_base"
                    ] > 0
                ).sum(),

            "same_score":
                (
                    comparison_df[
                        "delta_rag_minus_base"
                    ] == 0
                ).sum(),

            "rag_worse":
                (
                    comparison_df[
                        "delta_rag_minus_base"
                    ] < 0
                ).sum(),
        }
    )


rag_vs_base_summary = (

    pd.DataFrame(
        rag_comparison_records
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "8. RAG vs NON-RAG — SAME MODEL FAMILY"
)

print(
    "=" * 100
)


display(
    rag_vs_base_summary.round(3)
)


# =============================================================================
# 11. RAG DELTA BY EVALUATION TYPE
# =============================================================================

rag_type_records = []


for family_name, pair in (
    rag_family_pairs.items()
):

    rag_df = (

        track1_judge_df[

            track1_judge_df[
                "model_name"
            ] == pair["rag"]

        ][
            [
                "question_id",
                "evaluation_type",
                "overall_score",
            ]
        ]

        .rename(
            columns={
                "overall_score":
                    "rag_score"
            }
        )
    )


    base_df = (

        track1_judge_df[

            track1_judge_df[
                "model_name"
            ] == pair["base"]

        ][
            [
                "question_id",
                "overall_score",
            ]
        ]

        .rename(
            columns={
                "overall_score":
                    "base_score"
            }
        )
    )


    comparison_df = (

        rag_df

        .merge(
            base_df,
            on="question_id",
            how="inner",
        )
    )


    comparison_df[
        "delta"
    ] = (

        comparison_df[
            "rag_score"
        ]

        - comparison_df[
            "base_score"
        ]
    )


    for evaluation_type, group in (

        comparison_df

        .groupby(
            "evaluation_type"
        )
    ):

        rag_type_records.append(
            {

                "model_family":
                    family_name,

                "evaluation_type":
                    evaluation_type,

                "questions":
                    len(group),

                "rag_mean":
                    group[
                        "rag_score"
                    ].mean(),

                "base_mean":
                    group[
                        "base_score"
                    ].mean(),

                "mean_delta":
                    group[
                        "delta"
                    ].mean(),

                "rag_better":
                    (
                        group[
                            "delta"
                        ] > 0
                    ).sum(),

                "same_score":
                    (
                        group[
                            "delta"
                        ] == 0
                    ).sum(),

                "rag_worse":
                    (
                        group[
                            "delta"
                        ] < 0
                    ).sum(),
            }
        )


rag_by_type_summary = (

    pd.DataFrame(
        rag_type_records
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "9. RAG IMPACT BY EVALUATION TYPE"
)

print(
    "=" * 100
)


display(
    rag_by_type_summary.round(3)
)


# =============================================================================
# 12. RESPONSE STATUS BY MODEL
# =============================================================================

response_status_summary = (

    track1_judge_df

    .groupby(
        [
            "model_name",
            "response_status",
        ]
    )

    .size()

    .unstack(
        fill_value=0
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "10. RESPONSE STATUS BY MODEL"
)

print(
    "=" * 100
)


display(
    response_status_summary
)


# =============================================================================
# 13. FINAL SUMMARY
# =============================================================================

best_model = (
    overall_ranking
    .iloc[0]
)


print(
    "\n"
    + "=" * 100
)

print(
    "TRACK 1 PERFORMANCE SUMMARY COMPLETE"
)

print(
    "=" * 100
)


print(
    f"\nTop model by mean overall score : "
    f"{best_model['model_name']}"
)

print(
    f"Mean overall score              : "
    f"{best_model['mean_overall']:.3f}"
)

print(
    f"Median overall score            : "
    f"{best_model['median_overall']:.3f}"
)


print(
    "\nGenerated analysis objects:"
)

print(
    "• overall_ranking"
)

print(
    "• dimension_summary"
)

print(
    "• evaluation_type_summary"
)

print(
    "• evaluation_type_pivot"
)

print(
    "• question_score_matrix"
)

print(
    "• question_winners_df"
)

print(
    "• win_count_df"
)

print(
    "• rag_vs_base_summary"
)

print(
    "• rag_by_type_summary"
)

print(
    "• response_status_summary"
)

print(
    "\nMODULE 5.1 COMPLETE"
)

print("=" * 100)

MODULE 5.1 — TRACK 1 MODEL PERFORMANCE SUMMARY

Responses : 100
Questions : 20
Systems   : 5

1. OVERALL MODEL RANKING


,rank,model_name,responses,mean_overall,median_overall,min_overall,max_overall,std_overall
0,1,Otel 2.0 Only,20,9.20,9.5,7,10,0.951
1,2,Gemma 4 Only,20,9.00,9.0,5,10,1.214
2,3,Gemma 4 + RAG,20,6.75,7.5,1,10,2.633
3,4,Essential AI Only,20,5.80,5.0,3,9,1.852
4,5,Essential AI + RAG,20,5.65,6.0,1,10,2.434



2. MEAN SCORE BY EVALUATION DIMENSION


,technical_accuracy,completeness,relevance,engineering_reasoning,practical_applicability,factual_reliability,overall_score
model_name,,,,,,,
Otel 2.0 Only,9.30,8.55,9.95,9.45,9.60,9.55,9.20
Gemma 4 Only,9.40,8.15,9.90,9.30,9.15,9.50,9.00
Gemma 4 + RAG,7.00,6.25,7.95,6.75,6.70,7.10,6.75
Essential AI Only,5.55,5.45,7.70,5.85,5.80,5.70,5.80
Essential AI + RAG,5.35,5.60,7.55,5.55,5.65,5.50,5.65



3. PERFORMANCE BY EVALUATION TYPE


...


9. RAG IMPACT BY EVALUATION TYPE


,model_family,evaluation_type,questions,rag_mean,base_mean,mean_delta,rag_better,same_score,rag_worse
0,Essential AI,CORPUS_GROUNDED,15,6.133,6.0,0.133,6,4,5
1,Essential AI,ENGINEERING_SYNTHESIS,5,4.200,5.2,-1.000,3,0,2
2,Gemma 4,CORPUS_GROUNDED,15,7.067,9.4,-2.333,0,5,10
3,Gemma 4,ENGINEERING_SYNTHESIS,5,5.800,7.8,-2.000,0,1,4



10. RESPONSE STATUS BY MODEL


response_status,NON_RESPONSIVE,SUBSTANTIVE
model_name,,
Essential AI + RAG,1,19
Essential AI Only,0,20
Gemma 4 + RAG,1,19
Gemma 4 Only,0,20
Otel 2.0 Only,0,20



TRACK 1 PERFORMANCE SUMMARY COMPLETE

Top model by mean overall score : Otel 2.0 Only
Mean overall score              : 9.200
Median overall score            : 9.500

Generated analysis objects:
• overall_ranking
• dimension_summary
• evaluation_type_summary
• evaluation_type_pivot
• question_score_matrix
• question_winners_df
• win_count_df
• rag_vs_base_summary
• rag_by_type_summary
• response_status_summary

MODULE 5.1 COMPLETE


# **Track 1 Failure Pattern and RAG Impact Analysis**

In [52]:
# =============================================================================
# MODULE 5.2 — TRACK 1 FAILURE PATTERN AND RAG IMPACT ANALYSIS
# =============================================================================
#
# PURPOSE
# -------
# Analyse failure behaviour and RAG impact across Track 1.
#
# ANALYSIS INCLUDED
# -----------------
# 1. Non-responsive cases
# 2. Technical-error frequency by model
# 3. Missing-point frequency by model
# 4. Average errors / omissions per response
# 5. Performance by category
# 6. Question-level RAG deltas
# 7. Largest RAG improvements
# 8. Largest RAG degradations
# 9. RAG outcomes by corpus relevance
# 10. RAG outcomes by production K=7 support
# 11. Candidate cases for later grounding analysis
#
# INPUT
# -----
# track1_judge_df
#
# =============================================================================


import pandas as pd
import numpy as np


# =============================================================================
# 1. VALIDATE REQUIRED COLUMNS
# =============================================================================

required_columns = {

    "question_id",
    "category",
    "model_name",
    "evaluation_type",
    "corpus_relevance",
    "production_k7_support",

    "response_status",

    "technical_accuracy",
    "completeness",
    "engineering_reasoning",
    "factual_reliability",
    "overall_score",

    "significant_technical_errors",
    "missing_important_points",
    "strengths",
    "judge_summary",
}


missing_columns = (
    required_columns
    - set(track1_judge_df.columns)
)


if missing_columns:

    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )


print("=" * 100)
print("MODULE 5.2 — TRACK 1 FAILURE PATTERN AND RAG IMPACT ANALYSIS")
print("=" * 100)

print(
    f"\nResponses : "
    f"{len(track1_judge_df)}"
)

print(
    f"Questions : "
    f"{track1_judge_df['question_id'].nunique()}"
)

print(
    f"Systems   : "
    f"{track1_judge_df['model_name'].nunique()}"
)


# =============================================================================
# 2. NORMALISE LIST FIELDS
# =============================================================================

def safe_list(value):

    if isinstance(value, list):

        return value

    if value is None:

        return []

    if isinstance(value, float) and pd.isna(value):

        return []

    return [str(value)]


analysis_df = (
    track1_judge_df
    .copy()
)


analysis_df[
    "significant_technical_errors"
] = (
    analysis_df[
        "significant_technical_errors"
    ].apply(
        safe_list
    )
)


analysis_df[
    "missing_important_points"
] = (
    analysis_df[
        "missing_important_points"
    ].apply(
        safe_list
    )
)


analysis_df[
    "strengths"
] = (
    analysis_df[
        "strengths"
    ].apply(
        safe_list
    )
)


# =============================================================================
# 3. FAILURE COUNTS PER RESPONSE
# =============================================================================

analysis_df[
    "technical_error_count"
] = (
    analysis_df[
        "significant_technical_errors"
    ].apply(
        len
    )
)


analysis_df[
    "missing_point_count"
] = (
    analysis_df[
        "missing_important_points"
    ].apply(
        len
    )
)


analysis_df[
    "has_technical_error"
] = (
    analysis_df[
        "technical_error_count"
    ] > 0
)


analysis_df[
    "has_missing_points"
] = (
    analysis_df[
        "missing_point_count"
    ] > 0
)


# =============================================================================
# 4. NON-RESPONSIVE CASES
# =============================================================================

non_responsive_df = (

    analysis_df[

        analysis_df[
            "response_status"
        ] == "NON_RESPONSIVE"

    ][
        [
            "question_id",
            "category",
            "evaluation_type",
            "model_name",
            "corpus_relevance",
            "production_k7_support",
            "overall_score",
            "judge_summary",
        ]
    ]

    .sort_values(
        [
            "question_id",
            "model_name",
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "1. NON-RESPONSIVE CASES"
)

print(
    "=" * 100
)


if len(non_responsive_df) > 0:

    display(
        non_responsive_df
    )

else:

    print(
        "No non-responsive responses found."
    )


# =============================================================================
# 5. FAILURE PATTERN SUMMARY BY MODEL
# =============================================================================

failure_summary = (

    analysis_df

    .groupby(
        "model_name"
    )

    .agg(

        responses=(
            "question_id",
            "count",
        ),

        non_responsive=(
            "response_status",
            lambda x:
                (
                    x == "NON_RESPONSIVE"
                ).sum(),
        ),

        responses_with_errors=(
            "has_technical_error",
            "sum",
        ),

        total_technical_errors=(
            "technical_error_count",
            "sum",
        ),

        avg_errors_per_response=(
            "technical_error_count",
            "mean",
        ),

        responses_with_omissions=(
            "has_missing_points",
            "sum",
        ),

        total_missing_points=(
            "missing_point_count",
            "sum",
        ),

        avg_missing_per_response=(
            "missing_point_count",
            "mean",
        ),

        mean_overall_score=(
            "overall_score",
            "mean",
        ),

        mean_factual_reliability=(
            "factual_reliability",
            "mean",
        ),
    )

    .sort_values(
        "mean_overall_score",
        ascending=False,
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "2. FAILURE PATTERN SUMMARY BY MODEL"
)

print(
    "=" * 100
)


display(
    failure_summary.round(3)
)


# =============================================================================
# 6. FAILURE RATES BY MODEL
# =============================================================================

failure_rate_summary = (

    failure_summary[
        [
            "model_name",
            "responses",
            "non_responsive",
            "responses_with_errors",
            "responses_with_omissions",
        ]
    ]
    .copy()
)


failure_rate_summary[
    "technical_error_rate_pct"
] = (

    failure_rate_summary[
        "responses_with_errors"
    ]

    / failure_rate_summary[
        "responses"
    ]

    * 100
)


failure_rate_summary[
    "omission_rate_pct"
] = (

    failure_rate_summary[
        "responses_with_omissions"
    ]

    / failure_rate_summary[
        "responses"
    ]

    * 100
)


failure_rate_summary[
    "non_response_rate_pct"
] = (

    failure_rate_summary[
        "non_responsive"
    ]

    / failure_rate_summary[
        "responses"
    ]

    * 100
)


print(
    "\n"
    + "=" * 100
)

print(
    "3. FAILURE RATES BY MODEL"
)

print(
    "=" * 100
)


display(
    failure_rate_summary.round(2)
)


# =============================================================================
# 7. PERFORMANCE BY QUESTION CATEGORY
# =============================================================================

category_summary = (

    analysis_df

    .groupby(
        [
            "category",
            "model_name",
        ]
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        mean_overall=(
            "overall_score",
            "mean",
        ),

        mean_technical_accuracy=(
            "technical_accuracy",
            "mean",
        ),

        mean_engineering_reasoning=(
            "engineering_reasoning",
            "mean",
        ),

        mean_factual_reliability=(
            "factual_reliability",
            "mean",
        ),
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "4. PERFORMANCE BY QUESTION CATEGORY"
)

print(
    "=" * 100
)


display(
    category_summary.round(3)
)


# =============================================================================
# 8. CATEGORY SCORE MATRIX
# =============================================================================

category_score_matrix = (

    analysis_df

    .pivot_table(

        index="category",

        columns="model_name",

        values="overall_score",

        aggfunc="mean",
    )

    .round(3)
)


print(
    "\n"
    + "=" * 100
)

print(
    "5. CATEGORY × MODEL MEAN OVERALL SCORE"
)

print(
    "=" * 100
)


display(
    category_score_matrix
)


# =============================================================================
# 9. BUILD QUESTION-LEVEL RAG COMPARISONS
# =============================================================================

rag_family_pairs = {

    "Essential AI": {
        "rag":
            "Essential AI + RAG",

        "base":
            "Essential AI Only",
    },

    "Gemma 4": {
        "rag":
            "Gemma 4 + RAG",

        "base":
            "Gemma 4 Only",
    },
}


rag_question_records = []


for family_name, pair in (
    rag_family_pairs.items()
):

    rag_rows = (

        analysis_df[

            analysis_df[
                "model_name"
            ] == pair["rag"]

        ][
            [
                "question_id",
                "category",
                "evaluation_type",
                "corpus_relevance",
                "production_k7_support",
                "overall_score",
                "technical_accuracy",
                "completeness",
                "engineering_reasoning",
                "factual_reliability",
                "response_status",
                "technical_error_count",
                "missing_point_count",
            ]
        ]

        .rename(
            columns={

                "overall_score":
                    "rag_overall",

                "technical_accuracy":
                    "rag_technical_accuracy",

                "completeness":
                    "rag_completeness",

                "engineering_reasoning":
                    "rag_engineering_reasoning",

                "factual_reliability":
                    "rag_factual_reliability",

                "response_status":
                    "rag_response_status",

                "technical_error_count":
                    "rag_error_count",

                "missing_point_count":
                    "rag_missing_count",
            }
        )
    )


    base_rows = (

        analysis_df[

            analysis_df[
                "model_name"
            ] == pair["base"]

        ][
            [
                "question_id",
                "overall_score",
                "technical_accuracy",
                "completeness",
                "engineering_reasoning",
                "factual_reliability",
                "response_status",
                "technical_error_count",
                "missing_point_count",
            ]
        ]

        .rename(
            columns={

                "overall_score":
                    "base_overall",

                "technical_accuracy":
                    "base_technical_accuracy",

                "completeness":
                    "base_completeness",

                "engineering_reasoning":
                    "base_engineering_reasoning",

                "factual_reliability":
                    "base_factual_reliability",

                "response_status":
                    "base_response_status",

                "technical_error_count":
                    "base_error_count",

                "missing_point_count":
                    "base_missing_count",
            }
        )
    )


    family_comparison = (

        rag_rows

        .merge(
            base_rows,
            on="question_id",
            how="inner",
        )
    )


    family_comparison[
        "model_family"
    ] = (
        family_name
    )


    family_comparison[
        "overall_delta"
    ] = (

        family_comparison[
            "rag_overall"
        ]

        - family_comparison[
            "base_overall"
        ]
    )


    family_comparison[
        "technical_accuracy_delta"
    ] = (

        family_comparison[
            "rag_technical_accuracy"
        ]

        - family_comparison[
            "base_technical_accuracy"
        ]
    )


    family_comparison[
        "completeness_delta"
    ] = (

        family_comparison[
            "rag_completeness"
        ]

        - family_comparison[
            "base_completeness"
        ]
    )


    family_comparison[
        "engineering_reasoning_delta"
    ] = (

        family_comparison[
            "rag_engineering_reasoning"
        ]

        - family_comparison[
            "base_engineering_reasoning"
        ]
    )


    family_comparison[
        "factual_reliability_delta"
    ] = (

        family_comparison[
            "rag_factual_reliability"
        ]

        - family_comparison[
            "base_factual_reliability"
        ]
    )


    family_comparison[
        "error_count_delta"
    ] = (

        family_comparison[
            "rag_error_count"
        ]

        - family_comparison[
            "base_error_count"
        ]
    )


    family_comparison[
        "missing_count_delta"
    ] = (

        family_comparison[
            "rag_missing_count"
        ]

        - family_comparison[
            "base_missing_count"
        ]
    )


    rag_question_records.append(
        family_comparison
    )


rag_question_impact_df = (

    pd.concat(
        rag_question_records,
        ignore_index=True,
    )
)


# =============================================================================
# 10. CLASSIFY RAG OUTCOME
# =============================================================================

rag_question_impact_df[
    "rag_outcome"
] = np.select(

    [

        rag_question_impact_df[
            "overall_delta"
        ] > 0,

        rag_question_impact_df[
            "overall_delta"
        ] < 0,
    ],

    [
        "RAG_BETTER",
        "RAG_WORSE",
    ],

    default=
        "SAME_SCORE",
)


# =============================================================================
# 11. FULL QUESTION-LEVEL RAG IMPACT
# =============================================================================

rag_question_impact_display = (

    rag_question_impact_df[
        [
            "model_family",
            "question_id",
            "category",
            "evaluation_type",
            "corpus_relevance",
            "production_k7_support",

            "base_overall",
            "rag_overall",
            "overall_delta",

            "technical_accuracy_delta",
            "completeness_delta",
            "engineering_reasoning_delta",
            "factual_reliability_delta",

            "rag_outcome",
        ]
    ]

    .sort_values(
        [
            "model_family",
            "question_id",
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "6. QUESTION-LEVEL RAG IMPACT"
)

print(
    "=" * 100
)


display(
    rag_question_impact_display
)


# =============================================================================
# 12. LARGEST RAG IMPROVEMENTS
# =============================================================================

largest_rag_improvements = (

    rag_question_impact_df[

        rag_question_impact_df[
            "overall_delta"
        ] > 0

    ][
        [
            "model_family",
            "question_id",
            "category",
            "evaluation_type",
            "corpus_relevance",
            "production_k7_support",
            "base_overall",
            "rag_overall",
            "overall_delta",
        ]
    ]

    .sort_values(
        "overall_delta",
        ascending=False,
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "7. LARGEST RAG IMPROVEMENTS"
)

print(
    "=" * 100
)


display(
    largest_rag_improvements
)


# =============================================================================
# 13. LARGEST RAG DEGRADATIONS
# =============================================================================

largest_rag_degradations = (

    rag_question_impact_df[

        rag_question_impact_df[
            "overall_delta"
        ] < 0

    ][
        [
            "model_family",
            "question_id",
            "category",
            "evaluation_type",
            "corpus_relevance",
            "production_k7_support",
            "base_overall",
            "rag_overall",
            "overall_delta",
            "technical_accuracy_delta",
            "completeness_delta",
            "engineering_reasoning_delta",
            "factual_reliability_delta",
            "rag_response_status",
            "base_response_status",
        ]
    ]

    .sort_values(
        "overall_delta",
        ascending=True,
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "8. LARGEST RAG DEGRADATIONS"
)

print(
    "=" * 100
)


display(
    largest_rag_degradations
)


# =============================================================================
# 14. RAG OUTCOME BY CORPUS RELEVANCE
# =============================================================================

rag_by_corpus_relevance = (

    rag_question_impact_df

    .groupby(
        [
            "model_family",
            "corpus_relevance",
        ]
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        mean_delta=(
            "overall_delta",
            "mean",
        ),

        rag_better=(
            "rag_outcome",
            lambda x:
                (
                    x == "RAG_BETTER"
                ).sum(),
        ),

        same_score=(
            "rag_outcome",
            lambda x:
                (
                    x == "SAME_SCORE"
                ).sum(),
        ),

        rag_worse=(
            "rag_outcome",
            lambda x:
                (
                    x == "RAG_WORSE"
                ).sum(),
        ),
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "9. RAG IMPACT BY CORPUS RELEVANCE"
)

print(
    "=" * 100
)


display(
    rag_by_corpus_relevance.round(3)
)


# =============================================================================
# 15. RAG OUTCOME BY PRODUCTION K=7 SUPPORT
# =============================================================================

rag_by_k7_support = (

    rag_question_impact_df

    .groupby(
        [
            "model_family",
            "production_k7_support",
        ]
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        mean_delta=(
            "overall_delta",
            "mean",
        ),

        rag_better=(
            "rag_outcome",
            lambda x:
                (
                    x == "RAG_BETTER"
                ).sum(),
        ),

        same_score=(
            "rag_outcome",
            lambda x:
                (
                    x == "SAME_SCORE"
                ).sum(),
        ),

        rag_worse=(
            "rag_outcome",
            lambda x:
                (
                    x == "RAG_WORSE"
                ).sum(),
        ),
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "10. RAG IMPACT BY PRODUCTION K=7 SUPPORT"
)

print(
    "=" * 100
)


display(
    rag_by_k7_support.round(3)
)


# =============================================================================
# 16. RAG FAILURE CASES WITH INCREASED TECHNICAL ERRORS
# =============================================================================

rag_error_increase_cases = (

    rag_question_impact_df[

        rag_question_impact_df[
            "error_count_delta"
        ] > 0

    ][
        [
            "model_family",
            "question_id",
            "category",
            "evaluation_type",
            "corpus_relevance",
            "production_k7_support",

            "base_overall",
            "rag_overall",
            "overall_delta",

            "base_error_count",
            "rag_error_count",
            "error_count_delta",
        ]
    ]

    .sort_values(
        [
            "error_count_delta",
            "overall_delta",
        ],
        ascending=[
            False,
            True,
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "11. CASES WHERE RAG INCREASED TECHNICAL ERROR COUNT"
)

print(
    "=" * 100
)


display(
    rag_error_increase_cases
)


# =============================================================================
# 17. CANDIDATES FOR DEDICATED RAG GROUNDING ANALYSIS
# =============================================================================
#
# Prioritise:
#
# - RAG score <= base score - 3
# - RAG became NON_RESPONSIVE
# - RAG technical accuracy fell >= 3 points
# - RAG factual reliability fell >= 3 points
#
# =============================================================================

rag_grounding_candidates = (

    rag_question_impact_df[

        (
            rag_question_impact_df[
                "overall_delta"
            ] <= -3
        )

        |

        (
            rag_question_impact_df[
                "rag_response_status"
            ] == "NON_RESPONSIVE"
        )

        |

        (
            rag_question_impact_df[
                "technical_accuracy_delta"
            ] <= -3
        )

        |

        (
            rag_question_impact_df[
                "factual_reliability_delta"
            ] <= -3
        )

    ][
        [
            "model_family",
            "question_id",
            "category",
            "evaluation_type",
            "corpus_relevance",
            "production_k7_support",

            "base_overall",
            "rag_overall",
            "overall_delta",

            "technical_accuracy_delta",
            "completeness_delta",
            "engineering_reasoning_delta",
            "factual_reliability_delta",

            "base_response_status",
            "rag_response_status",
        ]
    ]

    .sort_values(
        [
            "overall_delta",
            "technical_accuracy_delta",
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "12. PRIORITY CASES FOR RAG GROUNDING DIAGNOSTIC"
)

print(
    "=" * 100
)


display(
    rag_grounding_candidates
)


# =============================================================================
# 18. SUMMARY COUNTS
# =============================================================================

rag_outcome_summary = (

    rag_question_impact_df

    .groupby(
        [
            "model_family",
            "rag_outcome",
        ]
    )

    .size()

    .unstack(
        fill_value=0
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "13. RAG OUTCOME SUMMARY"
)

print(
    "=" * 100
)


display(
    rag_outcome_summary
)


# =============================================================================
# 19. FINAL MODULE SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TRACK 1 FAILURE PATTERN + RAG IMPACT ANALYSIS COMPLETE"
)

print(
    "=" * 100
)


print(
    f"\nNon-responsive cases       : "
    f"{len(non_responsive_df)}"
)

print(
    f"RAG comparison cases       : "
    f"{len(rag_question_impact_df)}"
)

print(
    f"RAG improvement cases      : "
    f"{(
        rag_question_impact_df['overall_delta'] > 0
    ).sum()}"
)

print(
    f"RAG neutral cases          : "
    f"{(
        rag_question_impact_df['overall_delta'] == 0
    ).sum()}"
)

print(
    f"RAG degradation cases      : "
    f"{(
        rag_question_impact_df['overall_delta'] < 0
    ).sum()}"
)

print(
    f"Priority grounding cases   : "
    f"{len(rag_grounding_candidates)}"
)


print(
    "\nGenerated analysis objects:"
)

print(
    "• analysis_df"
)

print(
    "• non_responsive_df"
)

print(
    "• failure_summary"
)

print(
    "• failure_rate_summary"
)

print(
    "• category_summary"
)

print(
    "• category_score_matrix"
)

print(
    "• rag_question_impact_df"
)

print(
    "• largest_rag_improvements"
)

print(
    "• largest_rag_degradations"
)

print(
    "• rag_by_corpus_relevance"
)

print(
    "• rag_by_k7_support"
)

print(
    "• rag_error_increase_cases"
)

print(
    "• rag_grounding_candidates"
)

print(
    "• rag_outcome_summary"
)

print(
    "\nMODULE 5.2 COMPLETE"
)

print("=" * 100)

MODULE 5.2 — TRACK 1 FAILURE PATTERN AND RAG IMPACT ANALYSIS

Responses : 100
Questions : 20
Systems   : 5

1. NON-RESPONSIVE CASES


,question_id,category,evaluation_type,model_name,corpus_relevance,production_k7_support,overall_score,judge_summary
0,Q15,5G Throughput Troubleshooting,CORPUS_GROUNDED,Gemma 4 + RAG,PARTIAL,YES,1,The response incorrectly states that the docum...
1,Q19,Network Failure Isolation,ENGINEERING_SYNTHESIS,Essential AI + RAG,PARTIAL,PARTIAL,1,"The response is non-responsive, claiming 'the ..."



2. FAILURE PATTERN SUMMARY BY MODEL


,model_name,responses,non_responsive,responses_with_errors,total_technical_errors,avg_errors_per_response,responses_with_omissions,total_missing_points,avg_missing_per_response,mean_overall_score,mean_factual_reliability
0,Otel 2.0 Only,20,0,3,7,0.35,11,44,2.20,9.20,9.55
1,Gemma 4 Only,20,0,1,5,0.25,13,68,3.40,9.00,9.50
2,Gemma 4 + RAG,20,1,9,23,1.15,17,105,5.25,6.75,7.10
3,Essential AI Only,20,0,15,56,2.80,20,116,5.80,5.80,5.70
4,Essential AI + RAG,20,1,14,49,2.45,19,103,5.15,5.65,5.50



3. FAILURE RATES BY MODEL


...


12. PRIORITY CASES FOR RAG GROUNDING DIAGNOSTIC


,model_family,question_id,category,evaluation_type,corpus_relevance,production_k7_support,base_overall,rag_overall,overall_delta,technical_accuracy_delta,completeness_delta,engineering_reasoning_delta,factual_reliability_delta,base_response_status,rag_response_status
0,Gemma 4,Q15,5G Throughput Troubleshooting,CORPUS_GROUNDED,PARTIAL,YES,9,1,-8,-9,-6,-9,-9,SUBSTANTIVE,NON_RESPONSIVE
1,Essential AI,Q19,Network Failure Isolation,ENGINEERING_SYNTHESIS,PARTIAL,PARTIAL,8,1,-7,-7,-6,-7,-9,SUBSTANTIVE,NON_RESPONSIVE
2,Gemma 4,Q14,5G UL/DL Trade-off,CORPUS_GROUNDED,PARTIAL-STRONG,YES,9,4,-5,-6,-2,-5,-6,SUBSTANTIVE,SUBSTANTIVE
3,Gemma 4,Q17,Cloud-Native Telecom,CORPUS_GROUNDED,PARTIAL-STRONG,YES,9,4,-5,-6,-3,-6,-6,SUBSTANTIVE,SUBSTANTIVE
4,Gemma 4,Q05,5G SA Procedures,CORPUS_GROUNDED,STRONG,YES,10,5,-5,-4,-5,-5,-4,SUBSTANTIVE,SUBSTANTIVE
5,Gemma 4,Q13,5G Capacity Planning,ENGINEERING_SYNTHESIS,PARTIAL,YES,8,5,-3,-5,-2,-4,-5,SUBSTANTIVE,SUBSTANTIVE
6,Gemma 4,Q18,RAN Capacity Expansion,ENGINEERING_SYNTHESIS,WEAK,PARTIAL,8,5,-3,-5,-2,-3,-6,SUBSTANTIVE,SUBSTANTIVE
7,Gemma 4,Q06,5G SA Procedures,CORPUS_GROUNDED,STRONG,YES,8,5,-3,-4,-2,-4,-3,SUBSTANTIVE,SUBSTANTIVE
8,Gemma 4,Q16,Open RAN Deployment,CORPUS_GROUNDED,STRONG,YES,9,6,-3,-4,-2,-3,-5,SUBSTANTIVE,SUBSTANTIVE
9,Essential AI,Q06,5G SA Procedures,CORPUS_GROUNDED,STRONG,YES,5,2,-3,-2,-2,-3,-2,SUBSTANTIVE,SUBSTANTIVE



13. RAG OUTCOME SUMMARY


rag_outcome,RAG_BETTER,RAG_WORSE,SAME_SCORE
model_family,,,
Essential AI,9,7,4
Gemma 4,0,14,6



TRACK 1 FAILURE PATTERN + RAG IMPACT ANALYSIS COMPLETE

Non-responsive cases       : 2
RAG comparison cases       : 40
RAG improvement cases      : 9
RAG neutral cases          : 10
RAG degradation cases      : 21
Priority grounding cases   : 13

Generated analysis objects:
• analysis_df
• non_responsive_df
• failure_summary
• failure_rate_summary
• category_summary
• category_score_matrix
• rag_question_impact_df
• largest_rag_improvements
• largest_rag_degradations
• rag_by_corpus_relevance
• rag_by_k7_support
• rag_error_increase_cases
• rag_grounding_candidates
• rag_outcome_summary

MODULE 5.2 COMPLETE


In [53]:
# =============================================================================
# MODULE 5.3 — BUILD PRIORITY RAG GROUNDING DIAGNOSTIC DATASET
# =============================================================================
#
# PURPOSE
# -------
# Build a diagnostic dataset for the priority RAG degradation cases identified
# in Module 5.2.
#
# IMPORTANT EXPERIMENTAL PRINCIPLE
# --------------------------------
# DO NOT rerun Retriever V1 here.
#
# We use the retrieval evidence stored in the ORIGINAL RAG response payloads.
#
# This preserves exactly what each RAG system saw during its original inference.
#
# DIAGNOSTIC UNIT
# ---------------
# For each priority case we retain:
#
#   1. Question + expected points
#   2. Corpus relevance / production K=7 support
#   3. Standalone response
#   4. RAG response
#   5. Standalone judge result
#   6. RAG judge result
#   7. Original production Top-7 retrieved evidence
#
# INPUT OBJECTS EXPECTED
# ----------------------
# rag_grounding_candidates
# track1_evaluation_df
# track1_judge_df
# loaded_responses
#
# =============================================================================


import json
import pandas as pd
import numpy as np


# =============================================================================
# 1. VALIDATE PRIORITY CASES
# =============================================================================

if "rag_grounding_candidates" not in globals():

    raise RuntimeError(
        "rag_grounding_candidates not found. "
        "Run Module 5.2 first."
    )


if len(rag_grounding_candidates) == 0:

    raise RuntimeError(
        "No priority RAG grounding cases were identified."
    )


print("=" * 100)
print("MODULE 5.3 — PRIORITY RAG GROUNDING DIAGNOSTIC DATASET")
print("=" * 100)

print(
    f"\nPriority cases : "
    f"{len(rag_grounding_candidates)}"
)


# =============================================================================
# 2. MODEL FAMILY MAPPING
# =============================================================================

RAG_MODEL_MAP = {

    "Essential AI":
        "Essential AI + RAG",

    "Gemma 4":
        "Gemma 4 + RAG",
}


BASE_MODEL_MAP = {

    "Essential AI":
        "Essential AI Only",

    "Gemma 4":
        "Gemma 4 Only",
}


# =============================================================================
# 3. IDENTIFY TRACK 1 KEY IN loaded_responses
# =============================================================================

if "loaded_responses" not in globals():

    raise RuntimeError(
        "loaded_responses not found."
    )


print(
    "\nAvailable loaded response keys:"
)

print(
    list(
        loaded_responses.keys()
    )
)


track1_key = None


# Try likely names first
for candidate_key in [

    "Track 1",
    "track1",
    "TRACK1",
    "Track1",
    "track_1",

]:

    if candidate_key in loaded_responses:

        track1_key = (
            candidate_key
        )

        break


# If not found, identify the entry containing our model names
if track1_key is None:

    for key, value in (
        loaded_responses.items()
    ):

        if not isinstance(
            value,
            dict,
        ):

            continue


        model_names = set(
            value.keys()
        )


        if {
            "Essential AI + RAG",
            "Gemma 4 + RAG",
        }.issubset(
            model_names
        ):

            track1_key = (
                key
            )

            break


if track1_key is None:

    raise RuntimeError(
        "Could not automatically identify "
        "Track 1 inside loaded_responses."
    )


print(
    f"\n✅ Track 1 response key identified: "
    f"{track1_key}"
)


track1_raw_responses = (
    loaded_responses[
        track1_key
    ]
)


# =============================================================================
# 4. HELPER — GET QUESTION ID FROM RAW RECORD
# =============================================================================

def get_raw_question_id(record):

    if not isinstance(
        record,
        dict,
    ):

        return None


    return (
        record.get(
            "question_id"
        )

        or record.get(
            "id"
        )
    )


# =============================================================================
# 5. HELPER — LOCATE RAW RAG RECORD
# =============================================================================

def find_raw_record(
    model_name,
    question_id,
):

    if model_name not in (
        track1_raw_responses
    ):

        raise KeyError(
            f"Model not found in Track 1 raw responses: "
            f"{model_name}"
        )


    records = (
        track1_raw_responses[
            model_name
        ]
    )


    for record in records:

        if (
            get_raw_question_id(
                record
            )
            == question_id
        ):

            return record


    raise KeyError(
        f"Raw record not found: "
        f"{question_id} | {model_name}"
    )


# =============================================================================
# 6. HELPER — NORMALISE RETRIEVAL PAYLOAD
# =============================================================================
#
# The two RAG pipelines may store their retrieval payloads slightly
# differently. This helper handles common structures without changing
# the original evidence.
#
# =============================================================================

def extract_retrieval_items(
    retrieval_payload,
):

    if retrieval_payload is None:

        return []


    # -------------------------------------------------------------------------
    # Retrieval already stored as a list
    # -------------------------------------------------------------------------

    if isinstance(
        retrieval_payload,
        list,
    ):

        return retrieval_payload


    # -------------------------------------------------------------------------
    # Retrieval stored as a dictionary
    # -------------------------------------------------------------------------

    if isinstance(
        retrieval_payload,
        dict,
    ):

        candidate_keys = [

            "results",
            "retrieved_chunks",
            "chunks",
            "documents",
            "items",
            "contexts",
            "retrieval_results",
        ]


        for key in candidate_keys:

            value = (
                retrieval_payload.get(
                    key
                )
            )


            if isinstance(
                value,
                list,
            ):

                return value


        # Some pipelines may encode ranks directly in a dictionary.
        # Preserve dictionary values only when they themselves look
        # like retrieval records.

        dict_values = list(
            retrieval_payload.values()
        )


        if (
            dict_values
            and all(
                isinstance(
                    item,
                    dict,
                )
                for item
                in dict_values
            )
        ):

            return dict_values


    return []


# =============================================================================
# 7. HELPER — NORMALISE ONE RETRIEVED CHUNK
# =============================================================================

def normalise_retrieved_chunk(
    item,
    fallback_rank,
):

    if isinstance(
        item,
        str,
    ):

        return {

            "rank":
                fallback_rank,

            "score":
                None,

            "vector_id":
                None,

            "chunk_id":
                None,

            "document_id":
                None,

            "source":
                None,

            "title":
                None,

            "path":
                None,

            "text":
                item,
        }


    if not isinstance(
        item,
        dict,
    ):

        return {

            "rank":
                fallback_rank,

            "score":
                None,

            "vector_id":
                None,

            "chunk_id":
                None,

            "document_id":
                None,

            "source":
                None,

            "title":
                None,

            "path":
                None,

            "text":
                str(
                    item
                ),
        }


    text = (

        item.get(
            "text"
        )

        or item.get(
            "page_content"
        )

        or item.get(
            "content"
        )

        or item.get(
            "chunk_text"
        )

        or ""
    )


    return {

        "rank":
            item.get(
                "rank",
                fallback_rank,
            ),

        "score":
            item.get(
                "score"
            ),

        "vector_id":
            item.get(
                "vector_id"
            ),

        "chunk_id":
            item.get(
                "chunk_id"
            ),

        "document_id":
            item.get(
                "document_id"
            ),

        "source":
            item.get(
                "source"
            ),

        "title":
            item.get(
                "title"
            ),

        "path":
            item.get(
                "path"
            ),

        "text":
            text,
    }


# =============================================================================
# 8. HELPER — FORMAT TOP-7 EVIDENCE FOR LATER JUDGE
# =============================================================================

def format_top7_evidence(
    chunks,
):

    if not chunks:

        return (
            "[NO RETRIEVAL EVIDENCE FOUND "
            "IN STORED RESPONSE PAYLOAD]"
        )


    blocks = []


    for chunk in chunks:

        block = f"""
RANK: {chunk['rank']}
SCORE: {chunk['score']}
SOURCE: {chunk['source']}
TITLE: {chunk['title']}
CHUNK ID: {chunk['chunk_id']}

TEXT:
{chunk['text']}
""".strip()


        blocks.append(
            block
        )


    return (
        "\n\n"
        + "=" * 80
        + "\n\n"
    ).join(
        blocks
    )


# =============================================================================
# 9. BUILD PRIORITY DIAGNOSTIC RECORDS
# =============================================================================

diagnostic_records = []


for _, candidate in (
    rag_grounding_candidates
    .iterrows()
):

    model_family = (
        candidate[
            "model_family"
        ]
    )


    question_id = (
        candidate[
            "question_id"
        ]
    )


    rag_model_name = (
        RAG_MODEL_MAP[
            model_family
        ]
    )


    base_model_name = (
        BASE_MODEL_MAP[
            model_family
        ]
    )


    # -------------------------------------------------------------------------
    # Original evaluation rows
    # -------------------------------------------------------------------------

    rag_eval_match = (

        track1_evaluation_df[

            (
                track1_evaluation_df[
                    "question_id"
                ] == question_id
            )

            &

            (
                track1_evaluation_df[
                    "model_name"
                ] == rag_model_name
            )

        ]
    )


    base_eval_match = (

        track1_evaluation_df[

            (
                track1_evaluation_df[
                    "question_id"
                ] == question_id
            )

            &

            (
                track1_evaluation_df[
                    "model_name"
                ] == base_model_name
            )

        ]
    )


    if len(
        rag_eval_match
    ) != 1:

        raise RuntimeError(
            f"Expected one RAG evaluation row for "
            f"{question_id} | {rag_model_name}, "
            f"found {len(rag_eval_match)}."
        )


    if len(
        base_eval_match
    ) != 1:

        raise RuntimeError(
            f"Expected one base evaluation row for "
            f"{question_id} | {base_model_name}, "
            f"found {len(base_eval_match)}."
        )


    rag_eval_row = (
        rag_eval_match.iloc[0]
    )


    base_eval_row = (
        base_eval_match.iloc[0]
    )


    # -------------------------------------------------------------------------
    # Judge rows
    # -------------------------------------------------------------------------

    rag_judge_match = (

        track1_judge_df[

            (
                track1_judge_df[
                    "question_id"
                ] == question_id
            )

            &

            (
                track1_judge_df[
                    "model_name"
                ] == rag_model_name
            )

        ]
    )


    base_judge_match = (

        track1_judge_df[

            (
                track1_judge_df[
                    "question_id"
                ] == question_id
            )

            &

            (
                track1_judge_df[
                    "model_name"
                ] == base_model_name
            )

        ]
    )


    if len(
        rag_judge_match
    ) != 1:

        raise RuntimeError(
            f"Expected one RAG judge row for "
            f"{question_id} | {rag_model_name}."
        )


    if len(
        base_judge_match
    ) != 1:

        raise RuntimeError(
            f"Expected one base judge row for "
            f"{question_id} | {base_model_name}."
        )


    rag_judge_row = (
        rag_judge_match.iloc[0]
    )


    base_judge_row = (
        base_judge_match.iloc[0]
    )


    # -------------------------------------------------------------------------
    # Retrieve ORIGINAL stored retrieval payload
    # -------------------------------------------------------------------------

    raw_rag_record = (
        find_raw_record(
            rag_model_name,
            question_id,
        )
    )


    retrieval_payload = (
        raw_rag_record.get(
            "retrieval"
        )
    )


    raw_retrieval_items = (
        extract_retrieval_items(
            retrieval_payload
        )
    )


    # Production RAG used K=7.
    raw_retrieval_items = (
        raw_retrieval_items[
            :7
        ]
    )


    top7_chunks = [

        normalise_retrieved_chunk(
            item,
            fallback_rank=rank,
        )

        for rank, item
        in enumerate(
            raw_retrieval_items,
            start=1,
        )
    ]


    # -------------------------------------------------------------------------
    # Expected points
    # -------------------------------------------------------------------------

    expected_points = (
        rag_eval_row[
            "expected_points"
        ]
    )


    # -------------------------------------------------------------------------
    # Store one complete diagnostic unit
    # -------------------------------------------------------------------------

    diagnostic_records.append(
        {

            "model_family":
                model_family,

            "question_id":
                question_id,

            "category":
                candidate[
                    "category"
                ],

            "evaluation_type":
                candidate[
                    "evaluation_type"
                ],

            "corpus_relevance":
                candidate[
                    "corpus_relevance"
                ],

            "production_k7_support":
                candidate[
                    "production_k7_support"
                ],

            "question":
                rag_eval_row[
                    "question"
                ],

            "expected_points":
                expected_points,

            # -------------------------------------------------------------
            # MODEL IDENTITIES
            # -------------------------------------------------------------

            "base_model_name":
                base_model_name,

            "rag_model_name":
                rag_model_name,

            # -------------------------------------------------------------
            # RESPONSES
            # -------------------------------------------------------------

            "base_response":
                base_eval_row[
                    "response"
                ],

            "rag_response":
                rag_eval_row[
                    "response"
                ],

            # -------------------------------------------------------------
            # MAIN JUDGE SCORES
            # -------------------------------------------------------------

            "base_overall_score":
                base_judge_row[
                    "overall_score"
                ],

            "rag_overall_score":
                rag_judge_row[
                    "overall_score"
                ],

            "overall_delta":
                (
                    rag_judge_row[
                        "overall_score"
                    ]
                    -
                    base_judge_row[
                        "overall_score"
                    ]
                ),

            "base_response_status":
                base_judge_row[
                    "response_status"
                ],

            "rag_response_status":
                rag_judge_row[
                    "response_status"
                ],

            # -------------------------------------------------------------
            # JUDGE CRITIQUE
            # -------------------------------------------------------------

            "base_technical_errors":
                base_judge_row[
                    "significant_technical_errors"
                ],

            "rag_technical_errors":
                rag_judge_row[
                    "significant_technical_errors"
                ],

            "base_missing_points":
                base_judge_row[
                    "missing_important_points"
                ],

            "rag_missing_points":
                rag_judge_row[
                    "missing_important_points"
                ],

            "base_judge_summary":
                base_judge_row[
                    "judge_summary"
                ],

            "rag_judge_summary":
                rag_judge_row[
                    "judge_summary"
                ],

            # -------------------------------------------------------------
            # ORIGINAL RETRIEVAL EVIDENCE
            # -------------------------------------------------------------

            "retrieved_chunk_count":
                len(
                    top7_chunks
                ),

            "top7_retrieved_chunks":
                top7_chunks,

            "top7_evidence_text":
                format_top7_evidence(
                    top7_chunks
                ),

            # -------------------------------------------------------------
            # RAW RETRIEVAL CONFIG / METADATA
            # -------------------------------------------------------------

            "raw_retrieval_payload":
                retrieval_payload,
        }
    )


# =============================================================================
# 10. BUILD DIAGNOSTIC DATAFRAME
# =============================================================================

rag_grounding_diagnostic_df = (
    pd.DataFrame(
        diagnostic_records
    )
)


rag_grounding_diagnostic_df = (

    rag_grounding_diagnostic_df

    .sort_values(
        [
            "overall_delta",
            "model_family",
            "question_id",
        ],
        ascending=[
            True,
            True,
            True,
        ]
    )

    .reset_index(
        drop=True
    )
)


# =============================================================================
# 11. VALIDATE RETRIEVAL EVIDENCE
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "RETRIEVAL EVIDENCE VALIDATION"
)

print(
    "=" * 100
)


retrieval_validation = (

    rag_grounding_diagnostic_df[
        [
            "model_family",
            "question_id",
            "corpus_relevance",
            "production_k7_support",
            "base_overall_score",
            "rag_overall_score",
            "overall_delta",
            "retrieved_chunk_count",
        ]
    ]
)


display(
    retrieval_validation
)


# =============================================================================
# 12. RETRIEVAL COUNT CHECKS
# =============================================================================

cases_with_evidence = (

    rag_grounding_diagnostic_df[
        "retrieved_chunk_count"
    ] > 0
).sum()


cases_with_full_top7 = (

    rag_grounding_diagnostic_df[
        "retrieved_chunk_count"
    ] == 7
).sum()


cases_without_evidence = (

    rag_grounding_diagnostic_df[
        "retrieved_chunk_count"
    ] == 0
).sum()


print(
    "\n"
    + "=" * 100
)

print(
    "DIAGNOSTIC DATASET SUMMARY"
)

print(
    "=" * 100
)


print(
    f"\nPriority cases              : "
    f"{len(rag_grounding_diagnostic_df)}"
)

print(
    f"Cases with retrieval data   : "
    f"{cases_with_evidence}"
)

print(
    f"Cases with complete Top-7   : "
    f"{cases_with_full_top7}"
)

print(
    f"Cases without retrieval     : "
    f"{cases_without_evidence}"
)


# =============================================================================
# 13. RETRIEVAL SOURCE SUMMARY
# =============================================================================

source_records = []


for _, row in (
    rag_grounding_diagnostic_df
    .iterrows()
):

    for chunk in (
        row[
            "top7_retrieved_chunks"
        ]
    ):

        source_records.append(
            {

                "model_family":
                    row[
                        "model_family"
                    ],

                "question_id":
                    row[
                        "question_id"
                    ],

                "rank":
                    chunk[
                        "rank"
                    ],

                "score":
                    chunk[
                        "score"
                    ],

                "source":
                    chunk[
                        "source"
                    ],

                "title":
                    chunk[
                        "title"
                    ],

                "chunk_id":
                    chunk[
                        "chunk_id"
                    ],
            }
        )


rag_grounding_retrieval_df = (
    pd.DataFrame(
        source_records
    )
)


if len(
    rag_grounding_retrieval_df
) > 0:

    print(
        "\n"
        + "=" * 100
    )

    print(
        "RETRIEVAL SOURCE PREVIEW"
    )

    print(
        "=" * 100
    )


    display(
        rag_grounding_retrieval_df
        .head(
            30
        )
    )


# =============================================================================
# 14. PREVIEW MOST SEVERE FAILURE
# =============================================================================

worst_case = (
    rag_grounding_diagnostic_df
    .iloc[0]
)


print(
    "\n"
    + "=" * 100
)

print(
    "MOST SEVERE RAG DEGRADATION — PREVIEW"
)

print(
    "=" * 100
)


print(
    f"\nModel family       : "
    f"{worst_case['model_family']}"
)

print(
    f"Question           : "
    f"{worst_case['question_id']}"
)

print(
    f"Category           : "
    f"{worst_case['category']}"
)

print(
    f"Evaluation type    : "
    f"{worst_case['evaluation_type']}"
)

print(
    f"Corpus relevance   : "
    f"{worst_case['corpus_relevance']}"
)

print(
    f"K7 support         : "
    f"{worst_case['production_k7_support']}"
)

print(
    f"Base score         : "
    f"{worst_case['base_overall_score']}"
)

print(
    f"RAG score          : "
    f"{worst_case['rag_overall_score']}"
)

print(
    f"RAG delta          : "
    f"{worst_case['overall_delta']}"
)

print(
    f"Retrieved chunks   : "
    f"{worst_case['retrieved_chunk_count']}"
)


print(
    "\nQUESTION"
)

print(
    "-" * 100
)

print(
    worst_case[
        "question"
    ]
)


print(
    "\nRAG RESPONSE"
)

print(
    "-" * 100
)

print(
    worst_case[
        "rag_response"
    ]
)


print(
    "\nRAG JUDGE SUMMARY"
)

print(
    "-" * 100
)

print(
    worst_case[
        "rag_judge_summary"
    ]
)


print(
    "\nTOP-7 RETRIEVAL EVIDENCE"
)

print(
    "-" * 100
)

print(
    worst_case[
        "top7_evidence_text"
    ]
)


# =============================================================================
# 15. FINAL VALIDATION
# =============================================================================

validation_checks = {

    "Priority cases preserved":
        len(
            rag_grounding_diagnostic_df
        )
        == len(
            rag_grounding_candidates
        ),

    "No duplicate family/question":
        not rag_grounding_diagnostic_df
        .duplicated(
            subset=[
                "model_family",
                "question_id",
            ]
        )
        .any(),

    "All base responses present":
        rag_grounding_diagnostic_df[
            "base_response"
        ]
        .notna()
        .all(),

    "All RAG responses present":
        rag_grounding_diagnostic_df[
            "rag_response"
        ]
        .notna()
        .all(),

    "All judge summaries present":
        (
            rag_grounding_diagnostic_df[
                "base_judge_summary"
            ]
            .notna()
            .all()

            and

            rag_grounding_diagnostic_df[
                "rag_judge_summary"
            ]
            .notna()
            .all()
        ),
}


print(
    "\n"
    + "=" * 100
)

print(
    "MODULE 5.3 VALIDATION"
)

print(
    "=" * 100
)


for check_name, passed in (
    validation_checks.items()
):

    status = (
        "✅ PASS"
        if passed
        else "❌ FAIL"
    )

    print(
        f"{check_name:<40} : "
        f"{status}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 5.3 diagnostic dataset "
        "failed validation."
    )


print(
    "\nGenerated objects:"
)

print(
    "• rag_grounding_diagnostic_df"
)

print(
    "• rag_grounding_retrieval_df"
)


print(
    "\nMODULE 5.3 COMPLETE"
)

print("=" * 100)

MODULE 5.3 — PRIORITY RAG GROUNDING DIAGNOSTIC DATASET

Priority cases : 13

Available loaded response keys:
['Track 1', 'Track 2']

✅ Track 1 response key identified: Track 1

RETRIEVAL EVIDENCE VALIDATION


,model_family,question_id,corpus_relevance,production_k7_support,base_overall_score,rag_overall_score,overall_delta,retrieved_chunk_count
0,Gemma 4,Q15,PARTIAL,YES,9,1,-8,7
1,Essential AI,Q19,PARTIAL,PARTIAL,8,1,-7,7
2,Gemma 4,Q05,STRONG,YES,10,5,-5,7
3,Gemma 4,Q14,PARTIAL-STRONG,YES,9,4,-5,7
4,Gemma 4,Q17,PARTIAL-STRONG,YES,9,4,-5,7
5,Essential AI,Q06,STRONG,YES,5,2,-3,7
6,Essential AI,Q11,WEAK-PARTIAL,PARTIAL,5,2,-3,7
7,Gemma 4,Q06,STRONG,YES,8,5,-3,7
8,Gemma 4,Q13,PARTIAL,YES,8,5,-3,7
9,Gemma 4,Q16,STRONG,YES,9,6,-3,7



DIAGNOSTIC DATASET SUMMARY

Priority cases              : 13
Cases with retrieval data   : 13
Cases with complete Top-7   : 13
Cases without retrieval     : 0

RETRIEVAL SOURCE PREVIEW


,model_family,question_id,rank,score,source,title,chunk_id
0,Gemma 4,Q15,1,0.725836,standards,raw,standards/oran/marked/TIFG/O-RAN.TIFG.TS.E2E-T...
1,Gemma 4,Q15,2,0.718101,standards,raw,standards/oran/marked/TIFG/O-RAN.TIFG.TS.E2E-T...
2,Gemma 4,Q15,3,0.713253,standards,raw,standards/oran/marked/TIFG/O-RAN.TIFG.TS.E2E-T...
3,Gemma 4,Q15,4,0.710835,standards,raw,standards/oran/marked/TIFG/O-RAN.TIFG.TS.E2E-T...
4,Gemma 4,Q15,5,0.707226,standards,raw,standards/oran/marked/TIFG/O-RAN.TIFG.TS.E2E-T...
5,Gemma 4,Q15,6,0.705833,standards,raw,standards/etsi/marked/TR/tr/tr_138913v190000p/...
6,Gemma 4,Q15,7,0.704318,standards,raw,standards/oran/marked/TIFG/O-RAN.TIFG.TS.E2E-T...
7,Essential AI,Q19,1,0.691321,standards,23700-60-i00,standards/3gpp_rel18/original/23700-60-i00.doc...
8,Essential AI,Q19,2,0.687370,standards,raw,standards/etsi/marked/TS/ts/ts_123527v190500p/...
9,Essential AI,Q19,3,0.680232,open_source,20260211,open_source/free5gc_docs/docs/blog/20260211/20...



MOST SEVERE RAG DEGRADATION — PREVIEW

Model family       : Gemma 4
Question           : Q15
Category           : 5G Throughput Troubleshooting
Evaluation type    : CORPUS_GROUNDED
Corpus relevance   : PARTIAL
K7 support         : YES
Base score         : 9
RAG score          : 1
RAG delta          : -8
Retrieved chunks   : 7

QUESTION
----------------------------------------------------------------------------------------------------
A 5G SA UE has excellent RSRP and SINR but achieves only 300 Mbps DL when the expected peak throughput is above 1 Gbps. Develop a systematic troubleshooting methodology covering UE capability, NR configuration, scheduler, MIMO, PRBs, transport, UPF and 5GC. Explain how you would isolate the bottleneck.

RAG RESPONSE
----------------------------------------------------------------------------------------------------
The provided documentation does not contain information regarding a systematic troubleshooting methodology for 5G SA throughput issues, nor d

In [58]:
# =============================================================================
# MODULE 5.4 — PRIORITY RAG FAILURE MECHANISM CLASSIFICATION
#                CONTEXT-BOUNDED DIAGNOSTIC VERSION
# =============================================================================
#
# PURPOSE
# -------
# Diagnose WHY the 13 priority RAG cases degraded relative to their
# corresponding standalone model.
#
# IMPORTANT EXPERIMENTAL PRINCIPLE
# --------------------------------
# The ORIGINAL production Top-7 evidence remains untouched inside:
#
#     rag_grounding_diagnostic_df
#
# This module only creates a CONTEXT-BOUNDED diagnostic representation for
# the independent judge because the judge server has:
#
#     max_model_len = 8192
#
# We preserve:
#
#     - all seven retrieval ranks
#     - rank
#     - similarity score
#     - source
#     - title
#     - chunk ID
#     - bounded excerpt from every chunk
#
# Therefore this is NOT rerunning retrieval and NOT modifying the original
# production evidence.
#
# =============================================================================


import json
import pandas as pd


# =============================================================================
# 1. CONTEXT-SAFE SETTINGS
# =============================================================================

RAG_DIAGNOSTIC_MAX_TOKENS = 1400

MAX_RETRIEVAL_CHARS_PER_CHUNK = 900

MAX_BASE_RESPONSE_CHARS = 3500

MAX_RAG_RESPONSE_CHARS = 3500

MAX_EXPECTED_POINTS_CHARS = 2500


print("=" * 100)
print("MODULE 5.4 — CONTEXT-BOUNDED RAG FAILURE DIAGNOSTIC")
print("=" * 100)

print(
    f"\nDiagnostic output tokens       : "
    f"{RAG_DIAGNOSTIC_MAX_TOKENS}"
)

print(
    f"Retrieval chars per chunk      : "
    f"{MAX_RETRIEVAL_CHARS_PER_CHUNK}"
)

print(
    f"Standalone response max chars  : "
    f"{MAX_BASE_RESPONSE_CHARS}"
)

print(
    f"RAG response max chars         : "
    f"{MAX_RAG_RESPONSE_CHARS}"
)


# =============================================================================
# 2. VALID FAILURE TAXONOMY
# =============================================================================

VALID_FAILURE_MECHANISMS = {

    "RETRIEVAL_INSUFFICIENT",

    "RETRIEVAL_PARTIALLY_SUFFICIENT",

    "RETRIEVAL_MISLEADING",

    "GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE",

    "STRICT_GROUNDING_ABSTENTION",

    "CONTEXT_OVERCONSTRAINT",

    "EVIDENCE_SYNTHESIS_FAILURE",

    "BASE_MODEL_SUPPRESSED_BY_CONTEXT",

    "NO_CLEAR_RAG_FAILURE",
}


VALID_EVIDENCE_SUFFICIENCY = {

    "SUFFICIENT",

    "PARTIAL",

    "INSUFFICIENT",
}


# =============================================================================
# 3. COMPACT DIAGNOSTIC SYSTEM PROMPT
# =============================================================================

RAG_DIAGNOSTIC_SYSTEM_PROMPT = """
You are a senior telecom RAG diagnostic evaluator.

Determine WHY the RAG answer performed worse than the corresponding
standalone answer.

This is NOT another response-quality scoring task.

Distinguish:

1. retrieval quality,
2. evidence sufficiency,
3. grounding/context constraints,
4. evidence synthesis,
5. generation behaviour.

EVIDENCE SUFFICIENCY

SUFFICIENT:
The retrieved evidence contains enough information for a strong answer to
the core task.

PARTIAL:
Useful evidence exists, but important required elements are missing.

INSUFFICIENT:
The retrieved evidence does not contain enough information for an adequate
answer.

FAILURE MECHANISMS

RETRIEVAL_INSUFFICIENT
The retrieved set lacks enough relevant evidence.

RETRIEVAL_PARTIALLY_SUFFICIENT
Useful evidence exists but important task elements are absent.

RETRIEVAL_MISLEADING
Retrieved material materially steers the answer toward an incorrect conclusion.

GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE
Useful or sufficient evidence exists but the generator ignores, misuses,
contradicts or fails to exploit it.

STRICT_GROUNDING_ABSTENTION
The RAG model refuses despite usable retrieved evidence.

CONTEXT_OVERCONSTRAINT
The supplied context restricts the response and suppresses otherwise valid
reasoning.

EVIDENCE_SYNTHESIS_FAILURE
Relevant evidence exists across chunks but is not successfully combined.

BASE_MODEL_SUPPRESSED_BY_CONTEXT
The standalone model shows materially stronger valid knowledge or reasoning
that is lost after retrieval context is introduced.

NO_CLEAR_RAG_FAILURE
No specific mechanism can be established confidently.

RULES

- Judge semantic evidence, not similarity score alone.
- Poor RAG output does not automatically mean poor retrieval.
- Distinguish PARTIAL evidence from INSUFFICIENT evidence.
- Select exactly one primary mechanism.
- Select zero to two secondary mechanisms.
- Keep list items and rationale concise.
- Return exactly one valid JSON object.
- No markdown outside JSON.
""".strip()


# =============================================================================
# 4. OUTPUT SCHEMA
# =============================================================================

RAG_DIAGNOSTIC_SCHEMA = {

    "model_family":
        "Gemma 4",

    "question_id":
        "Q15",

    "evidence_sufficiency":
        "PARTIAL",

    "retrieval_at_fault":
        False,

    "generation_or_prompting_at_fault":
        True,

    "primary_failure_mechanism":
        "STRICT_GROUNDING_ABSTENTION",

    "secondary_failure_mechanisms": [
        "GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE"
    ],

    "retrieval_strengths": [],

    "retrieval_gaps": [],

    "rag_response_failures": [],

    "standalone_advantages": [],

    "diagnostic_rationale":
        "",
}


# =============================================================================
# 5. GENERIC TEXT BOUNDING HELPER
# =============================================================================

def bound_text(
    value,
    max_chars,
):

    if value is None:

        return ""


    text = str(
        value
    )


    if len(
        text
    ) <= max_chars:

        return text


    return (
        text[
            :max_chars
        ]
        + "\n...[TRUNCATED FOR DIAGNOSTIC CONTEXT]..."
    )


# =============================================================================
# 6. FORMAT EXPECTED POINTS
# =============================================================================

def format_expected_points(
    expected_points,
):

    if isinstance(
        expected_points,
        list,
    ):

        text = "\n".join(

            f"- {point}"

            for point
            in expected_points
        )

    else:

        text = str(
            expected_points
        )


    return bound_text(
        text,
        MAX_EXPECTED_POINTS_CHARS,
    )


# =============================================================================
# 7. BUILD CONTEXT-BOUNDED TOP-7 EVIDENCE
# =============================================================================
#
# IMPORTANT:
#
# We use the structured Top-7 chunks stored in Module 5.3 rather than
# modifying the original evidence.
#
# ALL SEVEN retrieval ranks remain represented.
#
# =============================================================================

def build_bounded_top7_evidence(
    chunks,
):

    if not isinstance(
        chunks,
        list,
    ):

        return (
            "[NO STRUCTURED RETRIEVAL EVIDENCE]"
        )


    evidence_blocks = []


    for fallback_rank, chunk in enumerate(
        chunks,
        start=1,
    ):

        if not isinstance(
            chunk,
            dict,
        ):

            evidence_blocks.append(
                f"""
RANK: {fallback_rank}

TEXT:
{bound_text(
    chunk,
    MAX_RETRIEVAL_CHARS_PER_CHUNK,
)}
""".strip()
            )

            continue


        rank = (
            chunk.get(
                "rank",
                fallback_rank,
            )
        )


        score = (
            chunk.get(
                "score"
            )
        )


        source = (
            chunk.get(
                "source"
            )
        )


        title = (
            chunk.get(
                "title"
            )
        )


        chunk_id = (
            chunk.get(
                "chunk_id"
            )
        )


        text = (
            chunk.get(
                "text",
                ""
            )
        )


        bounded_text = (
            bound_text(
                text,
                MAX_RETRIEVAL_CHARS_PER_CHUNK,
            )
        )


        block = f"""
RANK: {rank}
SCORE: {score}
SOURCE: {source}
TITLE: {title}
CHUNK ID: {chunk_id}

TEXT:
{bounded_text}
""".strip()


        evidence_blocks.append(
            block
        )


    return (
        "\n\n"
        + "-" * 70
        + "\n\n"
    ).join(
        evidence_blocks
    )


# =============================================================================
# 8. DIAGNOSTIC PROMPT BUILDER
# =============================================================================

def build_rag_diagnostic_prompt(
    row,
):

    expected_points_text = (
        format_expected_points(
            row[
                "expected_points"
            ]
        )
    )


    base_response_text = (
        bound_text(
            row[
                "base_response"
            ],
            MAX_BASE_RESPONSE_CHARS,
        )
    )


    rag_response_text = (
        bound_text(
            row[
                "rag_response"
            ],
            MAX_RAG_RESPONSE_CHARS,
        )
    )


    bounded_top7 = (
        build_bounded_top7_evidence(
            row[
                "top7_retrieved_chunks"
            ]
        )
    )


    return f"""
RAG FAILURE DIAGNOSTIC

CASE METADATA

Model family: {row['model_family']}
Question ID: {row['question_id']}
Category: {row['category']}
Evaluation type: {row['evaluation_type']}
Corpus relevance: {row['corpus_relevance']}
Production K7 support: {row['production_k7_support']}

Standalone score: {row['base_overall_score']}
RAG score: {row['rag_overall_score']}
Delta: {row['overall_delta']}

Standalone status: {row['base_response_status']}
RAG status: {row['rag_response_status']}


QUESTION

{row['question']}


EXPECTED TECHNICAL POINTS

{expected_points_text}


STANDALONE RESPONSE

{base_response_text}


RAG RESPONSE

{rag_response_text}


ORIGINAL PRODUCTION TOP-7 EVIDENCE
BOUNDED EXCERPT FROM EACH OF THE 7 ORIGINAL CHUNKS

{bounded_top7}


DIAGNOSTIC TASK

Determine:

- evidence sufficiency
- whether retrieval is materially at fault
- whether generation/prompting is materially at fault
- one primary failure mechanism
- up to two secondary mechanisms
- retrieval strengths
- retrieval gaps
- RAG response failures
- standalone advantages
- concise causal rationale

Return exactly this JSON structure:

{json.dumps(
    RAG_DIAGNOSTIC_SCHEMA,
    indent=2,
)}
""".strip()


# =============================================================================
# 9. VALIDATOR
# =============================================================================

def validate_rag_diagnostic_result(
    result,
    expected_model_family,
    expected_question_id,
):

    required_fields = {

        "model_family",
        "question_id",

        "evidence_sufficiency",

        "retrieval_at_fault",
        "generation_or_prompting_at_fault",

        "primary_failure_mechanism",
        "secondary_failure_mechanisms",

        "retrieval_strengths",
        "retrieval_gaps",

        "rag_response_failures",
        "standalone_advantages",

        "diagnostic_rationale",
    }


    missing = (

        required_fields
        -
        set(
            result.keys()
        )
    )


    if missing:

        raise ValueError(
            f"Missing fields: "
            f"{sorted(missing)}"
        )


    if (
        result[
            "model_family"
        ]
        != expected_model_family
    ):

        raise ValueError(
            "Model family mismatch."
        )


    if (
        result[
            "question_id"
        ]
        != expected_question_id
    ):

        raise ValueError(
            "Question ID mismatch."
        )


    if (
        result[
            "evidence_sufficiency"
        ]
        not in
        VALID_EVIDENCE_SUFFICIENCY
    ):

        raise ValueError(
            "Invalid evidence_sufficiency."
        )


    primary = (
        result[
            "primary_failure_mechanism"
        ]
    )


    if (
        primary
        not in
        VALID_FAILURE_MECHANISMS
    ):

        raise ValueError(
            f"Invalid primary mechanism: "
            f"{primary}"
        )


    secondary = (
        result[
            "secondary_failure_mechanisms"
        ]
    )


    if not isinstance(
        secondary,
        list,
    ):

        raise ValueError(
            "secondary_failure_mechanisms "
            "must be a list."
        )


    if len(
        secondary
    ) > 2:

        raise ValueError(
            "Maximum two secondary mechanisms allowed."
        )


    for mechanism in secondary:

        if (
            mechanism
            not in
            VALID_FAILURE_MECHANISMS
        ):

            raise ValueError(
                f"Invalid secondary mechanism: "
                f"{mechanism}"
            )


    for field in [

        "retrieval_at_fault",
        "generation_or_prompting_at_fault",

    ]:

        if not isinstance(
            result[
                field
            ],
            bool,
        ):

            raise ValueError(
                f"{field} must be boolean."
            )


    for field in [

        "retrieval_strengths",
        "retrieval_gaps",
        "rag_response_failures",
        "standalone_advantages",

    ]:

        if not isinstance(
            result[
                field
            ],
            list,
        ):

            raise ValueError(
                f"{field} must be a list."
            )


    if not isinstance(
        result[
            "diagnostic_rationale"
        ],
        str,
    ):

        raise ValueError(
            "diagnostic_rationale "
            "must be a string."
        )


    return True


# =============================================================================
# 10. INPUT VALIDATION
# =============================================================================

if (
    "rag_grounding_diagnostic_df"
    not in globals()
):

    raise RuntimeError(
        "rag_grounding_diagnostic_df "
        "not found. Run Module 5.3 first."
    )


if len(
    rag_grounding_diagnostic_df
) != 13:

    raise RuntimeError(
        f"Expected 13 priority cases, "
        f"found "
        f"{len(rag_grounding_diagnostic_df)}."
    )


# =============================================================================
# 11. PROMPT SIZE CHECK
# =============================================================================

prompt_size_records = []


for _, row in (
    rag_grounding_diagnostic_df
    .iterrows()
):

    diagnostic_prompt = (
        build_rag_diagnostic_prompt(
            row
        )
    )


    prompt_size_records.append(
        {

            "model_family":
                row[
                    "model_family"
                ],

            "question_id":
                row[
                    "question_id"
                ],

            "prompt_characters":
                len(
                    diagnostic_prompt
                ),

            "retrieved_chunks":
                len(
                    row[
                        "top7_retrieved_chunks"
                    ]
                ),
        }
    )


rag_diagnostic_prompt_sizes = (
    pd.DataFrame(
        prompt_size_records
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "CONTEXT-BOUNDED DIAGNOSTIC PROMPT SIZE"
)

print(
    "=" * 100
)


display(
    rag_diagnostic_prompt_sizes
)


print(
    "\nLargest diagnostic prompt:"
)

print(
    f"{rag_diagnostic_prompt_sizes['prompt_characters'].max():,} characters"
)


# =============================================================================
# 12. RUN PRIORITY DIAGNOSTICS
# =============================================================================

rag_failure_diagnostic_results = []


print(
    "\n"
    + "=" * 100
)

print(
    "RAG FAILURE MECHANISM CLASSIFICATION"
)

print(
    "=" * 100
)


print(
    f"\nCases to diagnose : "
    f"{len(rag_grounding_diagnostic_df)}"
)


for idx, row in (
    rag_grounding_diagnostic_df
    .iterrows()
):

    print(
        f"\n"
        f"[{idx + 1:02d}/"
        f"{len(rag_grounding_diagnostic_df):02d}] "
        f"{row['model_family']} | "
        f"{row['question_id']} | "
        f"Δ {row['overall_delta']}"
    )


    diagnostic_prompt = (
        build_rag_diagnostic_prompt(
            row
        )
    )


    response = (

        client
        .chat
        .completions
        .create(

            model=
                VLLM_MODEL,

            messages=[
                {
                    "role":
                        "system",

                    "content":
                        RAG_DIAGNOSTIC_SYSTEM_PROMPT,
                },
                {
                    "role":
                        "user",

                    "content":
                        diagnostic_prompt,
                },
            ],

            temperature=
                0.0,

            max_tokens=
                RAG_DIAGNOSTIC_MAX_TOKENS,
        )
    )


    raw_text = (

        response
        .choices[0]
        .message
        .content
        .strip()
    )


    try:

        diagnostic_result = (
            json.loads(
                raw_text
            )
        )


    except json.JSONDecodeError as exc:

        print(
            "\n❌ INVALID DIAGNOSTIC JSON"
        )

        print(
            raw_text
        )


        raise RuntimeError(
            f"Diagnostic JSON failed for "
            f"{row['model_family']} | "
            f"{row['question_id']}"
        ) from exc


    validate_rag_diagnostic_result(

        result=
            diagnostic_result,

        expected_model_family=
            row[
                "model_family"
            ],

        expected_question_id=
            row[
                "question_id"
            ],
    )


    rag_failure_diagnostic_results.append(
        {

            "model_family":
                row[
                    "model_family"
                ],

            "question_id":
                row[
                    "question_id"
                ],

            "category":
                row[
                    "category"
                ],

            "evaluation_type":
                row[
                    "evaluation_type"
                ],

            "corpus_relevance":
                row[
                    "corpus_relevance"
                ],

            "production_k7_support":
                row[
                    "production_k7_support"
                ],

            "base_overall_score":
                row[
                    "base_overall_score"
                ],

            "rag_overall_score":
                row[
                    "rag_overall_score"
                ],

            "overall_delta":
                row[
                    "overall_delta"
                ],

            "evidence_sufficiency":
                diagnostic_result[
                    "evidence_sufficiency"
                ],

            "retrieval_at_fault":
                diagnostic_result[
                    "retrieval_at_fault"
                ],

            "generation_or_prompting_at_fault":
                diagnostic_result[
                    "generation_or_prompting_at_fault"
                ],

            "primary_failure_mechanism":
                diagnostic_result[
                    "primary_failure_mechanism"
                ],

            "secondary_failure_mechanisms":
                diagnostic_result[
                    "secondary_failure_mechanisms"
                ],

            "retrieval_strengths":
                diagnostic_result[
                    "retrieval_strengths"
                ],

            "retrieval_gaps":
                diagnostic_result[
                    "retrieval_gaps"
                ],

            "rag_response_failures":
                diagnostic_result[
                    "rag_response_failures"
                ],

            "standalone_advantages":
                diagnostic_result[
                    "standalone_advantages"
                ],

            "diagnostic_rationale":
                diagnostic_result[
                    "diagnostic_rationale"
                ],

            "raw_diagnostic_json":
                diagnostic_result,
        }
    )


    print(
        f"    ✅ "
        f"{diagnostic_result['primary_failure_mechanism']} "
        f"| Evidence: "
        f"{diagnostic_result['evidence_sufficiency']} "
        f"| Retrieval fault: "
        f"{diagnostic_result['retrieval_at_fault']} "
        f"| Generation/prompt fault: "
        f"{diagnostic_result['generation_or_prompting_at_fault']}"
    )


# =============================================================================
# 13. BUILD RESULT DATAFRAME
# =============================================================================

rag_failure_diagnostic_df = (

    pd.DataFrame(
        rag_failure_diagnostic_results
    )

    .sort_values(
        [
            "overall_delta",
            "model_family",
            "question_id",
        ]
    )

    .reset_index(
        drop=True
    )
)


# =============================================================================
# 14. CORE DIAGNOSTIC VIEW
# =============================================================================

core_diagnostic_view = (

    rag_failure_diagnostic_df[
        [
            "model_family",
            "question_id",
            "evaluation_type",
            "corpus_relevance",
            "production_k7_support",

            "base_overall_score",
            "rag_overall_score",
            "overall_delta",

            "evidence_sufficiency",

            "retrieval_at_fault",
            "generation_or_prompting_at_fault",

            "primary_failure_mechanism",
            "secondary_failure_mechanisms",
        ]
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "1. PRIORITY RAG FAILURE CLASSIFICATION"
)

print(
    "=" * 100
)


display(
    core_diagnostic_view
)


# =============================================================================
# 15. PRIMARY FAILURE MECHANISM SUMMARY
# =============================================================================

primary_mechanism_summary = (

    rag_failure_diagnostic_df

    .groupby(
        [
            "model_family",
            "primary_failure_mechanism",
        ]
    )

    .size()

    .reset_index(
        name=
            "cases"
    )

    .sort_values(
        [
            "model_family",
            "cases",
        ],
        ascending=[
            True,
            False,
        ]
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "2. PRIMARY FAILURE MECHANISM SUMMARY"
)

print(
    "=" * 100
)


display(
    primary_mechanism_summary
)


# =============================================================================
# 16. EVIDENCE SUFFICIENCY SUMMARY
# =============================================================================

evidence_sufficiency_summary = (

    rag_failure_diagnostic_df

    .groupby(
        [
            "model_family",
            "evidence_sufficiency",
        ]
    )

    .size()

    .reset_index(
        name=
            "cases"
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "3. EVIDENCE SUFFICIENCY SUMMARY"
)

print(
    "=" * 100
)


display(
    evidence_sufficiency_summary
)


# =============================================================================
# 17. RETRIEVAL vs GENERATION/PROMPTING FAULT
# =============================================================================

fault_attribution_summary = (

    rag_failure_diagnostic_df

    .groupby(
        "model_family"
    )

    .agg(

        cases=(
            "question_id",
            "count",
        ),

        retrieval_fault_cases=(
            "retrieval_at_fault",
            "sum",
        ),

        generation_prompt_fault_cases=(
            "generation_or_prompting_at_fault",
            "sum",
        ),
    )

    .reset_index()
)


fault_attribution_summary[
    "retrieval_fault_pct"
] = (

    fault_attribution_summary[
        "retrieval_fault_cases"
    ]

    / fault_attribution_summary[
        "cases"
    ]

    * 100
)


fault_attribution_summary[
    "generation_prompt_fault_pct"
] = (

    fault_attribution_summary[
        "generation_prompt_fault_cases"
    ]

    / fault_attribution_summary[
        "cases"
    ]

    * 100
)


print(
    "\n"
    + "=" * 100
)

print(
    "4. RETRIEVAL vs GENERATION/PROMPTING FAULT"
)

print(
    "=" * 100
)


display(
    fault_attribution_summary.round(2)
)


# =============================================================================
# 18. SECONDARY FAILURE MECHANISM SUMMARY
# =============================================================================

secondary_records = []


for _, row in (
    rag_failure_diagnostic_df
    .iterrows()
):

    for mechanism in (
        row[
            "secondary_failure_mechanisms"
        ]
    ):

        secondary_records.append(
            {

                "model_family":
                    row[
                        "model_family"
                    ],

                "question_id":
                    row[
                        "question_id"
                    ],

                "secondary_failure_mechanism":
                    mechanism,
            }
        )


secondary_mechanism_df = (
    pd.DataFrame(
        secondary_records
    )
)


if len(
    secondary_mechanism_df
) > 0:

    secondary_mechanism_summary = (

        secondary_mechanism_df

        .groupby(
            [
                "model_family",
                "secondary_failure_mechanism",
            ]
        )

        .size()

        .reset_index(
            name=
                "cases"
        )

        .sort_values(
            [
                "model_family",
                "cases",
            ],
            ascending=[
                True,
                False,
            ]
        )
    )


else:

    secondary_mechanism_summary = (
        pd.DataFrame()
    )


print(
    "\n"
    + "=" * 100
)

print(
    "5. SECONDARY FAILURE MECHANISM SUMMARY"
)

print(
    "=" * 100
)


if len(
    secondary_mechanism_summary
) > 0:

    display(
        secondary_mechanism_summary
    )

else:

    print(
        "No secondary mechanisms recorded."
    )


# =============================================================================
# 19. CASE-LEVEL DIAGNOSTIC RATIONALE
# =============================================================================

case_rationale_view = (

    rag_failure_diagnostic_df[
        [
            "model_family",
            "question_id",

            "overall_delta",

            "evidence_sufficiency",

            "retrieval_at_fault",
            "generation_or_prompting_at_fault",

            "primary_failure_mechanism",

            "diagnostic_rationale",
        ]
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "6. CASE-LEVEL DIAGNOSTIC RATIONALE"
)

print(
    "=" * 100
)


display(
    case_rationale_view
)


# =============================================================================
# 20. VALIDATION
# =============================================================================

validation_checks = {

    "13 cases classified":
        (
            len(
                rag_failure_diagnostic_df
            )
            == 13
        ),

    "No duplicate cases":
        (
            not
            rag_failure_diagnostic_df
            .duplicated(
                subset=[
                    "model_family",
                    "question_id",
                ]
            )
            .any()
        ),

    "All evidence sufficiency classified":
        (
            rag_failure_diagnostic_df[
                "evidence_sufficiency"
            ]
            .notna()
            .all()
        ),

    "All primary mechanisms classified":
        (
            rag_failure_diagnostic_df[
                "primary_failure_mechanism"
            ]
            .notna()
            .all()
        ),

    "All rationales present":
        (
            rag_failure_diagnostic_df[
                "diagnostic_rationale"
            ]
            .notna()
            .all()
        ),
}


print(
    "\n"
    + "=" * 100
)

print(
    "MODULE 5.4 VALIDATION"
)

print(
    "=" * 100
)


for check_name, passed in (
    validation_checks.items()
):

    print(
        f"{check_name:<45} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 5.4 validation failed."
    )


# =============================================================================
# 21. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TRACK 1 PRIORITY RAG FAILURE DIAGNOSTIC COMPLETE"
)

print(
    "=" * 100
)


print(
    f"\nCases classified : "
    f"{len(rag_failure_diagnostic_df)}"
)


print(
    "\nGenerated objects:"
)

print(
    "• rag_diagnostic_prompt_sizes"
)

print(
    "• rag_failure_diagnostic_results"
)

print(
    "• rag_failure_diagnostic_df"
)

print(
    "• core_diagnostic_view"
)

print(
    "• primary_mechanism_summary"
)

print(
    "• evidence_sufficiency_summary"
)

print(
    "• fault_attribution_summary"
)

print(
    "• secondary_mechanism_df"
)

print(
    "• secondary_mechanism_summary"
)

print(
    "• case_rationale_view"
)


print(
    "\nMODULE 5.4 COMPLETE"
)

print(
    "=" * 100
)

MODULE 5.4 — CONTEXT-BOUNDED RAG FAILURE DIAGNOSTIC

Diagnostic output tokens       : 1400
Retrieval chars per chunk      : 900
Standalone response max chars  : 3500
RAG response max chars         : 3500

CONTEXT-BOUNDED DIAGNOSTIC PROMPT SIZE


,model_family,question_id,prompt_characters,retrieved_chunks
0,Gemma 4,Q15,13941,7
1,Essential AI,Q19,13727,7
2,Gemma 4,Q05,15547,7
3,Gemma 4,Q14,15099,7
4,Gemma 4,Q17,13691,7
5,Essential AI,Q06,13858,7
6,Essential AI,Q11,16557,7
7,Gemma 4,Q06,15662,7
8,Gemma 4,Q13,15315,7
9,Gemma 4,Q16,15853,7



Largest diagnostic prompt:
16,557 characters

RAG FAILURE MECHANISM CLASSIFICATION

Cases to diagnose : 13

[01/13] Gemma 4 | Q15 | Δ -8
    ✅ STRICT_GROUNDING_ABSTENTION | Evidence: PARTIAL | Retrieval fault: False | Generation/prompt fault: True

[02/13] Essential AI | Q19 | Δ -7
    ✅ STRICT_GROUNDING_ABSTENTION | Evidence: PARTIAL | Retrieval fault: False | Generation/prompt fault: True

[03/13] Gemma 4 | Q05 | Δ -5
    ✅ STRICT_GROUNDING_ABSTENTION | Evidence: PARTIAL | Retrieval fault: False | Generation/prompt fault: True

[04/13] Gemma 4 | Q14 | Δ -5


ValueError: Question ID mismatch.

In [59]:
# =============================================================================
# MODULE 5.4B — RESUME RAG FAILURE DIAGNOSTIC
#                 WITH ROBUST IDENTITY NORMALISATION
# =============================================================================
#
# PURPOSE
# -------
# Resume Module 5.4 after the Q14 identity mismatch.
#
# WHY THE PREVIOUS RUN STOPPED
# ----------------------------
# The diagnostic JSON was successfully generated, but the model returned a
# question_id that did not match the current case.
#
# This can happen because the output schema contains example values such as:
#
#     "question_id": "Q15"
#
# The identity of the case is already deterministically known from the row
# submitted to the judge. Therefore:
#
#     - model_family
#     - question_id
#
# are normalised from the source row after JSON parsing.
#
# All substantive diagnostic fields remain strictly validated.
#
# IMPORTANT
# ---------
# Existing successful results are preserved.
#
# Expected current state:
#
#     3 completed cases:
#       Gemma 4      | Q15
#       Essential AI | Q19
#       Gemma 4      | Q05
#
# Resume should therefore begin with:
#
#       Gemma 4 | Q14
#
# =============================================================================


import json
import os
import pandas as pd


# =============================================================================
# 1. SETTINGS
# =============================================================================

RAG_DIAGNOSTIC_MAX_TOKENS = 1400

RAG_DIAGNOSTIC_CHECKPOINT_FILE = (
    "track1_rag_failure_diagnostic_checkpoint.json"
)


# =============================================================================
# 2. VALID TAXONOMY
# =============================================================================

VALID_FAILURE_MECHANISMS = {

    "RETRIEVAL_INSUFFICIENT",

    "RETRIEVAL_PARTIALLY_SUFFICIENT",

    "RETRIEVAL_MISLEADING",

    "GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE",

    "STRICT_GROUNDING_ABSTENTION",

    "CONTEXT_OVERCONSTRAINT",

    "EVIDENCE_SYNTHESIS_FAILURE",

    "BASE_MODEL_SUPPRESSED_BY_CONTEXT",

    "NO_CLEAR_RAG_FAILURE",
}


VALID_EVIDENCE_SUFFICIENCY = {

    "SUFFICIENT",

    "PARTIAL",

    "INSUFFICIENT",
}


# =============================================================================
# 3. ROBUST VALIDATOR
# =============================================================================
#
# NOTE:
# model_family and question_id are no longer trusted as generated values.
# They are overwritten from the deterministic source row before validation.
#
# =============================================================================

def validate_rag_diagnostic_result(
    result,
):

    required_fields = {

        "model_family",
        "question_id",

        "evidence_sufficiency",

        "retrieval_at_fault",
        "generation_or_prompting_at_fault",

        "primary_failure_mechanism",
        "secondary_failure_mechanisms",

        "retrieval_strengths",
        "retrieval_gaps",

        "rag_response_failures",
        "standalone_advantages",

        "diagnostic_rationale",
    }


    missing = (
        required_fields
        -
        set(
            result.keys()
        )
    )


    if missing:

        raise ValueError(
            f"Missing fields: "
            f"{sorted(missing)}"
        )


    # -------------------------------------------------------------------------
    # Evidence sufficiency
    # -------------------------------------------------------------------------

    if (
        result[
            "evidence_sufficiency"
        ]
        not in
        VALID_EVIDENCE_SUFFICIENCY
    ):

        raise ValueError(
            "Invalid evidence_sufficiency: "
            f"{result['evidence_sufficiency']}"
        )


    # -------------------------------------------------------------------------
    # Primary mechanism
    # -------------------------------------------------------------------------

    primary = (
        result[
            "primary_failure_mechanism"
        ]
    )


    if (
        primary
        not in
        VALID_FAILURE_MECHANISMS
    ):

        raise ValueError(
            f"Invalid primary mechanism: "
            f"{primary}"
        )


    # -------------------------------------------------------------------------
    # Secondary mechanisms
    # -------------------------------------------------------------------------

    secondary = (
        result[
            "secondary_failure_mechanisms"
        ]
    )


    if not isinstance(
        secondary,
        list,
    ):

        raise ValueError(
            "secondary_failure_mechanisms "
            "must be a list."
        )


    if len(
        secondary
    ) > 2:

        raise ValueError(
            "Maximum two secondary mechanisms allowed."
        )


    for mechanism in secondary:

        if (
            mechanism
            not in
            VALID_FAILURE_MECHANISMS
        ):

            raise ValueError(
                f"Invalid secondary mechanism: "
                f"{mechanism}"
            )


    # -------------------------------------------------------------------------
    # Boolean fields
    # -------------------------------------------------------------------------

    for field in [

        "retrieval_at_fault",
        "generation_or_prompting_at_fault",

    ]:

        if not isinstance(
            result[
                field
            ],
            bool,
        ):

            raise ValueError(
                f"{field} must be boolean."
            )


    # -------------------------------------------------------------------------
    # List fields
    # -------------------------------------------------------------------------

    for field in [

        "retrieval_strengths",
        "retrieval_gaps",
        "rag_response_failures",
        "standalone_advantages",

    ]:

        if not isinstance(
            result[
                field
            ],
            list,
        ):

            raise ValueError(
                f"{field} must be a list."
            )


    # -------------------------------------------------------------------------
    # Rationale
    # -------------------------------------------------------------------------

    if not isinstance(
        result[
            "diagnostic_rationale"
        ],
        str,
    ):

        raise ValueError(
            "diagnostic_rationale must be a string."
        )


    return True


# =============================================================================
# 4. RECOVER EXISTING SUCCESSFUL RESULTS
# =============================================================================

if (
    "rag_failure_diagnostic_results"
    not in globals()
):

    rag_failure_diagnostic_results = []


print("=" * 100)
print("MODULE 5.4B — RESUME PRIORITY RAG FAILURE DIAGNOSTIC")
print("=" * 100)


print(
    f"\nResults currently in memory : "
    f"{len(rag_failure_diagnostic_results)}"
)


# =============================================================================
# 5. OPTIONAL CHECKPOINT RECOVERY
# =============================================================================

if os.path.exists(
    RAG_DIAGNOSTIC_CHECKPOINT_FILE
):

    with open(
        RAG_DIAGNOSTIC_CHECKPOINT_FILE,
        "r",
        encoding="utf-8",
    ) as f:

        checkpoint_results = (
            json.load(
                f
            )
        )


    print(
        f"Checkpoint records found     : "
        f"{len(checkpoint_results)}"
    )


    # -------------------------------------------------------------------------
    # Merge checkpoint + memory without duplicating cases
    # -------------------------------------------------------------------------

    merged = {}


    for item in (
        checkpoint_results
        +
        rag_failure_diagnostic_results
    ):

        key = (

            item[
                "model_family"
            ],

            item[
                "question_id"
            ],
        )


        merged[
            key
        ] = item


    rag_failure_diagnostic_results = (
        list(
            merged.values()
        )
    )


print(
    f"Recovered completed cases    : "
    f"{len(rag_failure_diagnostic_results)}"
)


# =============================================================================
# 6. COMPLETED CASE IDENTIFIERS
# =============================================================================

completed_cases = {

    (
        result[
            "model_family"
        ],
        result[
            "question_id"
        ],
    )

    for result
    in rag_failure_diagnostic_results
}


if completed_cases:

    print(
        "\nCompleted cases:"
    )


    for model_family, question_id in sorted(
        completed_cases
    ):

        print(
            f"  ✓ {model_family:<12} | "
            f"{question_id}"
        )


# =============================================================================
# 7. DETERMINE REMAINING CASES
# =============================================================================

remaining_indices = []


for idx, row in (
    rag_grounding_diagnostic_df
    .iterrows()
):

    key = (

        row[
            "model_family"
        ],

        row[
            "question_id"
        ],
    )


    if key not in completed_cases:

        remaining_indices.append(
            idx
        )


remaining_df = (

    rag_grounding_diagnostic_df
    .loc[
        remaining_indices
    ]
    .copy()
)


print(
    f"\nRemaining cases             : "
    f"{len(remaining_df)}"
)


if len(
    remaining_df
) > 0:

    print(
        "\nNext case:"
    )


    next_row = (
        remaining_df
        .iloc[0]
    )


    print(
        f"  {next_row['model_family']} | "
        f"{next_row['question_id']} | "
        f"Δ {next_row['overall_delta']}"
    )


# =============================================================================
# 8. RUN REMAINING CASES
# =============================================================================

for run_number, (_, row) in enumerate(

    remaining_df.iterrows(),

    start=1,
):

    print(
        f"\n"
        f"[{run_number:02d}/"
        f"{len(remaining_df):02d}] "
        f"{row['model_family']} | "
        f"{row['question_id']} | "
        f"Δ {row['overall_delta']}"
    )


    diagnostic_prompt = (
        build_rag_diagnostic_prompt(
            row
        )
    )


    response = (

        client
        .chat
        .completions
        .create(

            model=
                VLLM_MODEL,

            messages=[
                {
                    "role":
                        "system",

                    "content":
                        RAG_DIAGNOSTIC_SYSTEM_PROMPT,
                },
                {
                    "role":
                        "user",

                    "content":
                        diagnostic_prompt,
                },
            ],

            temperature=
                0.0,

            max_tokens=
                RAG_DIAGNOSTIC_MAX_TOKENS,
        )
    )


    raw_text = (

        response
        .choices[0]
        .message
        .content
        .strip()
    )


    # =========================================================================
    # JSON PARSE
    # =========================================================================

    try:

        diagnostic_result = (
            json.loads(
                raw_text
            )
        )


    except json.JSONDecodeError as exc:

        print(
            "\n❌ INVALID DIAGNOSTIC JSON"
        )

        print(
            raw_text
        )


        raise RuntimeError(
            f"Diagnostic JSON failed for "
            f"{row['model_family']} | "
            f"{row['question_id']}"
        ) from exc


    # =========================================================================
    # NORMALISE CASE IDENTITY
    # =========================================================================
    #
    # These values are deterministic metadata supplied by us.
    #
    # Do not rely on the model to reproduce them from the example schema.
    #
    # =========================================================================

    returned_model_family = (
        diagnostic_result.get(
            "model_family"
        )
    )


    returned_question_id = (
        diagnostic_result.get(
            "question_id"
        )
    )


    if (
        returned_model_family
        != row[
            "model_family"
        ]
    ):

        print(
            f"    ⚠️ Normalised model_family: "
            f"{returned_model_family} "
            f"→ {row['model_family']}"
        )


    if (
        returned_question_id
        != row[
            "question_id"
        ]
    ):

        print(
            f"    ⚠️ Normalised question_id: "
            f"{returned_question_id} "
            f"→ {row['question_id']}"
        )


    diagnostic_result[
        "model_family"
    ] = (
        row[
            "model_family"
        ]
    )


    diagnostic_result[
        "question_id"
    ] = (
        row[
            "question_id"
        ]
    )


    # =========================================================================
    # STRICT SUBSTANTIVE VALIDATION
    # =========================================================================

    validate_rag_diagnostic_result(
        diagnostic_result
    )


    # =========================================================================
    # STORE RESULT
    # =========================================================================

    result_record = {

        "model_family":
            row[
                "model_family"
            ],

        "question_id":
            row[
                "question_id"
            ],

        "category":
            row[
                "category"
            ],

        "evaluation_type":
            row[
                "evaluation_type"
            ],

        "corpus_relevance":
            row[
                "corpus_relevance"
            ],

        "production_k7_support":
            row[
                "production_k7_support"
            ],

        "base_overall_score":
            row[
                "base_overall_score"
            ],

        "rag_overall_score":
            row[
                "rag_overall_score"
            ],

        "overall_delta":
            row[
                "overall_delta"
            ],

        "evidence_sufficiency":
            diagnostic_result[
                "evidence_sufficiency"
            ],

        "retrieval_at_fault":
            diagnostic_result[
                "retrieval_at_fault"
            ],

        "generation_or_prompting_at_fault":
            diagnostic_result[
                "generation_or_prompting_at_fault"
            ],

        "primary_failure_mechanism":
            diagnostic_result[
                "primary_failure_mechanism"
            ],

        "secondary_failure_mechanisms":
            diagnostic_result[
                "secondary_failure_mechanisms"
            ],

        "retrieval_strengths":
            diagnostic_result[
                "retrieval_strengths"
            ],

        "retrieval_gaps":
            diagnostic_result[
                "retrieval_gaps"
            ],

        "rag_response_failures":
            diagnostic_result[
                "rag_response_failures"
            ],

        "standalone_advantages":
            diagnostic_result[
                "standalone_advantages"
            ],

        "diagnostic_rationale":
            diagnostic_result[
                "diagnostic_rationale"
            ],

        "raw_diagnostic_json":
            diagnostic_result,
    }


    rag_failure_diagnostic_results.append(
        result_record
    )


    # =========================================================================
    # CHECKPOINT AFTER EVERY SUCCESSFUL CASE
    # =========================================================================

    with open(
        RAG_DIAGNOSTIC_CHECKPOINT_FILE,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            rag_failure_diagnostic_results,
            f,
            indent=2,
            ensure_ascii=False,
        )


    print(
        f"    ✅ "
        f"{diagnostic_result['primary_failure_mechanism']} "
        f"| Evidence: "
        f"{diagnostic_result['evidence_sufficiency']} "
        f"| Retrieval fault: "
        f"{diagnostic_result['retrieval_at_fault']} "
        f"| Generation/prompt fault: "
        f"{diagnostic_result['generation_or_prompting_at_fault']}"
    )


# =============================================================================
# 9. BUILD FINAL DATAFRAME
# =============================================================================

rag_failure_diagnostic_df = (

    pd.DataFrame(
        rag_failure_diagnostic_results
    )

    .drop_duplicates(
        subset=[
            "model_family",
            "question_id",
        ],

        keep=
            "last",
    )

    .sort_values(
        [
            "overall_delta",
            "model_family",
            "question_id",
        ],
        ascending=[
            True,
            True,
            True,
        ]
    )

    .reset_index(
        drop=True
    )
)


# =============================================================================
# 10. CORE CLASSIFICATION VIEW
# =============================================================================

core_diagnostic_view = (

    rag_failure_diagnostic_df[
        [
            "model_family",
            "question_id",

            "evaluation_type",

            "corpus_relevance",
            "production_k7_support",

            "base_overall_score",
            "rag_overall_score",
            "overall_delta",

            "evidence_sufficiency",

            "retrieval_at_fault",
            "generation_or_prompting_at_fault",

            "primary_failure_mechanism",

            "secondary_failure_mechanisms",
        ]
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "1. PRIORITY RAG FAILURE CLASSIFICATION"
)

print(
    "=" * 100
)


display(
    core_diagnostic_view
)


# =============================================================================
# 11. PRIMARY FAILURE MECHANISM SUMMARY
# =============================================================================

primary_mechanism_summary = (

    rag_failure_diagnostic_df

    .groupby(
        [
            "model_family",
            "primary_failure_mechanism",
        ]
    )

    .size()

    .reset_index(
        name=
            "cases"
    )

    .sort_values(
        [
            "model_family",
            "cases",
        ],
        ascending=[
            True,
            False,
        ]
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "2. PRIMARY FAILURE MECHANISM SUMMARY"
)

print(
    "=" * 100
)


display(
    primary_mechanism_summary
)


# =============================================================================
# 12. EVIDENCE SUFFICIENCY SUMMARY
# =============================================================================

evidence_sufficiency_summary = (

    rag_failure_diagnostic_df

    .groupby(
        [
            "model_family",
            "evidence_sufficiency",
        ]
    )

    .size()

    .reset_index(
        name=
            "cases"
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "3. EVIDENCE SUFFICIENCY SUMMARY"
)

print(
    "=" * 100
)


display(
    evidence_sufficiency_summary
)


# =============================================================================
# 13. RETRIEVAL vs GENERATION/PROMPTING FAULT
# =============================================================================

fault_attribution_summary = (

    rag_failure_diagnostic_df

    .groupby(
        "model_family"
    )

    .agg(

        cases=(
            "question_id",
            "count",
        ),

        retrieval_fault_cases=(
            "retrieval_at_fault",
            "sum",
        ),

        generation_prompt_fault_cases=(
            "generation_or_prompting_at_fault",
            "sum",
        ),
    )

    .reset_index()
)


fault_attribution_summary[
    "retrieval_fault_pct"
] = (

    fault_attribution_summary[
        "retrieval_fault_cases"
    ]

    / fault_attribution_summary[
        "cases"
    ]

    * 100
)


fault_attribution_summary[
    "generation_prompt_fault_pct"
] = (

    fault_attribution_summary[
        "generation_prompt_fault_cases"
    ]

    / fault_attribution_summary[
        "cases"
    ]

    * 100
)


print(
    "\n"
    + "=" * 100
)

print(
    "4. RETRIEVAL vs GENERATION/PROMPTING FAULT"
)

print(
    "=" * 100
)


display(
    fault_attribution_summary.round(2)
)


# =============================================================================
# 14. SECONDARY MECHANISM SUMMARY
# =============================================================================

secondary_records = []


for _, row in (
    rag_failure_diagnostic_df
    .iterrows()
):

    for mechanism in (
        row[
            "secondary_failure_mechanisms"
        ]
    ):

        secondary_records.append(
            {

                "model_family":
                    row[
                        "model_family"
                    ],

                "question_id":
                    row[
                        "question_id"
                    ],

                "secondary_failure_mechanism":
                    mechanism,
            }
        )


secondary_mechanism_df = (
    pd.DataFrame(
        secondary_records
    )
)


if len(
    secondary_mechanism_df
) > 0:

    secondary_mechanism_summary = (

        secondary_mechanism_df

        .groupby(
            [
                "model_family",
                "secondary_failure_mechanism",
            ]
        )

        .size()

        .reset_index(
            name=
                "cases"
        )

        .sort_values(
            [
                "model_family",
                "cases",
            ],
            ascending=[
                True,
                False,
            ]
        )
    )


else:

    secondary_mechanism_summary = (
        pd.DataFrame()
    )


print(
    "\n"
    + "=" * 100
)

print(
    "5. SECONDARY FAILURE MECHANISM SUMMARY"
)

print(
    "=" * 100
)


if len(
    secondary_mechanism_summary
) > 0:

    display(
        secondary_mechanism_summary
    )

else:

    print(
        "No secondary mechanisms recorded."
    )


# =============================================================================
# 15. CASE-LEVEL RATIONALE
# =============================================================================

case_rationale_view = (

    rag_failure_diagnostic_df[
        [
            "model_family",
            "question_id",

            "overall_delta",

            "evidence_sufficiency",

            "retrieval_at_fault",

            "generation_or_prompting_at_fault",

            "primary_failure_mechanism",

            "diagnostic_rationale",
        ]
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "6. CASE-LEVEL DIAGNOSTIC RATIONALE"
)

print(
    "=" * 100
)


display(
    case_rationale_view
)


# =============================================================================
# 16. FINAL VALIDATION
# =============================================================================

validation_checks = {

    "13 cases classified":
        (
            len(
                rag_failure_diagnostic_df
            )
            == 13
        ),

    "No duplicate cases":
        (
            not
            rag_failure_diagnostic_df
            .duplicated(
                subset=[
                    "model_family",
                    "question_id",
                ]
            )
            .any()
        ),

    "All evidence sufficiency classified":
        (
            rag_failure_diagnostic_df[
                "evidence_sufficiency"
            ]
            .notna()
            .all()
        ),

    "All primary mechanisms classified":
        (
            rag_failure_diagnostic_df[
                "primary_failure_mechanism"
            ]
            .notna()
            .all()
        ),

    "All rationales present":
        (
            rag_failure_diagnostic_df[
                "diagnostic_rationale"
            ]
            .notna()
            .all()
        ),
}


print(
    "\n"
    + "=" * 100
)

print(
    "MODULE 5.4B VALIDATION"
)

print(
    "=" * 100
)


for check_name, passed in (
    validation_checks.items()
):

    print(
        f"{check_name:<45} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 5.4B validation failed."
    )


# =============================================================================
# 17. SAVE FINAL RESULTS
# =============================================================================

FINAL_JSON_FILE = (
    "track1_rag_failure_diagnostic_complete.json"
)

FINAL_CSV_FILE = (
    "track1_rag_failure_diagnostic_complete.csv"
)


with open(
    FINAL_JSON_FILE,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        rag_failure_diagnostic_results,
        f,
        indent=2,
        ensure_ascii=False,
    )


csv_export_df = (
    rag_failure_diagnostic_df
    .copy()
)


# Convert nested list/dict columns to JSON strings for CSV
for column in [

    "secondary_failure_mechanisms",
    "retrieval_strengths",
    "retrieval_gaps",
    "rag_response_failures",
    "standalone_advantages",
    "raw_diagnostic_json",

]:

    csv_export_df[
        column
    ] = (

        csv_export_df[
            column
        ]

        .apply(
            lambda value:
                json.dumps(
                    value,
                    ensure_ascii=False,
                )
        )
    )


csv_export_df.to_csv(
    FINAL_CSV_FILE,
    index=False,
)


# =============================================================================
# 18. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TRACK 1 PRIORITY RAG FAILURE DIAGNOSTIC COMPLETE"
)

print(
    "=" * 100
)


print(
    f"\nCases classified : "
    f"{len(rag_failure_diagnostic_df)}"
)


print(
    f"\nSaved JSON : "
    f"{FINAL_JSON_FILE}"
)

print(
    f"Saved CSV  : "
    f"{FINAL_CSV_FILE}"
)


print(
    "\nMODULE 5.4B COMPLETE"
)

print(
    "=" * 100
)

MODULE 5.4B — RESUME PRIORITY RAG FAILURE DIAGNOSTIC

Results currently in memory : 3
Recovered completed cases    : 3

Completed cases:
  ✓ Essential AI | Q19
  ✓ Gemma 4      | Q05
  ✓ Gemma 4      | Q15

Remaining cases             : 10

Next case:
  Gemma 4 | Q14 | Δ -5

[01/10] Gemma 4 | Q14 | Δ -5
    ⚠️ Normalised question_id: Q15 → Q14
    ✅ STRICT_GROUNDING_ABSTENTION | Evidence: PARTIAL | Retrieval fault: False | Generation/prompt fault: True

[02/10] Gemma 4 | Q17 | Δ -5
    ✅ STRICT_GROUNDING_ABSTENTION | Evidence: PARTIAL | Retrieval fault: False | Generation/prompt fault: True

[03/10] Essential AI | Q06 | Δ -3
    ✅ STRICT_GROUNDING_ABSTENTION | Evidence: PARTIAL | Retrieval fault: False | Generation/prompt fault: True

[04/10] Essential AI | Q11 | Δ -3
    ✅ STRICT_GROUNDING_ABSTENTION | Evidence: PARTIAL | Retrieval fault: False | Generation/prompt fault: True

[05/10] Gemma 4 | Q06 | Δ -3
    ✅ STRICT_GROUNDING_ABSTENTION | Evidence: PARTIAL | Retrieval fault: False |

,model_family,question_id,evaluation_type,corpus_relevance,production_k7_support,base_overall_score,rag_overall_score,overall_delta,evidence_sufficiency,retrieval_at_fault,generation_or_prompting_at_fault,primary_failure_mechanism,secondary_failure_mechanisms
0,Gemma 4,Q15,CORPUS_GROUNDED,PARTIAL,YES,9,1,-8,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
1,Essential AI,Q19,ENGINEERING_SYNTHESIS,PARTIAL,PARTIAL,8,1,-7,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
2,Gemma 4,Q05,CORPUS_GROUNDED,STRONG,YES,10,5,-5,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
3,Gemma 4,Q14,CORPUS_GROUNDED,PARTIAL-STRONG,YES,9,4,-5,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
4,Gemma 4,Q17,CORPUS_GROUNDED,PARTIAL-STRONG,YES,9,4,-5,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
5,Essential AI,Q06,CORPUS_GROUNDED,STRONG,YES,5,2,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
6,Essential AI,Q11,ENGINEERING_SYNTHESIS,WEAK-PARTIAL,PARTIAL,5,2,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
7,Gemma 4,Q06,CORPUS_GROUNDED,STRONG,YES,8,5,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
8,Gemma 4,Q13,ENGINEERING_SYNTHESIS,PARTIAL,YES,8,5,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]
9,Gemma 4,Q16,CORPUS_GROUNDED,STRONG,YES,9,6,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,[GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE]



2. PRIMARY FAILURE MECHANISM SUMMARY


,model_family,primary_failure_mechanism,cases
0,Essential AI,STRICT_GROUNDING_ABSTENTION,4
1,Gemma 4,STRICT_GROUNDING_ABSTENTION,9



3. EVIDENCE SUFFICIENCY SUMMARY


...


5. SECONDARY FAILURE MECHANISM SUMMARY


,model_family,secondary_failure_mechanism,cases
0,Essential AI,GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE,4
1,Gemma 4,GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE,9



6. CASE-LEVEL DIAGNOSTIC RATIONALE


,model_family,question_id,overall_delta,evidence_sufficiency,retrieval_at_fault,generation_or_prompting_at_fault,primary_failure_mechanism,diagnostic_rationale
0,Gemma 4,Q15,-8,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,"The retrieved evidence, while partial, contain..."
1,Essential AI,Q19,-7,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,"The retrieved evidence, while partial, contain..."
2,Gemma 4,Q05,-5,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,The retrieved evidence is PARTIAL—contains key...
3,Gemma 4,Q14,-5,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,The retrieved evidence contains sufficient inf...
4,Gemma 4,Q17,-5,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,Evidence sufficiency is PARTIAL: relevant tech...
5,Essential AI,Q06,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,"The retrieved evidence, while strong in covera..."
6,Essential AI,Q11,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,"The retrieved evidence, while weak and partial..."
7,Gemma 4,Q06,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,The retrieved evidence is PARTIAL: it contains...
8,Gemma 4,Q13,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,The retrieved evidence contains sufficient tec...
9,Gemma 4,Q16,-3,PARTIAL,False,True,STRICT_GROUNDING_ABSTENTION,"The retrieved evidence, while not fully compre..."



MODULE 5.4B VALIDATION
13 cases classified                           : ✅ PASS
No duplicate cases                            : ✅ PASS
All evidence sufficiency classified           : ✅ PASS
All primary mechanisms classified             : ✅ PASS
All rationales present                        : ✅ PASS

TRACK 1 PRIORITY RAG FAILURE DIAGNOSTIC COMPLETE

Cases classified : 13

Saved JSON : track1_rag_failure_diagnostic_complete.json
Saved CSV  : track1_rag_failure_diagnostic_complete.csv

MODULE 5.4B COMPLETE


# **Track 1 Consolidated Findings and RAG Design Recommendations**

In [60]:
# =============================================================================
# MODULE 5.5 — TRACK 1 CONSOLIDATED FINDINGS AND RAG DESIGN RECOMMENDATIONS
# =============================================================================
#
# PURPOSE
# -------
# Consolidate the quantitative Track 1 benchmark results and the RAG failure
# diagnostics into a concise technical interpretation.
#
# THIS MODULE SUMMARISES
# ----------------------
# 1. Overall model performance
# 2. Performance consistency
# 3. RAG vs standalone performance
# 4. CORPUS_GROUNDED vs ENGINEERING_SYNTHESIS behaviour
# 5. Failure patterns
# 6. Priority RAG failure diagnostics
# 7. RAG architecture implications
# 8. Recommended RAG design improvements
#
# IMPORTANT
# ---------
# This module does NOT rerun inference or judging.
#
# It uses the outputs already generated in:
#
#   Module 5.1
#   Module 5.2
#   Module 5.3
#   Module 5.4 / 5.4B
#
# =============================================================================


import pandas as pd
import numpy as np
import json


# =============================================================================
# 1. VALIDATE REQUIRED ANALYSIS OBJECTS
# =============================================================================

required_objects = [

    "overall_ranking",
    "dimension_summary",
    "evaluation_type_pivot",
    "rag_vs_base_summary",
    "rag_by_type_summary",
    "failure_summary",
    "failure_rate_summary",
    "rag_question_impact_df",
    "rag_failure_diagnostic_df",
    "fault_attribution_summary",
]


missing_objects = [

    obj

    for obj in required_objects

    if obj not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Missing required analysis objects: "
        f"{missing_objects}"
    )


print("=" * 110)
print("MODULE 5.5 — TRACK 1 CONSOLIDATED FINDINGS AND RAG DESIGN RECOMMENDATIONS")
print("=" * 110)


# =============================================================================
# 2. CORE TRACK 1 PERFORMANCE METRICS
# =============================================================================

ranking_lookup = (

    overall_ranking

    .set_index(
        "model_name"
    )
)


otel_mean = (
    ranking_lookup
    .loc[
        "Otel 2.0 Only",
        "mean_overall"
    ]
)


gemma_base_mean = (
    ranking_lookup
    .loc[
        "Gemma 4 Only",
        "mean_overall"
    ]
)


gemma_rag_mean = (
    ranking_lookup
    .loc[
        "Gemma 4 + RAG",
        "mean_overall"
    ]
)


essential_base_mean = (
    ranking_lookup
    .loc[
        "Essential AI Only",
        "mean_overall"
    ]
)


essential_rag_mean = (
    ranking_lookup
    .loc[
        "Essential AI + RAG",
        "mean_overall"
    ]
)


otel_std = (
    ranking_lookup
    .loc[
        "Otel 2.0 Only",
        "std_overall"
    ]
)


gemma_base_std = (
    ranking_lookup
    .loc[
        "Gemma 4 Only",
        "std_overall"
    ]
)


gemma_rag_std = (
    ranking_lookup
    .loc[
        "Gemma 4 + RAG",
        "std_overall"
    ]
)


essential_rag_std = (
    ranking_lookup
    .loc[
        "Essential AI + RAG",
        "std_overall"
    ]
)


# =============================================================================
# 3. RAG FAMILY DELTAS
# =============================================================================

essential_rag_summary = (

    rag_vs_base_summary[

        rag_vs_base_summary[
            "model_family"
        ] == "Essential AI"

    ]

    .iloc[0]
)


gemma_rag_summary = (

    rag_vs_base_summary[

        rag_vs_base_summary[
            "model_family"
        ] == "Gemma 4"

    ]

    .iloc[0]
)


# =============================================================================
# 4. EVALUATION-TYPE DELTAS
# =============================================================================

def get_rag_type_row(
    family,
    evaluation_type,
):

    return (

        rag_by_type_summary[

            (
                rag_by_type_summary[
                    "model_family"
                ] == family
            )

            &

            (
                rag_by_type_summary[
                    "evaluation_type"
                ] == evaluation_type
            )

        ]

        .iloc[0]
    )


essential_corpus = (
    get_rag_type_row(
        "Essential AI",
        "CORPUS_GROUNDED",
    )
)


essential_synthesis = (
    get_rag_type_row(
        "Essential AI",
        "ENGINEERING_SYNTHESIS",
    )
)


gemma_corpus = (
    get_rag_type_row(
        "Gemma 4",
        "CORPUS_GROUNDED",
    )
)


gemma_synthesis = (
    get_rag_type_row(
        "Gemma 4",
        "ENGINEERING_SYNTHESIS",
    )
)


# =============================================================================
# 5. FAILURE RATE LOOKUP
# =============================================================================

failure_lookup = (

    failure_rate_summary

    .set_index(
        "model_name"
    )
)


# =============================================================================
# 6. PRIORITY RAG DIAGNOSTIC METRICS
# =============================================================================

priority_cases = (
    len(
        rag_failure_diagnostic_df
    )
)


retrieval_fault_cases = int(

    rag_failure_diagnostic_df[
        "retrieval_at_fault"
    ].sum()
)


generation_fault_cases = int(

    rag_failure_diagnostic_df[
        "generation_or_prompting_at_fault"
    ].sum()
)


partial_evidence_cases = int(

    (
        rag_failure_diagnostic_df[
            "evidence_sufficiency"
        ]
        == "PARTIAL"
    )
    .sum()
)


sufficient_evidence_cases = int(

    (
        rag_failure_diagnostic_df[
            "evidence_sufficiency"
        ]
        == "SUFFICIENT"
    )
    .sum()
)


insufficient_evidence_cases = int(

    (
        rag_failure_diagnostic_df[
            "evidence_sufficiency"
        ]
        == "INSUFFICIENT"
    )
    .sum()
)


strict_grounding_cases = int(

    (
        rag_failure_diagnostic_df[
            "primary_failure_mechanism"
        ]
        == "STRICT_GROUNDING_ABSTENTION"
    )
    .sum()
)


# =============================================================================
# 7. FIND MOST SEVERE RAG DEGRADATIONS
# =============================================================================

top_degradations = (

    rag_question_impact_df

    .sort_values(
        "overall_delta"
    )

    .head(
        5
    )
)


# =============================================================================
# 8. BUILD CONSOLIDATED FINDINGS
# =============================================================================

track1_findings = []


track1_findings.append(

    {
        "finding_id":
            "F1",

        "title":
            "Standalone models dominated overall performance",

        "finding":
            (
                f"Otel 2.0 Only achieved the highest Track 1 mean overall "
                f"score at {otel_mean:.2f}, followed closely by Gemma 4 Only "
                f"at {gemma_base_mean:.2f}. "
                f"Gemma 4 + RAG scored {gemma_rag_mean:.2f}, "
                f"Essential AI Only {essential_base_mean:.2f}, and "
                f"Essential AI + RAG {essential_rag_mean:.2f}."
            ),

        "interpretation":
            (
                "The strongest benchmark performance came from the standalone "
                "models rather than the RAG configurations."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F2",

        "title":
            "Otel 2.0 was the strongest and most consistent system",

        "finding":
            (
                f"Otel 2.0 Only achieved a mean score of {otel_mean:.2f} "
                f"with standard deviation {otel_std:.3f}. "
                f"Gemma 4 Only scored {gemma_base_mean:.2f} with standard "
                f"deviation {gemma_base_std:.3f}."
            ),

        "interpretation":
            (
                "Otel 2.0 combined high technical quality with relatively "
                "low question-to-question variability."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F3",

        "title":
            "RAG impact was model-dependent",

        "finding":
            (
                f"For Essential AI, RAG changed the mean score from "
                f"{essential_base_mean:.2f} to {essential_rag_mean:.2f}, "
                f"a delta of {essential_rag_summary['mean_delta']:.2f}. "
                f"RAG improved {int(essential_rag_summary['rag_better'])} "
                f"questions, tied on "
                f"{int(essential_rag_summary['same_score'])}, and degraded "
                f"{int(essential_rag_summary['rag_worse'])}. "
                f"For Gemma 4, RAG reduced the mean score from "
                f"{gemma_base_mean:.2f} to {gemma_rag_mean:.2f}, "
                f"a delta of {gemma_rag_summary['mean_delta']:.2f}."
            ),

        "interpretation":
            (
                "RAG was not inherently beneficial. Its effect depended on "
                "the base model and the interaction between retrieved evidence, "
                "prompt constraints and generation behaviour."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F4",

        "title":
            "Gemma 4 showed systematic degradation after RAG",

        "finding":
            (
                f"Gemma 4 + RAG outperformed Gemma 4 Only on "
                f"{int(gemma_rag_summary['rag_better'])} of 20 questions, "
                f"tied on {int(gemma_rag_summary['same_score'])}, and "
                f"performed worse on {int(gemma_rag_summary['rag_worse'])}. "
                f"Its overall mean decreased by "
                f"{abs(gemma_rag_summary['mean_delta']):.2f} points."
            ),

        "interpretation":
            (
                "The result indicates a systematic RAG interaction problem "
                "rather than isolated question-level variance."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F5",

        "title":
            "Question type materially influenced RAG effectiveness",

        "finding":
            (
                f"Essential AI RAG produced a small positive mean delta of "
                f"{essential_corpus['mean_delta']:.3f} on CORPUS_GROUNDED "
                f"questions but a delta of "
                f"{essential_synthesis['mean_delta']:.3f} on "
                f"ENGINEERING_SYNTHESIS questions. "
                f"Gemma 4 RAG produced deltas of "
                f"{gemma_corpus['mean_delta']:.3f} and "
                f"{gemma_synthesis['mean_delta']:.3f}, respectively."
            ),

        "interpretation":
            (
                "Directly documented questions are more compatible with strict "
                "RAG grounding than questions requiring broader engineering "
                "reasoning and multi-domain synthesis."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F6",

        "title":
            "RAG configurations were less stable",

        "finding":
            (
                f"Gemma 4 + RAG had a standard deviation of "
                f"{gemma_rag_std:.3f}, while Essential AI + RAG had "
                f"{essential_rag_std:.3f}. Both were substantially more "
                f"variable than Otel 2.0 Only ({otel_std:.3f})."
            ),

        "interpretation":
            (
                "The RAG systems showed greater sensitivity to the individual "
                "question and retrieved context."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F7",

        "title":
            "RAG increased technical-error exposure in several cases",

        "finding":
            (
                f"Gemma 4 Only produced technical errors in "
                f"{int(failure_lookup.loc['Gemma 4 Only', 'responses_with_errors'])} "
                f"of 20 responses, compared with "
                f"{int(failure_lookup.loc['Gemma 4 + RAG', 'responses_with_errors'])} "
                f"for Gemma 4 + RAG. "
                f"Essential AI Only produced technical errors in "
                f"{int(failure_lookup.loc['Essential AI Only', 'responses_with_errors'])} "
                f"responses versus "
                f"{int(failure_lookup.loc['Essential AI + RAG', 'responses_with_errors'])} "
                f"for Essential AI + RAG."
            ),

        "interpretation":
            (
                "Retrieved context did not automatically increase factual "
                "reliability and, particularly for Gemma 4, could introduce "
                "new technical mistakes or incomplete reasoning."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F8",

        "title":
            "Two catastrophic RAG abstentions were observed",

        "finding":
            (
                "Gemma 4 + RAG became NON_RESPONSIVE on Q15, falling from "
                "a standalone score of 9 to 1. Essential AI + RAG became "
                "NON_RESPONSIVE on Q19, falling from 8 to 1."
            ),

        "interpretation":
            (
                "The fallback policy can convert partial evidence into a full "
                "refusal even when the system possesses useful information."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F9",

        "title":
            "Priority failures were not primarily retrieval failures",

        "finding":
            (
                f"Across the {priority_cases} priority degradation cases, "
                f"{retrieval_fault_cases} were classified as retrieval faults, "
                f"while {generation_fault_cases} were attributed to "
                f"generation/prompting behaviour."
            ),

        "interpretation":
            (
                "The first architecture component to improve should therefore "
                "be the grounding and generation policy rather than replacing "
                "the retriever."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F10",

        "title":
            "Retrieved evidence was generally usable but incomplete",

        "finding":
            (
                f"Among the {priority_cases} priority cases, "
                f"{partial_evidence_cases} had PARTIAL evidence, "
                f"{sufficient_evidence_cases} had SUFFICIENT evidence, and "
                f"{insufficient_evidence_cases} were classified as "
                f"INSUFFICIENT."
            ),

        "interpretation":
            (
                "The dominant problem was not absence of evidence, but the "
                "system's inability to make effective use of partial or "
                "sufficient evidence."
            ),
    }
)


track1_findings.append(

    {
        "finding_id":
            "F11",

        "title":
            "Strict grounding abstention dominated severe RAG failures",

        "finding":
            (
                f"{strict_grounding_cases} of {priority_cases} priority "
                f"failures were classified primarily as "
                f"STRICT_GROUNDING_ABSTENTION."
            ),

        "interpretation":
            (
                "The current grounding instruction is too binary: incomplete "
                "documentation can be interpreted as 'no answer available' "
                "instead of 'answer the supported parts and qualify the rest'."
            ),
    }
)


# =============================================================================
# 9. BUILD RAG DESIGN RECOMMENDATIONS
# =============================================================================

rag_design_recommendations = [

    {
        "priority":
            1,

        "recommendation":
            "Replace binary context-only fallback with graded grounding",

        "design_change":
            (
                "Use retrieved documentation as the primary evidence source, "
                "but allow the model to answer supported portions when "
                "evidence is partial. Abstain only when the retrieved context "
                "provides no meaningful support for the core question."
            ),

        "expected_benefit":
            (
                "Reduces catastrophic refusals such as Q15 and Q19 while "
                "maintaining grounding discipline."
            ),
    },

    {
        "priority":
            2,

        "recommendation":
            "Explicitly distinguish documented evidence from engineering reasoning",

        "design_change":
            (
                "Permit the model to supplement partial retrieved evidence "
                "with clearly identified engineering reasoning or pretrained "
                "knowledge when the task requires synthesis."
            ),

        "expected_benefit":
            (
                "Preserves the strong reasoning demonstrated by standalone "
                "Gemma 4 without presenting unsupported reasoning as "
                "documentation-derived fact."
            ),
    },

    {
        "priority":
            3,

        "recommendation":
            "Introduce an evidence-sufficiency decision before generation",

        "design_change":
            (
                "Classify retrieved context as SUFFICIENT, PARTIAL or "
                "INSUFFICIENT before final answer generation."
            ),

        "expected_benefit":
            (
                "Allows different response strategies rather than applying "
                "one strict grounding policy to every query."
            ),
    },

    {
        "priority":
            4,

        "recommendation":
            "Use query-type-aware RAG behaviour",

        "design_change":
            (
                "For CORPUS_GROUNDED questions, apply stronger evidence "
                "constraints. For ENGINEERING_SYNTHESIS questions, allow "
                "broader technical reasoning while retaining evidence "
                "attribution."
            ),

        "expected_benefit":
            (
                "Matches the generation strategy to the actual task rather "
                "than treating standards lookup and engineering diagnosis "
                "as the same problem."
            ),
    },

    {
        "priority":
            5,

        "recommendation":
            "Improve multi-chunk evidence synthesis",

        "design_change":
            (
                "Prompt the model to explicitly consolidate complementary "
                "facts across retrieved chunks before composing the answer."
            ),

        "expected_benefit":
            (
                "Reduces cases where individually useful chunks are retrieved "
                "but the final response fails to connect them into an "
                "end-to-end engineering explanation."
            ),
    },

    {
        "priority":
            6,

        "recommendation":
            "Prevent false 'documentation unavailable' conclusions",

        "design_change":
            (
                "Require the model to first identify what IS supported by the "
                "retrieved evidence before deciding whether any requested "
                "details remain unsupported."
            ),

        "expected_benefit":
            (
                "Avoids rejecting an entire query merely because some "
                "requested subtopics are missing."
            ),
    },

    {
        "priority":
            7,

        "recommendation":
            "Retain Retriever V1 as the baseline before major retriever changes",

        "design_change":
            (
                "Do not replace BGE-M3 / FAISS solely because RAG generation "
                "performed poorly. First retest with improved grounding and "
                "generation logic."
            ),

        "expected_benefit":
            (
                "Separates retrieval quality from generation-policy quality "
                "and avoids changing multiple architecture components at once."
            ),
    },

    {
        "priority":
            8,

        "recommendation":
            "Keep generation temperature low during engineering evaluation",

        "design_change":
            (
                "Continue using deterministic or near-deterministic decoding "
                "for technical evaluation. Address grounding-policy problems "
                "through prompt logic rather than increasing temperature."
            ),

        "expected_benefit":
            (
                "Maintains reproducibility while targeting the actual failure "
                "mechanism identified by the diagnostic."
            ),
    },

    {
        "priority":
            9,

        "recommendation":
            "Retest the revised RAG prompt using paired ablation",

        "design_change":
            (
                "Reuse the same Track 1 questions, retriever and base models. "
                "Change only the grounding/generation policy and compare "
                "paired score deltas."
            ),

        "expected_benefit":
            (
                "Provides causal evidence that any improvement comes from "
                "prompt and generation-policy changes rather than retrieval "
                "or dataset variation."
            ),
    },
]


# =============================================================================
# 10. BUILD DATAFRAMES
# =============================================================================

track1_consolidated_findings_df = (
    pd.DataFrame(
        track1_findings
    )
)


rag_design_recommendations_df = (
    pd.DataFrame(
        rag_design_recommendations
    )
)


# =============================================================================
# 11. DISPLAY CONSOLIDATED FINDINGS
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "1. CONSOLIDATED TRACK 1 FINDINGS"
)

print(
    "=" * 110
)


display(
    track1_consolidated_findings_df
)


# =============================================================================
# 12. DISPLAY RAG DESIGN RECOMMENDATIONS
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "2. RAG DESIGN RECOMMENDATIONS"
)

print(
    "=" * 110
)


display(
    rag_design_recommendations_df
)


# =============================================================================
# 13. MOST SEVERE RAG DEGRADATIONS
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "3. MOST SEVERE RAG DEGRADATION CASES"
)

print(
    "=" * 110
)


display(

    top_degradations[
        [
            "model_family",
            "question_id",
            "category",
            "evaluation_type",

            "corpus_relevance",
            "production_k7_support",

            "base_overall",
            "rag_overall",
            "overall_delta",

            "rag_response_status",
        ]
    ]
)


# =============================================================================
# 14. CONCISE REPORT-READY SUMMARY
# =============================================================================

TRACK1_REPORT_SUMMARY = f"""
TRACK 1 — CONSOLIDATED FINDINGS

The Track 1 benchmark evaluated five telecom AI configurations across
20 questions and 100 judged responses.

Otel 2.0 Only achieved the strongest overall performance with a mean score
of {otel_mean:.2f}, followed by Gemma 4 Only at {gemma_base_mean:.2f}.
The standalone models therefore substantially outperformed the two main
RAG configurations.

RAG impact was strongly model-dependent. Essential AI showed approximately
neutral overall behaviour, moving from {essential_base_mean:.2f} without RAG
to {essential_rag_mean:.2f} with RAG. Its RAG configuration improved
{int(essential_rag_summary['rag_better'])} questions, tied on
{int(essential_rag_summary['same_score'])}, and degraded
{int(essential_rag_summary['rag_worse'])}.

Gemma 4 showed a substantially different pattern. Its mean score decreased
from {gemma_base_mean:.2f} standalone to {gemma_rag_mean:.2f} with RAG.
Gemma 4 + RAG improved zero Track 1 questions, tied on
{int(gemma_rag_summary['same_score'])}, and degraded
{int(gemma_rag_summary['rag_worse'])}. This indicates systematic interaction
between the retrieved context and the generation policy.

Question type also influenced RAG effectiveness. Essential AI produced a
small positive RAG delta of {essential_corpus['mean_delta']:.3f} on
CORPUS_GROUNDED questions but a delta of
{essential_synthesis['mean_delta']:.3f} on ENGINEERING_SYNTHESIS questions.
Gemma 4 experienced negative deltas in both categories:
{gemma_corpus['mean_delta']:.3f} for CORPUS_GROUNDED and
{gemma_synthesis['mean_delta']:.3f} for ENGINEERING_SYNTHESIS.

The priority failure diagnostic examined {priority_cases} severe RAG
degradation cases using the original production Top-7 retrieved evidence.
None were classified primarily as retrieval failures.
{partial_evidence_cases} cases contained PARTIAL evidence and
{sufficient_evidence_cases} contained SUFFICIENT evidence, while
{insufficient_evidence_cases} were classified as INSUFFICIENT.

All {priority_cases} priority cases were attributed to generation or prompting
behaviour, with STRICT_GROUNDING_ABSTENTION identified as the primary failure
mechanism. This indicates that the current RAG policy is overly restrictive:
partial evidence can cause the model to suppress otherwise valid engineering
reasoning or incorrectly conclude that the requested information is unavailable.

The principal architectural recommendation is therefore not to replace
Retriever V1 immediately. The first redesign should focus on graded grounding,
evidence-sufficiency assessment, task-aware RAG behaviour, improved multi-chunk
synthesis, and a softer abstention policy.

For CORPUS_GROUNDED questions, retrieved evidence should remain the primary
source of truth. For ENGINEERING_SYNTHESIS questions, the model should be
allowed to combine retrieved evidence with clearly distinguished engineering
reasoning rather than being forced to abstain whenever the documentation does
not cover every requested technical element.

The next experimental iteration should retain the same retriever, models and
Track 1 questions while changing only the grounding/generation policy. This
paired ablation would provide a controlled measurement of whether the revised
RAG architecture recovers the standalone model capability while preserving
document grounding.
""".strip()


print(
    "\n"
    + "=" * 110
)

print(
    "4. REPORT-READY TRACK 1 SUMMARY"
)

print(
    "=" * 110
)


print(
    TRACK1_REPORT_SUMMARY
)


# =============================================================================
# 15. SAVE CONSOLIDATED OUTPUTS
# =============================================================================

FINDINGS_CSV = (
    "track1_consolidated_findings.csv"
)

RECOMMENDATIONS_CSV = (
    "track1_rag_design_recommendations.csv"
)

SUMMARY_TXT = (
    "track1_consolidated_report_summary.txt"
)


track1_consolidated_findings_df.to_csv(
    FINDINGS_CSV,
    index=False,
)


rag_design_recommendations_df.to_csv(
    RECOMMENDATIONS_CSV,
    index=False,
)


with open(
    SUMMARY_TXT,
    "w",
    encoding="utf-8",
) as f:

    f.write(
        TRACK1_REPORT_SUMMARY
    )


# =============================================================================
# 16. FINAL OUTPUT
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TRACK 1 CONSOLIDATION COMPLETE"
)

print(
    "=" * 110
)


print(
    f"\nSaved findings        : "
    f"{FINDINGS_CSV}"
)

print(
    f"Saved recommendations : "
    f"{RECOMMENDATIONS_CSV}"
)

print(
    f"Saved report summary  : "
    f"{SUMMARY_TXT}"
)


print(
    "\nGenerated objects:"
)

print(
    "• track1_consolidated_findings_df"
)

print(
    "• rag_design_recommendations_df"
)

print(
    "• TRACK1_REPORT_SUMMARY"
)


print(
    "\nMODULE 5.5 COMPLETE"
)

print(
    "=" * 110
)

MODULE 5.5 — TRACK 1 CONSOLIDATED FINDINGS AND RAG DESIGN RECOMMENDATIONS

1. CONSOLIDATED TRACK 1 FINDINGS


,finding_id,title,finding,interpretation
0,F1,Standalone models dominated overall performance,Otel 2.0 Only achieved the highest Track 1 mea...,The strongest benchmark performance came from ...
1,F2,Otel 2.0 was the strongest and most consistent...,Otel 2.0 Only achieved a mean score of 9.20 wi...,Otel 2.0 combined high technical quality with ...
2,F3,RAG impact was model-dependent,"For Essential AI, RAG changed the mean score f...",RAG was not inherently beneficial. Its effect ...
3,F4,Gemma 4 showed systematic degradation after RAG,Gemma 4 + RAG outperformed Gemma 4 Only on 0 o...,The result indicates a systematic RAG interact...
4,F5,Question type materially influenced RAG effect...,Essential AI RAG produced a small positive mea...,Directly documented questions are more compati...
5,F6,RAG configurations were less stable,Gemma 4 + RAG had a standard deviation of 2.63...,The RAG systems showed greater sensitivity to ...
6,F7,RAG increased technical-error exposure in seve...,Gemma 4 Only produced technical errors in 1 of...,Retrieved context did not automatically increa...
7,F8,Two catastrophic RAG abstentions were observed,"Gemma 4 + RAG became NON_RESPONSIVE on Q15, fa...",The fallback policy can convert partial eviden...
8,F9,Priority failures were not primarily retrieval...,"Across the 13 priority degradation cases, 0 we...",The first architecture component to improve sh...
9,F10,Retrieved evidence was generally usable but in...,"Among the 13 priority cases, 11 had PARTIAL ev...",The dominant problem was not absence of eviden...



2. RAG DESIGN RECOMMENDATIONS


,priority,recommendation,design_change,expected_benefit
0,1,Replace binary context-only fallback with grad...,Use retrieved documentation as the primary evi...,Reduces catastrophic refusals such as Q15 and ...
1,2,Explicitly distinguish documented evidence fro...,Permit the model to supplement partial retriev...,Preserves the strong reasoning demonstrated by...
2,3,Introduce an evidence-sufficiency decision bef...,"Classify retrieved context as SUFFICIENT, PART...",Allows different response strategies rather th...
3,4,Use query-type-aware RAG behaviour,"For CORPUS_GROUNDED questions, apply stronger ...",Matches the generation strategy to the actual ...
4,5,Improve multi-chunk evidence synthesis,Prompt the model to explicitly consolidate com...,Reduces cases where individually useful chunks...
5,6,Prevent false 'documentation unavailable' conc...,Require the model to first identify what IS su...,Avoids rejecting an entire query merely becaus...
6,7,Retain Retriever V1 as the baseline before maj...,Do not replace BGE-M3 / FAISS solely because R...,Separates retrieval quality from generation-po...
7,8,Keep generation temperature low during enginee...,Continue using deterministic or near-determini...,Maintains reproducibility while targeting the ...
8,9,Retest the revised RAG prompt using paired abl...,"Reuse the same Track 1 questions, retriever an...",Provides causal evidence that any improvement ...



3. MOST SEVERE RAG DEGRADATION CASES


,model_family,question_id,category,evaluation_type,corpus_relevance,production_k7_support,base_overall,rag_overall,overall_delta,rag_response_status
34,Gemma 4,Q15,5G Throughput Troubleshooting,CORPUS_GROUNDED,PARTIAL,YES,9,1,-8,NON_RESPONSIVE
18,Essential AI,Q19,Network Failure Isolation,ENGINEERING_SYNTHESIS,PARTIAL,PARTIAL,8,1,-7,NON_RESPONSIVE
33,Gemma 4,Q14,5G UL/DL Trade-off,CORPUS_GROUNDED,PARTIAL-STRONG,YES,9,4,-5,SUBSTANTIVE
24,Gemma 4,Q05,5G SA Procedures,CORPUS_GROUNDED,STRONG,YES,10,5,-5,SUBSTANTIVE
36,Gemma 4,Q17,Cloud-Native Telecom,CORPUS_GROUNDED,PARTIAL-STRONG,YES,9,4,-5,SUBSTANTIVE



4. REPORT-READY TRACK 1 SUMMARY
TRACK 1 — CONSOLIDATED FINDINGS

The Track 1 benchmark evaluated five telecom AI configurations across
20 questions and 100 judged responses.

Otel 2.0 Only achieved the strongest overall performance with a mean score
of 9.20, followed by Gemma 4 Only at 9.00.
The standalone models therefore substantially outperformed the two main
RAG configurations.

RAG impact was strongly model-dependent. Essential AI showed approximately
neutral overall behaviour, moving from 5.80 without RAG
to 5.65 with RAG. Its RAG configuration improved
9 questions, tied on
4, and degraded
7.

Gemma 4 showed a substantially different pattern. Its mean score decreased
from 9.00 standalone to 6.75 with RAG.
Gemma 4 + RAG improved zero Track 1 questions, tied on
6, and degraded
14. This indicates systematic interaction
between the retrieved context and the generation policy.

Question type also influenced RAG effectiveness. Essential AI produced a
small positive RAG delta of 0.133 

## **Track 1 — Key Observations**

### 1. Otel 2.0 Only Achieved the Strongest Overall Track 1 Performance

Otel 2.0 Only achieved the highest overall Track 1 mean score at **9.20/10**, followed closely by Gemma 4 Only at **9.00/10**.

The remaining systems achieved:

- Gemma 4 + RAG: **6.75**
- Essential AI Only: **5.80**
- Essential AI + RAG: **5.65**

This indicates that the strongest engineering responses were produced by the standalone telecom-capable models rather than by the current RAG configurations.

Otel 2.0 also showed the strongest consistency, combining a high mean score with relatively low score variation across the 20 engineering questions.


### 2. Otel 2.0 and Gemma 4 Only Dominated Question-Level Wins

At the question level:

- Otel 2.0 Only achieved the top score on **17/20 questions**
- Gemma 4 Only achieved the top score on **14/20 questions**
- Gemma 4 + RAG achieved the top score on **3/20 questions**
- Essential AI + RAG achieved the top score on **1/20 question**
- Essential AI Only achieved the top score on **0/20 questions**

Because ties were allowed, more than one model could win the same question.

The result reinforces that the strongest Track 1 performance came from the standalone Otel 2.0 and Gemma 4 models.


### 3. Gemma 4 Showed Strong Standalone Telecom Engineering Capability

Gemma 4 Only achieved a mean score of **9.00**, performing strongly across both corpus-grounded and engineering-synthesis questions.

It frequently produced technically complete, practical and well-structured telecom engineering responses, particularly on questions requiring:

- fault isolation,
- multi-domain reasoning,
- standards understanding,
- root-cause analysis,
- and practical troubleshooting methodology.

This establishes a strong standalone baseline against which the effect of RAG can be evaluated.


### 4. Gemma 4 Experienced Systematic Performance Degradation After RAG

Gemma 4 Only achieved **9.00**, while Gemma 4 + RAG achieved **6.75**, representing a **2.25-point reduction** in mean score.

At question level:

- RAG improved **0/20 questions**
- RAG produced the same score on **6/20**
- RAG degraded performance on **14/20**

This is one of the clearest findings in Track 1.

The current Gemma RAG configuration did not simply fail to improve performance; it systematically interfered with an already strong standalone model.


### 5. Essential AI Showed Smaller Overall RAG Degradation but Significant Question-Level Movement

Essential AI Only achieved **5.80**, while Essential AI + RAG achieved **5.65**, a relatively small mean difference of **-0.15**.

However, the small aggregate change masks substantial question-level variation:

- RAG improved **9/20 questions**
- RAG produced the same score on **4/20**
- RAG degraded **7/20 questions**

Therefore, the near-equal mean scores should not be interpreted as RAG having little effect.

RAG materially changed the quality of individual answers, but positive and negative effects approximately offset each other at the aggregate level.


### 6. RAG Performance Was Sensitive to Question Type

Track 1 contained two different evaluation types:

- `CORPUS_GROUNDED`
- `ENGINEERING_SYNTHESIS`

The RAG systems generally performed better where the answer could be derived relatively directly from retrieved documentation.

Performance was less reliable on questions requiring broader engineering synthesis, where the model needed to combine evidence across multiple domains and apply practical telecom reasoning.

This indicates that a single strict retrieval-and-generation policy is not equally appropriate for all engineering question types.


### 7. Strict Document Grounding Became a Major Failure Mechanism

The RAG systems were intentionally designed to answer using retrieved telecom documentation and to abstain when the required information was not sufficiently supported by the retrieved context.

This improves factual grounding and reduces unsupported generation.

However, the Track 1 diagnostic showed that the grounding policy could become too restrictive.

Among the **13 priority RAG degradation cases investigated**:

- **13/13** were classified primarily as `STRICT_GROUNDING_ABSTENTION`
- Essential AI accounted for **4 cases**
- Gemma 4 accounted for **9 cases**

This indicates that the RAG systems frequently failed not because they lacked all useful evidence, but because the generation policy treated partially supported evidence too conservatively.


### 8. Retrieved Evidence Was Usually Available in the Priority Failure Cases

The detailed RAG diagnostic found that, among the 13 priority degradation cases:

- **11/13** had `PARTIAL` evidence
- **2/13** had `SUFFICIENT` evidence
- **0/13** had evidence classified as insufficient

This is an important distinction.

The dominant problem was not that the retriever consistently returned irrelevant or unusable content.

Instead, the model often failed to convert useful or partially useful retrieved evidence into a strong engineering answer.


### 9. Generation and Prompting Were Diagnosed as the Primary Failure Point in the Priority Cases

Among the 13 investigated priority RAG degradation cases:

- Retrieval-at-fault: **0/13**
- Generation/prompting-at-fault: **13/13**

All 13 cases were also classified with the secondary failure mechanism:

`GENERATION_FAILURE_DESPITE_GOOD_EVIDENCE`

This strongly suggests that, for these severe degradation cases, the immediate optimization priority should be the generation and grounding policy rather than rebuilding the retrieval layer.

This conclusion applies specifically to the investigated priority degradation cases and should not be generalized to every RAG response without further analysis.


### 10. Catastrophic Abstentions Demonstrated the Risk of Binary Grounding

Two particularly severe RAG failures occurred where the model produced non-responsive fallback behaviour despite the standalone model providing a strong answer.

Examples included:

- Gemma 4 + RAG on **Q15**
- Essential AI + RAG on **Q19**

These cases demonstrate how a binary grounding policy can transform a partially supported question into a complete abstention, causing a very large quality drop.


### 11. Groundedness Remains a Strength, but Over-Grounding Is a Failure Mode

The goal of grounding the RAG systems in controlled telecom documentation remains valid.

The benchmark does not indicate that grounding should be removed.

Instead, it shows that the current implementation is too binary:

- explicitly supported → answer
- not explicitly supported → abstain

A better design should preserve evidence-first generation while allowing the model to perform clearly identified engineering inference when retrieved evidence is partial but useful.

The desired behaviour is therefore **grounded reasoning**, not merely **document extraction**.


### 12. Track 1 Demonstrates That Retrieval Success Does Not Guarantee RAG Success

The priority diagnostic shows an important architectural lesson:

> Relevant retrieval alone is not sufficient for successful RAG.

A pipeline can retrieve useful documentation and still produce a weak response if:

- the model cannot synthesize multiple chunks,
- the prompt is overly restrictive,
- useful pretrained knowledge is suppressed,
- the grounding policy is too binary,
- or engineering reasoning is not explicitly permitted.

Track 1 therefore demonstrates that RAG quality depends on both retrieval quality and the model's ability to use retrieved evidence effectively.

## **Track 1 — Issues Identified**

### 1. The Current RAG Configuration Does Not Consistently Improve Engineering Response Quality

The primary Track 1 issue is that retrieval augmentation did not consistently improve model performance.

For Essential AI:

- Standalone mean: **5.80**
- RAG mean: **5.65**
- Delta: **-0.15**

For Gemma 4:

- Standalone mean: **9.00**
- RAG mean: **6.75**
- Delta: **-2.25**

This demonstrates that simply adding retrieved context to an LLM does not guarantee improved telecom engineering performance.


### 2. Gemma 4 RAG Exhibited Systematic Degradation

Gemma 4 RAG was particularly problematic.

Across all 20 questions:

- RAG improved **0**
- tied **6**
- degraded **14**

The standalone Gemma model was already highly capable, and the RAG pipeline frequently reduced answer quality by constraining, distracting or suppressing its existing engineering knowledge.

This indicates a strong interaction problem between the current retrieval-grounding architecture and the model's native reasoning capability.


### 3. Binary Grounding Encouraged Excessive Abstention

The current RAG prompt strongly prioritizes using only retrieved documentation.

This is appropriate for controlling unsupported generation, but the current implementation is overly binary.

When evidence is incomplete or distributed across multiple chunks, the model may treat the question as unsupported rather than synthesizing the available evidence.

The result is unnecessary fallback behaviour even when useful information is present.


### 4. Partial Evidence Was Treated Too Similarly to Missing Evidence

Among the 13 priority failure cases:

- 11 contained partial evidence
- 2 contained sufficient evidence

Despite this, all 13 were classified as strict-grounding abstention failures.

This suggests that the generation policy does not sufficiently differentiate between:

- partial evidence,
- sufficient evidence,
- and genuinely insufficient evidence.

This reduces the system's ability to reason under realistic telecom documentation conditions, where the full answer may rarely exist in one exact chunk.


### 5. Generation and Prompting Failed to Exploit Retrieved Evidence

For all 13 priority degradation cases, the diagnostic identified generation or prompting as the dominant fault rather than retrieval.

This means the system frequently had useful evidence available but failed to convert it into a high-quality answer.

Possible causes include:

- overly restrictive system instructions,
- insufficient multi-chunk synthesis,
- excessive literalism,
- fallback triggering too early,
- and suppression of model reasoning.


### 6. The Same Grounding Policy Was Applied to Different Engineering Task Types

Track 1 included both directly documented questions and broader engineering-synthesis questions.

These require different response strategies.

For `CORPUS_GROUNDED` questions, strict evidence dependence is appropriate.

For `ENGINEERING_SYNTHESIS` questions, the model may need to combine:

- RAN evidence,
- transport evidence,
- 5GC evidence,
- Kubernetes/platform evidence,
- configuration evidence,
- and practical engineering reasoning.

Applying the same document-only policy to both task classes limits the system's usefulness.


### 7. Retrieved Context Can Override Strong Pretrained Knowledge

The Gemma 4 results show that retrieval can negatively influence a highly capable standalone model.

Possible mechanisms include:

- retrieved chunks containing incomplete details,
- context ordering bias,
- irrelevant but semantically similar evidence,
- excessive model attention to retrieved content,
- and explicit instructions preventing use of valid pretrained knowledge.

The result is that external knowledge can sometimes displace rather than complement correct internal model knowledge.


### 8. RAG Increased Variability and Reduced Stability

The RAG systems produced more variable performance across questions.

Gemma 4 + RAG had a much wider spread in scores than Gemma 4 Only.

This indicates that the current RAG architecture is more sensitive to:

- retrieval quality,
- query formulation,
- chunk composition,
- evidence completeness,
- and prompt interpretation.

A production system requires more predictable behaviour.


### 9. Non-Responsive Failures Carry Disproportionately High Cost

The severe fallback cases demonstrate that abstention is not merely a small reduction in answer quality.

A strong standalone answer can become effectively unusable once the RAG system refuses to answer.

For engineering assistants, such non-responsive behaviour can be more operationally damaging than a partially incomplete but otherwise technically useful answer.


### 10. Similarity-Based Retrieval Alone Cannot Determine Evidence Sufficiency

High semantic similarity does not guarantee that a retrieved chunk contains the exact evidence needed to answer a question.

Likewise, a partial answer may be spread across several lower-ranked chunks.

Therefore, Retriever V1 similarity scores should not be interpreted directly as proof that a question is either fully supported or unsupported.

A separate evidence-sufficiency decision is needed.


### 11. RAG Evaluation Must Distinguish Retrieval Failure from Generation Failure

Without the targeted diagnostic, poor RAG scores could easily be attributed to the retriever.

Track 1 demonstrates that this would have been misleading for the priority failure cases.

Future evaluation should explicitly separate:

- retrieval relevance,
- evidence sufficiency,
- generation quality,
- grounding behaviour,
- and final answer correctness.

This enables targeted architecture improvements rather than changing multiple components unnecessarily.


### 12. Track 1 Results Should Not Be Interpreted as Evidence That RAG Is Inherently Inferior

The results evaluate the current RAG implementation, not the theoretical value of RAG.

The benchmark indicates that the present combination of:

- retrieval,
- strict grounding,
- prompt behaviour,
- and generation strategy

is suboptimal for several Track 1 tasks.

The appropriate conclusion is therefore to redesign the interaction between retrieval and generation rather than to abandon RAG.

## Track 1 — Recommendations

### 1. Replace Binary Document-Only Grounding with Graded Grounding

The highest-priority change should be to replace the current binary grounding policy.

Instead of:

- answer only when explicitly documented
- otherwise return fallback

the model should distinguish:

1. **Directly supported evidence**
2. **Partially supported evidence**
3. **Reasonable engineering inference**
4. **Unsupported claims**
5. **Insufficient evidence requiring abstention**

This preserves document grounding while reducing unnecessary refusals.


### 2. Introduce an Explicit Evidence-Sufficiency Stage

Before generation, the RAG pipeline should classify retrieved context as:

- `SUFFICIENT`
- `PARTIAL`
- `INSUFFICIENT`

The generation strategy should then change accordingly.

**SUFFICIENT**
- answer directly from retrieved evidence.

**PARTIAL**
- synthesize available evidence and allow clearly identified engineering reasoning.

**INSUFFICIENT**
- abstain or request additional information.

This directly addresses the dominant failure pattern observed in the 13 priority degradation cases.


### 3. Distinguish Retrieved Evidence from Engineering Inference in the Final Answer

The model should be allowed to supplement retrieved documentation with technical reasoning when appropriate, but the distinction should be clear.

For example:

- documented fact,
- evidence-based interpretation,
- engineering inference,
- recommendation.

This provides transparency without forcing the system into unnecessary abstention.


### 4. Introduce Query-Type-Aware RAG Behaviour

The system should first classify the question.

For `CORPUS_GROUNDED` questions:

- documentation should remain the primary authority,
- citations or evidence should dominate,
- unsupported claims should be minimized.

For `ENGINEERING_SYNTHESIS` questions:

- retrieved evidence should inform the answer,
- but the model should also be allowed to perform domain reasoning,
- synthesize across multiple layers,
- and construct practical fault-isolation methodologies.

This is particularly important for multi-domain telecom troubleshooting questions.


### 5. Preserve Retriever V1 for the Next Controlled Experiment

The current BGE-M3 + FAISS Retriever V1 should remain unchanged during the next RAG experiment.

Among the 13 investigated priority degradation cases:

- 11 had partial evidence
- 2 had sufficient evidence
- 0 were classified primarily as retrieval failures

Therefore, changing the retriever at the same time as the grounding strategy would make it difficult to identify the cause of any improvement.

The next ablation should focus on generation behaviour.


### 6. Test Revised Graded Grounding Against the Current Strict Baseline

The next controlled experiment should compare:

**Current Baseline**
- strict context-only answering
- binary fallback
- limited use of external model reasoning

**Revised RAG**
- evidence-first generation
- evidence-sufficiency classification
- graded grounding
- multi-chunk synthesis
- clearly distinguished engineering inference
- abstention only when evidence is genuinely insufficient

The same questions, Retriever V1, Top-K and models should be retained for a clean comparison.


### 7. Improve Multi-Chunk Evidence Synthesis

Telecom engineering answers often span multiple documents or network domains.

The RAG generator should be explicitly instructed to:

- combine related evidence across retrieved chunks,
- reconcile overlapping information,
- identify complementary evidence,
- and construct a coherent engineering response.

This is especially important for fault-isolation and troubleshooting questions.


### 8. Avoid Automatically Suppressing Strong Model Knowledge

For models with strong telecom capability, such as Gemma 4 and Otel 2.0, retrieved evidence should complement rather than automatically replace pretrained knowledge.

A safer hybrid policy is:

- retrieved evidence is authoritative where available,
- model knowledge may be used for reasoning,
- unsupported factual claims should be marked or avoided,
- and conflicts between model knowledge and retrieved documentation should favor the controlled corpus.

This reduces the risk of RAG degrading an otherwise correct standalone answer.


### 9. Introduce Selective RAG Routing

Not every telecom question requires retrieval.

A routing layer could classify questions as:

- documentation lookup,
- standards lookup,
- engineering synthesis,
- calculation,
- log diagnosis,
- troubleshooting,
- general telecom knowledge.

RAG should be activated when the expected benefit of retrieved evidence is high.

Standalone reasoning or specialized tools may be more appropriate for other task types.


### 10. Continue Using Low-Temperature Generation

The Track 1 findings do not suggest that simply increasing temperature will solve the RAG problem.

The dominant failures were linked to grounding and generation policy rather than insufficient creativity.

Low-temperature generation around **0.0–0.1** should therefore remain appropriate while the prompting and evidence-use strategy is redesigned.


### 11. Add Retrieval and Generation Diagnostics to Every Future RAG Evaluation

Future evaluation should retain explicit diagnostic fields such as:

- retrieval relevance,
- retrieved chunk count,
- evidence sufficiency,
- retrieval-at-fault,
- generation-at-fault,
- failure mechanism,
- abstention reason.

This allows RAG failures to be traced to the correct system component.


### 12. Evaluate End-to-End RAG Quality Separately from Retriever Quality

Retriever quality and final RAG quality should be treated as related but separate metrics.

A retriever may return useful evidence while the final answer remains weak.

Future reporting should therefore distinguish:

- retrieval success,
- evidence sufficiency,
- generation quality,
- grounding correctness,
- and final engineering usefulness.


### 13. Prioritize the Gemma 4 RAG Degradation Cases for Controlled Ablation

Gemma 4 provides the clearest test case because:

- standalone mean = **9.00**
- RAG mean = **6.75**
- RAG better = **0**
- RAG worse = **14**

This makes Gemma 4 an ideal model for testing whether a revised grounding policy can recover standalone capability while preserving retrieval benefits.


### 14. Preserve Groundedness as a Core Design Principle

The solution should not remove grounding.

The objective remains to ensure that telecom answers are supported by controlled, authoritative documentation.

The architecture should instead evolve from:

> strict document-only generation

to:

> evidence-first, grounded engineering reasoning.

This approach preserves factual reliability while allowing the model to operate effectively when retrieved evidence is partial, distributed or requires technical synthesis.


### 15. Treat RAG as a Selective Knowledge Architecture Rather Than a Universal Model Upgrade

Track 1 demonstrates that RAG is not automatically beneficial simply because additional context is provided.

Its value depends on:

- retrieval relevance,
- evidence completeness,
- task type,
- corpus coverage,
- prompt design,
- grounding policy,
- generation behaviour,
- and model capability.

The preferred production architecture should therefore use **selective, task-aware RAG with graded grounding and engineering synthesis**, rather than forcing every telecom query through a strict document-only response path.

# **Build and Validate the Unified Track 2 Evaluation Dataset**

In [23]:
# =============================================================================
# MODULE 6.1 — BUILD AND VALIDATE UNIFIED TRACK 2 EVALUATION DATASET
# =============================================================================
#
# PURPOSE
# -------
# Build the complete Track 2 evaluation dataframe across all five systems
# before independent LLM judging.
#
# TRACK 2
# -------
# Industry-curated telecom benchmark
#
# Expected:
#
#     32 questions
#     5 systems
#     160 total responses
#
# SYSTEMS
# -------
# 1. Essential AI + RAG
# 2. Essential AI Only
# 3. Otel 2.0 Only
# 4. Gemma 4 Only
# 5. Gemma 4 + RAG
#
# IMPORTANT
# ---------
# - No inference is rerun.
# - Existing saved responses are reused.
# - The same judge model and frozen Track 1 rubric will be used later.
# - This module ONLY prepares and validates Track 2.
#
# =============================================================================


import pandas as pd
import numpy as np
import json
from collections import Counter


# =============================================================================
# 1. VALIDATE LOADED RESPONSE OBJECT
# =============================================================================

if "loaded_responses" not in globals():

    raise RuntimeError(
        "loaded_responses not found. "
        "Run the response-loading module first."
    )


if "Track 2" not in loaded_responses:

    raise RuntimeError(
        "Track 2 not found inside loaded_responses."
    )


track2_loaded = (
    loaded_responses[
        "Track 2"
    ]
)


print("=" * 105)
print("MODULE 6.1 — TRACK 2 UNIFIED EVALUATION DATASET")
print("=" * 105)

print(
    f"\nLoaded Track 2 systems : "
    f"{len(track2_loaded)}"
)


# =============================================================================
# 2. INSPECT AVAILABLE SYSTEM KEYS
# =============================================================================

print(
    "\nAvailable Track 2 systems:"
)


for system_name in (
    track2_loaded.keys()
):

    print(
        f"• {system_name}"
    )


# =============================================================================
# 3. EXPECTED SYSTEM NAMES
# =============================================================================
#
# These should match the canonical model names used in Track 1.
#
# =============================================================================

EXPECTED_TRACK2_SYSTEMS = [

    "Essential AI + RAG",

    "Essential AI Only",

    "Otel 2.0 Only",

    "Gemma 4 Only",

    "Gemma 4 + RAG",
]


missing_systems = [

    system_name

    for system_name
    in EXPECTED_TRACK2_SYSTEMS

    if system_name
    not in track2_loaded
]


if missing_systems:

    raise RuntimeError(
        "Missing Track 2 systems: "
        f"{missing_systems}"
    )


# =============================================================================
# 4. HELPER — GET FIRST AVAILABLE FIELD
# =============================================================================

def first_available(
    item,
    field_names,
    default=None,
):

    for field_name in field_names:

        if (
            field_name in item
            and
            item[
                field_name
            ]
            is not None
        ):

            return (
                item[
                    field_name
                ]
            )


    return default


# =============================================================================
# 5. HELPER — NORMALISE RETRIEVAL CHUNKS
# =============================================================================

def normalise_retrieval_chunks(
    item,
):

    possible_fields = [

        "retrieved_chunks",
        "retrieval_results",
        "retrieved_documents",
        "source_documents",
        "context_chunks",
        "top_k_results",
        "top7_retrieved_chunks",
    ]


    chunks = None


    for field in possible_fields:

        if (
            field in item
            and
            item[
                field
            ] is not None
        ):

            chunks = (
                item[
                    field
                ]
            )

            break


    if chunks is None:

        return []


    if not isinstance(
        chunks,
        list,
    ):

        return []


    normalised = []


    for rank, chunk in enumerate(
        chunks,
        start=1,
    ):

        if isinstance(
            chunk,
            dict,
        ):

            normalised.append(
                {

                    "rank":
                        chunk.get(
                            "rank",
                            rank,
                        ),

                    "score":
                        first_available(
                            chunk,
                            [
                                "score",
                                "similarity_score",
                                "similarity",
                                "distance",
                            ],
                        ),

                    "source":
                        first_available(
                            chunk,
                            [
                                "source",
                                "file",
                                "filename",
                                "document",
                            ],
                        ),

                    "title":
                        first_available(
                            chunk,
                            [
                                "title",
                                "section",
                                "heading",
                            ],
                        ),

                    "chunk_id":
                        first_available(
                            chunk,
                            [
                                "chunk_id",
                                "id",
                                "document_id",
                            ],
                        ),

                    "text":
                        first_available(
                            chunk,
                            [
                                "text",
                                "content",
                                "page_content",
                                "chunk_text",
                            ],
                            default="",
                        ),
                }
            )


        else:

            normalised.append(
                {

                    "rank":
                        rank,

                    "score":
                        None,

                    "source":
                        None,

                    "title":
                        None,

                    "chunk_id":
                        None,

                    "text":
                        str(
                            chunk
                        ),
                }
            )


    return normalised


# =============================================================================
# 6. BUILD UNIFIED TRACK 2 RECORDS
# =============================================================================

track2_records = []


for model_name in (
    EXPECTED_TRACK2_SYSTEMS
):

    system_results = (
        track2_loaded[
            model_name
        ]
    )


    print(
        f"\nProcessing "
        f"{model_name:<22} : "
        f"{len(system_results)} responses"
    )


    for result in system_results:

        question_id = (
            first_available(
                result,
                [
                    "question_id",
                    "id",
                    "qid",
                ],
            )
        )


        question = (
            first_available(
                result,
                [
                    "question",
                    "query",
                    "prompt",
                    "user_query",
                ],
                default="",
            )
        )


        category = (
            first_available(
                result,
                [
                    "category",
                    "topic",
                    "domain",
                ],
                default="",
            )
        )


        expected_points = (
            first_available(
                result,
                [
                    "expected_points",
                    "key_points",
                    "reference_points",
                    "expected_answer_points",
                ],
                default=[],
            )
        )


        response_text = (
            first_available(
                result,
                [
                    "response",
                    "answer",
                    "generated_response",
                    "output",
                    "model_response",
                ],
                default="",
            )
        )


        retrieval_chunks = (
            normalise_retrieval_chunks(
                result
            )
        )


        is_rag = (
            "+ RAG"
            in model_name
        )


        track2_records.append(
            {

                "track":
                    "Track 2",

                "question_id":
                    question_id,

                "category":
                    category,

                "question":
                    question,

                "expected_points":
                    expected_points,

                "model_name":
                    model_name,

                "is_rag":
                    is_rag,

                "response":
                    response_text,

                "retrieval_chunk_count":
                    len(
                        retrieval_chunks
                    ),

                "top7_retrieved_chunks":
                    retrieval_chunks,

                "raw_result":
                    result,
            }
        )


# =============================================================================
# 7. CREATE DATAFRAME
# =============================================================================

track2_evaluation_df = (
    pd.DataFrame(
        track2_records
    )
)


# =============================================================================
# 8. SORT QUESTION IDS SAFELY
# =============================================================================

def track2_question_sort_key(
    question_id,
):

    if question_id is None:

        return 9999


    text = str(
        question_id
    )


    digits = "".join(

        char

        for char in text

        if char.isdigit()
    )


    if digits:

        return int(
            digits
        )


    return 9999


track2_evaluation_df[
    "_question_order"
] = (

    track2_evaluation_df[
        "question_id"
    ]

    .apply(
        track2_question_sort_key
    )
)


track2_evaluation_df = (

    track2_evaluation_df

    .sort_values(
        [
            "_question_order",
            "model_name",
        ]
    )

    .drop(
        columns=[
            "_question_order"
        ]
    )

    .reset_index(
        drop=True
    )
)


# =============================================================================
# 9. CORE DATASET SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 105
)

print(
    "1. TRACK 2 DATASET SUMMARY"
)

print(
    "=" * 105
)


print(
    f"\nRows              : "
    f"{len(track2_evaluation_df)}"
)

print(
    f"Unique questions  : "
    f"{track2_evaluation_df['question_id'].nunique()}"
)

print(
    f"Unique models     : "
    f"{track2_evaluation_df['model_name'].nunique()}"
)

print(
    f"RAG responses     : "
    f"{int(track2_evaluation_df['is_rag'].sum())}"
)

print(
    f"Non-RAG responses : "
    f"{int((~track2_evaluation_df['is_rag']).sum())}"
)


# =============================================================================
# 10. RESPONSES PER MODEL
# =============================================================================

model_counts = (

    track2_evaluation_df[
        "model_name"
    ]

    .value_counts()

    .rename_axis(
        "model_name"
    )

    .reset_index(
        name=
            "responses"
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "2. RESPONSES PER MODEL"
)

print(
    "=" * 105
)


display(
    model_counts
)


# =============================================================================
# 11. RESPONSES PER QUESTION
# =============================================================================

question_counts = (

    track2_evaluation_df

    .groupby(
        "question_id"
    )

    .size()

    .reset_index(
        name=
            "responses"
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "3. RESPONSES PER QUESTION"
)

print(
    "=" * 105
)


display(
    question_counts
)


# =============================================================================
# 12. RAG RETRIEVAL METADATA VALIDATION
# =============================================================================

rag_rows = (

    track2_evaluation_df[

        track2_evaluation_df[
            "is_rag"
        ]

    ]

    .copy()
)


non_rag_rows = (

    track2_evaluation_df[

        ~track2_evaluation_df[
            "is_rag"
        ]

    ]

    .copy()
)


rag_with_retrieval = int(

    (
        rag_rows[
            "retrieval_chunk_count"
        ]
        > 0
    )
    .sum()
)


rag_with_top7 = int(

    (
        rag_rows[
            "retrieval_chunk_count"
        ]
        >= 7
    )
    .sum()
)


print(
    "\n"
    + "=" * 105
)

print(
    "4. RAG RETRIEVAL METADATA VALIDATION"
)

print(
    "=" * 105
)


print(
    f"\nRAG rows                    : "
    f"{len(rag_rows)}"
)

print(
    f"RAG rows with retrieval     : "
    f"{rag_with_retrieval}"
)

print(
    f"RAG rows with ≥7 chunks     : "
    f"{rag_with_top7}"
)

print(
    f"Non-RAG rows                : "
    f"{len(non_rag_rows)}"
)


# =============================================================================
# 13. CHECK EMPTY RESPONSES
# =============================================================================

track2_evaluation_df[
    "response_is_empty"
] = (

    track2_evaluation_df[
        "response"
    ]

    .fillna(
        ""
    )

    .astype(
        str
    )

    .str
    .strip()

    .eq(
        ""
    )
)


empty_response_rows = (

    track2_evaluation_df[

        track2_evaluation_df[
            "response_is_empty"
        ]

    ]
)


print(
    "\n"
    + "=" * 105
)

print(
    "5. EMPTY RESPONSE CHECK"
)

print(
    "=" * 105
)


print(
    f"\nEmpty responses : "
    f"{len(empty_response_rows)}"
)


if len(
    empty_response_rows
) > 0:

    display(
        empty_response_rows[
            [
                "question_id",
                "model_name",
            ]
        ]
    )


# =============================================================================
# 14. CHECK DUPLICATES
# =============================================================================

duplicate_rows = (

    track2_evaluation_df[

        track2_evaluation_df
        .duplicated(
            subset=[
                "question_id",
                "model_name",
            ],
            keep=False,
        )

    ]
)


print(
    "\n"
    + "=" * 105
)

print(
    "6. DUPLICATE CHECK"
)

print(
    "=" * 105
)


print(
    f"\nDuplicate question/model rows : "
    f"{len(duplicate_rows)}"
)


if len(
    duplicate_rows
) > 0:

    display(
        duplicate_rows[
            [
                "question_id",
                "model_name",
            ]
        ]
    )


# =============================================================================
# 15. CHECK QUESTION ALIGNMENT
# =============================================================================

question_model_matrix = (

    track2_evaluation_df

    .pivot_table(
        index=
            "question_id",

        columns=
            "model_name",

        values=
            "response",

        aggfunc=
            "count",

        fill_value=
            0,
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "7. QUESTION × MODEL COVERAGE"
)

print(
    "=" * 105
)


display(
    question_model_matrix
)


# =============================================================================
# 16. CATEGORY DISTRIBUTION
# =============================================================================

category_summary = (

    track2_evaluation_df[
        [
            "question_id",
            "category",
        ]
    ]

    .drop_duplicates()

    .groupby(
        "category"
    )

    .size()

    .reset_index(
        name=
            "questions"
    )

    .sort_values(
        "questions",
        ascending=False,
    )
)


print(
    "\n"
    + "=" * 105
)

print(
    "8. TRACK 2 CATEGORY DISTRIBUTION"
)

print(
    "=" * 105
)


display(
    category_summary
)


# =============================================================================
# 17. SAMPLE RECORDS
# =============================================================================

print(
    "\n"
    + "=" * 105
)

print(
    "9. SAMPLE TRACK 2 RECORDS"
)

print(
    "=" * 105
)


display(

    track2_evaluation_df[
        [
            "question_id",
            "category",
            "model_name",
            "is_rag",
            "retrieval_chunk_count",
            "response_is_empty",
        ]
    ]

    .head(
        15
    )
)


# =============================================================================
# 18. FINAL VALIDATION
# =============================================================================

validation_checks = {

    "160 total responses":
        (
            len(
                track2_evaluation_df
            )
            == 160
        ),

    "32 unique questions":
        (
            track2_evaluation_df[
                "question_id"
            ]
            .nunique()
            == 32
        ),

    "5 unique models":
        (
            track2_evaluation_df[
                "model_name"
            ]
            .nunique()
            == 5
        ),

    "32 responses per model":
        (
            model_counts[
                "responses"
            ]
            .eq(
                32
            )
            .all()
        ),

    "5 responses per question":
        (
            question_counts[
                "responses"
            ]
            .eq(
                5
            )
            .all()
        ),

    "No duplicate question/model rows":
        (
            len(
                duplicate_rows
            )
            == 0
        ),

    "No empty responses":
        (
            len(
                empty_response_rows
            )
            == 0
        ),

    "64 RAG responses":
        (
            len(
                rag_rows
            )
            == 64
        ),

    "96 standalone responses":
        (
            len(
                non_rag_rows
            )
            == 96
        ),
}


print(
    "\n"
    + "=" * 105
)

print(
    "MODULE 6.1 VALIDATION"
)

print(
    "=" * 105
)


for check_name, passed in (
    validation_checks.items()
):

    print(
        f"{check_name:<45} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 6.1 validation failed. "
        "Resolve Track 2 dataset alignment before judging."
    )


# =============================================================================
# 19. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 105
)

print(
    "TRACK 2 UNIFIED EVALUATION DATASET READY"
)

print(
    "=" * 105
)


print(
    f"\nRows             : "
    f"{len(track2_evaluation_df)}"
)

print(
    f"Questions        : "
    f"{track2_evaluation_df['question_id'].nunique()}"
)

print(
    f"Models           : "
    f"{track2_evaluation_df['model_name'].nunique()}"
)


print(
    "\nGenerated object:"
)

print(
    "• track2_evaluation_df"
)


print(
    "\nNext:"
)

print(
    "Module 6.2 — Track 2 Independent LLM Judge Evaluation"
)


print(
    "\nMODULE 6.1 COMPLETE"
)

print(
    "=" * 105
)

MODULE 6.1 — TRACK 2 UNIFIED EVALUATION DATASET

Loaded Track 2 systems : 5

Available Track 2 systems:
• Essential AI + RAG
• Essential AI Only
• Otel 2.0 Only
• Gemma 4 Only
• Gemma 4 + RAG

Processing Essential AI + RAG     : 32 responses

Processing Essential AI Only      : 32 responses

Processing Otel 2.0 Only          : 32 responses

Processing Gemma 4 Only           : 32 responses

Processing Gemma 4 + RAG          : 32 responses

1. TRACK 2 DATASET SUMMARY

Rows              : 160
Unique questions  : 32
Unique models     : 5
RAG responses     : 64
Non-RAG responses : 96

2. RESPONSES PER MODEL


,model_name,responses
0,Essential AI + RAG,32
1,Essential AI Only,32
2,Gemma 4 + RAG,32
3,Gemma 4 Only,32
4,Otel 2.0 Only,32



3. RESPONSES PER QUESTION


,question_id,responses
0,T2-01,5
1,T2-02,5
2,T2-03,5
3,T2-04,5
4,T2-05,5
5,T2-06,5
6,T2-07,5
7,T2-08,5
8,T2-09,5
9,T2-10,5



4. RAG RETRIEVAL METADATA VALIDATION

RAG rows                    : 64
RAG rows with retrieval     : 0
RAG rows with ≥7 chunks     : 0
Non-RAG rows                : 96

5. EMPTY RESPONSE CHECK

Empty responses : 9


...

,category,questions
0,,32
1,General,32



9. SAMPLE TRACK 2 RECORDS


,question_id,category,model_name,is_rag,retrieval_chunk_count,response_is_empty
0,T2-01,,Essential AI + RAG,True,0,True
1,T2-01,,Essential AI Only,False,0,False
2,T2-01,,Gemma 4 + RAG,True,0,False
3,T2-01,General,Gemma 4 Only,False,0,False
4,T2-01,,Otel 2.0 Only,False,0,False
5,T2-02,,Essential AI + RAG,True,0,True
6,T2-02,,Essential AI Only,False,0,False
7,T2-02,,Gemma 4 + RAG,True,0,False
8,T2-02,General,Gemma 4 Only,False,0,False
9,T2-02,,Otel 2.0 Only,False,0,False



MODULE 6.1 VALIDATION
160 total responses                           : ✅ PASS
32 unique questions                           : ✅ PASS
5 unique models                               : ✅ PASS
32 responses per model                        : ✅ PASS
5 responses per question                      : ✅ PASS
No duplicate question/model rows              : ✅ PASS
No empty responses                            : ❌ FAIL
64 RAG responses                              : ✅ PASS
96 standalone responses                       : ✅ PASS


RuntimeError: Module 6.1 validation failed. Resolve Track 2 dataset alignment before judging.

In [24]:
# =============================================================================
# MODULE 6.1A — INSPECT TRACK 2 RAW JSON STRUCTURE
# =============================================================================
#
# PURPOSE
# -------
# Identify the exact response and retrieval field names used by each
# Track 2 system before rebuilding track2_evaluation_df.
#
# No inference or judging is performed.
#
# =============================================================================

import json
import pandas as pd


print("=" * 110)
print("MODULE 6.1A — TRACK 2 RAW JSON STRUCTURE INSPECTION")
print("=" * 110)


# =============================================================================
# 1. TOP-LEVEL STRUCTURE FOR EACH SYSTEM
# =============================================================================

for model_name, results in track2_loaded.items():

    print("\n" + "=" * 110)
    print(f"SYSTEM: {model_name}")
    print("=" * 110)

    print(f"Records: {len(results)}")

    if len(results) == 0:

        print("No records.")
        continue

    first_record = results[0]

    print("\nFirst-record keys:")

    for key in first_record.keys():

        value = first_record[key]

        print(
            f"  {key:<35} "
            f"type={type(value).__name__:<15} "
            f"preview={str(value)[:180]!r}"
        )


# =============================================================================
# 2. ESSENTIAL AI + RAG — INSPECT THE NINE APPARENTLY EMPTY CASES
# =============================================================================

problem_ids = [

    "T2-01",
    "T2-02",
    "T2-03",
    "T2-17",
    "T2-18",
    "T2-19",
    "T2-20",
    "T2-22",
    "T2-32",
]


essential_rag_results = (
    track2_loaded[
        "Essential AI + RAG"
    ]
)


print("\n" + "=" * 110)
print("ESSENTIAL AI + RAG — APPARENTLY EMPTY RESPONSE CASES")
print("=" * 110)


for record in essential_rag_results:

    qid = (
        record.get("question_id")
        or record.get("id")
        or record.get("qid")
    )

    if qid not in problem_ids:

        continue

    print("\n" + "-" * 110)
    print(f"QUESTION: {qid}")
    print("-" * 110)

    for key, value in record.items():

        print(
            f"\nKEY: {key}"
        )

        print(
            f"TYPE: {type(value).__name__}"
        )

        if isinstance(value, (dict, list)):

            try:

                preview = json.dumps(
                    value,
                    indent=2,
                    ensure_ascii=False,
                )

            except Exception:

                preview = str(value)

        else:

            preview = str(value)

        print(
            preview[:3000]
        )


# =============================================================================
# 3. INSPECT ONE RAG RECORD FROM EACH FAMILY IN DETAIL
# =============================================================================

print("\n" + "=" * 110)
print("FULL RAG RECORD STRUCTURE — FIRST RECORD OF EACH RAG SYSTEM")
print("=" * 110)


for model_name in [

    "Essential AI + RAG",
    "Gemma 4 + RAG",

]:

    record = (
        track2_loaded[
            model_name
        ][0]
    )

    print("\n" + "-" * 110)
    print(model_name)
    print("-" * 110)

    print(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False,
            default=str,
        )[:12000]
    )


# =============================================================================
# 4. AUTOMATIC FIELD INVENTORY
# =============================================================================

field_inventory = []


for model_name, results in track2_loaded.items():

    all_keys = set()

    non_null_counts = {}


    for record in results:

        for key, value in record.items():

            all_keys.add(key)

            if value is not None:

                if isinstance(value, str):

                    if value.strip() != "":

                        non_null_counts[key] = (
                            non_null_counts.get(key, 0)
                            + 1
                        )

                elif isinstance(value, (list, dict)):

                    if len(value) > 0:

                        non_null_counts[key] = (
                            non_null_counts.get(key, 0)
                            + 1
                        )

                else:

                    non_null_counts[key] = (
                        non_null_counts.get(key, 0)
                        + 1
                    )


    for key in sorted(all_keys):

        field_inventory.append(
            {
                "model_name":
                    model_name,

                "field":
                    key,

                "non_empty_records":
                    non_null_counts.get(
                        key,
                        0,
                    ),

                "total_records":
                    len(results),
            }
        )


track2_field_inventory_df = (
    pd.DataFrame(
        field_inventory
    )
)


print("\n" + "=" * 110)
print("FIELD INVENTORY BY SYSTEM")
print("=" * 110)


display(
    track2_field_inventory_df
)


# =============================================================================
# 5. LOOK FOR LIKELY RESPONSE / RETRIEVAL FIELD NAMES
# =============================================================================

search_terms = [

    "response",
    "answer",
    "output",
    "generation",
    "text",

    "retriev",
    "context",
    "chunk",
    "document",
    "source",

]


likely_fields = (

    track2_field_inventory_df[

        track2_field_inventory_df[
            "field"
        ]
        .str.lower()
        .apply(
            lambda field:
                any(
                    term in field
                    for term in search_terms
                )
        )

    ]

    .sort_values(
        [
            "model_name",
            "field",
        ]
    )
)


print("\n" + "=" * 110)
print("LIKELY RESPONSE / RETRIEVAL FIELDS")
print("=" * 110)


display(
    likely_fields
)


print("\nMODULE 6.1A COMPLETE")
print("=" * 110)

MODULE 6.1A — TRACK 2 RAW JSON STRUCTURE INSPECTION

SYSTEM: Essential AI + RAG
Records: 32

First-record keys:
  question_id                         type=str             preview='T2-01'
  benchmark                           type=str             preview='3gpp_tsg'
  question                            type=str             preview='As a distinguished expert in telecommunication domain you are skilled in understanding and classifying 3GPP techincal documents. Please help user to classify text into 3GPP working'
  choices                             type=NoneType        preview='None'
  expected_answer                     type=str             preview='RAN1'
  explanation                         type=NoneType        preview='None'
  candidate_selection_score           type=float           preview='3.0'
  retriever                           type=dict            preview="{'version': 'V1', 'k': 7}"
  status                              type=str             preview='FAIL'
  answer             

,model_name,field,non_empty_records,total_records
0,Essential AI + RAG,answer,23,32
1,Essential AI + RAG,benchmark,32,32
2,Essential AI + RAG,candidate_selection_score,32,32
3,Essential AI + RAG,choices,20,32
4,Essential AI + RAG,error,9,32
...,...,...,...,...
71,Gemma 4 + RAG,retrieval,32,32
72,Gemma 4 + RAG,retriever,32,32
73,Gemma 4 + RAG,status,32,32
74,Gemma 4 + RAG,timestamp_utc,32,32



LIKELY RESPONSE / RETRIEVAL FIELDS


,model_name,field,non_empty_records,total_records
0,Essential AI + RAG,answer,23,32
5,Essential AI + RAG,expected_answer,32,32
7,Essential AI + RAG,generation_config,32,32
8,Essential AI + RAG,generation_time_sec,32,32
10,Essential AI + RAG,output_tokens,23,32
13,Essential AI + RAG,retrieval,23,32
14,Essential AI + RAG,retriever,32,32
17,Essential AI Only,answer,32,32
22,Essential AI Only,expected_answer,32,32
24,Essential AI Only,generation_config,32,32



MODULE 6.1A COMPLETE


In [25]:
# =============================================================================
# MODULE 6.1 — BUILD AND VALIDATE UNIFIED TRACK 2 EVALUATION DATASET
#              FINAL STRUCTURE-AWARE VERSION
# =============================================================================
#
# PURPOSE
# -------
# Build the complete Track 2 evaluation dataframe across all five systems.
#
# IMPORTANT FINDINGS FROM RAW JSON INSPECTION
# ------------------------------------------
# 1. Track 2 contains:
#
#       32 questions
#       5 systems
#       160 system/question records
#
# 2. Essential AI + RAG contains 9 genuine generation failures caused by
#    CUDA OutOfMemoryError.
#
# 3. Those failures MUST NOT be rerun or silently removed.
#
# 4. They are represented as NON_RESPONSIVE benchmark outcomes.
#
# 5. Successful RAG records store retrieval metadata under a DICTIONARY
#    field called:
#
#       retrieval
#
#    rather than directly as a list of chunks.
#
# 6. Question metadata is canonicalised from Essential AI Only because
#    that result file contains complete:
#
#       question_id
#       benchmark
#       question
#       choices
#       expected_answer
#       explanation
#
#    for all 32 questions.
#
# =============================================================================


import pandas as pd
import numpy as np
import json


# =============================================================================
# 1. EXPECTED SYSTEMS
# =============================================================================

EXPECTED_TRACK2_SYSTEMS = [

    "Essential AI + RAG",
    "Essential AI Only",
    "Otel 2.0 Only",
    "Gemma 4 Only",
    "Gemma 4 + RAG",
]


# =============================================================================
# 2. INPUT VALIDATION
# =============================================================================

if "loaded_responses" not in globals():

    raise RuntimeError(
        "loaded_responses not found."
    )


if "Track 2" not in loaded_responses:

    raise RuntimeError(
        "Track 2 not found inside loaded_responses."
    )


track2_loaded = (
    loaded_responses[
        "Track 2"
    ]
)


missing_systems = [

    model_name

    for model_name
    in EXPECTED_TRACK2_SYSTEMS

    if model_name
    not in track2_loaded
]


if missing_systems:

    raise RuntimeError(
        f"Missing Track 2 systems: "
        f"{missing_systems}"
    )


print("=" * 110)
print("MODULE 6.1 — TRACK 2 UNIFIED EVALUATION DATASET")
print("=" * 110)


# =============================================================================
# 3. BUILD CANONICAL QUESTION METADATA
# =============================================================================
#
# Essential AI Only contains the complete Track 2 benchmark metadata and
# succeeded for all 32 questions.
#
# This prevents category/question metadata from varying across model files.
#
# =============================================================================

canonical_source = (
    track2_loaded[
        "Essential AI Only"
    ]
)


track2_question_metadata = {}


for item in canonical_source:

    question_id = (
        item[
            "question_id"
        ]
    )


    track2_question_metadata[
        question_id
    ] = {

        "question_id":
            question_id,

        "benchmark":
            item.get(
                "benchmark"
            ),

        "question":
            item.get(
                "question",
                "",
            ),

        "choices":
            item.get(
                "choices"
            ),

        "expected_answer":
            item.get(
                "expected_answer"
            ),

        "explanation":
            item.get(
                "explanation"
            ),

        "candidate_selection_score":
            item.get(
                "candidate_selection_score"
            ),
    }


print(
    f"\nCanonical questions loaded : "
    f"{len(track2_question_metadata)}"
)


if len(
    track2_question_metadata
) != 32:

    raise RuntimeError(
        "Expected 32 canonical Track 2 questions."
    )


# =============================================================================
# 4. HELPER — QUESTION ID
# =============================================================================

def extract_question_id(
    item,
):

    return (

        item.get(
            "question_id"
        )

        or

        item.get(
            "id"
        )
    )


# =============================================================================
# 5. HELPER — RESPONSE TEXT
# =============================================================================

def extract_track2_response(
    model_name,
    item,
):

    # -------------------------------------------------------------------------
    # Gemma standalone uses "response"
    # -------------------------------------------------------------------------

    if model_name == "Gemma 4 Only":

        value = (
            item.get(
                "response"
            )
        )


    # -------------------------------------------------------------------------
    # Other four Track 2 result formats use "answer"
    # -------------------------------------------------------------------------

    else:

        value = (
            item.get(
                "answer"
            )
        )


    if value is None:

        return None


    text = str(
        value
    ).strip()


    if text == "":

        return None


    return text


# =============================================================================
# 6. HELPER — SOURCE STATUS
# =============================================================================

def normalise_source_status(
    model_name,
    item,
):

    status = str(
        item.get(
            "status",
            "",
        )
    ).strip().upper()


    # Gemma standalone uses "success"
    if status == "SUCCESS":

        return "PASS"


    if status == "PASS":

        return "PASS"


    if status == "FAIL":

        return "FAIL"


    # Fallback based on response existence
    response = (
        extract_track2_response(
            model_name,
            item,
        )
    )


    if response is not None:

        return "PASS"


    return "UNKNOWN"


# =============================================================================
# 7. HELPER — GENERATION FAILURE
# =============================================================================

def detect_generation_failure(
    model_name,
    item,
):

    source_status = (
        normalise_source_status(
            model_name,
            item,
        )
    )


    response = (
        extract_track2_response(
            model_name,
            item,
        )
    )


    error = (
        item.get(
            "error"
        )
    )


    return (

        source_status == "FAIL"

        or

        (
            response is None
            and
            error is not None
        )
    )


# =============================================================================
# 8. HELPER — ERROR TYPE
# =============================================================================

def extract_error_type(
    item,
):

    error = (
        item.get(
            "error"
        )
    )


    if error is None:

        return None


    error_text = str(
        error
    )


    if "OutOfMemoryError" in error_text:

        return "CUDA_OUT_OF_MEMORY"


    return (
        error_text
        .split(
            ":",
            1,
        )[0]
        .strip()
    )


# =============================================================================
# 9. HELPER — RETRIEVAL METADATA
# =============================================================================

def extract_retrieval_metadata(
    item,
):

    retrieval = (
        item.get(
            "retrieval"
        )
    )


    if not isinstance(
        retrieval,
        dict,
    ):

        return None


    if len(
        retrieval
    ) == 0:

        return None


    return retrieval


# =============================================================================
# 10. HELPER — RETRIEVAL AVAILABLE
# =============================================================================

def has_retrieval(
    item,
):

    retrieval = (
        extract_retrieval_metadata(
            item
        )
    )


    return (
        retrieval
        is not None
    )


# =============================================================================
# 11. HELPER — RETRIEVER K
# =============================================================================

def extract_retriever_k(
    item,
):

    retriever = (
        item.get(
            "retriever"
        )
    )


    if not isinstance(
        retriever,
        dict,
    ):

        return None


    return (
        retriever.get(
            "k"
        )
    )


# =============================================================================
# 12. BUILD RAW MODEL LOOKUPS
# =============================================================================

model_result_lookup = {}


for model_name in (
    EXPECTED_TRACK2_SYSTEMS
):

    model_result_lookup[
        model_name
    ] = {}


    for item in (
        track2_loaded[
            model_name
        ]
    ):

        question_id = (
            extract_question_id(
                item
            )
        )


        if question_id is None:

            raise RuntimeError(
                f"Question ID missing for "
                f"{model_name}"
            )


        if (
            question_id
            in
            model_result_lookup[
                model_name
            ]
        ):

            raise RuntimeError(
                f"Duplicate raw result: "
                f"{model_name} | "
                f"{question_id}"
            )


        model_result_lookup[
            model_name
        ][
            question_id
        ] = item


# =============================================================================
# 13. BUILD UNIFIED RECORDS
# =============================================================================

track2_records = []


for question_id in sorted(
    track2_question_metadata.keys()
):

    metadata = (
        track2_question_metadata[
            question_id
        ]
    )


    for model_name in (
        EXPECTED_TRACK2_SYSTEMS
    ):

        if (
            question_id
            not in
            model_result_lookup[
                model_name
            ]
        ):

            raise RuntimeError(
                f"Missing raw record: "
                f"{model_name} | "
                f"{question_id}"
            )


        raw_result = (

            model_result_lookup[
                model_name
            ][
                question_id
            ]
        )


        raw_response = (
            extract_track2_response(
                model_name,
                raw_result,
            )
        )


        generation_failed = (
            detect_generation_failure(
                model_name,
                raw_result,
            )
        )


        source_status = (
            normalise_source_status(
                model_name,
                raw_result,
            )
        )


        error_type = (
            extract_error_type(
                raw_result
            )
        )


        retrieval_metadata = (
            extract_retrieval_metadata(
                raw_result
            )
        )


        retrieval_available = (
            retrieval_metadata
            is not None
        )


        is_rag = (
            "+ RAG"
            in model_name
        )


        # ---------------------------------------------------------------------
        # IMPORTANT
        #
        # Never expose CUDA/OOM error detail as the model answer presented to
        # the judge.
        #
        # The judge should evaluate that the system produced NO RESPONSE.
        #
        # ---------------------------------------------------------------------

        if generation_failed:

            judge_response = (
                "[NO MODEL RESPONSE GENERATED]"
            )

            benchmark_response_status = (
                "NON_RESPONSIVE"
            )


        elif raw_response is None:

            judge_response = (
                "[NO MODEL RESPONSE GENERATED]"
            )

            benchmark_response_status = (
                "NON_RESPONSIVE"
            )


        else:

            judge_response = (
                raw_response
            )

            benchmark_response_status = (
                "SUBSTANTIVE"
            )


        track2_records.append(
            {

                # -------------------------------------------------------------
                # BENCHMARK IDENTITY
                # -------------------------------------------------------------

                "track":
                    "Track 2",

                "question_id":
                    question_id,

                "benchmark":
                    metadata[
                        "benchmark"
                    ],

                "category":
                    metadata[
                        "benchmark"
                    ],

                "question":
                    metadata[
                        "question"
                    ],

                "choices":
                    metadata[
                        "choices"
                    ],

                "expected_answer":
                    metadata[
                        "expected_answer"
                    ],

                "explanation":
                    metadata[
                        "explanation"
                    ],

                "candidate_selection_score":
                    metadata[
                        "candidate_selection_score"
                    ],

                # -------------------------------------------------------------
                # MODEL
                # -------------------------------------------------------------

                "model_name":
                    model_name,

                "is_rag":
                    is_rag,

                # -------------------------------------------------------------
                # RESPONSE
                # -------------------------------------------------------------

                "response":
                    judge_response,

                "raw_response":
                    raw_response,

                "response_status":
                    benchmark_response_status,

                # -------------------------------------------------------------
                # ORIGINAL RUN STATUS
                # -------------------------------------------------------------

                "source_status":
                    source_status,

                "generation_failed":
                    generation_failed,

                "generation_error_type":
                    error_type,

                "generation_error":
                    raw_result.get(
                        "error"
                    ),

                # -------------------------------------------------------------
                # GENERATION METADATA
                # -------------------------------------------------------------

                "input_tokens":
                    raw_result.get(
                        "input_tokens"
                    ),

                "output_tokens":
                    raw_result.get(
                        "output_tokens"
                    ),

                "generation_time_sec":
                    raw_result.get(
                        "generation_time_sec"
                    ),

                "generation_config":
                    raw_result.get(
                        "generation_config"
                    ),

                # -------------------------------------------------------------
                # RAG METADATA
                # -------------------------------------------------------------

                "retrieval_available":
                    retrieval_available,

                "retriever_k":
                    extract_retriever_k(
                        raw_result
                    ),

                "retrieval":
                    retrieval_metadata,

                # -------------------------------------------------------------
                # RAW RECORD
                # -------------------------------------------------------------

                "raw_result":
                    raw_result,
            }
        )


# =============================================================================
# 14. CREATE DATAFRAME
# =============================================================================

track2_evaluation_df = (

    pd.DataFrame(
        track2_records
    )

    .reset_index(
        drop=True
    )
)


# =============================================================================
# 15. CORE DATASET SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "1. TRACK 2 DATASET SUMMARY"
)

print(
    "=" * 110
)


print(
    f"\nRows              : "
    f"{len(track2_evaluation_df)}"
)

print(
    f"Unique questions  : "
    f"{track2_evaluation_df['question_id'].nunique()}"
)

print(
    f"Unique models     : "
    f"{track2_evaluation_df['model_name'].nunique()}"
)

print(
    f"RAG responses     : "
    f"{int(track2_evaluation_df['is_rag'].sum())}"
)

print(
    f"Standalone        : "
    f"{int((~track2_evaluation_df['is_rag']).sum())}"
)


# =============================================================================
# 16. RESPONSES PER MODEL
# =============================================================================

model_counts = (

    track2_evaluation_df

    .groupby(
        "model_name"
    )

    .size()

    .reset_index(
        name=
            "responses"
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "2. RESPONSES PER MODEL"
)

print(
    "=" * 110
)


display(
    model_counts
)


# =============================================================================
# 17. GENERATION STATUS BY MODEL
# =============================================================================

generation_status_summary = (

    track2_evaluation_df

    .groupby(
        [
            "model_name",
            "response_status",
        ]
    )

    .size()

    .reset_index(
        name=
            "responses"
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "3. RESPONSE STATUS BY MODEL"
)

print(
    "=" * 110
)


display(
    generation_status_summary
)


# =============================================================================
# 18. GENERATION FAILURE SUMMARY
# =============================================================================

generation_failure_summary = (

    track2_evaluation_df

    .groupby(
        "model_name"
    )

    .agg(

        total_responses=(
            "question_id",
            "count",
        ),

        generation_failures=(
            "generation_failed",
            "sum",
        ),
    )

    .reset_index()
)


generation_failure_summary[
    "generation_failure_pct"
] = (

    generation_failure_summary[
        "generation_failures"
    ]

    /
    generation_failure_summary[
        "total_responses"
    ]

    * 100
)


print(
    "\n"
    + "=" * 110
)

print(
    "4. GENERATION FAILURE SUMMARY"
)

print(
    "=" * 110
)


display(
    generation_failure_summary.round(2)
)


# =============================================================================
# 19. FAILURE CASES
# =============================================================================

generation_failure_cases = (

    track2_evaluation_df[

        track2_evaluation_df[
            "generation_failed"
        ]

    ][
        [
            "question_id",
            "benchmark",
            "model_name",
            "generation_error_type",
            "response_status",
        ]
    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "5. GENERATION FAILURE CASES"
)

print(
    "=" * 110
)


if len(
    generation_failure_cases
) > 0:

    display(
        generation_failure_cases
    )

else:

    print(
        "No generation failures."
    )


# =============================================================================
# 20. RETRIEVAL AVAILABILITY
# =============================================================================

rag_rows = (

    track2_evaluation_df[

        track2_evaluation_df[
            "is_rag"
        ]

    ]

    .copy()
)


retrieval_summary = (

    rag_rows

    .groupby(
        "model_name"
    )

    .agg(

        rag_records=(
            "question_id",
            "count",
        ),

        retrieval_available=(
            "retrieval_available",
            "sum",
        ),

        generation_failures=(
            "generation_failed",
            "sum",
        ),
    )

    .reset_index()
)


retrieval_summary[
    "retrieval_available_pct"
] = (

    retrieval_summary[
        "retrieval_available"
    ]

    /
    retrieval_summary[
        "rag_records"
    ]

    * 100
)


print(
    "\n"
    + "=" * 110
)

print(
    "6. RAG RETRIEVAL AVAILABILITY"
)

print(
    "=" * 110
)


display(
    retrieval_summary.round(2)
)


# =============================================================================
# 21. BENCHMARK DISTRIBUTION
# =============================================================================

benchmark_summary = (

    track2_evaluation_df[
        [
            "question_id",
            "benchmark",
        ]
    ]

    .drop_duplicates()

    .groupby(
        "benchmark"
    )

    .size()

    .reset_index(
        name=
            "questions"
    )

    .sort_values(
        [
            "questions",
            "benchmark",
        ],
        ascending=[
            False,
            True,
        ]
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "7. TRACK 2 BENCHMARK DISTRIBUTION"
)

print(
    "=" * 110
)


display(
    benchmark_summary
)


# =============================================================================
# 22. QUESTION × MODEL COVERAGE
# =============================================================================

coverage_matrix = (

    track2_evaluation_df

    .pivot_table(
        index=
            "question_id",

        columns=
            "model_name",

        values=
            "response",

        aggfunc=
            "count",

        fill_value=
            0,
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "8. QUESTION × MODEL COVERAGE"
)

print(
    "=" * 110
)


display(
    coverage_matrix
)


# =============================================================================
# 23. SAMPLE DATASET
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "9. SAMPLE TRACK 2 RECORDS"
)

print(
    "=" * 110
)


display(

    track2_evaluation_df[
        [
            "question_id",
            "benchmark",
            "model_name",
            "is_rag",
            "source_status",
            "response_status",
            "generation_failed",
            "retrieval_available",
        ]
    ]

    .head(
        20
    )
)


# =============================================================================
# 24. FINAL VALIDATION
# =============================================================================

question_counts = (

    track2_evaluation_df

    .groupby(
        "question_id"
    )

    .size()
)


model_response_counts = (

    track2_evaluation_df

    .groupby(
        "model_name"
    )

    .size()
)


duplicate_count = (

    track2_evaluation_df

    .duplicated(
        subset=[
            "question_id",
            "model_name",
        ]
    )

    .sum()
)


essential_rag_failure_count = int(

    (
        (
            track2_evaluation_df[
                "model_name"
            ]
            == "Essential AI + RAG"
        )

        &

        (
            track2_evaluation_df[
                "generation_failed"
            ]
        )
    )

    .sum()
)


validation_checks = {

    "160 total records":
        (
            len(
                track2_evaluation_df
            )
            == 160
        ),

    "32 unique questions":
        (
            track2_evaluation_df[
                "question_id"
            ]
            .nunique()
            == 32
        ),

    "5 unique models":
        (
            track2_evaluation_df[
                "model_name"
            ]
            .nunique()
            == 5
        ),

    "32 records per model":
        (
            model_response_counts
            .eq(
                32
            )
            .all()
        ),

    "5 records per question":
        (
            question_counts
            .eq(
                5
            )
            .all()
        ),

    "No duplicate question/model records":
        (
            duplicate_count
            == 0
        ),

    "64 RAG records":
        (
            int(
                track2_evaluation_df[
                    "is_rag"
                ]
                .sum()
            )
            == 64
        ),

    "96 standalone records":
        (
            int(
                (
                    ~track2_evaluation_df[
                        "is_rag"
                    ]
                )
                .sum()
            )
            == 96
        ),

    "9 Essential AI RAG generation failures":
        (
            essential_rag_failure_count
            == 9
        ),

    "All failed generations marked NON_RESPONSIVE":
        (
            track2_evaluation_df.loc[
                track2_evaluation_df[
                    "generation_failed"
                ],
                "response_status",
            ]

            .eq(
                "NON_RESPONSIVE"
            )

            .all()
        ),

    "No null judge response strings":
        (
            track2_evaluation_df[
                "response"
            ]
            .notna()
            .all()
        ),
}


print(
    "\n"
    + "=" * 110
)

print(
    "MODULE 6.1 VALIDATION"
)

print(
    "=" * 110
)


for check_name, passed in (
    validation_checks.items()
):

    print(
        f"{check_name:<55} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 6.1 validation failed."
    )


# =============================================================================
# 25. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TRACK 2 UNIFIED EVALUATION DATASET READY"
)

print(
    "=" * 110
)


print(
    f"\nRecords              : "
    f"{len(track2_evaluation_df)}"
)

print(
    f"Questions            : "
    f"{track2_evaluation_df['question_id'].nunique()}"
)

print(
    f"Models               : "
    f"{track2_evaluation_df['model_name'].nunique()}"
)

print(
    f"Generation failures  : "
    f"{int(track2_evaluation_df['generation_failed'].sum())}"
)

print(
    f"Substantive responses: "
    f"{int((track2_evaluation_df['response_status'] == 'SUBSTANTIVE').sum())}"
)

print(
    f"Non-responsive       : "
    f"{int((track2_evaluation_df['response_status'] == 'NON_RESPONSIVE').sum())}"
)


print(
    "\nGenerated objects:"
)

print(
    "• track2_question_metadata"
)

print(
    "• track2_evaluation_df"
)

print(
    "• generation_status_summary"
)

print(
    "• generation_failure_summary"
)

print(
    "• generation_failure_cases"
)

print(
    "• retrieval_summary"
)

print(
    "• benchmark_summary"
)


print(
    "\nNext:"
)

print(
    "Module 6.2 — Track 2 Independent LLM Judge Evaluation"
)


print(
    "\nMODULE 6.1 COMPLETE"
)

print(
    "=" * 110
)

MODULE 6.1 — TRACK 2 UNIFIED EVALUATION DATASET

Canonical questions loaded : 32

1. TRACK 2 DATASET SUMMARY

Rows              : 160
Unique questions  : 32
Unique models     : 5
RAG responses     : 64
Standalone        : 96

2. RESPONSES PER MODEL


,model_name,responses
0,Essential AI + RAG,32
1,Essential AI Only,32
2,Gemma 4 + RAG,32
3,Gemma 4 Only,32
4,Otel 2.0 Only,32



3. RESPONSE STATUS BY MODEL


,model_name,response_status,responses
0,Essential AI + RAG,NON_RESPONSIVE,9
1,Essential AI + RAG,SUBSTANTIVE,23
2,Essential AI Only,SUBSTANTIVE,32
3,Gemma 4 + RAG,SUBSTANTIVE,32
4,Gemma 4 Only,SUBSTANTIVE,32
5,Otel 2.0 Only,SUBSTANTIVE,32



4. GENERATION FAILURE SUMMARY


...


8. QUESTION × MODEL COVERAGE


model_name,Essential AI + RAG,Essential AI Only,Gemma 4 + RAG,Gemma 4 Only,Otel 2.0 Only
question_id,,,,,
T2-01,1,1,1,1,1
T2-02,1,1,1,1,1
T2-03,1,1,1,1,1
T2-04,1,1,1,1,1
T2-05,1,1,1,1,1
T2-06,1,1,1,1,1
T2-07,1,1,1,1,1
T2-08,1,1,1,1,1
T2-09,1,1,1,1,1



9. SAMPLE TRACK 2 RECORDS


,question_id,benchmark,model_name,is_rag,source_status,response_status,generation_failed,retrieval_available
0,T2-01,3gpp_tsg,Essential AI + RAG,True,FAIL,NON_RESPONSIVE,True,False
1,T2-01,3gpp_tsg,Essential AI Only,False,PASS,SUBSTANTIVE,False,False
2,T2-01,3gpp_tsg,Otel 2.0 Only,False,PASS,SUBSTANTIVE,False,False
3,T2-01,3gpp_tsg,Gemma 4 Only,False,PASS,SUBSTANTIVE,False,False
4,T2-01,3gpp_tsg,Gemma 4 + RAG,True,PASS,SUBSTANTIVE,False,True
5,T2-02,3gpp_tsg,Essential AI + RAG,True,FAIL,NON_RESPONSIVE,True,False
6,T2-02,3gpp_tsg,Essential AI Only,False,PASS,SUBSTANTIVE,False,False
7,T2-02,3gpp_tsg,Otel 2.0 Only,False,PASS,SUBSTANTIVE,False,False
8,T2-02,3gpp_tsg,Gemma 4 Only,False,PASS,SUBSTANTIVE,False,False
9,T2-02,3gpp_tsg,Gemma 4 + RAG,True,PASS,SUBSTANTIVE,False,True



MODULE 6.1 VALIDATION
160 total records                                       : ✅ PASS
32 unique questions                                     : ✅ PASS
5 unique models                                         : ✅ PASS
32 records per model                                    : ✅ PASS
5 records per question                                  : ✅ PASS
No duplicate question/model records                     : ✅ PASS
64 RAG records                                          : ✅ PASS
96 standalone records                                   : ✅ PASS
9 Essential AI RAG generation failures                  : ✅ PASS
All failed generations marked NON_RESPONSIVE            : ✅ PASS
No null judge response strings                          : ✅ PASS

TRACK 2 UNIFIED EVALUATION DATASET READY

Records              : 160
Questions            : 32
Models               : 5
Generation failures  : 9
Substantive responses: 151
Non-responsive       : 9

Generated objects:
• track2_question_metadata
• track2_evaluati

# **Track 2 Answer Normalization and Scoring Preparation**

In [26]:
# =============================================================================
# MODULE 6.2 — TRACK 2 ANSWER NORMALIZATION
# =============================================================================
#
# PURPOSE
# -------
# Normalize Track 2 model outputs into a comparable form before objective
# expected-answer scoring.
#
# TRACK 2 contains multiple answer styles, including:
#
#   - 3GPP working-group labels      e.g. RAN1
#   - Root-cause labels              e.g. C4
#   - Numerical answers              e.g. 1.0
#   - Multiple-choice selections
#   - JSON-formatted answers
#   - Boxed answers                  e.g. \boxed{4}
#
# IMPORTANT
# ---------
# This module DOES NOT judge correctness yet.
#
# It:
#
#   1. Inspects benchmark answer formats
#   2. Normalizes expected answers
#   3. Extracts normalized model answers
#   4. Preserves raw responses
#   5. Flags non-responsive cases
#
# =============================================================================


import pandas as pd
import numpy as np
import json
import re
import math


print("=" * 110)
print("MODULE 6.2 — TRACK 2 ANSWER NORMALIZATION")
print("=" * 110)


# =============================================================================
# 1. VALIDATE INPUT
# =============================================================================

if "track2_evaluation_df" not in globals():

    raise RuntimeError(
        "track2_evaluation_df not found. "
        "Run Module 6.1 first."
    )


required_columns = [

    "question_id",
    "benchmark",
    "model_name",
    "response",
    "raw_response",
    "expected_answer",
    "choices",
    "response_status",
]


missing_columns = [

    column

    for column in required_columns

    if column
    not in track2_evaluation_df.columns
]


if missing_columns:

    raise RuntimeError(
        f"Missing required columns: "
        f"{missing_columns}"
    )


# =============================================================================
# 2. BASIC TEXT NORMALIZATION
# =============================================================================

def normalize_basic_text(
    value,
):

    if value is None:

        return None


    if isinstance(
        value,
        float,
    ):

        if np.isnan(
            value
        ):

            return None


    text = str(
        value
    ).strip()


    if text == "":

        return None


    return text


# =============================================================================
# 3. REMOVE MARKDOWN CODE FENCES
# =============================================================================

def remove_code_fences(
    text,
):

    if text is None:

        return None


    cleaned = (
        text
        .strip()
    )


    cleaned = re.sub(
        r"^```(?:json|python|text)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    return (
        cleaned
        .strip()
    )


# =============================================================================
# 4. JSON VALUE EXTRACTION
# =============================================================================

def extract_json_value(
    text,
):

    if text is None:

        return None


    cleaned = (
        remove_code_fences(
            text
        )
    )


    try:

        parsed = json.loads(
            cleaned
        )

    except Exception:

        return None


    if isinstance(
        parsed,
        dict,
    ):

        # ---------------------------------------------------------------------
        # Common Track 2 output fields
        # ---------------------------------------------------------------------

        preferred_keys = [

            "WORKING GROUP",
            "working_group",
            "working group",

            "ANSWER",
            "answer",

            "CHOICE",
            "choice",

            "LABEL",
            "label",

            "RESULT",
            "result",
        ]


        for key in (
            preferred_keys
        ):

            if key in parsed:

                return (
                    parsed[
                        key
                    ]
                )


        # ---------------------------------------------------------------------
        # If dictionary contains only one value, use it
        # ---------------------------------------------------------------------

        if len(
            parsed
        ) == 1:

            return (
                next(
                    iter(
                        parsed.values()
                    )
                )
            )


    if isinstance(
        parsed,
        (
            str,
            int,
            float,
        ),
    ):

        return parsed


    return None


# =============================================================================
# 5. EXTRACT BOXED ANSWER
# =============================================================================

def extract_boxed_answer(
    text,
):

    if text is None:

        return None


    match = re.search(
        r"\\boxed\s*\{\s*([^{}]+?)\s*\}",
        text,
        flags=re.IGNORECASE,
    )


    if match:

        return (
            match
            .group(1)
            .strip()
        )


    return None


# =============================================================================
# 6. NORMALIZE NUMERIC VALUE
# =============================================================================

def normalize_numeric_value(
    value,
):

    if value is None:

        return None


    try:

        number = float(
            value
        )


        if math.isclose(
            number,
            round(
                number
            ),
            rel_tol=0,
            abs_tol=1e-12,
        ):

            return str(
                int(
                    round(
                        number
                    )
                )
            )


        return (
            f"{number:.10f}"
            .rstrip("0")
            .rstrip(".")
        )


    except Exception:

        return None


# =============================================================================
# 7. EXTRACT PERCENTAGE
# =============================================================================

def extract_percentage(
    text,
):

    if text is None:

        return None


    matches = re.findall(
        r"(-?\d+(?:\.\d+)?)\s*%",
        text,
    )


    if not matches:

        return None


    # Prefer the final percentage in the response
    return (
        normalize_numeric_value(
            matches[-1]
        )
    )


# =============================================================================
# 8. EXTRACT C-LABEL
# =============================================================================

def extract_c_label(
    text,
):

    if text is None:

        return None


    # e.g. C4
    matches = re.findall(
        r"\bC\s*([1-9]\d*)\b",
        text,
        flags=re.IGNORECASE,
    )


    if matches:

        return (
            f"C{matches[-1]}"
            .upper()
        )


    # e.g. \boxed{4}
    boxed = (
        extract_boxed_answer(
            text
        )
    )


    if boxed is not None:

        boxed_match = re.fullmatch(
            r"C?\s*([1-9]\d*)",
            boxed,
            flags=re.IGNORECASE,
        )


        if boxed_match:

            return (
                f"C{boxed_match.group(1)}"
                .upper()
            )


    return None


# =============================================================================
# 9. EXTRACT 3GPP WORKING GROUP LABEL
# =============================================================================

TRACK2_3GPP_WORKING_GROUPS = [

    "CT1",
    "CT3",
    "CT4",
    "CT6",

    "RAN1",
    "RAN2",
    "RAN3",
    "RAN4",
    "RAN5",
    "RAN_AH1",

    "SA1",
    "SA2",
    "SA3",
    "SA4",
    "SA5",
    "SA6",
]


def extract_3gpp_working_group(
    text,
):

    if text is None:

        return None


    # -------------------------------------------------------------------------
    # First attempt structured JSON extraction
    # -------------------------------------------------------------------------

    json_value = (
        extract_json_value(
            text
        )
    )


    if json_value is not None:

        candidate = (
            str(
                json_value
            )
            .strip()
            .upper()
        )


        if candidate in (
            TRACK2_3GPP_WORKING_GROUPS
        ):

            return candidate


    # -------------------------------------------------------------------------
    # Search text directly
    # -------------------------------------------------------------------------

    upper_text = (
        text
        .upper()
    )


    # Match longer label first
    for label in sorted(
        TRACK2_3GPP_WORKING_GROUPS,
        key=len,
        reverse=True,
    ):

        pattern = (
            r"(?<![A-Z0-9_])"
            +
            re.escape(
                label
            )
            +
            r"(?![A-Z0-9_])"
        )


        if re.search(
            pattern,
            upper_text,
        ):

            return label


    return None


# =============================================================================
# 10. NORMALIZE CHOICE LIST
# =============================================================================

def normalize_choices(
    choices,
):

    if choices is None:

        return []


    if isinstance(
        choices,
        np.ndarray,
    ):

        choices = (
            choices.tolist()
        )


    if isinstance(
        choices,
        tuple,
    ):

        choices = list(
            choices
        )


    if isinstance(
        choices,
        list,
    ):

        return choices


    return []


# =============================================================================
# 11. MAP EXPECTED ANSWER TO CHOICE VALUE
# =============================================================================
#
# Some benchmark records store expected_answer as an integer index.
#
# Example:
#
# choices:
#   ["8%", "10.00%", "14.00%", ...]
#
# expected_answer:
#   0
#
# This means the expected semantic answer is:
#
#   "8%"
#
# =============================================================================

def resolve_expected_choice(
    expected_answer,
    choices,
):

    choices = (
        normalize_choices(
            choices
        )
    )


    if not choices:

        return None


    # -------------------------------------------------------------------------
    # Integer index
    # -------------------------------------------------------------------------

    if isinstance(
        expected_answer,
        (
            int,
            np.integer,
        ),
    ):

        index = int(
            expected_answer
        )


        if (
            0
            <= index
            < len(
                choices
            )
        ):

            return (
                choices[
                    index
                ]
            )


    # -------------------------------------------------------------------------
    # Float representing integer index
    # -------------------------------------------------------------------------

    if isinstance(
        expected_answer,
        (
            float,
            np.floating,
        ),
    ):

        if (
            not np.isnan(
                expected_answer
            )
            and
            float(
                expected_answer
            ).is_integer()
        ):

            index = int(
                expected_answer
            )


            if (
                0
                <= index
                < len(
                    choices
                )
            ):

                return (
                    choices[
                        index
                    ]
                )


    return None


# =============================================================================
# 12. EXPECTED ANSWER NORMALIZER
# =============================================================================

def normalize_expected_answer(
    benchmark,
    expected_answer,
    choices,
):

    benchmark = (
        normalize_basic_text(
            benchmark
        )
        or ""
    ).lower()


    # -------------------------------------------------------------------------
    # 3GPP working-group classification
    # -------------------------------------------------------------------------

    if benchmark == "3gpp_tsg":

        return (
            str(
                expected_answer
            )
            .strip()
            .upper()
        )


    # -------------------------------------------------------------------------
    # Telelogs root-cause labels
    # -------------------------------------------------------------------------

    if benchmark == "telelogs":

        text = (
            str(
                expected_answer
            )
            .strip()
            .upper()
        )


        match = re.search(
            r"C?\s*(\d+)",
            text,
        )


        if match:

            return (
                f"C{match.group(1)}"
            )


        return text


    # -------------------------------------------------------------------------
    # MCQ / choice-based expected answer
    # -------------------------------------------------------------------------

    resolved_choice = (
        resolve_expected_choice(
            expected_answer,
            choices,
        )
    )


    if resolved_choice is not None:

        resolved_text = (
            str(
                resolved_choice
            )
            .strip()
        )


        # percentage option
        pct = (
            extract_percentage(
                resolved_text
            )
        )


        if pct is not None:

            return (
                f"{pct}%"
            )


        return (
            resolved_text
            .strip()
            .upper()
        )


    # -------------------------------------------------------------------------
    # Numeric expected answer
    # -------------------------------------------------------------------------

    numeric = (
        normalize_numeric_value(
            expected_answer
        )
    )


    if numeric is not None:

        return numeric


    # -------------------------------------------------------------------------
    # General text / label
    # -------------------------------------------------------------------------

    return (
        str(
            expected_answer
        )
        .strip()
        .upper()
    )


# =============================================================================
# 13. GENERAL MCQ CHOICE MATCHER
# =============================================================================

def extract_choice_match(
    text,
    choices,
):

    if text is None:

        return None


    choices = (
        normalize_choices(
            choices
        )
    )


    if not choices:

        return None


    cleaned_text = (
        remove_code_fences(
            text
        )
    )


    # -------------------------------------------------------------------------
    # Exact answer text appearing in response
    # -------------------------------------------------------------------------

    upper_text = (
        cleaned_text
        .upper()
    )


    choice_matches = []


    for index, choice in enumerate(
        choices
    ):

        choice_text = (
            str(
                choice
            )
            .strip()
        )


        if (
            choice_text
            and
            choice_text.upper()
            in upper_text
        ):

            choice_matches.append(
                (
                    index,
                    choice_text,
                )
            )


    # Prefer last matching explicit choice
    if choice_matches:

        return (
            choice_matches[-1][1]
        )


    # -------------------------------------------------------------------------
    # A/B/C/D/E format
    # -------------------------------------------------------------------------

    letter_matches = re.findall(
        r"\b(?:OPTION\s*)?([A-E])\b",
        upper_text,
    )


    if letter_matches:

        letter = (
            letter_matches[-1]
        )


        index = (
            ord(
                letter
            )
            - ord(
                "A"
            )
        )


        if (
            0
            <= index
            < len(
                choices
            )
        ):

            return (
                str(
                    choices[
                        index
                    ]
                )
                .strip()
            )


    return None


# =============================================================================
# 14. MODEL ANSWER NORMALIZER
# =============================================================================

def normalize_model_answer(
    benchmark,
    response,
    choices,
    response_status,
):

    # -------------------------------------------------------------------------
    # Non-responsive cases
    # -------------------------------------------------------------------------

    if (
        response_status
        == "NON_RESPONSIVE"
    ):

        return None


    text = (
        normalize_basic_text(
            response
        )
    )


    if text is None:

        return None


    benchmark = (
        str(
            benchmark
        )
        .strip()
        .lower()
    )


    # -------------------------------------------------------------------------
    # 3GPP TSG classification
    # -------------------------------------------------------------------------

    if benchmark == "3gpp_tsg":

        return (
            extract_3gpp_working_group(
                text
            )
        )


    # -------------------------------------------------------------------------
    # Telelogs root-cause benchmark
    # -------------------------------------------------------------------------

    if benchmark == "telelogs":

        return (
            extract_c_label(
                text
            )
        )


    # -------------------------------------------------------------------------
    # Choice-based benchmark
    # -------------------------------------------------------------------------

    choice_match = (
        extract_choice_match(
            text,
            choices,
        )
    )


    if choice_match is not None:

        pct = (
            extract_percentage(
                choice_match
            )
        )


        if pct is not None:

            return (
                f"{pct}%"
            )


        return (
            str(
                choice_match
            )
            .strip()
            .upper()
        )


    # -------------------------------------------------------------------------
    # JSON structured answer
    # -------------------------------------------------------------------------

    json_value = (
        extract_json_value(
            text
        )
    )


    if json_value is not None:

        pct = (
            extract_percentage(
                str(
                    json_value
                )
            )
        )


        if pct is not None:

            return (
                f"{pct}%"
            )


        numeric = (
            normalize_numeric_value(
                json_value
            )
        )


        if numeric is not None:

            return numeric


        return (
            str(
                json_value
            )
            .strip()
            .upper()
        )


    # -------------------------------------------------------------------------
    # Boxed answer
    # -------------------------------------------------------------------------

    boxed = (
        extract_boxed_answer(
            text
        )
    )


    if boxed is not None:

        pct = (
            extract_percentage(
                boxed
            )
        )


        if pct is not None:

            return (
                f"{pct}%"
            )


        numeric = (
            normalize_numeric_value(
                boxed
            )
        )


        if numeric is not None:

            return numeric


        return (
            boxed
            .strip()
            .upper()
        )


    # -------------------------------------------------------------------------
    # Percentage
    # -------------------------------------------------------------------------

    pct = (
        extract_percentage(
            text
        )
    )


    if pct is not None:

        return (
            f"{pct}%"
        )


    # -------------------------------------------------------------------------
    # Numeric answer
    # -------------------------------------------------------------------------

    number_matches = re.findall(
        r"(?<![A-Za-z0-9])"
        r"[-+]?\d+(?:\.\d+)?"
        r"(?![A-Za-z0-9])",
        text,
    )


    if number_matches:

        # Prefer final numeric value
        numeric = (
            normalize_numeric_value(
                number_matches[-1]
            )
        )


        if numeric is not None:

            return numeric


    # -------------------------------------------------------------------------
    # Final fallback — short response only
    # -------------------------------------------------------------------------

    cleaned = (
        remove_code_fences(
            text
        )
        .strip()
    )


    if len(
        cleaned
    ) <= 100:

        return (
            cleaned
            .upper()
        )


    return None


# =============================================================================
# 15. APPLY NORMALIZATION
# =============================================================================

track2_scoring_df = (
    track2_evaluation_df
    .copy()
)


track2_scoring_df[
    "normalized_expected_answer"
] = (

    track2_scoring_df

    .apply(
        lambda row:

            normalize_expected_answer(

                benchmark=
                    row[
                        "benchmark"
                    ],

                expected_answer=
                    row[
                        "expected_answer"
                    ],

                choices=
                    row[
                        "choices"
                    ],
            ),

        axis=1,
    )
)


track2_scoring_df[
    "normalized_model_answer"
] = (

    track2_scoring_df

    .apply(
        lambda row:

            normalize_model_answer(

                benchmark=
                    row[
                        "benchmark"
                    ],

                response=
                    row[
                        "response"
                    ],

                choices=
                    row[
                        "choices"
                    ],

                response_status=
                    row[
                        "response_status"
                    ],
            ),

        axis=1,
    )
)


# =============================================================================
# 16. NORMALIZATION STATUS
# =============================================================================

track2_scoring_df[
    "normalization_status"
] = np.where(

    track2_scoring_df[
        "response_status"
    ]
    .eq(
        "NON_RESPONSIVE"
    ),

    "NON_RESPONSIVE",

    np.where(

        track2_scoring_df[
            "normalized_model_answer"
        ]
        .isna(),

        "UNRESOLVED",

        "RESOLVED",
    )
)


# =============================================================================
# 17. BENCHMARK FORMAT SUMMARY
# =============================================================================

benchmark_format_summary = (

    track2_scoring_df[
        [
            "question_id",
            "benchmark",
            "expected_answer",
            "choices",
            "normalized_expected_answer",
        ]
    ]

    .drop_duplicates(
        subset=[
            "question_id"
        ]
    )

    .groupby(
        "benchmark"
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        questions_with_choices=(
            "choices",
            lambda x:
                sum(
                    isinstance(
                        value,
                        list,
                    )
                    and
                    len(
                        value
                    ) > 0

                    for value
                    in x
                ),
        ),
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 110
)

print(
    "1. BENCHMARK ANSWER FORMAT SUMMARY"
)

print(
    "=" * 110
)


display(
    benchmark_format_summary
)


# =============================================================================
# 18. NORMALIZATION SUMMARY
# =============================================================================

normalization_summary = (

    track2_scoring_df

    .groupby(
        [
            "model_name",
            "normalization_status",
        ]
    )

    .size()

    .reset_index(
        name=
            "responses"
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "2. NORMALIZATION STATUS BY MODEL"
)

print(
    "=" * 110
)


display(
    normalization_summary
)


# =============================================================================
# 19. SAMPLE NORMALIZED ANSWERS
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "3. SAMPLE NORMALIZED ANSWERS"
)

print(
    "=" * 110
)


display(

    track2_scoring_df[
        [
            "question_id",
            "benchmark",
            "model_name",
            "expected_answer",
            "normalized_expected_answer",
            "raw_response",
            "normalized_model_answer",
            "normalization_status",
        ]
    ]

    .head(
        30
    )
)


# =============================================================================
# 20. UNRESOLVED RESPONSES
# =============================================================================

unresolved_track2_answers = (

    track2_scoring_df[

        track2_scoring_df[
            "normalization_status"
        ]
        .eq(
            "UNRESOLVED"
        )

    ][
        [
            "question_id",
            "benchmark",
            "model_name",
            "expected_answer",
            "choices",
            "raw_response",
        ]
    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "4. UNRESOLVED SUBSTANTIVE RESPONSES"
)

print(
    "=" * 110
)


print(
    f"\nUnresolved substantive responses : "
    f"{len(unresolved_track2_answers)}"
)


if len(
    unresolved_track2_answers
) > 0:

    display(
        unresolved_track2_answers
    )


# =============================================================================
# 21. QUESTION-LEVEL EXPECTED ANSWER INSPECTION
# =============================================================================

track2_expected_answer_reference_df = (

    track2_scoring_df[
        [
            "question_id",
            "benchmark",
            "choices",
            "expected_answer",
            "normalized_expected_answer",
        ]
    ]

    .drop_duplicates(
        subset=[
            "question_id"
        ]
    )

    .sort_values(
        "question_id"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "5. CANONICAL EXPECTED ANSWER REFERENCE"
)

print(
    "=" * 110
)


display(
    track2_expected_answer_reference_df
)


# =============================================================================
# 22. VALIDATION
# =============================================================================

expected_answer_missing = int(

    track2_scoring_df[
        "normalized_expected_answer"
    ]
    .isna()
    .sum()
)


nonresponsive_count = int(

    track2_scoring_df[
        "normalization_status"
    ]
    .eq(
        "NON_RESPONSIVE"
    )
    .sum()
)


unresolved_count = int(

    track2_scoring_df[
        "normalization_status"
    ]
    .eq(
        "UNRESOLVED"
    )
    .sum()
)


resolved_count = int(

    track2_scoring_df[
        "normalization_status"
    ]
    .eq(
        "RESOLVED"
    )
    .sum()
)


validation_checks = {

    "160 total scoring records":
        (
            len(
                track2_scoring_df
            )
            == 160
        ),

    "All expected answers normalized":
        (
            expected_answer_missing
            == 0
        ),

    "9 NON_RESPONSIVE preserved":
        (
            nonresponsive_count
            == 9
        ),

    "Normalization totals equal 160":
        (
            resolved_count
            +
            unresolved_count
            +
            nonresponsive_count
            == 160
        ),
}


print(
    "\n"
    + "=" * 110
)

print(
    "MODULE 6.2 VALIDATION"
)

print(
    "=" * 110
)


for check_name, passed in (
    validation_checks.items()
):

    print(
        f"{check_name:<50} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 6.2 validation failed."
    )


# =============================================================================
# 23. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TRACK 2 ANSWER NORMALIZATION COMPLETE"
)

print(
    "=" * 110
)


print(
    f"\nResolved answers      : "
    f"{resolved_count}"
)

print(
    f"Unresolved answers    : "
    f"{unresolved_count}"
)

print(
    f"Non-responsive        : "
    f"{nonresponsive_count}"
)


print(
    "\nGenerated objects:"
)

print(
    "• track2_scoring_df"
)

print(
    "• benchmark_format_summary"
)

print(
    "• normalization_summary"
)

print(
    "• unresolved_track2_answers"
)

print(
    "• track2_expected_answer_reference_df"
)


print(
    "\nNext:"
)

print(
    "Module 6.3 — Objective Track 2 Correctness Scoring"
)


print(
    "\nMODULE 6.2 COMPLETE"
)

print(
    "=" * 110
)

MODULE 6.2 — TRACK 2 ANSWER NORMALIZATION

1. BENCHMARK ANSWER FORMAT SUMMARY


,benchmark,questions,questions_with_choices
0,3gpp_tsg,4,0
1,oranbench,4,4
2,sixg_bench,4,4
3,srsranbench,4,4
4,telelogs,4,0
5,telemath,4,0
6,teleqna,4,4
7,teletables,4,4



2. NORMALIZATION STATUS BY MODEL


,model_name,normalization_status,responses
0,Essential AI + RAG,NON_RESPONSIVE,9
1,Essential AI + RAG,RESOLVED,22
2,Essential AI + RAG,UNRESOLVED,1
3,Essential AI Only,RESOLVED,32
4,Gemma 4 + RAG,RESOLVED,31
5,Gemma 4 + RAG,UNRESOLVED,1
6,Gemma 4 Only,RESOLVED,32
7,Otel 2.0 Only,RESOLVED,32



3. SAMPLE NORMALIZED ANSWERS


...


4. UNRESOLVED SUBSTANTIVE RESPONSES

Unresolved substantive responses : 2


,question_id,benchmark,model_name,expected_answer,choices,raw_response
0,T2-06,oranbench,Gemma 4 + RAG,1,[1. Ensuring the X2 interface is properly conf...,Verifying that the user plane data transferred...
1,T2-15,srsranbench,Essential AI + RAG,0,"[1. To initialize the node size byte., 2. To c...",The `specific_init()` function in the `ldpc_de...



5. CANONICAL EXPECTED ANSWER REFERENCE


,question_id,benchmark,choices,expected_answer,normalized_expected_answer
0,T2-01,3gpp_tsg,None,RAN1,RAN1
1,T2-02,3gpp_tsg,None,RAN4,RAN4
2,T2-03,3gpp_tsg,None,RAN1,RAN1
3,T2-04,3gpp_tsg,None,CT3,CT3
4,T2-05,oranbench,"[1. Intra-gNB handover, 2. F1 conditional hand...",2,3. XN/X2 OR NG OR INTER-RAT CONDITIONAL HANDOVERS
5,T2-06,oranbench,[1. Ensuring the X2 interface is properly conf...,1,2. VERIFYING THAT THE USER PLANE DATA TRANSFER...
6,T2-07,oranbench,"[1. alarmId, 2. additionalInformation, 3. perc...",2,3. PERCEIVEDSEVERITY
7,T2-08,oranbench,"[1. CTIClientServerConnStatus, 2. CTISessionGr...",2,3. CTICONNPROFILEREF
8,T2-09,sixg_bench,[Continue for two turns on URLLC at current al...,0,CONTINUE FOR TWO TURNS ON URLLC AT CURRENT ALT...
9,T2-10,sixg_bench,[Keep all telemetry and commands on URLLC with...,3,"SPLIT COMMANDS ON URLLC AND TELEMETRY ON EMBB,..."



MODULE 6.2 VALIDATION
160 total scoring records                          : ✅ PASS
All expected answers normalized                    : ✅ PASS
9 NON_RESPONSIVE preserved                         : ✅ PASS
Normalization totals equal 160                     : ✅ PASS

TRACK 2 ANSWER NORMALIZATION COMPLETE

Resolved answers      : 149
Unresolved answers    : 2
Non-responsive        : 9

Generated objects:
• track2_scoring_df
• benchmark_format_summary
• normalization_summary
• unresolved_track2_answers
• track2_expected_answer_reference_df

Next:
Module 6.3 — Objective Track 2 Correctness Scoring

MODULE 6.2 COMPLETE


In [28]:
# =============================================================================
# MODULE 6.3 — OBJECTIVE TRACK 2 CORRECTNESS SCORING
# =============================================================================
#
# PURPOSE
# -------
# Score each Track 2 response objectively against the canonical expected answer.
#
# SCORING
# -------
# Correct answer        = 1
# Incorrect answer      = 0
# Non-responsive        = 0
# Unresolved formatting = flagged for review, not silently scored
#
# =============================================================================


import pandas as pd
import numpy as np


print("=" * 110)
print("MODULE 6.3 — OBJECTIVE TRACK 2 CORRECTNESS SCORING")
print("=" * 110)


# =============================================================================
# 1. VALIDATE INPUT
# =============================================================================

if "track2_scoring_df" not in globals():

    raise RuntimeError(
        "track2_scoring_df not found. "
        "Run Module 6.2 first."
    )


required_columns = [

    "question_id",
    "benchmark",
    "model_name",
    "normalized_expected_answer",
    "normalized_model_answer",
    "normalization_status",
    "response_status",
]


missing_columns = [

    column

    for column in required_columns

    if column not in track2_scoring_df.columns
]


if missing_columns:

    raise RuntimeError(
        f"Missing required columns: {missing_columns}"
    )


# =============================================================================
# 2. OBJECTIVE CORRECTNESS FUNCTION
# =============================================================================

def score_track2_answer(row):

    # -------------------------------------------------------------------------
    # Non-responsive system output
    # -------------------------------------------------------------------------

    if row["normalization_status"] == "NON_RESPONSIVE":

        return 0


    # -------------------------------------------------------------------------
    # Unresolved substantive answer
    # -------------------------------------------------------------------------

    if row["normalization_status"] == "UNRESOLVED":

        return np.nan


    expected = (
        str(
            row["normalized_expected_answer"]
        )
        .strip()
        .upper()
    )


    predicted = (
        str(
            row["normalized_model_answer"]
        )
        .strip()
        .upper()
    )


    # -------------------------------------------------------------------------
    # Exact normalized match
    # -------------------------------------------------------------------------

    if predicted == expected:

        return 1


    return 0


# =============================================================================
# 3. APPLY OBJECTIVE SCORING
# =============================================================================

track2_scoring_df[
    "correct"
] = (

    track2_scoring_df

    .apply(
        score_track2_answer,
        axis=1,
    )
)


# =============================================================================
# 4. CREATE RESULT LABEL
# =============================================================================

track2_scoring_df[
    "score_status"
] = np.select(

    [

        track2_scoring_df[
            "normalization_status"
        ]
        .eq(
            "NON_RESPONSIVE"
        ),

        track2_scoring_df[
            "normalization_status"
        ]
        .eq(
            "UNRESOLVED"
        ),

        track2_scoring_df[
            "correct"
        ]
        .eq(
            1
        ),

        track2_scoring_df[
            "correct"
        ]
        .eq(
            0
        ),
    ],

    [

        "NON_RESPONSIVE",

        "REVIEW_REQUIRED",

        "CORRECT",

        "INCORRECT",
    ],

    default=
        "UNKNOWN",
)


# =============================================================================
# 5. OVERALL MODEL ACCURACY
# =============================================================================

track2_model_accuracy = (

    track2_scoring_df

    .groupby(
        "model_name"
    )

    .agg(

        total_questions=(
            "question_id",
            "count",
        ),

        correct_answers=(
            "correct",
            lambda x:
                int(
                    np.nansum(
                        x
                    )
                ),
        ),

        unresolved=(
            "normalization_status",
            lambda x:
                int(
                    (
                        x
                        == "UNRESOLVED"
                    )
                    .sum()
                ),
        ),

        non_responsive=(
            "normalization_status",
            lambda x:
                int(
                    (
                        x
                        == "NON_RESPONSIVE"
                    )
                    .sum()
                ),
        ),
    )

    .reset_index()
)


track2_model_accuracy[
    "accuracy_pct"
] = (

    track2_model_accuracy[
        "correct_answers"
    ]

    /

    track2_model_accuracy[
        "total_questions"
    ]

    * 100
)


track2_model_accuracy = (

    track2_model_accuracy

    .sort_values(
        [
            "accuracy_pct",
            "correct_answers",
        ],
        ascending=[
            False,
            False,
        ]
    )

    .reset_index(
        drop=True
    )
)


track2_model_accuracy[
    "rank"
] = (

    np.arange(
        1,
        len(
            track2_model_accuracy
        )
        + 1
    )
)


track2_model_accuracy = (

    track2_model_accuracy[
        [
            "rank",
            "model_name",
            "correct_answers",
            "total_questions",
            "accuracy_pct",
            "non_responsive",
            "unresolved",
        ]
    ]
)


print(
    "\n"
    + "=" * 110
)

print(
    "1. OVERALL TRACK 2 MODEL ACCURACY"
)

print(
    "=" * 110
)


display(
    track2_model_accuracy.round(2)
)


# =============================================================================
# 6. ACCURACY BY BENCHMARK
# =============================================================================

track2_benchmark_accuracy = (

    track2_scoring_df

    .groupby(
        [
            "benchmark",
            "model_name",
        ]
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        correct_answers=(
            "correct",
            lambda x:
                int(
                    np.nansum(
                        x
                    )
                ),
        ),

        non_responsive=(
            "normalization_status",
            lambda x:
                int(
                    (
                        x
                        == "NON_RESPONSIVE"
                    )
                    .sum()
                ),
        ),

        unresolved=(
            "normalization_status",
            lambda x:
                int(
                    (
                        x
                        == "UNRESOLVED"
                    )
                    .sum()
                ),
        ),
    )

    .reset_index()
)


track2_benchmark_accuracy[
    "accuracy_pct"
] = (

    track2_benchmark_accuracy[
        "correct_answers"
    ]

    /

    track2_benchmark_accuracy[
        "questions"
    ]

    * 100
)


track2_benchmark_accuracy = (

    track2_benchmark_accuracy

    .sort_values(
        [
            "benchmark",
            "accuracy_pct",
        ],
        ascending=[
            True,
            False,
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "2. ACCURACY BY BENCHMARK FAMILY"
)

print(
    "=" * 110
)


display(
    track2_benchmark_accuracy.round(2)
)


# =============================================================================
# 7. BENCHMARK × MODEL PIVOT
# =============================================================================

track2_accuracy_pivot = (

    track2_benchmark_accuracy

    .pivot(
        index=
            "benchmark",

        columns=
            "model_name",

        values=
            "accuracy_pct",
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "3. BENCHMARK × MODEL ACCURACY (%)"
)

print(
    "=" * 110
)


display(
    track2_accuracy_pivot.round(2)
)


# =============================================================================
# 8. QUESTION-LEVEL RESULTS
# =============================================================================

track2_question_results = (

    track2_scoring_df[
        [
            "question_id",
            "benchmark",
            "model_name",
            "normalized_expected_answer",
            "normalized_model_answer",
            "score_status",
            "correct",
        ]
    ]

    .sort_values(
        [
            "question_id",
            "model_name",
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "4. QUESTION-LEVEL OBJECTIVE RESULTS"
)

print(
    "=" * 110
)


display(
    track2_question_results
)


# =============================================================================
# 9. INCORRECT ANSWERS
# =============================================================================

track2_incorrect_answers = (

    track2_scoring_df[

        track2_scoring_df[
            "score_status"
        ]
        .eq(
            "INCORRECT"
        )

    ][
        [
            "question_id",
            "benchmark",
            "model_name",
            "normalized_expected_answer",
            "normalized_model_answer",
            "raw_response",
        ]
    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "5. INCORRECT ANSWERS"
)

print(
    "=" * 110
)


print(
    f"\nIncorrect substantive answers : "
    f"{len(track2_incorrect_answers)}"
)


if len(
    track2_incorrect_answers
) > 0:

    display(
        track2_incorrect_answers
    )


# =============================================================================
# 10. REVIEW-REQUIRED ANSWERS
# =============================================================================

track2_review_required = (

    track2_scoring_df[

        track2_scoring_df[
            "score_status"
        ]
        .eq(
            "REVIEW_REQUIRED"
        )

    ][
        [
            "question_id",
            "benchmark",
            "model_name",
            "expected_answer",
            "normalized_expected_answer",
            "raw_response",
        ]
    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "6. REVIEW-REQUIRED ANSWERS"
)

print(
    "=" * 110
)


print(
    f"\nReview-required responses : "
    f"{len(track2_review_required)}"
)


if len(
    track2_review_required
) > 0:

    display(
        track2_review_required
    )


# =============================================================================
# 11. MODEL RESULT COUNTS
# =============================================================================

track2_result_status_summary = (

    track2_scoring_df

    .groupby(
        [
            "model_name",
            "score_status",
        ]
    )

    .size()

    .reset_index(
        name=
            "responses"
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "7. RESULT STATUS SUMMARY"
)

print(
    "=" * 110
)


display(
    track2_result_status_summary
)


# =============================================================================
# 12. VALIDATION
# =============================================================================

review_required_count = int(

    track2_scoring_df[
        "score_status"
    ]
    .eq(
        "REVIEW_REQUIRED"
    )
    .sum()
)


nonresponsive_count = int(

    track2_scoring_df[
        "score_status"
    ]
    .eq(
        "NON_RESPONSIVE"
    )
    .sum()
)


scored_count = int(

    track2_scoring_df[
        "correct"
    ]
    .notna()
    .sum()
)


validation_checks = {

    "160 total records":
        (
            len(
                track2_scoring_df
            )
            == 160
        ),

    "9 NON_RESPONSIVE preserved":
        (
            nonresponsive_count
            == 9
        ),

    "All non-responsive scored 0":
        (
            track2_scoring_df.loc[
                track2_scoring_df[
                    "score_status"
                ]
                .eq(
                    "NON_RESPONSIVE"
                ),
                "correct",
            ]
            .eq(
                0
            )
            .all()
        ),

    "Scoring accounting equals 160":
        (
            scored_count
            +
            review_required_count
            == 160
        ),
}


print(
    "\n"
    + "=" * 110
)

print(
    "MODULE 6.3 VALIDATION"
)

print(
    "=" * 110
)


for check_name, passed in (
    validation_checks.items()
):

    print(
        f"{check_name:<50} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 6.3 validation failed."
    )


# =============================================================================
# 13. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TRACK 2 OBJECTIVE SCORING COMPLETE"
)

print(
    "=" * 110
)


print(
    f"\nTotal records        : "
    f"{len(track2_scoring_df)}"
)

print(
    f"Scored records       : "
    f"{scored_count}"
)

print(
    f"Review required      : "
    f"{review_required_count}"
)

print(
    f"Non-responsive       : "
    f"{nonresponsive_count}"
)


print(
    "\nGenerated objects:"
)

print(
    "• track2_model_accuracy"
)

print(
    "• track2_benchmark_accuracy"
)

print(
    "• track2_accuracy_pivot"
)

print(
    "• track2_question_results"
)

print(
    "• track2_incorrect_answers"
)

print(
    "• track2_review_required"
)

print(
    "• track2_result_status_summary"
)


print(
    "\nNext:"
)

print(
    "Module 6.4 — Track 2 Performance Analysis and RAG Impact"
)


print(
    "\nMODULE 6.3 COMPLETE"
)

print(
    "=" * 110
)

MODULE 6.3 — OBJECTIVE TRACK 2 CORRECTNESS SCORING

1. OVERALL TRACK 2 MODEL ACCURACY


,rank,model_name,correct_answers,total_questions,accuracy_pct,non_responsive,unresolved
0,1,Otel 2.0 Only,18,32,56.25,0,0
1,2,Essential AI Only,13,32,40.62,0,0
2,3,Gemma 4 Only,8,32,25.00,0,0
3,4,Essential AI + RAG,7,32,21.88,9,1
4,5,Gemma 4 + RAG,7,32,21.88,0,1



2. ACCURACY BY BENCHMARK FAMILY


,benchmark,model_name,questions,correct_answers,non_responsive,unresolved,accuracy_pct
0,3gpp_tsg,Gemma 4 + RAG,4,3,0,0,75.0
1,3gpp_tsg,Otel 2.0 Only,4,3,0,0,75.0
2,3gpp_tsg,Essential AI Only,4,2,0,0,50.0
3,3gpp_tsg,Gemma 4 Only,4,2,0,0,50.0
4,3gpp_tsg,Essential AI + RAG,4,0,3,0,0.0
5,oranbench,Otel 2.0 Only,4,4,0,0,100.0
6,oranbench,Essential AI Only,4,2,0,0,50.0
7,oranbench,Essential AI + RAG,4,1,0,0,25.0
8,oranbench,Gemma 4 + RAG,4,0,0,1,0.0
9,oranbench,Gemma 4 Only,4,0,0,0,0.0



3. BENCHMARK × MODEL ACCURACY (%)


...


6. REVIEW-REQUIRED ANSWERS

Review-required responses : 2


,question_id,benchmark,model_name,expected_answer,normalized_expected_answer,raw_response
0,T2-06,oranbench,Gemma 4 + RAG,1,2. VERIFYING THAT THE USER PLANE DATA TRANSFER...,Verifying that the user plane data transferred...
1,T2-15,srsranbench,Essential AI + RAG,0,1. TO INITIALIZE THE NODE SIZE BYTE.,The `specific_init()` function in the `ldpc_de...



7. RESULT STATUS SUMMARY


,model_name,score_status,responses
0,Essential AI + RAG,CORRECT,7
1,Essential AI + RAG,INCORRECT,15
2,Essential AI + RAG,NON_RESPONSIVE,9
3,Essential AI + RAG,REVIEW_REQUIRED,1
4,Essential AI Only,CORRECT,13
5,Essential AI Only,INCORRECT,19
6,Gemma 4 + RAG,CORRECT,7
7,Gemma 4 + RAG,INCORRECT,24
8,Gemma 4 + RAG,REVIEW_REQUIRED,1
9,Gemma 4 Only,CORRECT,8



MODULE 6.3 VALIDATION
160 total records                                  : ✅ PASS
9 NON_RESPONSIVE preserved                         : ✅ PASS
All non-responsive scored 0                        : ✅ PASS
Scoring accounting equals 160                      : ✅ PASS

TRACK 2 OBJECTIVE SCORING COMPLETE

Total records        : 160
Scored records       : 158
Review required      : 2
Non-responsive       : 9

Generated objects:
• track2_model_accuracy
• track2_benchmark_accuracy
• track2_accuracy_pivot
• track2_question_results
• track2_incorrect_answers
• track2_review_required
• track2_result_status_summary

Next:
Module 6.4 — Track 2 Performance Analysis and RAG Impact

MODULE 6.3 COMPLETE


In [30]:
# =============================================================================
# MODULE 6.3B — RESOLVE REVIEW-REQUIRED CASES AND LOCK TRACK 2 SCORING
# =============================================================================
#
# MANUAL / DETERMINISTIC RESOLUTION
# ---------------------------------
#
# T2-06 — Gemma 4 + RAG
#   Expected: Option 2
#   Model response semantically matches Option 2 exactly.
#   Resolution: CORRECT
#
# T2-15 — Essential AI + RAG
#   Expected: Option 1
#   Model response explicitly states "initialize the node size byte".
#   Resolution: CORRECT
#
# These are normalization fixes only.
# No LLM judge is used.
#
# =============================================================================

import pandas as pd
import numpy as np


print("=" * 110)
print("MODULE 6.3B — LOCK FINAL TRACK 2 OBJECTIVE SCORING")
print("=" * 110)


# =============================================================================
# 1. DEFINE RESOLUTIONS
# =============================================================================

manual_resolutions = {

    (
        "T2-06",
        "Gemma 4 + RAG",
    ):
        {
            "normalized_model_answer":
                "2. VERIFYING THAT THE USER PLANE DATA TRANSFERRED OVER THE X2 "
                "INTERFACE IN BOTH DIRECTIONS IS SUCCESSFULLY RECEIVED WITHOUT "
                "PACKET LOSSES.",

            "correct":
                1,

            "score_status":
                "CORRECT",

            "normalization_status":
                "RESOLVED",
        },

    (
        "T2-15",
        "Essential AI + RAG",
    ):
        {
            "normalized_model_answer":
                "1. TO INITIALIZE THE NODE SIZE BYTE.",

            "correct":
                1,

            "score_status":
                "CORRECT",

            "normalization_status":
                "RESOLVED",
        },
}


# =============================================================================
# 2. APPLY RESOLUTIONS
# =============================================================================

for (
    question_id,
    model_name,
), resolution in manual_resolutions.items():

    mask = (

        track2_scoring_df[
            "question_id"
        ]
        .eq(
            question_id
        )

        &

        track2_scoring_df[
            "model_name"
        ]
        .eq(
            model_name
        )
    )


    matches = int(
        mask.sum()
    )


    if matches != 1:

        raise RuntimeError(
            f"Expected exactly one row for "
            f"{question_id} | {model_name}, "
            f"found {matches}."
        )


    for column, value in resolution.items():

        track2_scoring_df.loc[
            mask,
            column,
        ] = value


# =============================================================================
# 3. VERIFY NO REVIEW-REQUIRED CASES REMAIN
# =============================================================================

remaining_reviews = (

    track2_scoring_df[

        track2_scoring_df[
            "score_status"
        ]
        .eq(
            "REVIEW_REQUIRED"
        )

    ]
)


print(
    f"\nRemaining REVIEW_REQUIRED responses : "
    f"{len(remaining_reviews)}"
)


if len(
    remaining_reviews
) > 0:

    display(
        remaining_reviews[
            [
                "question_id",
                "benchmark",
                "model_name",
                "raw_response",
            ]
        ]
    )

    raise RuntimeError(
        "Unresolved Track 2 responses remain."
    )


# =============================================================================
# 4. REBUILD FINAL OVERALL MODEL ACCURACY
# =============================================================================

track2_model_accuracy = (

    track2_scoring_df

    .groupby(
        "model_name"
    )

    .agg(

        total_questions=(
            "question_id",
            "count",
        ),

        correct_answers=(
            "correct",
            "sum",
        ),

        non_responsive=(
            "score_status",
            lambda x:
                int(
                    (
                        x
                        == "NON_RESPONSIVE"
                    )
                    .sum()
                ),
        ),

        incorrect_answers=(
            "score_status",
            lambda x:
                int(
                    (
                        x
                        == "INCORRECT"
                    )
                    .sum()
                ),
        ),
    )

    .reset_index()
)


track2_model_accuracy[
    "correct_answers"
] = (

    track2_model_accuracy[
        "correct_answers"
    ]
    .astype(
        int
    )
)


track2_model_accuracy[
    "accuracy_pct"
] = (

    track2_model_accuracy[
        "correct_answers"
    ]

    /

    track2_model_accuracy[
        "total_questions"
    ]

    * 100
)


track2_model_accuracy = (

    track2_model_accuracy

    .sort_values(
        [
            "accuracy_pct",
            "correct_answers",
        ],
        ascending=[
            False,
            False,
        ]
    )

    .reset_index(
        drop=True
    )
)


track2_model_accuracy.insert(

    0,

    "rank",

    np.arange(
        1,
        len(
            track2_model_accuracy
        )
        + 1
    )
)


# =============================================================================
# 5. REBUILD BENCHMARK ACCURACY
# =============================================================================

track2_benchmark_accuracy = (

    track2_scoring_df

    .groupby(
        [
            "benchmark",
            "model_name",
        ]
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        correct_answers=(
            "correct",
            "sum",
        ),

        non_responsive=(
            "score_status",
            lambda x:
                int(
                    (
                        x
                        == "NON_RESPONSIVE"
                    )
                    .sum()
                ),
        ),
    )

    .reset_index()
)


track2_benchmark_accuracy[
    "correct_answers"
] = (

    track2_benchmark_accuracy[
        "correct_answers"
    ]
    .astype(
        int
    )
)


track2_benchmark_accuracy[
    "accuracy_pct"
] = (

    track2_benchmark_accuracy[
        "correct_answers"
    ]

    /

    track2_benchmark_accuracy[
        "questions"
    ]

    * 100
)


track2_benchmark_accuracy = (

    track2_benchmark_accuracy

    .sort_values(
        [
            "benchmark",
            "accuracy_pct",
        ],
        ascending=[
            True,
            False,
        ]
    )

    .reset_index(
        drop=True
    )
)


# =============================================================================
# 6. REBUILD PIVOT
# =============================================================================

track2_accuracy_pivot = (

    track2_benchmark_accuracy

    .pivot(
        index=
            "benchmark",

        columns=
            "model_name",

        values=
            "accuracy_pct",
    )
)


# =============================================================================
# 7. REBUILD RESULT STATUS SUMMARY
# =============================================================================

track2_result_status_summary = (

    track2_scoring_df

    .groupby(
        [
            "model_name",
            "score_status",
        ]
    )

    .size()

    .reset_index(
        name=
            "responses"
    )
)


# =============================================================================
# 8. FINAL QUESTION-LEVEL RESULTS
# =============================================================================

track2_question_results = (

    track2_scoring_df[
        [
            "question_id",
            "benchmark",
            "model_name",
            "normalized_expected_answer",
            "normalized_model_answer",
            "score_status",
            "correct",
        ]
    ]

    .sort_values(
        [
            "question_id",
            "model_name",
        ]
    )

    .reset_index(
        drop=True
    )
)


# =============================================================================
# 9. DISPLAY FINAL OVERALL RANKING
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "1. FINAL TRACK 2 MODEL ACCURACY"
)

print(
    "=" * 110
)


display(
    track2_model_accuracy.round(2)
)


# =============================================================================
# 10. DISPLAY FINAL BENCHMARK ACCURACY
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "2. FINAL ACCURACY BY BENCHMARK FAMILY"
)

print(
    "=" * 110
)


display(
    track2_benchmark_accuracy.round(2)
)


print(
    "\n"
    + "=" * 110
)

print(
    "3. FINAL BENCHMARK × MODEL ACCURACY (%)"
)

print(
    "=" * 110
)


display(
    track2_accuracy_pivot.round(2)
)


# =============================================================================
# 11. FINAL STATUS SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "4. FINAL RESULT STATUS SUMMARY"
)

print(
    "=" * 110
)


display(
    track2_result_status_summary
)


# =============================================================================
# 12. VALIDATION
# =============================================================================

validation_checks = {

    "160 total responses":
        (
            len(
                track2_scoring_df
            )
            == 160
        ),

    "0 review-required responses":
        (
            track2_scoring_df[
                "score_status"
            ]
            .eq(
                "REVIEW_REQUIRED"
            )
            .sum()
            == 0
        ),

    "9 non-responsive responses preserved":
        (
            track2_scoring_df[
                "score_status"
            ]
            .eq(
                "NON_RESPONSIVE"
            )
            .sum()
            == 9
        ),

    "All 160 cases have final correctness":
        (
            track2_scoring_df[
                "correct"
            ]
            .notna()
            .all()
        ),

    "T2-06 Gemma RAG corrected":
        (
            track2_scoring_df.loc[
                (
                    track2_scoring_df[
                        "question_id"
                    ]
                    == "T2-06"
                )
                &
                (
                    track2_scoring_df[
                        "model_name"
                    ]
                    == "Gemma 4 + RAG"
                ),
                "correct",
            ]
            .iloc[0]
            == 1
        ),

    "T2-15 Essential RAG corrected":
        (
            track2_scoring_df.loc[
                (
                    track2_scoring_df[
                        "question_id"
                    ]
                    == "T2-15"
                )
                &
                (
                    track2_scoring_df[
                        "model_name"
                    ]
                    == "Essential AI + RAG"
                ),
                "correct",
            ]
            .iloc[0]
            == 1
        ),
}


print(
    "\n"
    + "=" * 110
)

print(
    "MODULE 6.3B VALIDATION"
)

print(
    "=" * 110
)


for check_name, passed in (
    validation_checks.items()
):

    print(
        f"{check_name:<55} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Final Track 2 scoring validation failed."
    )


# =============================================================================
# 13. SAVE FINAL OBJECTIVE RESULTS
# =============================================================================

TRACK2_FINAL_CSV = (
    "track2_objective_scoring_final.csv"
)


track2_scoring_df.to_csv(
    TRACK2_FINAL_CSV,
    index=False,
)


print(
    "\n"
    + "=" * 110
)

print(
    "TRACK 2 OBJECTIVE SCORING LOCKED"
)

print(
    "=" * 110
)


print(
    f"\nSaved final scoring : "
    f"{TRACK2_FINAL_CSV}"
)


print(
    "\nNext:"
)

print(
    "Module 6.4 — Track 2 Performance Analysis and RAG Impact"
)


print(
    "\nMODULE 6.3B COMPLETE"
)

print(
    "=" * 110
)

MODULE 6.3B — LOCK FINAL TRACK 2 OBJECTIVE SCORING

Remaining REVIEW_REQUIRED responses : 0

1. FINAL TRACK 2 MODEL ACCURACY


,rank,model_name,total_questions,correct_answers,non_responsive,incorrect_answers,accuracy_pct
0,1,Otel 2.0 Only,32,18,0,14,56.25
1,2,Essential AI Only,32,13,0,19,40.62
2,3,Essential AI + RAG,32,8,9,15,25.00
3,4,Gemma 4 + RAG,32,8,0,24,25.00
4,5,Gemma 4 Only,32,8,0,24,25.00



2. FINAL ACCURACY BY BENCHMARK FAMILY


,benchmark,model_name,questions,correct_answers,non_responsive,accuracy_pct
0,3gpp_tsg,Gemma 4 + RAG,4,3,0,75.0
1,3gpp_tsg,Otel 2.0 Only,4,3,0,75.0
2,3gpp_tsg,Essential AI Only,4,2,0,50.0
3,3gpp_tsg,Gemma 4 Only,4,2,0,50.0
4,3gpp_tsg,Essential AI + RAG,4,0,3,0.0
5,oranbench,Otel 2.0 Only,4,4,0,100.0
6,oranbench,Essential AI Only,4,2,0,50.0
7,oranbench,Essential AI + RAG,4,1,0,25.0
8,oranbench,Gemma 4 + RAG,4,1,0,25.0
9,oranbench,Gemma 4 Only,4,0,0,0.0



3. FINAL BENCHMARK × MODEL ACCURACY (%)


model_name,Essential AI + RAG,Essential AI Only,Gemma 4 + RAG,Gemma 4 Only,Otel 2.0 Only
benchmark,,,,,
3gpp_tsg,0.0,50.0,75.0,50.0,75.0
oranbench,25.0,50.0,25.0,0.0,100.0
sixg_bench,50.0,50.0,25.0,25.0,75.0
srsranbench,75.0,75.0,0.0,75.0,100.0
telelogs,0.0,25.0,25.0,0.0,25.0
telemath,0.0,25.0,0.0,0.0,0.0
teleqna,25.0,25.0,25.0,25.0,75.0
teletables,25.0,25.0,25.0,25.0,0.0



4. FINAL RESULT STATUS SUMMARY


,model_name,score_status,responses
0,Essential AI + RAG,CORRECT,8
1,Essential AI + RAG,INCORRECT,15
2,Essential AI + RAG,NON_RESPONSIVE,9
3,Essential AI Only,CORRECT,13
4,Essential AI Only,INCORRECT,19
5,Gemma 4 + RAG,CORRECT,8
6,Gemma 4 + RAG,INCORRECT,24
7,Gemma 4 Only,CORRECT,8
8,Gemma 4 Only,INCORRECT,24
9,Otel 2.0 Only,CORRECT,18



MODULE 6.3B VALIDATION
160 total responses                                     : ✅ PASS
0 review-required responses                             : ✅ PASS
9 non-responsive responses preserved                    : ✅ PASS
All 160 cases have final correctness                    : ✅ PASS
T2-06 Gemma RAG corrected                               : ✅ PASS
T2-15 Essential RAG corrected                           : ✅ PASS

TRACK 2 OBJECTIVE SCORING LOCKED

Saved final scoring : track2_objective_scoring_final.csv

Next:
Module 6.4 — Track 2 Performance Analysis and RAG Impact

MODULE 6.3B COMPLETE


# **Track 2 Performance Analysis and RAG Impact**

In [31]:
# =============================================================================
# MODULE 6.4 — TRACK 2 PERFORMANCE ANALYSIS AND RAG IMPACT
# =============================================================================
#
# PURPOSE
# -------
# Analyse the final objective Track 2 results across:
#
#   1. Overall model performance
#   2. Benchmark-family performance
#   3. RAG vs standalone impact
#   4. Question-level RAG improvements / degradations
#   5. Model wins by question
#   6. Operational generation-failure impact
#   7. Benchmark difficulty
#
# IMPORTANT
# ---------
# - Uses locked objective Track 2 scores.
# - No LLM judge.
# - No inference.
# - No retrieval rerun.
#
# =============================================================================


import pandas as pd
import numpy as np


print("=" * 115)
print("MODULE 6.4 — TRACK 2 PERFORMANCE ANALYSIS AND RAG IMPACT")
print("=" * 115)


# =============================================================================
# 1. VALIDATE INPUT
# =============================================================================

if "track2_scoring_df" not in globals():

    raise RuntimeError(
        "track2_scoring_df not found."
    )


required_columns = [

    "question_id",
    "benchmark",
    "model_name",
    "is_rag",
    "correct",
    "score_status",
    "generation_failed",
]


missing_columns = [

    column

    for column in required_columns

    if column not in track2_scoring_df.columns
]


if missing_columns:

    raise RuntimeError(
        f"Missing required columns: {missing_columns}"
    )


if track2_scoring_df["correct"].isna().any():

    raise RuntimeError(
        "Track 2 contains unresolved correctness values. "
        "Run Module 6.3B first."
    )


# =============================================================================
# 2. FINAL OVERALL MODEL PERFORMANCE
# =============================================================================

track2_overall_performance = (

    track2_scoring_df

    .groupby(
        "model_name"
    )

    .agg(

        total_questions=(
            "question_id",
            "count",
        ),

        correct_answers=(
            "correct",
            "sum",
        ),

        incorrect_answers=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "INCORRECT"
                    ).sum()
                ),
        ),

        non_responsive=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "NON_RESPONSIVE"
                    ).sum()
                ),
        ),

        generation_failures=(
            "generation_failed",
            "sum",
        ),
    )

    .reset_index()
)


track2_overall_performance[
    "correct_answers"
] = (

    track2_overall_performance[
        "correct_answers"
    ]
    .astype(int)
)


track2_overall_performance[
    "accuracy_pct"
] = (

    track2_overall_performance[
        "correct_answers"
    ]

    /

    track2_overall_performance[
        "total_questions"
    ]

    * 100
)


track2_overall_performance[
    "response_success_pct"
] = (

    (
        track2_overall_performance[
            "total_questions"
        ]

        -

        track2_overall_performance[
            "non_responsive"
        ]
    )

    /

    track2_overall_performance[
        "total_questions"
    ]

    * 100
)


track2_overall_performance = (

    track2_overall_performance

    .sort_values(
        [
            "accuracy_pct",
            "correct_answers",
        ],
        ascending=[
            False,
            False,
        ]
    )

    .reset_index(
        drop=True
    )
)


track2_overall_performance.insert(

    0,

    "rank",

    np.arange(
        1,
        len(track2_overall_performance) + 1
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "1. FINAL OVERALL MODEL PERFORMANCE"
)

print(
    "=" * 115
)


display(
    track2_overall_performance.round(2)
)


# =============================================================================
# 3. FINAL PERFORMANCE BY BENCHMARK FAMILY
# =============================================================================

track2_performance_by_benchmark = (

    track2_scoring_df

    .groupby(
        [
            "benchmark",
            "model_name",
        ]
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        correct_answers=(
            "correct",
            "sum",
        ),

        non_responsive=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "NON_RESPONSIVE"
                    ).sum()
                ),
        ),
    )

    .reset_index()
)


track2_performance_by_benchmark[
    "correct_answers"
] = (

    track2_performance_by_benchmark[
        "correct_answers"
    ]
    .astype(int)
)


track2_performance_by_benchmark[
    "accuracy_pct"
] = (

    track2_performance_by_benchmark[
        "correct_answers"
    ]

    /

    track2_performance_by_benchmark[
        "questions"
    ]

    * 100
)


print(
    "\n"
    + "=" * 115
)

print(
    "2. PERFORMANCE BY BENCHMARK FAMILY"
)

print(
    "=" * 115
)


display(

    track2_performance_by_benchmark

    .sort_values(
        [
            "benchmark",
            "accuracy_pct",
        ],
        ascending=[
            True,
            False,
        ]
    )

    .round(2)
)


# =============================================================================
# 4. BENCHMARK × MODEL ACCURACY MATRIX
# =============================================================================

track2_benchmark_accuracy_matrix = (

    track2_performance_by_benchmark

    .pivot(
        index=
            "benchmark",

        columns=
            "model_name",

        values=
            "accuracy_pct",
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "3. BENCHMARK × MODEL ACCURACY (%)"
)

print(
    "=" * 115
)


display(
    track2_benchmark_accuracy_matrix.round(2)
)


# =============================================================================
# 5. DEFINE BASE ↔ RAG PAIRS
# =============================================================================

TRACK2_RAG_PAIRS = {

    "Essential AI":
        {
            "base":
                "Essential AI Only",

            "rag":
                "Essential AI + RAG",
        },

    "Gemma 4":
        {
            "base":
                "Gemma 4 Only",

            "rag":
                "Gemma 4 + RAG",
        },
}


# =============================================================================
# 6. OVERALL RAG VS STANDALONE IMPACT
# =============================================================================

rag_impact_records = []


for family, pair in TRACK2_RAG_PAIRS.items():

    base_name = pair["base"]

    rag_name = pair["rag"]


    base_rows = (

        track2_scoring_df[
            track2_scoring_df[
                "model_name"
            ].eq(base_name)
        ]

        .set_index(
            "question_id"
        )
    )


    rag_rows = (

        track2_scoring_df[
            track2_scoring_df[
                "model_name"
            ].eq(rag_name)
        ]

        .set_index(
            "question_id"
        )
    )


    common_ids = sorted(

        set(base_rows.index)

        &

        set(rag_rows.index)
    )


    pair_rows = []


    for question_id in common_ids:

        base_correct = int(
            base_rows.loc[
                question_id,
                "correct"
            ]
        )

        rag_correct = int(
            rag_rows.loc[
                question_id,
                "correct"
            ]
        )


        delta = (
            rag_correct
            -
            base_correct
        )


        if delta > 0:

            impact = "RAG_BETTER"

        elif delta < 0:

            impact = "RAG_WORSE"

        else:

            impact = "SAME"


        pair_rows.append(
            {
                "question_id":
                    question_id,

                "benchmark":
                    base_rows.loc[
                        question_id,
                        "benchmark"
                    ],

                "family":
                    family,

                "base_model":
                    base_name,

                "rag_model":
                    rag_name,

                "base_correct":
                    base_correct,

                "rag_correct":
                    rag_correct,

                "delta":
                    delta,

                "impact":
                    impact,

                "rag_non_responsive":
                    (
                        rag_rows.loc[
                            question_id,
                            "score_status"
                        ]
                        == "NON_RESPONSIVE"
                    ),

                "rag_generation_failed":
                    bool(
                        rag_rows.loc[
                            question_id,
                            "generation_failed"
                        ]
                    ),
            }
        )


    family_df = pd.DataFrame(
        pair_rows
    )


    rag_impact_records.extend(
        pair_rows
    )


track2_rag_question_impact_df = (

    pd.DataFrame(
        rag_impact_records
    )

    .sort_values(
        [
            "family",
            "question_id",
        ]
    )

    .reset_index(
        drop=True
    )
)


track2_rag_vs_base_summary = (

    track2_rag_question_impact_df

    .groupby(
        "family"
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        base_correct=(
            "base_correct",
            "sum",
        ),

        rag_correct=(
            "rag_correct",
            "sum",
        ),

        rag_better=(
            "impact",
            lambda x:
                int(
                    (
                        x == "RAG_BETTER"
                    ).sum()
                ),
        ),

        same=(
            "impact",
            lambda x:
                int(
                    (
                        x == "SAME"
                    ).sum()
                ),
        ),

        rag_worse=(
            "impact",
            lambda x:
                int(
                    (
                        x == "RAG_WORSE"
                    ).sum()
                ),
        ),

        rag_non_responsive=(
            "rag_non_responsive",
            "sum",
        ),

        rag_generation_failures=(
            "rag_generation_failed",
            "sum",
        ),
    )

    .reset_index()
)


track2_rag_vs_base_summary[
    "base_accuracy_pct"
] = (

    track2_rag_vs_base_summary[
        "base_correct"
    ]

    /

    track2_rag_vs_base_summary[
        "questions"
    ]

    * 100
)


track2_rag_vs_base_summary[
    "rag_accuracy_pct"
] = (

    track2_rag_vs_base_summary[
        "rag_correct"
    ]

    /

    track2_rag_vs_base_summary[
        "questions"
    ]

    * 100
)


track2_rag_vs_base_summary[
    "accuracy_delta_pp"
] = (

    track2_rag_vs_base_summary[
        "rag_accuracy_pct"
    ]

    -

    track2_rag_vs_base_summary[
        "base_accuracy_pct"
    ]
)


print(
    "\n"
    + "=" * 115
)

print(
    "4. OVERALL RAG VS STANDALONE IMPACT"
)

print(
    "=" * 115
)


display(
    track2_rag_vs_base_summary[
        [
            "family",
            "questions",
            "base_correct",
            "rag_correct",
            "base_accuracy_pct",
            "rag_accuracy_pct",
            "accuracy_delta_pp",
            "rag_better",
            "same",
            "rag_worse",
            "rag_non_responsive",
            "rag_generation_failures",
        ]
    ].round(2)
)


# =============================================================================
# 7. RAG IMPACT BY BENCHMARK FAMILY
# =============================================================================

track2_rag_by_benchmark = (

    track2_rag_question_impact_df

    .groupby(
        [
            "family",
            "benchmark",
        ]
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        base_correct=(
            "base_correct",
            "sum",
        ),

        rag_correct=(
            "rag_correct",
            "sum",
        ),

        rag_better=(
            "impact",
            lambda x:
                int(
                    (
                        x == "RAG_BETTER"
                    ).sum()
                ),
        ),

        same=(
            "impact",
            lambda x:
                int(
                    (
                        x == "SAME"
                    ).sum()
                ),
        ),

        rag_worse=(
            "impact",
            lambda x:
                int(
                    (
                        x == "RAG_WORSE"
                    ).sum()
                ),
        ),

        rag_non_responsive=(
            "rag_non_responsive",
            "sum",
        ),
    )

    .reset_index()
)


track2_rag_by_benchmark[
    "base_accuracy_pct"
] = (

    track2_rag_by_benchmark[
        "base_correct"
    ]

    /

    track2_rag_by_benchmark[
        "questions"
    ]

    * 100
)


track2_rag_by_benchmark[
    "rag_accuracy_pct"
] = (

    track2_rag_by_benchmark[
        "rag_correct"
    ]

    /

    track2_rag_by_benchmark[
        "questions"
    ]

    * 100
)


track2_rag_by_benchmark[
    "accuracy_delta_pp"
] = (

    track2_rag_by_benchmark[
        "rag_accuracy_pct"
    ]

    -

    track2_rag_by_benchmark[
        "base_accuracy_pct"
    ]
)


print(
    "\n"
    + "=" * 115
)

print(
    "5. RAG IMPACT BY BENCHMARK FAMILY"
)

print(
    "=" * 115
)


display(

    track2_rag_by_benchmark

    .sort_values(
        [
            "family",
            "benchmark",
        ]
    )

    .round(2)
)


# =============================================================================
# 8. QUESTION-LEVEL RAG CHANGES
# =============================================================================

print(
    "\n"
    + "=" * 115
)

print(
    "6. QUESTION-LEVEL RAG IMPACT"
)

print(
    "=" * 115
)


display(

    track2_rag_question_impact_df[
        [
            "family",
            "question_id",
            "benchmark",
            "base_correct",
            "rag_correct",
            "delta",
            "impact",
            "rag_non_responsive",
            "rag_generation_failed",
        ]
    ]
)


# =============================================================================
# 9. RAG IMPROVEMENTS
# =============================================================================

track2_rag_improvements = (

    track2_rag_question_impact_df[

        track2_rag_question_impact_df[
            "impact"
        ]
        .eq(
            "RAG_BETTER"
        )

    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "7. QUESTIONS WHERE RAG IMPROVED THE BASE MODEL"
)

print(
    "=" * 115
)


print(
    f"\nRAG improvement cases : "
    f"{len(track2_rag_improvements)}"
)


if len(
    track2_rag_improvements
) > 0:

    display(
        track2_rag_improvements
    )


# =============================================================================
# 10. RAG DEGRADATIONS
# =============================================================================

track2_rag_degradations = (

    track2_rag_question_impact_df[

        track2_rag_question_impact_df[
            "impact"
        ]
        .eq(
            "RAG_WORSE"
        )

    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "8. QUESTIONS WHERE RAG DEGRADED THE BASE MODEL"
)

print(
    "=" * 115
)


print(
    f"\nRAG degradation cases : "
    f"{len(track2_rag_degradations)}"
)


if len(
    track2_rag_degradations
) > 0:

    display(
        track2_rag_degradations
    )


# =============================================================================
# 11. QUESTION-LEVEL WIN COUNTS ACROSS ALL FIVE SYSTEMS
# =============================================================================
#
# A model receives a win when:
#
#   - it answers the question correctly
#   - and its correctness equals the maximum correctness on that question
#
# Since Track 2 scoring is binary, all correct models share the win.
#
# Questions where every model is incorrect have no winner.
#
# =============================================================================

question_max_correct = (

    track2_scoring_df

    .groupby(
        "question_id"
    )[
        "correct"
    ]

    .max()
)


winner_records = []


for question_id, max_correct in (
    question_max_correct.items()
):

    if int(max_correct) == 0:

        continue


    winners = (

        track2_scoring_df[
            (
                track2_scoring_df[
                    "question_id"
                ]
                == question_id
            )
            &
            (
                track2_scoring_df[
                    "correct"
                ]
                == max_correct
            )
        ]
    )


    for _, winner in winners.iterrows():

        winner_records.append(
            {
                "question_id":
                    question_id,

                "benchmark":
                    winner[
                        "benchmark"
                    ],

                "model_name":
                    winner[
                        "model_name"
                    ],
            }
        )


track2_question_winners_df = (
    pd.DataFrame(
        winner_records
    )
)


if len(
    track2_question_winners_df
) > 0:

    track2_question_win_summary = (

        track2_question_winners_df

        .groupby(
            "model_name"
        )

        .size()

        .reset_index(
            name=
                "question_wins"
        )

        .sort_values(
            "question_wins",
            ascending=False,
        )

        .reset_index(
            drop=True
        )
    )

else:

    track2_question_win_summary = (
        pd.DataFrame(
            columns=[
                "model_name",
                "question_wins",
            ]
        )
    )


print(
    "\n"
    + "=" * 115
)

print(
    "9. QUESTION WIN COUNTS"
)

print(
    "=" * 115
)


display(
    track2_question_win_summary
)


# =============================================================================
# 12. QUESTIONS ANSWERED CORRECTLY BY NO MODEL
# =============================================================================

question_accuracy_summary = (

    track2_scoring_df

    .groupby(
        [
            "question_id",
            "benchmark",
        ]
    )

    .agg(

        models_correct=(
            "correct",
            "sum",
        ),

        models_attempted=(
            "model_name",
            "count",
        ),

        non_responsive=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "NON_RESPONSIVE"
                    ).sum()
                ),
        ),
    )

    .reset_index()
)


question_accuracy_summary[
    "model_success_rate_pct"
] = (

    question_accuracy_summary[
        "models_correct"
    ]

    /

    question_accuracy_summary[
        "models_attempted"
    ]

    * 100
)


track2_zero_correct_questions = (

    question_accuracy_summary[

        question_accuracy_summary[
            "models_correct"
        ]
        .eq(
            0
        )

    ]

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "10. QUESTIONS ANSWERED CORRECTLY BY NO MODEL"
)

print(
    "=" * 115
)


print(
    f"\nZero-correct questions : "
    f"{len(track2_zero_correct_questions)}"
)


if len(
    track2_zero_correct_questions
) > 0:

    display(
        track2_zero_correct_questions
    )


# =============================================================================
# 13. QUESTION DIFFICULTY DISTRIBUTION
# =============================================================================

def classify_question_difficulty(
    models_correct,
):

    if models_correct == 0:

        return "VERY_HARD"

    if models_correct == 1:

        return "HARD"

    if models_correct in [
        2,
        3,
    ]:

        return "MODERATE"

    return "EASY"


question_accuracy_summary[
    "difficulty"
] = (

    question_accuracy_summary[
        "models_correct"
    ]

    .apply(
        classify_question_difficulty
    )
)


track2_question_difficulty_summary = (

    question_accuracy_summary

    .groupby(
        "difficulty"
    )

    .size()

    .reset_index(
        name=
            "questions"
    )
)


difficulty_order = {

    "EASY": 1,
    "MODERATE": 2,
    "HARD": 3,
    "VERY_HARD": 4,
}


track2_question_difficulty_summary[
    "_order"
] = (

    track2_question_difficulty_summary[
        "difficulty"
    ]

    .map(
        difficulty_order
    )
)


track2_question_difficulty_summary = (

    track2_question_difficulty_summary

    .sort_values(
        "_order"
    )

    .drop(
        columns=[
            "_order"
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "11. QUESTION DIFFICULTY DISTRIBUTION"
)

print(
    "=" * 115
)


display(
    track2_question_difficulty_summary
)


# =============================================================================
# 14. BENCHMARK DIFFICULTY
# =============================================================================

track2_benchmark_difficulty = (

    question_accuracy_summary

    .groupby(
        "benchmark"
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        total_correct_across_models=(
            "models_correct",
            "sum",
        ),

        avg_models_correct=(
            "models_correct",
            "mean",
        ),

        avg_model_success_rate_pct=(
            "model_success_rate_pct",
            "mean",
        ),
    )

    .reset_index()

    .sort_values(
        "avg_model_success_rate_pct",
        ascending=False,
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 115
)

print(
    "12. BENCHMARK DIFFICULTY"
)

print(
    "=" * 115
)


display(
    track2_benchmark_difficulty.round(2)
)


# =============================================================================
# 15. OPERATIONAL FAILURE IMPACT
# =============================================================================

track2_operational_failure_summary = (

    track2_scoring_df

    .groupby(
        "model_name"
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        generation_failures=(
            "generation_failed",
            "sum",
        ),

        non_responsive=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "NON_RESPONSIVE"
                    ).sum()
                ),
        ),

        correct_answers=(
            "correct",
            "sum",
        ),
    )

    .reset_index()
)


track2_operational_failure_summary[
    "generation_failure_pct"
] = (

    track2_operational_failure_summary[
        "generation_failures"
    ]

    /

    track2_operational_failure_summary[
        "questions"
    ]

    * 100
)


track2_operational_failure_summary[
    "response_success_pct"
] = (

    (
        track2_operational_failure_summary[
            "questions"
        ]

        -

        track2_operational_failure_summary[
            "non_responsive"
        ]
    )

    /

    track2_operational_failure_summary[
        "questions"
    ]

    * 100
)


print(
    "\n"
    + "=" * 115
)

print(
    "13. OPERATIONAL GENERATION-FAILURE IMPACT"
)

print(
    "=" * 115
)


display(
    track2_operational_failure_summary.round(2)
)


# =============================================================================
# 16. ESSENTIAL AI RAG — CONDITIONAL ACCURACY WHEN RESPONSE WAS GENERATED
# =============================================================================
#
# This does NOT replace the official 32-question benchmark score.
#
# It is a diagnostic metric that separates:
#
#   answer-quality failures
#
# from
#
#   infrastructure / generation failures.
#
# =============================================================================

essential_rag_rows = (

    track2_scoring_df[

        track2_scoring_df[
            "model_name"
        ]
        .eq(
            "Essential AI + RAG"
        )

    ]
)


essential_rag_substantive = (

    essential_rag_rows[

        essential_rag_rows[
            "score_status"
        ]
        .ne(
            "NON_RESPONSIVE"
        )

    ]
)


essential_rag_conditional_accuracy = (

    essential_rag_substantive[
        "correct"
    ]
    .mean()

    * 100
)


print(
    "\n"
    + "=" * 115
)

print(
    "14. ESSENTIAL AI + RAG CONDITIONAL RESPONSE ACCURACY"
)

print(
    "=" * 115
)


print(
    f"\nOfficial benchmark accuracy       : "
    f"{essential_rag_rows['correct'].mean() * 100:.2f}%"
)

print(
    f"Generated substantive responses   : "
    f"{len(essential_rag_substantive)} / "
    f"{len(essential_rag_rows)}"
)

print(
    f"Accuracy when response generated  : "
    f"{essential_rag_conditional_accuracy:.2f}%"
)


# =============================================================================
# 17. FINAL ANALYSIS SUMMARY OBJECT
# =============================================================================

TRACK2_ANALYSIS_SUMMARY = {

    "best_model":
        track2_overall_performance.iloc[0][
            "model_name"
        ],

    "best_accuracy_pct":
        float(
            track2_overall_performance.iloc[0][
                "accuracy_pct"
            ]
        ),

    "essential_base_accuracy_pct":
        float(
            track2_overall_performance.loc[
                track2_overall_performance[
                    "model_name"
                ]
                == "Essential AI Only",
                "accuracy_pct",
            ]
            .iloc[0]
        ),

    "essential_rag_accuracy_pct":
        float(
            track2_overall_performance.loc[
                track2_overall_performance[
                    "model_name"
                ]
                == "Essential AI + RAG",
                "accuracy_pct",
            ]
            .iloc[0]
        ),

    "gemma_base_accuracy_pct":
        float(
            track2_overall_performance.loc[
                track2_overall_performance[
                    "model_name"
                ]
                == "Gemma 4 Only",
                "accuracy_pct",
            ]
            .iloc[0]
        ),

    "gemma_rag_accuracy_pct":
        float(
            track2_overall_performance.loc[
                track2_overall_performance[
                    "model_name"
                ]
                == "Gemma 4 + RAG",
                "accuracy_pct",
            ]
            .iloc[0]
        ),

    "zero_correct_questions":
        int(
            len(
                track2_zero_correct_questions
            )
        ),

    "essential_rag_generation_failures":
        int(
            essential_rag_rows[
                "generation_failed"
            ]
            .sum()
        ),

    "essential_rag_conditional_accuracy_pct":
        float(
            essential_rag_conditional_accuracy
        ),
}


# =============================================================================
# 18. SAVE ANALYSIS TABLES
# =============================================================================

track2_overall_performance.to_csv(
    "track2_overall_performance.csv",
    index=False,
)


track2_performance_by_benchmark.to_csv(
    "track2_performance_by_benchmark.csv",
    index=False,
)


track2_rag_vs_base_summary.to_csv(
    "track2_rag_vs_base_summary.csv",
    index=False,
)


track2_rag_by_benchmark.to_csv(
    "track2_rag_by_benchmark.csv",
    index=False,
)


track2_rag_question_impact_df.to_csv(
    "track2_rag_question_impact.csv",
    index=False,
)


question_accuracy_summary.to_csv(
    "track2_question_difficulty.csv",
    index=False,
)


# =============================================================================
# 19. FINAL VALIDATION
# =============================================================================

validation_checks = {

    "5 models analysed":
        (
            track2_overall_performance[
                "model_name"
            ]
            .nunique()
            == 5
        ),

    "32 questions analysed":
        (
            question_accuracy_summary[
                "question_id"
            ]
            .nunique()
            == 32
        ),

    "64 paired RAG/base comparisons":
        (
            len(
                track2_rag_question_impact_df
            )
            == 64
        ),

    "Essential AI pair has 32 comparisons":
        (
            (
                track2_rag_question_impact_df[
                    "family"
                ]
                == "Essential AI"
            )
            .sum()
            == 32
        ),

    "Gemma 4 pair has 32 comparisons":
        (
            (
                track2_rag_question_impact_df[
                    "family"
                ]
                == "Gemma 4"
            )
            .sum()
            == 32
        ),

    "9 generation failures preserved":
        (
            int(
                track2_scoring_df[
                    "generation_failed"
                ]
                .sum()
            )
            == 9
        ),
}


print(
    "\n"
    + "=" * 115
)

print(
    "MODULE 6.4 VALIDATION"
)

print(
    "=" * 115
)


for check_name, passed in (
    validation_checks.items()
):

    print(
        f"{check_name:<60} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 6.4 validation failed."
    )


# =============================================================================
# 20. FINAL OUTPUT
# =============================================================================

print(
    "\n"
    + "=" * 115
)

print(
    "TRACK 2 PERFORMANCE ANALYSIS COMPLETE"
)

print(
    "=" * 115
)


print(
    "\nGenerated objects:"
)

print(
    "• track2_overall_performance"
)

print(
    "• track2_performance_by_benchmark"
)

print(
    "• track2_benchmark_accuracy_matrix"
)

print(
    "• track2_rag_vs_base_summary"
)

print(
    "• track2_rag_by_benchmark"
)

print(
    "• track2_rag_question_impact_df"
)

print(
    "• track2_rag_improvements"
)

print(
    "• track2_rag_degradations"
)

print(
    "• track2_question_win_summary"
)

print(
    "• track2_zero_correct_questions"
)

print(
    "• track2_question_difficulty_summary"
)

print(
    "• track2_benchmark_difficulty"
)

print(
    "• track2_operational_failure_summary"
)

print(
    "• TRACK2_ANALYSIS_SUMMARY"
)


print(
    "\nSaved:"
)

print(
    "• track2_overall_performance.csv"
)

print(
    "• track2_performance_by_benchmark.csv"
)

print(
    "• track2_rag_vs_base_summary.csv"
)

print(
    "• track2_rag_by_benchmark.csv"
)

print(
    "• track2_rag_question_impact.csv"
)

print(
    "• track2_question_difficulty.csv"
)


print(
    "\nNext:"
)

print(
    "Module 6.5 — Track 2 Error Patterns and Consolidated Findings"
)


print(
    "\nMODULE 6.4 COMPLETE"
)

print(
    "=" * 115
)

MODULE 6.4 — TRACK 2 PERFORMANCE ANALYSIS AND RAG IMPACT

1. FINAL OVERALL MODEL PERFORMANCE


,rank,model_name,total_questions,correct_answers,incorrect_answers,non_responsive,generation_failures,accuracy_pct,response_success_pct
0,1,Otel 2.0 Only,32,18,14,0,0,56.25,100.00
1,2,Essential AI Only,32,13,19,0,0,40.62,100.00
2,3,Essential AI + RAG,32,8,15,9,9,25.00,71.88
3,4,Gemma 4 + RAG,32,8,24,0,0,25.00,100.00
4,5,Gemma 4 Only,32,8,24,0,0,25.00,100.00



2. PERFORMANCE BY BENCHMARK FAMILY


,benchmark,model_name,questions,correct_answers,non_responsive,accuracy_pct
2,3gpp_tsg,Gemma 4 + RAG,4,3,0,75.0
4,3gpp_tsg,Otel 2.0 Only,4,3,0,75.0
1,3gpp_tsg,Essential AI Only,4,2,0,50.0
3,3gpp_tsg,Gemma 4 Only,4,2,0,50.0
0,3gpp_tsg,Essential AI + RAG,4,0,3,0.0
9,oranbench,Otel 2.0 Only,4,4,0,100.0
6,oranbench,Essential AI Only,4,2,0,50.0
5,oranbench,Essential AI + RAG,4,1,0,25.0
7,oranbench,Gemma 4 + RAG,4,1,0,25.0
8,oranbench,Gemma 4 Only,4,0,0,0.0



3. BENCHMARK × MODEL ACCURACY (%)


...


12. BENCHMARK DIFFICULTY


,benchmark,questions,total_correct_across_models,avg_models_correct,avg_model_success_rate_pct
0,srsranbench,4,13.0,3.25,65.0
1,3gpp_tsg,4,10.0,2.50,50.0
2,sixg_bench,4,9.0,2.25,45.0
3,oranbench,4,8.0,2.00,40.0
4,teleqna,4,7.0,1.75,35.0
5,teletables,4,4.0,1.00,20.0
6,telelogs,4,3.0,0.75,15.0
7,telemath,4,1.0,0.25,5.0



13. OPERATIONAL GENERATION-FAILURE IMPACT


,model_name,questions,generation_failures,non_responsive,correct_answers,generation_failure_pct,response_success_pct
0,Essential AI + RAG,32,9,9,8.0,28.12,71.88
1,Essential AI Only,32,0,0,13.0,0.00,100.00
2,Gemma 4 + RAG,32,0,0,8.0,0.00,100.00
3,Gemma 4 Only,32,0,0,8.0,0.00,100.00
4,Otel 2.0 Only,32,0,0,18.0,0.00,100.00



14. ESSENTIAL AI + RAG CONDITIONAL RESPONSE ACCURACY

Official benchmark accuracy       : 25.00%
Generated substantive responses   : 23 / 32
Accuracy when response generated  : 34.78%

MODULE 6.4 VALIDATION
5 models analysed                                            : ✅ PASS
32 questions analysed                                        : ✅ PASS
64 paired RAG/base comparisons                               : ✅ PASS
Essential AI pair has 32 comparisons                         : ✅ PASS
Gemma 4 pair has 32 comparisons                              : ✅ PASS
9 generation failures preserved                              : ✅ PASS

TRACK 2 PERFORMANCE ANALYSIS COMPLETE

Generated objects:
• track2_overall_performance
• track2_performance_by_benchmark
• track2_benchmark_accuracy_matrix
• track2_rag_vs_base_summary
• track2_rag_by_benchmark
• track2_rag_question_impact_df
• track2_rag_improvements
• track2_rag_degradations
• track2_question_win_summary
• track2_zero_correct_questions
• track2_quest

# **Track 2 Error Patterns and Consolidated Findings**

In [32]:
# =============================================================================
# MODULE 6.5 — TRACK 2 ERROR PATTERNS AND CONSOLIDATED FINDINGS
# =============================================================================
#
# PURPOSE
# -------
# Consolidate the final Track 2 objective results into:
#
#   1. Error-pattern summaries
#   2. RAG improvement/degradation patterns
#   3. Operational failure impact
#   4. Benchmark-specific strengths and weaknesses
#   5. Final Track 2 findings
#   6. Track 2 recommendations
#
# IMPORTANT
# ---------
# - No LLM judge
# - No inference
# - No retrieval rerun
# - Uses only locked Track 2 objective results
#
# =============================================================================


import pandas as pd
import numpy as np


print("=" * 118)
print("MODULE 6.5 — TRACK 2 ERROR PATTERNS AND CONSOLIDATED FINDINGS")
print("=" * 118)


# =============================================================================
# 1. VALIDATE REQUIRED OBJECTS
# =============================================================================

required_objects = [

    "track2_scoring_df",
    "track2_overall_performance",
    "track2_performance_by_benchmark",
    "track2_rag_vs_base_summary",
    "track2_rag_by_benchmark",
    "track2_rag_question_impact_df",
    "track2_rag_improvements",
    "track2_rag_degradations",
    "track2_zero_correct_questions",
    "track2_benchmark_difficulty",
    "track2_operational_failure_summary",
]


missing_objects = [

    obj

    for obj in required_objects

    if obj not in globals()
]


if missing_objects:

    raise RuntimeError(
        f"Missing required objects: {missing_objects}"
    )


# =============================================================================
# 2. MODEL ERROR PROFILE
# =============================================================================

track2_model_error_profile = (

    track2_scoring_df

    .groupby(
        "model_name"
    )

    .agg(

        total_questions=(
            "question_id",
            "count",
        ),

        correct=(
            "correct",
            "sum",
        ),

        incorrect=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "INCORRECT"
                    ).sum()
                ),
        ),

        non_responsive=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "NON_RESPONSIVE"
                    ).sum()
                ),
        ),

        generation_failures=(
            "generation_failed",
            "sum",
        ),
    )

    .reset_index()
)


track2_model_error_profile[
    "accuracy_pct"
] = (

    track2_model_error_profile[
        "correct"
    ]

    /

    track2_model_error_profile[
        "total_questions"
    ]

    * 100
)


track2_model_error_profile[
    "incorrect_pct"
] = (

    track2_model_error_profile[
        "incorrect"
    ]

    /

    track2_model_error_profile[
        "total_questions"
    ]

    * 100
)


track2_model_error_profile[
    "non_responsive_pct"
] = (

    track2_model_error_profile[
        "non_responsive"
    ]

    /

    track2_model_error_profile[
        "total_questions"
    ]

    * 100
)


track2_model_error_profile = (

    track2_model_error_profile

    .sort_values(
        "accuracy_pct",
        ascending=False,
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "1. MODEL ERROR PROFILE"
)

print(
    "=" * 118
)


display(
    track2_model_error_profile.round(2)
)


# =============================================================================
# 3. ERROR PROFILE BY BENCHMARK
# =============================================================================

track2_benchmark_error_profile = (

    track2_scoring_df

    .groupby(
        [
            "benchmark",
            "model_name",
        ]
    )

    .agg(

        questions=(
            "question_id",
            "count",
        ),

        correct=(
            "correct",
            "sum",
        ),

        incorrect=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "INCORRECT"
                    ).sum()
                ),
        ),

        non_responsive=(
            "score_status",
            lambda x:
                int(
                    (
                        x == "NON_RESPONSIVE"
                    ).sum()
                ),
        ),
    )

    .reset_index()
)


track2_benchmark_error_profile[
    "accuracy_pct"
] = (

    track2_benchmark_error_profile[
        "correct"
    ]

    /

    track2_benchmark_error_profile[
        "questions"
    ]

    * 100
)


print(
    "\n"
    + "=" * 118
)

print(
    "2. ERROR PROFILE BY BENCHMARK FAMILY"
)

print(
    "=" * 118
)


display(

    track2_benchmark_error_profile

    .sort_values(
        [
            "benchmark",
            "accuracy_pct",
        ],
        ascending=[
            True,
            False,
        ]
    )

    .round(2)
)


# =============================================================================
# 4. RAG IMPACT DISTRIBUTION
# =============================================================================

track2_rag_impact_distribution = (

    track2_rag_question_impact_df

    .groupby(
        [
            "family",
            "impact",
        ]
    )

    .size()

    .reset_index(
        name=
            "questions"
    )
)


track2_rag_impact_distribution[
    "percentage"
] = (

    track2_rag_impact_distribution

    .groupby(
        "family"
    )[
        "questions"
    ]

    .transform(
        lambda x:
            x
            /
            x.sum()
            *
            100
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "3. RAG IMPACT DISTRIBUTION"
)

print(
    "=" * 118
)


display(
    track2_rag_impact_distribution.round(2)
)


# =============================================================================
# 5. RAG DEGRADATION — OPERATIONAL VS SUBSTANTIVE
# =============================================================================
#
# This separates RAG degradation caused by:
#
#   A. generation / OOM / non-responsive failure
#
# from:
#
#   B. substantive wrong-answer degradation
#
# =============================================================================

rag_degradation_rows = (
    track2_rag_degradations
    .copy()
)


rag_degradation_rows[
    "degradation_type"
] = np.where(

    rag_degradation_rows[
        "rag_generation_failed"
    ],

    "OPERATIONAL_GENERATION_FAILURE",

    "SUBSTANTIVE_WRONG_ANSWER",
)


track2_rag_degradation_type_summary = (

    rag_degradation_rows

    .groupby(
        [
            "family",
            "degradation_type",
        ]
    )

    .size()

    .reset_index(
        name=
            "cases"
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "4. RAG DEGRADATION TYPE"
)

print(
    "=" * 118
)


display(
    track2_rag_degradation_type_summary
)


# =============================================================================
# 6. ESSENTIAL AI RAG DEGRADATION BREAKDOWN
# =============================================================================

essential_rag_degradations = (

    rag_degradation_rows[

        rag_degradation_rows[
            "family"
        ]
        .eq(
            "Essential AI"
        )

    ]

    .copy()
)


essential_operational_degradations = int(

    essential_rag_degradations[
        "rag_generation_failed"
    ]
    .sum()
)


essential_substantive_degradations = int(

    (
        ~essential_rag_degradations[
            "rag_generation_failed"
        ]
    )
    .sum()
)


print(
    "\n"
    + "=" * 118
)

print(
    "5. ESSENTIAL AI + RAG DEGRADATION BREAKDOWN"
)

print(
    "=" * 118
)


print(
    f"\nTotal degradation cases             : "
    f"{len(essential_rag_degradations)}"
)

print(
    f"Operational generation degradation  : "
    f"{essential_operational_degradations}"
)

print(
    f"Substantive answer degradation       : "
    f"{essential_substantive_degradations}"
)


# =============================================================================
# 7. GEMMA RAG DEGRADATION BREAKDOWN
# =============================================================================

gemma_rag_degradations = (

    rag_degradation_rows[

        rag_degradation_rows[
            "family"
        ]
        .eq(
            "Gemma 4"
        )

    ]

    .copy()
)


gemma_operational_degradations = int(

    gemma_rag_degradations[
        "rag_generation_failed"
    ]
    .sum()
)


gemma_substantive_degradations = int(

    (
        ~gemma_rag_degradations[
            "rag_generation_failed"
        ]
    )
    .sum()
)


print(
    "\n"
    + "=" * 118
)

print(
    "6. GEMMA 4 + RAG DEGRADATION BREAKDOWN"
)

print(
    "=" * 118
)


print(
    f"\nTotal degradation cases             : "
    f"{len(gemma_rag_degradations)}"
)

print(
    f"Operational generation degradation  : "
    f"{gemma_operational_degradations}"
)

print(
    f"Substantive answer degradation       : "
    f"{gemma_substantive_degradations}"
)


# =============================================================================
# 8. BENCHMARK-SPECIFIC BEST MODELS
# =============================================================================

benchmark_best_records = []


for benchmark in sorted(
    track2_performance_by_benchmark[
        "benchmark"
    ].unique()
):

    subset = (

        track2_performance_by_benchmark[

            track2_performance_by_benchmark[
                "benchmark"
            ]
            .eq(
                benchmark
            )

        ]

        .copy()
    )


    best_accuracy = (
        subset[
            "accuracy_pct"
        ]
        .max()
    )


    winners = (

        subset[

            subset[
                "accuracy_pct"
            ]
            .eq(
                best_accuracy
            )

        ]
    )


    for _, row in winners.iterrows():

        benchmark_best_records.append(
            {

                "benchmark":
                    benchmark,

                "model_name":
                    row[
                        "model_name"
                    ],

                "accuracy_pct":
                    row[
                        "accuracy_pct"
                    ],
            }
        )


track2_benchmark_best_models = (

    pd.DataFrame(
        benchmark_best_records
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "7. BEST MODEL(S) BY BENCHMARK FAMILY"
)

print(
    "=" * 118
)


display(
    track2_benchmark_best_models.round(2)
)


# =============================================================================
# 9. RAG BENCHMARK GAINS / LOSSES
# =============================================================================

track2_rag_benchmark_gain_loss = (

    track2_rag_by_benchmark[
        [
            "family",
            "benchmark",
            "base_accuracy_pct",
            "rag_accuracy_pct",
            "accuracy_delta_pp",
            "rag_better",
            "same",
            "rag_worse",
            "rag_non_responsive",
        ]
    ]

    .copy()
)


track2_rag_benchmark_gain_loss[
    "impact_class"
] = np.select(

    [

        track2_rag_benchmark_gain_loss[
            "accuracy_delta_pp"
        ]
        > 0,

        track2_rag_benchmark_gain_loss[
            "accuracy_delta_pp"
        ]
        < 0,
    ],

    [

        "RAG_IMPROVED",
        "RAG_DEGRADED",
    ],

    default=
        "NO_NET_CHANGE",
)


print(
    "\n"
    + "=" * 118
)

print(
    "8. RAG BENCHMARK-LEVEL GAINS / LOSSES"
)

print(
    "=" * 118
)


display(
    track2_rag_benchmark_gain_loss.round(2)
)


# =============================================================================
# 10. ZERO-CORRECT QUESTION DISTRIBUTION
# =============================================================================

track2_zero_correct_by_benchmark = (

    track2_zero_correct_questions

    .groupby(
        "benchmark"
    )

    .size()

    .reset_index(
        name=
            "zero_correct_questions"
    )

    .sort_values(
        "zero_correct_questions",
        ascending=False,
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "9. ZERO-CORRECT QUESTIONS BY BENCHMARK"
)

print(
    "=" * 118
)


display(
    track2_zero_correct_by_benchmark
)


# =============================================================================
# 11. BUILD CONSOLIDATED FINDINGS
# =============================================================================

track2_findings = [

    {
        "finding":
            "Otel 2.0 Only achieved the strongest overall Track 2 performance.",

        "evidence":
            "18/32 correct (56.25%), the highest accuracy among all five systems.",

        "interpretation":
            "Otel 2.0 demonstrated the strongest objective telecom benchmark performance."
    },

    {
        "finding":
            "Essential AI Only ranked second.",

        "evidence":
            "13/32 correct (40.62%).",

        "interpretation":
            "Essential AI demonstrated useful telecom-domain capability without retrieval augmentation."
    },

    {
        "finding":
            "Essential AI + RAG underperformed Essential AI Only.",

        "evidence":
            "Accuracy decreased from 40.62% to 25.00%, a -15.62 percentage-point change.",

        "interpretation":
            "The current Essential AI RAG configuration reduced overall benchmark performance."
    },

    {
        "finding":
            "Essential AI + RAG suffered significant operational instability.",

        "evidence":
            "9/32 generation failures (28.12%), reducing response success to 71.88%.",

        "interpretation":
            "Infrastructure/runtime limitations materially affected the observed RAG benchmark score."
    },

    {
        "finding":
            "Essential AI + RAG remained weaker even when only generated responses were considered.",

        "evidence":
            "Conditional accuracy on 23 generated responses was 34.78%, below Essential AI Only at 40.62%.",

        "interpretation":
            "Operational failures explain part, but not all, of the Essential AI RAG degradation."
    },

    {
        "finding":
            "Gemma 4 + RAG produced no net overall accuracy improvement.",

        "evidence":
            "Gemma 4 Only and Gemma 4 + RAG both achieved 8/32 correct (25.00%).",

        "interpretation":
            "RAG redistributed Gemma successes rather than improving aggregate correctness."
    },

    {
        "finding":
            "Gemma RAG showed strong question-level volatility.",

        "evidence":
            "7 questions improved, 18 were unchanged, and 7 degraded.",

        "interpretation":
            "Retrieval helped some questions while harming others, producing zero net overall benefit."
    },

    {
        "finding":
            "RAG effects were highly benchmark-dependent.",

        "evidence":
            "Gemma RAG improved 3gpp_tsg and oranbench by +25 pp, but reduced srsranbench by -75 pp.",

        "interpretation":
            "A single universal RAG strategy is unlikely to be optimal across heterogeneous telecom benchmark tasks."
    },

    {
        "finding":
            "Otel 2.0 showed particularly strong performance on implementation-oriented benchmarks.",

        "evidence":
            "100% accuracy on oranbench and srsranbench.",

        "interpretation":
            "The model appears especially strong on standards/software implementation knowledge represented in these subsets."
    },

    {
        "finding":
            "Some benchmark families were difficult for all systems.",

        "evidence":
            "telemath had only 5% average model success, telelogs 15%, and teletables 20%.",

        "interpretation":
            "The low scores are not exclusively model-specific and indicate benchmark/task difficulty."
    },

    {
        "finding":
            "Seven Track 2 questions were answered correctly by no system.",

        "evidence":
            "T2-10, T2-20, T2-21, T2-23, T2-24, T2-29 and T2-30 had zero correct responses.",

        "interpretation":
            "These cases should be reviewed separately for reasoning complexity, benchmark formatting, corpus coverage, and possible task ambiguity."
    },
]


track2_consolidated_findings_df = (

    pd.DataFrame(
        track2_findings
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "10. CONSOLIDATED TRACK 2 FINDINGS"
)

print(
    "=" * 118
)


display(
    track2_consolidated_findings_df
)


# =============================================================================
# 12. BUILD RECOMMENDATIONS
# =============================================================================

track2_recommendations = [

    {
        "priority":
            1,

        "recommendation":
            "Retain objective expected-answer scoring as the primary Track 2 evaluation method.",

        "reason":
            "Track 2 already provides authoritative expected answers, making deterministic scoring more appropriate than an LLM judge."
    },

    {
        "priority":
            2,

        "recommendation":
            "Separate operational reliability from answer-quality diagnostics while retaining failures in the official score.",

        "reason":
            "Essential AI + RAG experienced 9 OOM failures, which are valid system outcomes but should also be analysed separately from substantive wrong answers."
    },

    {
        "priority":
            3,

        "recommendation":
            "Do not treat RAG as universally beneficial across benchmark families.",

        "reason":
            "RAG impact varied substantially by task family and model."
    },

    {
        "priority":
            4,

        "recommendation":
            "Introduce benchmark/task-aware RAG routing.",

        "reason":
            "Some task families benefited from retrieval while others showed significant degradation."
    },

    {
        "priority":
            5,

        "recommendation":
            "Investigate Gemma RAG srsranbench degradation.",

        "reason":
            "Accuracy fell from 75% standalone to 0% with RAG, representing the largest benchmark-family RAG degradation."
    },

    {
        "priority":
            6,

        "recommendation":
            "Investigate the seven zero-correct questions independently.",

        "reason":
            "Failure across all five systems may indicate unusually difficult reasoning, answer-normalization issues, benchmark ambiguity, or missing supporting knowledge."
    },

    {
        "priority":
            7,

        "recommendation":
            "Maintain Retriever V1 unchanged for the next controlled RAG ablation.",

        "reason":
            "Changing both retrieval and generation simultaneously would make causal interpretation difficult."
    },

    {
        "priority":
            8,

        "recommendation":
            "Test revised graded-grounding RAG against the existing strict-grounding baseline.",

        "reason":
            "Track 1 identified strict grounding as a major generation-side failure mechanism, while Track 2 also shows task-dependent RAG degradation."
    },

    {
        "priority":
            9,

        "recommendation":
            "Preserve benchmark-level analysis rather than relying only on aggregate accuracy.",

        "reason":
            "Aggregate scores conceal major variation across 3gpp_tsg, oranbench, srsranbench, telelogs, telemath, teleqna and teletables."
    },
]


track2_recommendations_df = (

    pd.DataFrame(
        track2_recommendations
    )

    .sort_values(
        "priority"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "11. TRACK 2 RECOMMENDATIONS"
)

print(
    "=" * 118
)


display(
    track2_recommendations_df
)


# =============================================================================
# 13. FINAL REPORT SUMMARY
# =============================================================================

TRACK2_REPORT_SUMMARY = f"""
TRACK 2 — FINAL OBJECTIVE EVALUATION SUMMARY
============================================

Benchmark Design
----------------
Track 2 consists of 32 industry-curated telecom benchmark questions across
8 benchmark families:

- 3gpp_tsg
- oranbench
- sixg_bench
- srsranbench
- telelogs
- telemath
- teleqna
- teletables

Five systems were evaluated, producing 160 system/question results.

Primary Evaluation
------------------
Track 2 uses authoritative expected answers supplied by the benchmark.
Correctness was therefore evaluated objectively rather than using an LLM judge.

Final Overall Accuracy
----------------------
1. Otel 2.0 Only      : 56.25% (18/32)
2. Essential AI Only  : 40.62% (13/32)
3. Essential AI + RAG : 25.00% (8/32)
4. Gemma 4 + RAG      : 25.00% (8/32)
5. Gemma 4 Only       : 25.00% (8/32)

Essential AI RAG Impact
-----------------------
Essential AI Only : 40.62%
Essential AI + RAG: 25.00%

Delta: -15.62 percentage points.

Essential AI + RAG experienced 9 generation failures, corresponding to
28.12% of Track 2 questions.

When only successfully generated responses are considered, Essential AI + RAG
achieved 34.78% accuracy. This remains below Essential AI Only at 40.62%.

Gemma 4 RAG Impact
------------------
Gemma 4 Only : 25.00%
Gemma 4 + RAG: 25.00%

Overall delta: 0.00 percentage points.

However, question-level behaviour changed substantially:

- RAG better : 7 questions
- Same       : 18 questions
- RAG worse  : 7 questions

Therefore, the zero aggregate delta does not imply that RAG had no effect.
Rather, positive and negative effects approximately cancelled each other.

Benchmark-Specific RAG Behaviour
--------------------------------
Gemma 4 + RAG:

- 3gpp_tsg   : +25 percentage points
- oranbench  : +25 percentage points
- sixg_bench : 0
- srsranbench: -75 percentage points
- telelogs   : +25 percentage points
- telemath   : 0
- teleqna    : 0
- teletables : 0

This demonstrates strong task dependence in RAG effectiveness.

Benchmark Difficulty
--------------------
Average model success rate:

- srsranbench : 65%
- 3gpp_tsg    : 50%
- sixg_bench  : 45%
- oranbench   : 40%
- teleqna     : 35%
- teletables  : 20%
- telelogs    : 15%
- telemath    : 5%

Seven questions were answered correctly by no system.

Track 2 Conclusion
------------------
Otel 2.0 Only achieved the strongest objective benchmark performance.

Essential AI Only ranked second.

The current Essential AI RAG configuration underperformed the standalone
model due to both operational generation failures and substantive answer
degradation.

Gemma 4 RAG produced no overall accuracy improvement but materially changed
which questions were answered correctly, demonstrating that retrieval can
both help and harm depending on task type.

Track 2 therefore reinforces the Track 1 conclusion that RAG effectiveness
depends on retrieval evidence, task type, grounding policy, generation
behaviour and operational robustness rather than simply adding retrieved
context to the model prompt.
"""


print(
    "\n"
    + "=" * 118
)

print(
    "12. FINAL TRACK 2 REPORT SUMMARY"
)

print(
    "=" * 118
)


print(
    TRACK2_REPORT_SUMMARY
)


# =============================================================================
# 14. SAVE OUTPUTS
# =============================================================================

track2_model_error_profile.to_csv(
    "track2_model_error_profile.csv",
    index=False,
)


track2_rag_degradation_type_summary.to_csv(
    "track2_rag_degradation_type_summary.csv",
    index=False,
)


track2_benchmark_best_models.to_csv(
    "track2_benchmark_best_models.csv",
    index=False,
)


track2_consolidated_findings_df.to_csv(
    "track2_consolidated_findings.csv",
    index=False,
)


track2_recommendations_df.to_csv(
    "track2_recommendations.csv",
    index=False,
)


with open(
    "track2_final_report_summary.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        TRACK2_REPORT_SUMMARY
    )


# =============================================================================
# 15. VALIDATION
# =============================================================================

validation_checks = {

    "11 consolidated findings created":
        (
            len(
                track2_consolidated_findings_df
            )
            == 11
        ),

    "9 recommendations created":
        (
            len(
                track2_recommendations_df
            )
            == 9
        ),

    "16 total RAG degradation cases":
        (
            len(
                track2_rag_degradations
            )
            == 16
        ),

    "11 total RAG improvement cases":
        (
            len(
                track2_rag_improvements
            )
            == 11
        ),

    "7 zero-correct questions":
        (
            len(
                track2_zero_correct_questions
            )
            == 7
        ),

    "Track 2 report summary created":
        (
            len(
                TRACK2_REPORT_SUMMARY
            )
            > 500
        ),
}


print(
    "\n"
    + "=" * 118
)

print(
    "MODULE 6.5 VALIDATION"
)

print(
    "=" * 118
)


for check_name, passed in validation_checks.items():

    print(
        f"{check_name:<60} : "
        f"{'✅ PASS' if passed else '❌ FAIL'}"
    )


if not all(
    validation_checks.values()
):

    raise RuntimeError(
        "Module 6.5 validation failed."
    )


# =============================================================================
# 16. FINAL OUTPUT
# =============================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "TRACK 2 ERROR ANALYSIS AND CONSOLIDATION COMPLETE"
)

print(
    "=" * 118
)


print(
    "\nGenerated objects:"
)

print(
    "• track2_model_error_profile"
)

print(
    "• track2_benchmark_error_profile"
)

print(
    "• track2_rag_impact_distribution"
)

print(
    "• track2_rag_degradation_type_summary"
)

print(
    "• track2_benchmark_best_models"
)

print(
    "• track2_rag_benchmark_gain_loss"
)

print(
    "• track2_zero_correct_by_benchmark"
)

print(
    "• track2_consolidated_findings_df"
)

print(
    "• track2_recommendations_df"
)

print(
    "• TRACK2_REPORT_SUMMARY"
)


print(
    "\nSaved:"
)

print(
    "• track2_model_error_profile.csv"
)

print(
    "• track2_rag_degradation_type_summary.csv"
)

print(
    "• track2_benchmark_best_models.csv"
)

print(
    "• track2_consolidated_findings.csv"
)

print(
    "• track2_recommendations.csv"
)

print(
    "• track2_final_report_summary.txt"
)


print(
    "\nMODULE 6.5 COMPLETE"
)

print(
    "=" * 118
)

MODULE 6.5 — TRACK 2 ERROR PATTERNS AND CONSOLIDATED FINDINGS

1. MODEL ERROR PROFILE


,model_name,total_questions,correct,incorrect,non_responsive,generation_failures,accuracy_pct,incorrect_pct,non_responsive_pct
0,Otel 2.0 Only,32,18.0,14,0,0,56.25,43.75,0.00
1,Essential AI Only,32,13.0,19,0,0,40.62,59.38,0.00
2,Essential AI + RAG,32,8.0,15,9,9,25.00,46.88,28.12
3,Gemma 4 + RAG,32,8.0,24,0,0,25.00,75.00,0.00
4,Gemma 4 Only,32,8.0,24,0,0,25.00,75.00,0.00



2. ERROR PROFILE BY BENCHMARK FAMILY


,benchmark,model_name,questions,correct,incorrect,non_responsive,accuracy_pct
2,3gpp_tsg,Gemma 4 + RAG,4,3.0,1,0,75.0
4,3gpp_tsg,Otel 2.0 Only,4,3.0,1,0,75.0
1,3gpp_tsg,Essential AI Only,4,2.0,2,0,50.0
3,3gpp_tsg,Gemma 4 Only,4,2.0,2,0,50.0
0,3gpp_tsg,Essential AI + RAG,4,0.0,1,3,0.0
9,oranbench,Otel 2.0 Only,4,4.0,0,0,100.0
6,oranbench,Essential AI Only,4,2.0,2,0,50.0
5,oranbench,Essential AI + RAG,4,1.0,3,0,25.0
7,oranbench,Gemma 4 + RAG,4,1.0,3,0,25.0
8,oranbench,Gemma 4 Only,4,0.0,4,0,0.0



3. RAG IMPACT DISTRIBUTION


...


10. CONSOLIDATED TRACK 2 FINDINGS


,finding,evidence,interpretation
0,Otel 2.0 Only achieved the strongest overall T...,"18/32 correct (56.25%), the highest accuracy a...",Otel 2.0 demonstrated the strongest objective ...
1,Essential AI Only ranked second.,13/32 correct (40.62%).,Essential AI demonstrated useful telecom-domai...
2,Essential AI + RAG underperformed Essential AI...,"Accuracy decreased from 40.62% to 25.00%, a -1...",The current Essential AI RAG configuration red...
3,Essential AI + RAG suffered significant operat...,"9/32 generation failures (28.12%), reducing re...",Infrastructure/runtime limitations materially ...
4,Essential AI + RAG remained weaker even when o...,Conditional accuracy on 23 generated responses...,"Operational failures explain part, but not all..."
5,Gemma 4 + RAG produced no net overall accuracy...,Gemma 4 Only and Gemma 4 + RAG both achieved 8...,RAG redistributed Gemma successes rather than ...
6,Gemma RAG showed strong question-level volatil...,"7 questions improved, 18 were unchanged, and 7...",Retrieval helped some questions while harming ...
7,RAG effects were highly benchmark-dependent.,Gemma RAG improved 3gpp_tsg and oranbench by +...,A single universal RAG strategy is unlikely to...
8,Otel 2.0 showed particularly strong performanc...,100% accuracy on oranbench and srsranbench.,The model appears especially strong on standar...
9,Some benchmark families were difficult for all...,"telemath had only 5% average model success, te...",The low scores are not exclusively model-speci...



11. TRACK 2 RECOMMENDATIONS


,priority,recommendation,reason
0,1,Retain objective expected-answer scoring as th...,Track 2 already provides authoritative expecte...
1,2,Separate operational reliability from answer-q...,"Essential AI + RAG experienced 9 OOM failures,..."
2,3,Do not treat RAG as universally beneficial acr...,RAG impact varied substantially by task family...
3,4,Introduce benchmark/task-aware RAG routing.,Some task families benefited from retrieval wh...
4,5,Investigate Gemma RAG srsranbench degradation.,Accuracy fell from 75% standalone to 0% with R...
5,6,Investigate the seven zero-correct questions i...,Failure across all five systems may indicate u...
6,7,Maintain Retriever V1 unchanged for the next c...,Changing both retrieval and generation simulta...
7,8,Test revised graded-grounding RAG against the ...,Track 1 identified strict grounding as a major...
8,9,Preserve benchmark-level analysis rather than ...,Aggregate scores conceal major variation acros...



12. FINAL TRACK 2 REPORT SUMMARY

TRACK 2 — FINAL OBJECTIVE EVALUATION SUMMARY

Benchmark Design
----------------
Track 2 consists of 32 industry-curated telecom benchmark questions across
8 benchmark families:

- 3gpp_tsg
- oranbench
- sixg_bench
- srsranbench
- telelogs
- telemath
- teleqna
- teletables

Five systems were evaluated, producing 160 system/question results.

Primary Evaluation
------------------
Track 2 uses authoritative expected answers supplied by the benchmark.
Correctness was therefore evaluated objectively rather than using an LLM judge.

Final Overall Accuracy
----------------------
1. Otel 2.0 Only      : 56.25% (18/32)
2. Essential AI Only  : 40.62% (13/32)
3. Essential AI + RAG : 25.00% (8/32)
4. Gemma 4 + RAG      : 25.00% (8/32)
5. Gemma 4 Only       : 25.00% (8/32)

Essential AI RAG Impact
-----------------------
Essential AI Only : 40.62%
Essential AI + RAG: 25.00%

Delta: -15.62 percentage points.

Essential AI + RAG experienced 9 generation failures, co

# **Track 2 Observations, Issues and Recommendations**

## **Track 2 — Key Observations**

### 1. Otel 2.0 Only Achieved the Strongest Objective Benchmark Performance

Otel 2.0 Only achieved the highest Track 2 accuracy with **18 correct answers out of 32 (56.25%)**, followed by Essential AI Only with **13/32 (40.62%)**.

The remaining three systems — Essential AI + RAG, Gemma 4 Only and Gemma 4 + RAG — each achieved **8/32 (25.00%)**.

This indicates that the telecom-specialized Otel 2.0 model demonstrated the strongest ability to answer the industry-curated objective benchmark questions without retrieval augmentation.


### 2. Essential AI Performed Better Without RAG

Essential AI Only achieved **40.62% accuracy**, while Essential AI + RAG achieved **25.00%**, representing a **15.62 percentage-point reduction** after introducing retrieval.

At the question level:

- RAG improved **4/32 questions**
- RAG produced the same outcome on **19/32**
- RAG degraded performance on **9/32**

The decline was therefore not caused solely by retrieval failing to add value; in several cases the RAG pipeline changed a previously correct standalone answer into an incorrect or non-responsive result.


### 3. Essential AI + RAG Was Also Affected by Operational Generation Failures

Essential AI + RAG experienced **9 generation failures out of 32 questions (28.12%)**, resulting in a response success rate of only **71.88%**.

Five of the nine cases where Essential AI RAG performed worse than the standalone model were directly associated with generation failures.

However, infrastructure failure does not fully explain the RAG degradation. When only the **23 successfully generated responses** are considered, Essential AI + RAG achieved **34.78% conditional accuracy**, which remained below Essential AI Only at **40.62%**.

This separates two distinct effects:

1. **Operational reliability problems**, particularly GPU memory / generation failures.
2. **Substantive RAG answer-quality degradation** on successfully generated responses.


### 4. Gemma 4 RAG Produced No Net Accuracy Gain but Substantially Changed Individual Outcomes

Gemma 4 Only and Gemma 4 + RAG both achieved **25.00% overall accuracy**.

However, the identical aggregate score hides significant question-level movement:

- RAG improved **7 questions**
- RAG produced the same result on **18 questions**
- RAG degraded **7 questions**

Therefore, retrieval was not neutral. It materially changed Gemma's behaviour, but the positive and negative effects approximately cancelled each other.


### 5. RAG Effectiveness Was Strongly Task-Dependent

Gemma 4 + RAG showed substantially different behaviour across benchmark families.

Compared with Gemma 4 Only:

- `3gpp_tsg`: **50% → 75%** (+25 pp)
- `oranbench`: **0% → 25%** (+25 pp)
- `telelogs`: **0% → 25%** (+25 pp)
- `sixg_bench`: no net change
- `telemath`: no net change
- `teleqna`: no net change
- `teletables`: no net change
- `srsranbench`: **75% → 0%** (-75 pp)

This demonstrates that retrieval augmentation should not be assumed to improve every telecom task equally.


### 6. Strict Document Grounding Appears to Influence RAG Answer Behaviour

The RAG systems were intentionally designed to answer from retrieved documentation and to abstain when the requested information was not supported by that evidence.

This behaviour improves grounding discipline, but Track 2 shows that it can also suppress otherwise answerable responses.

For example, on T2-15 the standalone Gemma 4 model correctly answered that `specific_init()` initializes the node size byte, whereas Gemma 4 + RAG returned:

> "The requested details are not available in the documentation."

The benchmark answer was available and the standalone model answered correctly, showing that strict grounding can convert an answerable question into an abstention when the retrieved evidence does not sufficiently support the answer. :contentReference[oaicite:0]{index=0}

This behaviour is consistent with the strict-grounding pattern already identified during the Track 1 RAG diagnostics.


### 7. Otel 2.0 Demonstrated Particularly Strong Performance on Several Telecom-Specific Benchmark Families

Otel 2.0 achieved:

- **100%** on `oranbench`
- **100%** on `srsranbench`
- **75%** on `3gpp_tsg`
- **75%** on `sixg_bench`
- **75%** on `teleqna`

This suggests strong pretrained telecom-domain capability, particularly for standards, RAN implementation and telecom technical knowledge tasks.


### 8. Some Track 2 Benchmark Families Were Difficult for All Models

Average model success rates varied substantially:

- `srsranbench`: **65%**
- `3gpp_tsg`: **50%**
- `sixg_bench`: **45%**
- `oranbench`: **40%**
- `teleqna`: **35%**
- `teletables`: **20%**
- `telelogs`: **15%**
- `telemath`: **5%**

Seven questions were answered correctly by **none of the five systems**, with three of these coming from `telemath`.

This indicates that part of the observed low accuracy reflects benchmark difficulty rather than only weaknesses in individual models. :contentReference[oaicite:1]{index=1}

## **Track 2 — Issues Identified**

### 1. RAG Did Not Consistently Improve Objective Answer Accuracy

The central Track 2 issue is that retrieval augmentation did not produce a consistent accuracy improvement.

Essential AI declined from **40.62% to 25.00%**, while Gemma 4 remained unchanged at **25.00% overall** despite substantial question-level changes.

Across both model families, there were:

- **11 RAG improvement cases**
- **16 RAG degradation cases**

This confirms that adding retrieved context alone does not guarantee improved benchmark correctness.


### 2. Strict Grounding Can Cause Over-Abstention

The current RAG architecture strongly prioritizes answering only from retrieved documentation.

Although this reduces unsupported generation, it can become too restrictive when:

- the retriever provides only partial evidence,
- the exact answer is not explicitly stated in the retrieved chunks,
- multiple chunks must be synthesized,
- or the model already possesses sufficient domain knowledge to answer correctly.

In these situations, the RAG model may return the configured fallback response instead of performing reasonable technical reasoning.

T2-15 provides a clear example: Gemma 4 Only produced the correct answer, while Gemma 4 + RAG abstained because the required details were considered unavailable in the retrieved documentation. :contentReference[oaicite:2]{index=2}

This mirrors the Track 1 finding that **strict grounding / abstention behaviour can become a generation-side limitation even when retrieval is not necessarily the primary problem**.


### 3. Retrieved Context Can Override Correct Standalone Knowledge

RAG sometimes changed a correct standalone answer into an incorrect result.

For Gemma 4, seven previously correct answers became incorrect after retrieval augmentation, including three of the four `srsranbench` questions.

The most significant benchmark-level degradation was:

**Gemma 4 `srsranbench`: 75% standalone → 0% with RAG.**

This suggests that retrieved context may sometimes:

- distract the model,
- introduce competing information,
- cause excessive literal interpretation of documentation,
- or override useful pretrained model knowledge.


### 4. Essential AI + RAG Had an Infrastructure Reliability Problem

Essential AI + RAG failed to generate responses for **9/32 questions** because of CUDA out-of-memory failures.

This produced a **28.12% generation failure rate** and reduced end-to-end response success to **71.88%**.

Five RAG degradation cases were attributable directly to operational generation failure, while four were substantive wrong-answer degradations.

The benchmark therefore exposes both an **AI quality issue** and a **runtime reliability issue**, which should be reported separately.


### 5. RAG Behaviour Was Not Adapted to Different Task Types

The same retrieval and grounding strategy was applied across highly heterogeneous benchmark types, including:

- standards classification,
- RAN software implementation,
- log diagnosis,
- numerical reasoning,
- technical Q&A,
- and table-based questions.

The results show that these tasks do not benefit equally from external retrieval.

A universal RAG policy therefore creates unnecessary retrieval overhead and, in some cases, reduces accuracy.


### 6. Numerical and Structured Reasoning Remain Weak Areas

`telemath` achieved only a **5% average model success rate**, with three of its four questions answered incorrectly by every model.

`teletables` also achieved only **20% average model success**.

These tasks may require stronger deterministic reasoning, calculation or structured-data handling than normal natural-language generation.

The results suggest that an LLM-only or RAG-only architecture may not be sufficient for all telecom question types.


### 7. Aggregate Accuracy Can Hide Important RAG Behaviour

Gemma 4 Only and Gemma 4 + RAG both achieved **25%**, which could incorrectly suggest that RAG had no effect.

In reality, RAG changed the correctness outcome on **14 of 32 Gemma questions**:

- 7 improved
- 7 degraded

Therefore, aggregate accuracy alone is insufficient for evaluating RAG systems; question-level paired analysis is necessary.


### 8. Some Benchmark Questions Require Separate Investigation

Seven questions were answered correctly by no model:

- T2-10
- T2-20
- T2-21
- T2-23
- T2-24
- T2-29
- T2-30

These questions should be reviewed separately to determine whether the failures arise from:

- genuine reasoning difficulty,
- benchmark ambiguity,
- answer-format expectations,
- insufficient corpus coverage,
- numerical reasoning limitations,
- or weaknesses shared across all evaluated models.

## **Track 2 — Recommendations**

### 1. Retain Objective Expected-Answer Scoring for Track 2

Track 2 provides authoritative expected answers, so deterministic correctness scoring should remain the primary evaluation method.

An LLM judge is unnecessary for primary scoring and would introduce additional subjectivity into an otherwise objective benchmark.

LLM-based analysis may still be useful later for diagnosing *why* particular answers failed.


### 2. Replace Strict Binary Grounding with Graded Grounding

The RAG system should remain evidence-aware, but the current grounding policy should be softened.

Instead of requiring every answer to be explicitly recoverable from retrieved documents, the generation policy should distinguish between:

1. **Directly supported facts**
2. **Reasonable engineering inference from retrieved evidence**
3. **Model knowledge not supported by retrieved evidence**
4. **Insufficient evidence requiring abstention**

For telecom engineering questions, the model should be permitted to perform clearly identified technical reasoning from partially sufficient evidence rather than automatically returning the fallback response.

The goal should be **grounded reasoning**, not merely **document extraction**.


### 3. Introduce Evidence-Sufficiency Assessment Before Generation

Before forcing strict grounding, the RAG pipeline should assess whether the retrieved context is:

- **SUFFICIENT**
- **PARTIAL**
- **INSUFFICIENT**

If evidence is sufficient, the model should answer directly from the documents.

If evidence is partial, the model should synthesize the retrieved evidence and use clearly distinguished engineering reasoning where appropriate.

Only genuinely insufficient evidence should trigger abstention.

This directly addresses the strict-grounding behaviour observed in both Track 1 and Track 2.


### 4. Introduce Task-Aware RAG Routing

RAG should not automatically be activated for every query.

The system should first classify the task, for example:

- standards/document lookup → RAG likely useful
- technical implementation lookup → RAG potentially useful
- general telecom reasoning → standalone model or hybrid reasoning
- numerical calculation → calculator/tool execution
- structured table reasoning → structured-data tool
- log/root-cause diagnosis → RAG + reasoning / agent workflow

This would avoid injecting retrieved context into tasks where it provides little benefit or can actively reduce accuracy.


### 5. Preserve Retriever V1 for the Next Controlled Experiment

The current BGE-M3 + FAISS Retriever V1 should remain unchanged during the next RAG experiment.

Changing retrieval, prompt behaviour and generation policy simultaneously would make it difficult to determine which modification caused any performance improvement.

The next controlled ablation should therefore focus primarily on **generation and grounding behaviour**.


### 6. Test a Revised RAG Prompt Against the Current Strict-Grounding Baseline

The next experiment should compare:

**Baseline**
- strict document-only grounding
- fallback when retrieved evidence does not explicitly support the answer

**Revised RAG**
- retrieved evidence remains primary
- graded evidence sufficiency
- multi-chunk synthesis
- engineering inference allowed when clearly identified
- abstention only when evidence is genuinely insufficient

The same Retriever V1 and benchmark questions should be retained so that the effect of the prompt/grounding change can be measured directly.


### 7. Investigate Gemma 4 + RAG `srsranbench` as a Priority Failure Case

Gemma 4 Only achieved **75%** on `srsranbench`, while Gemma 4 + RAG achieved **0%**.

This is the clearest Track 2 example of retrieval harming an otherwise capable model and should be investigated through:

- retrieved Top-7 chunks,
- evidence relevance,
- conflicting retrieved information,
- context ordering,
- prompt interpretation,
- and abstention / grounding behaviour.

This focused analysis may reveal mechanisms that also explain other RAG degradation cases.


### 8. Resolve Essential AI + RAG Runtime Reliability Before Further Comparison

The **9 CUDA OOM failures** should be addressed before future benchmarking.

Potential actions include:

- reducing maximum model context where appropriate,
- lowering concurrency,
- reducing `max_num_seqs`,
- managing KV-cache allocation,
- reducing prompt/context length,
- or using more GPU memory for the Essential AI + RAG configuration.

Operational failure should continue to count against end-to-end system performance, but future experiments should minimize infrastructure instability so that architecture quality can be evaluated more cleanly.


### 9. Add Deterministic Tools for Numerical and Structured Tasks

The very low `telemath` performance suggests that numerical questions should not rely exclusively on free-form LLM generation.

A production telecom AI architecture should route appropriate questions to:

- calculator / Python execution,
- deterministic formulas,
- structured table parsers,
- or specialized analytical tools.

The LLM can then interpret and explain the result rather than being responsible for the entire calculation.


### 10. Continue Reporting Both End-to-End Accuracy and Conditional Accuracy

For systems with operational failures, two metrics should be retained:

**End-to-End Accuracy**
- correct answers / all benchmark questions
- reflects real system performance

**Conditional Answer Accuracy**
- correct answers / successfully generated substantive responses
- helps isolate model-answer quality from runtime reliability

For Essential AI + RAG these were **25.00%** and **34.78%**, respectively. :contentReference[oaicite:3]{index=3}


### 11. Preserve Benchmark-Level and Question-Level Analysis

Overall accuracy alone is not sufficient for evaluating telecom RAG systems.

Future evaluation should continue to include:

- overall model ranking,
- benchmark-family accuracy,
- RAG vs standalone paired comparison,
- question-level improvements,
- question-level degradations,
- operational failures,
- and zero-correct questions.

The Gemma result demonstrates why this is important: overall accuracy remained unchanged at 25%, even though RAG changed the outcome on 14 of 32 questions.


### 12. Treat RAG as a Selective Knowledge Architecture Rather Than a Universal Model Upgrade

The combined Track 1 and Track 2 results indicate that RAG should not be treated as an automatic improvement applied to every telecom query.

Its value depends on:

- retrieval quality,
- evidence sufficiency,
- task type,
- corpus coverage,
- grounding policy,
- prompt design,
- generation behaviour,
- and runtime reliability.

The preferred architecture should therefore use **selective, task-aware RAG with graded grounding and engineering reasoning**, rather than forcing every query through a strict document-only generation path.

# **Cross-Track Consolidated Findings**

## **Cross-Track Consolidated Findings**

### 1. Otel 2.0 Only Was the Strongest Overall System Across Both Tracks

Otel 2.0 Only achieved the highest Track 1 mean score at **9.20/10** and the highest Track 2 objective accuracy at **56.25% (18/32)**.

This consistency across two different evaluation regimes is significant:

- Track 1 measured engineering quality, completeness, reasoning and practical applicability.
- Track 2 measured deterministic expected-answer correctness.

The result indicates that Otel 2.0 demonstrated both strong engineering response quality and strong objective telecom benchmark performance.


### 2. Strong Standalone Models Generally Outperformed Their Current RAG Variants

Across both tracks, the current RAG configurations did not consistently outperform their standalone counterparts.

For Gemma 4:

- Track 1: **9.00 standalone → 6.75 RAG**
- Track 2: **25.00% standalone → 25.00% RAG**

For Essential AI:

- Track 1: **5.80 standalone → 5.65 RAG**
- Track 2: **40.62% standalone → 25.00% RAG**

This indicates that retrieval augmentation should not be treated as an automatic model upgrade.


### 3. Aggregate Scores Concealed Significant Question-Level RAG Effects

In several cases, similar aggregate scores masked large changes at question level.

For Essential AI in Track 1:

- RAG better: **9**
- Same: **4**
- RAG worse: **7**

For Gemma 4 in Track 2:

- RAG better: **7**
- Same: **18**
- RAG worse: **7**

This shows that RAG can materially redistribute model successes and failures even when the final average or accuracy appears unchanged.


### 4. Gemma 4 Provided the Clearest Evidence of RAG Interference

Gemma 4 Only was one of the strongest Track 1 systems with a mean score of **9.00**.

After adding RAG:

- Track 1 mean fell to **6.75**
- RAG improved **0/20 questions**
- RAG degraded **14/20 questions**

Track 2 showed a more balanced result overall, but still demonstrated substantial task-level volatility.

This suggests that retrieved context and grounding instructions can interfere with a model that already possesses strong telecom-domain reasoning capability.


### 5. Strict Grounding Was Identified as a Major RAG V1 Failure Mechanism

The Track 1 targeted diagnostic examined **13 priority RAG degradation cases**.

Among those cases:

- **13/13** were classified as `STRICT_GROUNDING_ABSTENTION`
- **11/13** had `PARTIAL` evidence
- **2/13** had `SUFFICIENT` evidence
- **0/13** were classified primarily as retrieval failures
- **13/13** were classified as generation/prompting failures

This indicates that, for the investigated severe failures, useful retrieved evidence was generally available but the generation policy did not exploit it effectively.

The issue was therefore not simply retrieval quality, but the interaction between retrieved evidence, grounding instructions and model generation behaviour.


### 6. Groundedness Remains Valuable, but Over-Grounding Reduced Answer Utility

The objective of grounding responses in controlled telecom documentation remains important because it reduces unsupported technical claims.

However, the current V1 policy can behave too conservatively when:

- evidence is partial,
- information is distributed across multiple chunks,
- engineering inference is required,
- or the standalone model already knows the answer.

The desired future behaviour should therefore be **grounded engineering reasoning**, not strict document extraction.


### 7. RAG Effectiveness Was Highly Task-Dependent

Both tracks showed that retrieval does not benefit all telecom question types equally.

RAG was more suitable where answers were directly supported by documentation.

It was less reliable where questions required:

- engineering synthesis,
- multi-domain fault isolation,
- numerical reasoning,
- structured table interpretation,
- or broader model reasoning.

This supports a selective, task-aware RAG architecture rather than routing every query through the same retrieval pipeline.


### 8. Runtime Reliability Also Affected RAG Performance

Essential AI + RAG experienced **9 generation failures in Track 2**, corresponding to **28.12%** of the benchmark.

Its official accuracy was **25.00%**, while conditional accuracy on successfully generated responses was **34.78%**.

This shows that future RAG evaluation must distinguish between:

- answer-quality failure,
- grounding/generation failure,
- retrieval failure,
- and infrastructure/runtime failure.


### 9. Numerical and Structured Reasoning Require Additional Capabilities

Track 2 showed particularly weak performance on:

- `telemath`
- `teletables`
- `telelogs`

`telemath` achieved only **5% average model success** across the evaluated systems.

This suggests that the next architecture should not rely exclusively on free-form LLM generation for every task.

Deterministic calculators, Python execution, structured-data handling and specialized diagnostic tools should be considered where appropriate.


### 10. RAG V1 Should Be Treated as the Frozen Experimental Baseline

The current system has now been:

- implemented,
- benchmarked,
- evaluated,
- and diagnostically analysed.

Its limitations are sufficiently characterized to support a controlled second iteration.

RAG V1 should therefore remain frozen as the baseline against which future RAG V2 improvements are measured.

# **Cross Track Unified RAG Version 1 Issues**

## **Unified RAG V1 Issues**

### 1. Binary Grounding Was Too Restrictive

The current RAG policy effectively behaves as:

- sufficiently explicit evidence → answer
- otherwise → abstain

This is too restrictive for many practical telecom engineering questions where evidence may be partial or distributed across multiple documents.


### 2. Evidence Sufficiency Was Not Explicitly Distinguished

RAG V1 did not sufficiently differentiate between:

- sufficient evidence,
- partial evidence,
- and insufficient evidence.

This resulted in partial evidence sometimes being treated similarly to missing evidence.


### 3. Generation Did Not Consistently Exploit Retrieved Context

The Track 1 priority diagnostic showed that severe degradation cases were primarily associated with generation and prompting rather than retrieval.

The generator therefore needs better instructions for:

- evidence synthesis,
- partial-context reasoning,
- and combining retrieved evidence with engineering inference.


### 4. Retrieved Context Could Override Strong Standalone Knowledge

Gemma 4 demonstrated that RAG can reduce the quality of a capable standalone model.

Retrieved context should complement model knowledge rather than automatically suppressing valid internal knowledge and reasoning.


### 5. A Single RAG Policy Was Applied Across Heterogeneous Tasks

The same pipeline was used for questions requiring very different capabilities, including:

- standards lookup,
- document-grounded factual retrieval,
- software implementation knowledge,
- engineering synthesis,
- troubleshooting,
- numerical reasoning,
- logs,
- and structured tables.

This reduced efficiency and, in some cases, accuracy.


### 6. Multi-Chunk Evidence Synthesis May Be Insufficient

Many telecom questions require information from multiple retrieved chunks.

The high number of `PARTIAL` evidence cases suggests that the full answer may sometimes be distributed across retrieved documents rather than missing entirely.

This should be tested explicitly in RAG V2.


### 7. Runtime Stability Was Not Fully Production-Ready

The Essential AI + RAG CUDA OOM failures demonstrate that architectural quality and runtime reliability cannot be evaluated independently in an end-to-end system.

Future experiments should reduce infrastructure instability while continuing to count real runtime failures in end-to-end benchmark results.


### 8. LLM-Only Reasoning Is Not Ideal for Every Telecom Task

The poor `telemath` and structured benchmark results indicate that some tasks are better handled using deterministic tools or specialized processing rather than relying solely on LLM reasoning.


### 9. Overall Scores Alone Are Insufficient for RAG Evaluation

Future evaluation must preserve:

- overall performance,
- benchmark-level performance,
- question-level RAG deltas,
- retrieval evidence quality,
- generation failure mechanisms,
- abstentions,
- and runtime failures.

This is necessary because aggregate metrics can hide substantial architecture-level behaviour.

# **RAG Version 2 Design Priorities and Next-Phase Plan**

## **RAG V2 Design Priorities and Next-Phase Plan**

### 1. Retain RAG V1 as the Frozen Baseline

The current RAG V1 configuration should remain unchanged and be preserved as the experimental baseline.

Future improvements should be compared directly against V1 using the same Track 1 and Track 2 benchmark datasets.


### 2. Introduce Graded Grounding

Replace binary grounding with three evidence states:

- `SUFFICIENT`
- `PARTIAL`
- `INSUFFICIENT`

Generation behaviour should depend on the evidence state rather than using a single fallback policy.


### 3. Allow Controlled Engineering Reasoning

When retrieved evidence is partial but relevant, the model should be allowed to perform clearly identified engineering inference.

The final answer should distinguish:

- documented evidence,
- interpretation,
- engineering reasoning,
- and unsupported information.


### 4. Improve Multi-Chunk Evidence Synthesis

RAG V2 should explicitly instruct the model to combine complementary information across retrieved chunks.

This should be tested particularly on cases previously classified as `PARTIAL` evidence to determine whether the required answer is collectively supported across multiple chunks.


### 5. Introduce Task-Aware RAG Routing

The system should classify queries before selecting the response strategy.

Possible routes include:

- documentation / standards lookup → RAG
- engineering synthesis → hybrid RAG + reasoning
- general telecom knowledge → standalone model where appropriate
- numerical tasks → calculator / Python
- structured tables → structured-data processing
- logs / diagnostics → reasoning + retrieval + diagnostic tools


### 6. Preserve Retriever V1 During Initial V2 Ablation

The current BGE-M3 + FAISS Retriever V1 should remain fixed during the first RAG V2 experiments.

This allows the effect of grounding and generation changes to be measured independently.

Retriever tuning should only follow if later diagnostics identify clear retrieval limitations.


### 7. Improve Runtime Stability

Before final V2 comparison, runtime issues should be minimized through:

- context-length control,
- GPU memory optimization,
- concurrency tuning,
- KV-cache management,
- and generation configuration review.

Operational failures should still remain part of end-to-end scoring.


### 8. Maintain Low-Temperature Generation Initially

The current findings do not indicate that higher temperature is the primary solution.

RAG V2 should initially retain low-temperature generation while modifying grounding, evidence handling and synthesis behaviour.


### 9. Re-Evaluate Using the Same Frozen Benchmarks

RAG V2 should be evaluated using the same:

- 20 Track 1 questions
- 32 Track 2 questions
- evaluation methodology
- judge configuration for Track 1
- deterministic expected-answer scoring for Track 2

This provides a controlled comparison between V1 and V2.


### 10. Add RAG V2 as an Additional Experimental Configuration

A future benchmark comparison could include:

- Essential AI Only
- Essential AI + RAG V1
- Essential AI + RAG V2
- Gemma 4 Only
- Gemma 4 + RAG V1
- Gemma 4 + RAG V2
- Otel 2.0 Only

This would directly measure whether V2 recovers the capability lost under V1 while preserving useful retrieval benefits.


### 11. Define RAG V2 Success Beyond Simple Score Improvement

The objective should not only be to increase the benchmark score.

RAG V2 should aim to:

- increase grounded correct answers,
- reduce unnecessary abstentions,
- reduce RAG-induced degradation,
- preserve useful standalone reasoning,
- improve multi-domain engineering synthesis,
- maintain factual reliability,
- and improve operational stability.

A successful V2 should therefore produce **more useful grounded answers without materially increasing unsupported technical claims**.


### 12. Next-Phase Experimental Objective

The next phase should test the following central hypothesis:

> A task-aware RAG architecture with graded grounding, evidence-sufficiency assessment and controlled engineering reasoning can outperform the strict document-only RAG V1 architecture while preserving factual groundedness.

The experimental sequence should be:

**RAG V1 baseline → V2 design changes → controlled rerun → V1 vs V2 comparison → failure analysis → production architecture recommendation**